# QW V8 — Quanto e Onde na autodestilação

Este Jupyter Notebook integra o material computacional da tese **IA Explicável na Autodestilação de Conhecimentos em Cenários Desbalanceados**, de **Bruna de Vargas Guterres**, apresentada ao **Programa de Pós-Graduação em Modelagem Computacional da Universidade Federal do Rio Grande - FURG**, como requisito para a obtenção do grau de **Doutora em Modelagem Computacional**.

O seletor `EXPERIMENT_PROFILE`, na próxima célula, escolhe **um** dos experimentos:

| Perfil | Arquitetura e dados | Classes | Seeds padrão |
|---|---|---|---|
| `resnet32_cifar100_lt` | ResNet32 / CIFAR-100-LT | 100 | 42, 3407, 2024, 2025, 2026 |
| `mobilenetv2_imagenet_lt` | MobileNetV2 / ImageNet-LT | 1000 | 42, 3407, 2026 |

O professor é pré-treinado e congelado; o estudante utiliza a mesma arquitetura e inicialização aleatória por padrão. A implementação selecionada mantém seus dados, transformações, perdas, geradores aleatórios, precisão, seleção de checkpoint e formato de saída. Somente as células do perfil escolhido são executadas.

## Variantes disponíveis nos dois perfis

| Experimento | Quanto | Onde | Descrição |
|---|---|---|---|
| `QW0_BASE_RAW` | Não | Não | Entropia cruzada e alinhamento uniforme das CAMs assinadas originais |
| `QW1_AMOUNT_ONLY` | Sim | Não | Referência do professor retificada por intensidade |
| `QW2_WHERE_ONLY` | Não | Sim | Referência original e retificação por localidade |
| `QW3_AMOUNT_WHERE` | Sim | Sim | Referência retificada por intensidade e localidade|


In [ ]:
# Selecione um perfil e execute todas as células em ordem.
# No Colab, EXPERIMENT_PROFILE também aparece como lista de seleção.
EXPERIMENT_PROFILE = "resnet32_cifar100_lt" # @param ["resnet32_cifar100_lt", "mobilenetv2_imagenet_lt"]
MOUNT_GOOGLE_DRIVE = False # @param {type:"boolean"}

import os
import tempfile
from pathlib import Path

# Pasta base pública. Pode apontar ao Drive, por exemplo:
# /content/drive/MyDrive/qw_project
PROJECT_BASE = os.path.abspath(os.path.expanduser(os.environ.get("QW_PROJECT_ROOT", "./qw_project")))

# Somente ImageNet-LT: cópia de trabalho em disco local da sessão.
# A pasta contém imagenet_lt/ após extrair o arquivo de origem.
_default_runtime_root = Path("/content/qw_data") if Path("/content").is_dir() else Path(tempfile.gettempdir()) / "qw_data"
IMAGENET_WORK_DIR = os.path.abspath(os.path.expanduser(os.environ.get("QW_IMAGENET_WORK_DIR", str(_default_runtime_root))))

# None preserva a seleção do respectivo anexo. Para executar as quatro variantes:
# EXPERIMENTS_OVERRIDE = ["QW0_BASE_RAW", "QW1_AMOUNT_ONLY", "QW2_WHERE_ONLY", "QW3_AMOUNT_WHERE"]
EXPERIMENTS_OVERRIDE = None
SEEDS_OVERRIDE = None

_allowed_profiles = {"resnet32_cifar100_lt", "mobilenetv2_imagenet_lt"}
if EXPERIMENT_PROFILE not in _allowed_profiles:
    raise ValueError("Perfil não reconhecido: " + str(EXPERIMENT_PROFILE))
if "_ACTIVE_EXPERIMENT_PROFILE" in globals():
    if _ACTIVE_EXPERIMENT_PROFILE != EXPERIMENT_PROFILE:
        raise RuntimeError("Para trocar de perfil, reinicie o kernel/ambiente e execute o notebook desde o início.")
elif "CONFIG" in globals() or "qw_cam_loss" in globals():
    raise RuntimeError("Há funções ou uma configuração de outra execução neste kernel. Reinicie o ambiente antes de iniciar este notebook.")
_ACTIVE_EXPERIMENT_PROFILE = EXPERIMENT_PROFILE

def assert_active_profile():
    if globals().get("EXPERIMENT_PROFILE") != _ACTIVE_EXPERIMENT_PROFILE:
        raise RuntimeError("O perfil mudou após a inicialização. Reinicie o kernel/ambiente e execute desde o início.")

print("Perfil selecionado:", EXPERIMENT_PROFILE)
print("Pasta base:", PROJECT_BASE)
if EXPERIMENT_PROFILE == "mobilenetv2_imagenet_lt":
    print("ImageNet-LT: pasta de trabalho local:", IMAGENET_WORK_DIR)


In [ ]:
assert_active_profile()

if MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
    except ImportError as exc:
        raise RuntimeError("Google Drive requer Colab. Para Jupyter local, use MOUNT_GOOGLE_DRIVE=False.") from exc
    drive.mount("/content/drive")


In [ ]:
assert_active_profile()

if EXPERIMENT_PROFILE == "resnet32_cifar100_lt":
    # Core imports and reproducibility utilities.
    import os
    import sys
    import json
    import random
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    import torch.optim as optim

    from datetime import datetime
    from zoneinfo import ZoneInfo

    from torch.utils.data import DataLoader, Dataset, Subset
    from torchvision import transforms, datasets
    from torchvision.transforms import functional as TF

    from sklearn.metrics import (
        f1_score,
        balanced_accuracy_score,
        top_k_accuracy_score,
    )
    from sklearn.model_selection import train_test_split

    from PIL import Image
    from IPython.display import display


    def set_seed(seed):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)

        if torch.cuda.is_available():
            torch.cuda.manual_seed(seed)
            torch.cuda.manual_seed_all(seed)

        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


    def seed_worker(worker_id):
        worker_seed = (
            torch.initial_seed() %
            2**32
        )

        np.random.seed(
            worker_seed
        )

        random.seed(
            worker_seed
        )


    def make_generator(seed):
        generator = torch.Generator()
        generator.manual_seed(
            seed
        )
        return generator

    import hashlib
    import importlib
    import time
    from pathlib import Path
else:  # mobilenetv2_imagenet_lt
    # Core imports and reproducibility utilities.
    import os
    import sys
    import json
    import random
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    import torch.optim as optim

    from datetime import datetime
    from zoneinfo import ZoneInfo

    from torch.utils.data import DataLoader, Dataset, Subset
    import torchvision
    from torchvision import transforms
    from torchvision.models import mobilenet_v2

    from sklearn.metrics import (
        f1_score,
        balanced_accuracy_score,
        top_k_accuracy_score,
    )

    from PIL import Image
    from IPython.display import display


    def set_seed(seed):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)

        if torch.cuda.is_available():
            torch.cuda.manual_seed(seed)
            torch.cuda.manual_seed_all(seed)

        torch.backends.cudnn.deterministic = DETERMINISTIC
        torch.backends.cudnn.benchmark = not DETERMINISTIC


    def seed_worker(worker_id):
        worker_seed = (
            torch.initial_seed() %
            2**32
        )

        np.random.seed(
            worker_seed
        )

        random.seed(
            worker_seed
        )


    def make_generator(seed):
        generator = torch.Generator()
        generator.manual_seed(
            seed
        )
        return generator

    import hashlib
    import importlib
    import time
    from pathlib import Path

    import gc
    import shutil
    import subprocess


<a id="configuracao"></a>
## Configuração, caminhos e entradas

São necessários Python/Jupyter, PyTorch e torchvision compatíveis, NumPy, pandas, matplotlib, Pillow, scikit-learn e IPython. O perfil ResNet32 também utiliza `openpyxl` na exportação XLSX. O notebook não instala bibliotecas nem fixa versões automaticamente.

Todos os caminhos são genéricos. A organização a partir de `PROJECT_BASE` é:

```text
qw_project/
├── resnet32_cifar100_lt/
│   ├── data/
│   ├── external/LDAM-DRW/imbalance_cifar.py
│   ├── checkpoints/teachers/seed_<seed>/best_checkpoint.pth
│   └── results/QW_V8/
└── mobilenetv2_imagenet_lt/
    ├── data_archive/imagenet_lt.tar
    ├── checkpoints/teachers/seed_<seed>/best_checkpoint.pth
    ├── checkpoints/teachers/seed_<seed>/metrics.json
    └── results/QW_V8_MobileNet/
```

Para **ResNet32/CIFAR-100-LT**, forneça o módulo `imbalance_cifar.py` com `IMBALANCECIFAR100`, baseado no [LDAM-DRW](https://github.com/kaidic/LDAM-DRW), na mesma versão utilizada para preparar os professores. O download do CIFAR-100 é feito pelo torchvision quando necessário. Cada professor deve corresponder à divisão da mesma seed e ao mesmo protocolo de construção do pool.

Para **MobileNetV2/ImageNet-LT**, forneça as imagens e os manifestos oficiais de treino, validação e teste. O arquivo de origem deve conter a pasta `imagenet_lt/`, com `train/`, `val/`, `ImageNet_LT_train.txt`, `ImageNet_LT_val.txt` e `ImageNet_LT_test.txt`. Os manifestos definem os caminhos relativos das imagens. O arquivo não é baixado por este notebook.

O ImageNet-LT é extraído para `IMAGENET_WORK_DIR/imagenet_lt`, no disco local da sessão. Se essa estrutura já estiver preparada, a cópia e a extração são dispensadas. A pasta temporária e a pasta do arquivo de origem devem ser distintas. O professor MobileNetV2 deve ser o baseline correspondente ao protocolo ImageNet-LT, não os pesos padrão do torchvision.

Com `RUN_TEACHER_CHECK=True`, cada professor MobileNetV2 precisa também de um `metrics.json` contendo `best_monitor_metric`, o F1 macro de validação registrado no treinamento do baseline. A validação do professor mantém a tolerância e a política de precisão do anexo.

### Principais parâmetros

| Nome no texto | Variável | Significado |
|---|---|---|
| Alpha | `CAM_ALPHA` | Peso da perda de destilação por CAM |
| Beta Quanto | `AMOUNT_POWER` | Expoente da compensação de intensidade por grupo e sinal |
| Beta Onde | `WHERE_BETA` | Mistura entre atenção uniforme e orientada pelas evidências |
| Top-K | `WHERE_TOP_K` | Classe verdadeira e K−1 concorrentes do professor |
| Filtro de Quanto | `AMOUNT_APPLY_TO` | Aplicação em `all`, `correct` ou `error` |
| Filtro de Onde | `WHERE_APPLY_TO` | Seleção independente de desfechos para ponderação espacial |
| Modos de Onde | `WHERE_CORRECT_MODE`, `WHERE_ERROR_MODE` | `raw`, `difference` ou `coincidence` |
| Máscaras | `WHERE_CORRECT_MASK_MODE`, `WHERE_ERROR_MASK_MODE` | `aligned`, `uniform` ou `shuffled` |
| Grupos elegíveis | `WHERE_GROUPS` | Grupos que podem receber ponderação espacial |

Os dois anexos usam inicialmente Alpha 1, Beta Quanto 12, Beta Onde 0,5 e filtros `all/all`. As demais configurações, inclusive precisão, workers e aumentos, são mantidas por perfil na próxima célula. Os campos genéricos do seletor não convertem um protocolo no outro.


In [ ]:
assert_active_profile()

if EXPERIMENT_PROFILE == "resnet32_cifar100_lt":
    # Edite esta célula e execute o notebook desde o início.
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    SEEDS = [42, 3407, 2024, 2025, 2026]
    EXPERIMENTS_TO_RUN = ["QW1_AMOUNT_ONLY", "QW3_AMOUNT_WHERE"]  # disponíveis também QW0_BASE_RAW e QW2_WHERE_ONLY
    STUDENT_INIT_MODES = ["random"]
    LT_POOL_SEED = 0
    BATCH_SIZE, EPOCHS = 128, 240
    LR, WEIGHT_DECAY, ADAM_BETAS = 0.001, 1e-4, (0.9, 0.999)
    PLATEAU_PATIENCE, PLATEAU_FACTOR = 10, 0.1
    USE_EARLY_STOPPING, ES_PATIENCE, ES_MIN_DELTA = True, 20, 1e-4
    CKPT_INTERVAL = 40
    CE_WEIGHT, CAM_ALPHA = 1.0, 1

    # QUANTO V8: médias positivas/negativas por classe e grupo; calibração conjunta por sinal.
    # s_T é global e está ativo nos quatro métodos; QUANTO controla somente q_G.
    # Fora do desfecho escolhido, q_aplicado=1 exatamente (sem nova normalização).
    AMOUNT_POWER = 12
    AMOUNT_APPLY_TO = "all"       # all, correct, error; só atua em QW1/QW3
    AMOUNT_FACTOR_MIN, AMOUNT_FACTOR_MAX = None, None
    AMOUNT_NORMALIZATION = "sample_expectation"
    CAM_SCALE_EPS = 1e-8

    # ONDE preservado: somente o alvo da MSE recebe os fatores de intensidade por sinal.
    # Somente os pesos espaciais usam CAMs do professor normalizadas individualmente.
    WHERE_TOP_K = 5
    WHERE_GROUPS = ("medium", "few")
    WHERE_APPLY_TO = "all"        # all, correct, error; só atua em QW2/QW3
    WHERE_CORRECT_MODE = "raw"       # raw, difference, coincidence
    WHERE_ERROR_MODE = "difference" # mesmas três opções, independentemente dos acertos
    WHERE_BETA = 0.5                # mistura uniforme/evidência; beta=0 recupera raw
    WHERE_NORM_EPS = 1e-8
    # Controles espaciais separados: raw ignora a máscara do seu ramo.
    WHERE_CORRECT_MASK_MODE = "aligned"  # aligned, uniform, shuffled
    WHERE_ERROR_MASK_MODE = "aligned"    # aligned, uniform, shuffled

    # Uma imagem aumentada por exemplo, compartilhada por professor e aluno.
    USE_TRAIN_AUG = True
    TRAIN_CROP_PADDING, TRAIN_HORIZONTAL_FLIP_P = 4, 0.5
    MANY_SHOT_THRESHOLD, FEW_SHOT_THRESHOLD = 100, 20
    IMB_TYPE, IMB_FACTOR, NUM_CLASSES, VAL_RATIO = "exp", 0.01, 100, 0.20
    NUM_WORKERS = 0
    PIN_MEMORY = DEVICE.type == "cuda"

    # Tabelas de treino: todas as imagens. Probes: subconjunto fixo sem aumento.
    DIAGNOSTIC_MAX_PER_CLASS = 8
    SAVE_CAM_FIGURES = True
    FIGURES_PER_GROUP_OUTCOME = 1
    GRAD_DIAG_MAX_BATCHES = 1        # primeiros batches por época; 0 desliga esse diagnóstico
    RUN_SELF_TESTS = True
    SKIP_COMPLETED_RUNS = True      # reutiliza apenas execuções concluídas; não retoma épocas
    RESUME_TIMESTAMP = None

    # Caminhos genéricos para a distribuição pública; adapte à organização dos arquivos.
    # Exemplo no Colab com Drive: QW_PROJECT_ROOT=/content/drive/MyDrive/qw_project
    PROJECT_ROOT = str(Path(PROJECT_BASE) / "resnet32_cifar100_lt")
    DATA_ROOT = os.path.join(PROJECT_ROOT, "data")
    IMBALANCE_CIFAR_CODE_ROOT = os.path.join(PROJECT_ROOT, "external", "LDAM-DRW")
    TEACHER_CKPT_TEMPLATE = os.path.join(
        PROJECT_ROOT, "checkpoints", "teachers", "seed_{seed}", "best_checkpoint.pth")

    EXPERIMENT_DEFINITIONS = {
        "QW0_BASE_RAW": {"use_amount": False, "use_where": False,
            "description": "CE + raw signed target-CAM MSE"},
        "QW1_AMOUNT_ONLY": {"use_amount": True, "use_where": False,
            "description": "CE + signed target-CAM MSE against teacher reference rectified by group and sign; global s_T"},
        "QW2_WHERE_ONLY": {"use_amount": False, "use_where": True,
            "description": "CE + signed target-CAM MSE with correctness-dependent spatial modes"},
        "QW3_AMOUNT_WHERE": {"use_amount": True, "use_where": True,
            "description": "CE + unchanged spatial weighting and independently outcome-selected sign/group teacher reference; global s_T"},
    }
    # Substituições opcionais; os defaults de cada anexo ficam intactos com None.
    if SEEDS_OVERRIDE is not None:
        SEEDS = list(SEEDS_OVERRIDE)
    if EXPERIMENTS_OVERRIDE is not None:
        EXPERIMENTS_TO_RUN = list(EXPERIMENTS_OVERRIDE)

    assert AMOUNT_NORMALIZATION == "sample_expectation"
    assert np.isfinite(AMOUNT_POWER)
    assert AMOUNT_APPLY_TO in {"all", "correct", "error"}
    assert AMOUNT_FACTOR_MIN is None or np.isfinite(AMOUNT_FACTOR_MIN) and AMOUNT_FACTOR_MIN >= 0
    assert AMOUNT_FACTOR_MAX is None or np.isfinite(AMOUNT_FACTOR_MAX) and AMOUNT_FACTOR_MAX > 0
    assert AMOUNT_FACTOR_MIN is None or AMOUNT_FACTOR_MAX is None or AMOUNT_FACTOR_MIN <= AMOUNT_FACTOR_MAX
    assert np.isfinite(CAM_SCALE_EPS) and CAM_SCALE_EPS > 0
    assert 2 <= WHERE_TOP_K <= NUM_CLASSES
    assert np.isfinite(WHERE_BETA) and 0 <= WHERE_BETA <= 1
    assert np.isfinite(WHERE_NORM_EPS) and WHERE_NORM_EPS > 0
    assert WHERE_CORRECT_MODE in {"raw", "difference", "coincidence"}
    assert WHERE_ERROR_MODE in {"raw", "difference", "coincidence"}
    assert WHERE_APPLY_TO in {"all", "correct", "error"}
    assert WHERE_CORRECT_MASK_MODE in {"aligned", "uniform", "shuffled"}
    assert WHERE_ERROR_MASK_MODE in {"aligned", "uniform", "shuffled"}
    assert set(WHERE_GROUPS) <= {"many", "medium", "few"}
    assert len(WHERE_GROUPS) == len(set(WHERE_GROUPS))
    assert np.isfinite(CAM_ALPHA) and CAM_ALPHA >= 0 and np.isfinite(CE_WEIGHT) and CE_WEIGHT > 0
    assert TRAIN_CROP_PADDING >= 0 and 0 <= TRAIN_HORIZONTAL_FLIP_P <= 1
    assert set(EXPERIMENTS_TO_RUN) <= set(EXPERIMENT_DEFINITIONS)
    assert len(EXPERIMENTS_TO_RUN) == len(set(EXPERIMENTS_TO_RUN)) and len(EXPERIMENTS_TO_RUN) > 0
    assert set(STUDENT_INIT_MODES) <= {"random", "teacher_init"}
    assert len(STUDENT_INIT_MODES) == len(set(STUDENT_INIT_MODES)) and len(STUDENT_INIT_MODES) > 0
    assert len(SEEDS) == len(set(SEEDS)) and len(SEEDS) > 0
    assert GRAD_DIAG_MAX_BATCHES >= 0 and isinstance(GRAD_DIAG_MAX_BATCHES, int)
    assert DIAGNOSTIC_MAX_PER_CLASS is None or isinstance(DIAGNOSTIC_MAX_PER_CLASS, int) and DIAGNOSTIC_MAX_PER_CLASS > 0
    assert FIGURES_PER_GROUP_OUTCOME >= 0 and isinstance(FIGURES_PER_GROUP_OUTCOME, int)

    # Identificadores históricos preservados para compatibilidade dos registros de pesquisa.
    SOURCE_NOTEBOOK_SHA256 = "8719e0f92aeb274ea86a699211bdb8db4c01fdffe61ecb6c098eafcc5d12952c"
    IMPLEMENTATION_SHA256 = "000baaa1f08ee7a65eee3949e382a57e01ccdc0577d3d5dce5d215ceea852bd0"
    CONFIG_KEYS = [
        "SEEDS", "EXPERIMENTS_TO_RUN", "STUDENT_INIT_MODES", "LT_POOL_SEED",
        "BATCH_SIZE", "EPOCHS", "LR", "WEIGHT_DECAY", "ADAM_BETAS", "PLATEAU_PATIENCE", "PLATEAU_FACTOR",
        "USE_EARLY_STOPPING", "ES_PATIENCE", "ES_MIN_DELTA", "CKPT_INTERVAL", "CE_WEIGHT", "CAM_ALPHA",
        "AMOUNT_POWER", "AMOUNT_APPLY_TO", "AMOUNT_FACTOR_MIN", "AMOUNT_FACTOR_MAX", "AMOUNT_NORMALIZATION", "CAM_SCALE_EPS",
        "WHERE_TOP_K", "WHERE_GROUPS", "WHERE_APPLY_TO", "WHERE_CORRECT_MODE", "WHERE_ERROR_MODE", "WHERE_BETA", "WHERE_NORM_EPS",
        "WHERE_CORRECT_MASK_MODE", "WHERE_ERROR_MASK_MODE", "USE_TRAIN_AUG", "TRAIN_CROP_PADDING", "TRAIN_HORIZONTAL_FLIP_P",
        "MANY_SHOT_THRESHOLD", "FEW_SHOT_THRESHOLD", "IMB_TYPE", "IMB_FACTOR", "NUM_CLASSES", "VAL_RATIO",
        "NUM_WORKERS", "DIAGNOSTIC_MAX_PER_CLASS", "SAVE_CAM_FIGURES", "FIGURES_PER_GROUP_OUTCOME", "GRAD_DIAG_MAX_BATCHES",
        "PROJECT_ROOT", "DATA_ROOT", "IMBALANCE_CIFAR_CODE_ROOT", "TEACHER_CKPT_TEMPLATE",
        "SOURCE_NOTEBOOK_SHA256", "IMPLEMENTATION_SHA256"]
    CONFIG = {key: globals()[key] for key in CONFIG_KEYS}
    CONFIG.update({
        "method_version": "QW_V8_signed_spatial_cam_sign_scaled_target_outcome_controls",
        "teacher_architecture": "resnet32_cifar_ce_baseline",
        "student_architecture": "resnet32_cifar",
        "group_f1_definition": "historical_true_group_subset_macro_f1_preserved",
        "amount_definition": "raw deterministic TRAIN target CAMs; separate positive/negative class and group means; joint sign/group normalization enforces E_image[mean_pixel(r_base)]=1",
        "amount_application_definition": "teacher_target=r_applied*teacher_raw; r selects q_G_positive/q_G_negative/q_G_zero from teacher signs; unselected outcomes use exactly one; no additional outcome normalization",
        "amount_expected_weight_definition": "E_image[mean_pixel(r_base)]=1 in deterministic TRAIN; q in legacy summaries denotes the spatial mean of r, not a scalar target multiplier; selected/augmented means need not equal one",
        "teacher_outcome_definition": "teacher argmax equals the true label on the SAME current image used in the loss; teacher is frozen and detached",
        "where_application_definition": "spatial mode applies only when use_where, true-label group is eligible and teacher outcome matches WHERE_APPLY_TO; otherwise spatially uniform MSE with the independently selected teacher target",
        "cam_scale_policy": "global_teacher_mean_square_all_methods",
        "difference_definition": "mean_over_rivals(abs(normalized_target_CAM-normalized_rival_CAM))",
        "coincidence_definition": "mean_over_rivals(min(abs(target),abs(rival)) if same nonzero sign else 0)",
        "spatial_weight_definition": "(1-beta)+beta*N*evidence/sum(evidence); zero evidence => ones",
        "loss_definition": "CE_WEIGHT*CE + CAM_ALPHA*mean(mean_spatial(weight*(raw_student_y-r_applied*raw_teacher_y)^2)/s_T); global s_T=max(raw_cam_mean_sq,CAM_SCALE_EPS) in ALL QW methods",
        "mass_definition": "mean_spatial_abs_signed_target_CAM; static_class_balanced_and_epoch_sample_weighted_separate",
        "loss_balance_definition": "full_epoch_per_image_sums; weighted_CE_and_method; ratios_of_sums",
        "gradient_scope": "first configured batches per epoch; last convolutional feature tensor"})
    HASH_CONFIG = {key: value for key, value in CONFIG.items()
                   if key not in {"SEEDS", "EXPERIMENTS_TO_RUN", "STUDENT_INIT_MODES"}}
    CONFIG_HASH = hashlib.sha256(json.dumps(HASH_CONFIG, sort_keys=True).encode()).hexdigest()[:12]
    RUN_TIMESTAMP = RESUME_TIMESTAMP or datetime.now(ZoneInfo("America/Sao_Paulo")).strftime("%Y%m%d_%H%M%S")
    PROTOCOL_TAG = (f"{RUN_TIMESTAMP}_QW_V8_K{WHERE_TOP_K}_b{WHERE_BETA:g}"
                    f"_A{AMOUNT_APPLY_TO}_W{WHERE_APPLY_TO}"
                    f"_C{WHERE_CORRECT_MODE}-{WHERE_CORRECT_MASK_MODE}"
                    f"_E{WHERE_ERROR_MODE}-{WHERE_ERROR_MASK_MODE}_{CONFIG_HASH}")
    RESULTS_DIR = os.path.join(PROJECT_ROOT, "results", "QW_V8")
    RUNS_ROOT = os.path.join(RESULTS_DIR, PROTOCOL_TAG)
    RESULTS_XLSX_PATH = os.path.join(RUNS_ROOT, "complete_results.xlsx")
    SUMMARY_XLSX_PATH = os.path.join(RUNS_ROOT, "summary_diagnostics.xlsx")
    os.makedirs(RUNS_ROOT, exist_ok=True)
    with open(os.path.join(RUNS_ROOT, "config.json"), "w", encoding="utf-8") as file:
        json.dump(CONFIG, file, ensure_ascii=False, indent=2, allow_nan=False)
    print("Device:", DEVICE, "| seeds:", SEEDS, "| experiments:", EXPERIMENTS_TO_RUN)
    print("V8 | correct:", WHERE_CORRECT_MODE, WHERE_CORRECT_MASK_MODE,
          "| error:", WHERE_ERROR_MODE, WHERE_ERROR_MASK_MODE,
          "| alpha:", CAM_ALPHA, "| beta:", WHERE_BETA)
    print("Apply QUANTO:", AMOUNT_APPLY_TO, "| apply ONDE:", WHERE_APPLY_TO,
          "| global s_T: all QW0/QW1/QW2/QW3")
    print("Config hash:", CONFIG_HASH, "| output:", RUNS_ROOT)


    def validate_v8_configuration(_config_keys=tuple(CONFIG_KEYS)):
        """Reject mixed notebook versions, stale globals and mismatched output folders."""
        expected_method = "QW_V8_signed_spatial_cam_sign_scaled_target_outcome_controls"
        if CONFIG.get("method_version") != expected_method:
            raise RuntimeError("Configuração incompatível com a V8. Reinicie o ambiente e execute este notebook em ordem.")
        changed = [key for key in _config_keys
                   if key not in CONFIG or key not in globals()
                   or json.dumps(CONFIG[key], sort_keys=True) != json.dumps(globals()[key], sort_keys=True)]
        if changed:
            raise RuntimeError("Parâmetros diferem da configuração registrada: " + ", ".join(changed)
                               + ". Edite a configuração e execute desde o início.")
        hash_config = {key: value for key, value in CONFIG.items()
                       if key not in {"SEEDS", "EXPERIMENTS_TO_RUN", "STUDENT_INIT_MODES"}}
        expected_hash = hashlib.sha256(json.dumps(hash_config, sort_keys=True).encode()).hexdigest()[:12]
        expected_tag = (f"{RUN_TIMESTAMP}_QW_V8_K{WHERE_TOP_K}_b{WHERE_BETA:g}"
                        f"_A{AMOUNT_APPLY_TO}_W{WHERE_APPLY_TO}"
                        f"_C{WHERE_CORRECT_MODE}-{WHERE_CORRECT_MASK_MODE}"
                        f"_E{WHERE_ERROR_MODE}-{WHERE_ERROR_MASK_MODE}_{expected_hash}")
        expected_dir = os.path.join(PROJECT_ROOT, "results", "QW_V8")
        expected_root = os.path.join(expected_dir, expected_tag)
        expected_paths = {"RESULTS_DIR": expected_dir, "RUNS_ROOT": expected_root,
            "RESULTS_XLSX_PATH": os.path.join(expected_root, "complete_results.xlsx"),
            "SUMMARY_XLSX_PATH": os.path.join(expected_root, "summary_diagnostics.xlsx")}
        if CONFIG_HASH != expected_hash or PROTOCOL_TAG != expected_tag or any(
                os.path.normpath(str(globals()[key])) != os.path.normpath(path)
                for key, path in expected_paths.items()):
            raise RuntimeError("Hash, pasta ou tag não corresponde à configuração ativa da V8.")
        if getattr(globals().get("qw_cam_loss"), "qw_method_version", None) != expected_method:
            raise RuntimeError("A função qw_cam_loss ativa não é a V8. Execute as células deste notebook em ordem.")
        print(f"CONFIGURAÇÃO VALIDADA | V8 | alpha={CAM_ALPHA:g} | beta={WHERE_BETA:g} | "
              f"amount_apply_to={AMOUNT_APPLY_TO} | where_apply_to={WHERE_APPLY_TO} | "
              f"correct={WHERE_CORRECT_MODE}/{WHERE_CORRECT_MASK_MODE} | "
              f"error={WHERE_ERROR_MODE}/{WHERE_ERROR_MASK_MODE}")
else:  # mobilenetv2_imagenet_lt
    # Edite esta célula e execute o notebook desde o início.
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    SEEDS = [42, 3407, 2026]
    EXPERIMENTS_TO_RUN = ["QW0_BASE_RAW", "QW1_AMOUNT_ONLY"]  # disponíveis também QW0_BASE_RAW e QW2_WHERE_ONLY
    STUDENT_INIT_MODES = ["random"]
    BATCH_SIZE, EPOCHS = 128, 240
    LR, WEIGHT_DECAY, ADAM_BETAS = 0.001, 1e-4, (0.9, 0.999)
    PLATEAU_PATIENCE, PLATEAU_FACTOR = 10, 0.1
    USE_EARLY_STOPPING, ES_PATIENCE, ES_MIN_DELTA = True, 20, 1e-4
    CKPT_INTERVAL = 40
    CE_WEIGHT, CAM_ALPHA = 1.0, 1

    # QUANTO V8: médias positivas/negativas por classe e grupo; calibração conjunta por sinal.
    # s_T é global e está ativo nos quatro métodos; QUANTO controla somente q_G.
    # Fora do desfecho escolhido, q_aplicado=1 exatamente (sem nova normalização).
    AMOUNT_POWER = 12
    AMOUNT_APPLY_TO = "all"       # all, correct, error; só atua em QW1/QW3
    AMOUNT_FACTOR_MIN, AMOUNT_FACTOR_MAX = None, None
    AMOUNT_NORMALIZATION = "sample_expectation"
    CAM_SCALE_EPS = 1e-8

    # ONDE preservado: somente o alvo da MSE recebe os fatores de intensidade por sinal.
    # Somente os pesos espaciais usam CAMs do professor normalizadas individualmente.
    WHERE_TOP_K = 5
    WHERE_GROUPS = ("medium", "few")
    WHERE_APPLY_TO = "all"        # all, correct, error; só atua em QW2/QW3
    WHERE_CORRECT_MODE = "raw"       # raw, difference, coincidence
    WHERE_ERROR_MODE = "difference" # mesmas três opções, independentemente dos acertos
    WHERE_BETA = 0.5                # mistura uniforme/evidência; beta=0 recupera raw
    WHERE_NORM_EPS = 1e-8
    # Controles espaciais separados: raw ignora a máscara do seu ramo.
    WHERE_CORRECT_MASK_MODE = "aligned"  # aligned, uniform, shuffled
    WHERE_ERROR_MASK_MODE = "aligned"    # aligned, uniform, shuffled

    # Uma imagem aumentada por exemplo, compartilhada por professor e aluno.
    USE_TRAIN_AUG = True
    TRAIN_HORIZONTAL_FLIP_P = 0.5
    MANY_SHOT_THRESHOLD, FEW_SHOT_THRESHOLD = 100, 20
    NUM_CLASSES, IMG_SIZE, RESIZE_TEST = 1000, 224, 256
    USE_COLOR_JITTER = False
    EXPECTED_SPLIT_SIZES = {"train": 115846, "val": 20000, "test": 50000}
    NUM_WORKERS = os.cpu_count() or 2
    PREFETCH_FACTOR = 4
    PIN_MEMORY = DEVICE.type == "cuda"
    USE_AMP = DEVICE.type == "cuda" and torch.cuda.get_device_capability(0)[0] >= 8
    CHANNELS_LAST = DEVICE.type == "cuda"
    DETERMINISTIC = True  # reprodutibilidade da V8; a mesma política em todos os métodos.
    RUN_TEACHER_CHECK = True
    TEACHER_CHECK_TOL = 2e-3
    if DEVICE.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = False  # cabeça/CAM float32 consistente.

    # Tabelas de treino: todas as imagens. Probes: subconjunto fixo sem aumento.
    DIAGNOSTIC_MAX_PER_CLASS = 8
    SAVE_CAM_FIGURES = True  # legado/proveniência: probes/figuras não executados neste perfil
    FIGURES_PER_GROUP_OUTCOME = 1
    GRAD_DIAG_MAX_BATCHES = 1        # legado/proveniência: não executado no perfil essencial
    RUN_SELF_TESTS = True
    SKIP_COMPLETED_RUNS = True      # reutiliza apenas execuções concluídas; não retoma épocas
    RESUME_TIMESTAMP = None

    # Caminhos genéricos do perfil MobileNetV2 / ImageNet-LT.
    DRIVE_DIR = Path(PROJECT_BASE) / "mobilenetv2_imagenet_lt"
    PROJECT_ROOT = str(DRIVE_DIR)
    DATA_ARCHIVE = DRIVE_DIR / "data_archive" / "imagenet_lt.tar"
    LOCAL_DIR = Path(IMAGENET_WORK_DIR)
    LOCAL_DIR.mkdir(parents=True, exist_ok=True)
    DATA_ROOT = LOCAL_DIR / "imagenet_lt"
    TRAIN_TXT = DATA_ROOT / "ImageNet_LT_train.txt"
    VAL_TXT = DATA_ROOT / "ImageNet_LT_val.txt"
    TEST_TXT = DATA_ROOT / "ImageNet_LT_test.txt"
    TEACHER_ROOT = DRIVE_DIR / "checkpoints" / "teachers"
    TEACHER_CKPT_TEMPLATE = str(TEACHER_ROOT / "seed_{seed}" / "best_checkpoint.pth")
    TEACHER_METRICS_TEMPLATE = str(TEACHER_ROOT / "seed_{seed}" / "metrics.json")

    EXPERIMENT_DEFINITIONS = {
        "QW0_BASE_RAW": {"use_amount": False, "use_where": False,
            "description": "CE + raw signed target-CAM MSE"},
        "QW1_AMOUNT_ONLY": {"use_amount": True, "use_where": False,
            "description": "CE + signed target-CAM MSE against teacher reference rectified by group and sign; global s_T"},
        "QW2_WHERE_ONLY": {"use_amount": False, "use_where": True,
            "description": "CE + signed target-CAM MSE with correctness-dependent spatial modes"},
        "QW3_AMOUNT_WHERE": {"use_amount": True, "use_where": True,
            "description": "CE + unchanged spatial weighting and independently outcome-selected sign/group teacher reference; global s_T"},
    }
    # Substituições opcionais; os defaults de cada anexo ficam intactos com None.
    if SEEDS_OVERRIDE is not None:
        SEEDS = list(SEEDS_OVERRIDE)
    if EXPERIMENTS_OVERRIDE is not None:
        EXPERIMENTS_TO_RUN = list(EXPERIMENTS_OVERRIDE)

    assert AMOUNT_NORMALIZATION == "sample_expectation"
    assert np.isfinite(AMOUNT_POWER)
    assert AMOUNT_APPLY_TO in {"all", "correct", "error"}
    assert AMOUNT_FACTOR_MIN is None or np.isfinite(AMOUNT_FACTOR_MIN) and AMOUNT_FACTOR_MIN >= 0
    assert AMOUNT_FACTOR_MAX is None or np.isfinite(AMOUNT_FACTOR_MAX) and AMOUNT_FACTOR_MAX > 0
    assert AMOUNT_FACTOR_MIN is None or AMOUNT_FACTOR_MAX is None or AMOUNT_FACTOR_MIN <= AMOUNT_FACTOR_MAX
    assert np.isfinite(CAM_SCALE_EPS) and CAM_SCALE_EPS > 0
    assert 2 <= WHERE_TOP_K <= NUM_CLASSES
    assert np.isfinite(WHERE_BETA) and 0 <= WHERE_BETA <= 1
    assert np.isfinite(WHERE_NORM_EPS) and WHERE_NORM_EPS > 0
    assert WHERE_CORRECT_MODE in {"raw", "difference", "coincidence"}
    assert WHERE_ERROR_MODE in {"raw", "difference", "coincidence"}
    assert WHERE_APPLY_TO in {"all", "correct", "error"}
    assert WHERE_CORRECT_MASK_MODE in {"aligned", "uniform", "shuffled"}
    assert WHERE_ERROR_MASK_MODE in {"aligned", "uniform", "shuffled"}
    assert set(WHERE_GROUPS) <= {"many", "medium", "few"}
    assert len(WHERE_GROUPS) == len(set(WHERE_GROUPS))
    assert np.isfinite(CAM_ALPHA) and CAM_ALPHA >= 0 and np.isfinite(CE_WEIGHT) and CE_WEIGHT > 0
    assert 0 <= TRAIN_HORIZONTAL_FLIP_P <= 1
    assert IMG_SIZE > 0 and RESIZE_TEST >= IMG_SIZE
    assert NUM_WORKERS >= 0 and PREFETCH_FACTOR > 0
    assert np.isfinite(TEACHER_CHECK_TOL) and TEACHER_CHECK_TOL >= 0
    assert set(EXPERIMENTS_TO_RUN) <= set(EXPERIMENT_DEFINITIONS)
    assert len(EXPERIMENTS_TO_RUN) == len(set(EXPERIMENTS_TO_RUN)) and len(EXPERIMENTS_TO_RUN) > 0
    assert STUDENT_INIT_MODES == ["random"], "V8_mobilenet: estudante aleatório, sem teacher_init."
    assert len(STUDENT_INIT_MODES) == len(set(STUDENT_INIT_MODES)) and len(STUDENT_INIT_MODES) > 0
    assert len(SEEDS) == len(set(SEEDS)) and len(SEEDS) > 0
    assert GRAD_DIAG_MAX_BATCHES >= 0 and isinstance(GRAD_DIAG_MAX_BATCHES, int)
    assert DIAGNOSTIC_MAX_PER_CLASS is None or isinstance(DIAGNOSTIC_MAX_PER_CLASS, int) and DIAGNOSTIC_MAX_PER_CLASS > 0
    assert FIGURES_PER_GROUP_OUTCOME >= 0 and isinstance(FIGURES_PER_GROUP_OUTCOME, int)

    SOURCE_NOTEBOOK_SHA256 = "c51c895ce4badd722e38c21ae04426548ad9fce530777f015e62a101be62e208"
    MOBILE_SOURCE_NOTEBOOK_SHA256 = "f941ea1154f9b0e5409e64a196b24217ea37e4205c1053a7f9e0c899ee4fd263"
    IMPLEMENTATION_SHA256 = "dfd548656b146e71987063c1ed4b14dda0af3f49398066ed5c77e56e7220d4b1"
    CONFIG_KEYS = [
        "SEEDS", "EXPERIMENTS_TO_RUN", "STUDENT_INIT_MODES",
        "BATCH_SIZE", "EPOCHS", "LR", "WEIGHT_DECAY", "ADAM_BETAS", "PLATEAU_PATIENCE", "PLATEAU_FACTOR",
        "USE_EARLY_STOPPING", "ES_PATIENCE", "ES_MIN_DELTA", "CKPT_INTERVAL", "CE_WEIGHT", "CAM_ALPHA",
        "AMOUNT_POWER", "AMOUNT_APPLY_TO", "AMOUNT_FACTOR_MIN", "AMOUNT_FACTOR_MAX", "AMOUNT_NORMALIZATION", "CAM_SCALE_EPS",
        "WHERE_TOP_K", "WHERE_GROUPS", "WHERE_APPLY_TO", "WHERE_CORRECT_MODE", "WHERE_ERROR_MODE", "WHERE_BETA", "WHERE_NORM_EPS",
        "WHERE_CORRECT_MASK_MODE", "WHERE_ERROR_MASK_MODE", "USE_TRAIN_AUG", "TRAIN_HORIZONTAL_FLIP_P",
        "MANY_SHOT_THRESHOLD", "FEW_SHOT_THRESHOLD", "NUM_CLASSES", "IMG_SIZE", "RESIZE_TEST", "USE_COLOR_JITTER",
        "EXPECTED_SPLIT_SIZES", "PREFETCH_FACTOR", "USE_AMP", "CHANNELS_LAST", "DETERMINISTIC",
        "RUN_TEACHER_CHECK", "TEACHER_CHECK_TOL",
        "NUM_WORKERS", "DIAGNOSTIC_MAX_PER_CLASS", "SAVE_CAM_FIGURES", "FIGURES_PER_GROUP_OUTCOME", "GRAD_DIAG_MAX_BATCHES",
        "PROJECT_ROOT", "DATA_ROOT", "DATA_ARCHIVE", "LOCAL_DIR", "DRIVE_DIR", "TEACHER_ROOT",
        "TRAIN_TXT", "VAL_TXT", "TEST_TXT", "TEACHER_CKPT_TEMPLATE", "TEACHER_METRICS_TEMPLATE",
        "SOURCE_NOTEBOOK_SHA256", "MOBILE_SOURCE_NOTEBOOK_SHA256", "IMPLEMENTATION_SHA256"]
    def config_value(value):
        return str(value) if isinstance(value, Path) else value

    CONFIG = {key: config_value(globals()[key]) for key in CONFIG_KEYS}
    CONFIG.update({
        "method_version": "QW_V8_mobilenet_signed_spatial_cam_sign_scaled_target_outcome_controls",
        "teacher_architecture": "torchvision_mobilenetv2_width1_ce_imagenet_lt_baseline",
        "student_architecture": "torchvision_mobilenetv2_width1_random",
        "dataset": "ImageNet-LT_official_splits",
        "precision_policy": "optional_bf16_extractor_fp32_GAP_classifier_CAM_loss",
        "cam_layer": "last_features_before_GAP_no_extra_ReLU",
        "classifier_policy": "original_dropout_0.2_and_linear_preserved",
        "group_f1_definition": "historical_true_group_subset_macro_f1_preserved",
        "amount_definition": "raw deterministic TRAIN target CAMs; separate positive/negative class and group means; joint sign/group normalization enforces E_image[mean_pixel(r_base)]=1",
        "amount_application_definition": "teacher_target=r_applied*teacher_raw; r selects q_G_positive/q_G_negative/q_G_zero from teacher signs; unselected outcomes use exactly one; no additional outcome normalization",
        "amount_expected_weight_definition": "E_image[mean_pixel(r_base)]=1 in deterministic TRAIN; q in legacy summaries denotes the spatial mean of r, not a scalar target multiplier; selected/augmented means need not equal one",
        "teacher_outcome_definition": "teacher argmax equals the true label on the SAME current image used in the loss; teacher is frozen and detached",
        "where_application_definition": "spatial mode applies only when use_where, true-label group is eligible and teacher outcome matches WHERE_APPLY_TO; otherwise spatially uniform MSE with the independently selected teacher target",
        "cam_scale_policy": "global_teacher_mean_square_all_methods",
        "difference_definition": "mean_over_rivals(abs(normalized_target_CAM-normalized_rival_CAM))",
        "coincidence_definition": "mean_over_rivals(min(abs(target),abs(rival)) if same nonzero sign else 0)",
        "spatial_weight_definition": "(1-beta)+beta*N*evidence/sum(evidence); zero evidence => ones",
        "loss_definition": "CE_WEIGHT*CE + CAM_ALPHA*mean(mean_spatial(weight*(raw_student_y-r_applied*raw_teacher_y)^2)/s_T); global s_T=max(raw_cam_mean_sq,CAM_SCALE_EPS) in ALL QW methods",
        "mass_definition": "mean_spatial_abs_signed_target_CAM; static_class_balanced_and_epoch_sample_weighted_separate",
        "loss_balance_definition": "full_epoch_per_image_sums; weighted_CE_and_method; ratios_of_sums",
        "gradient_scope": "first configured batches per epoch; last convolutional feature tensor"})
    HASH_CONFIG = {key: value for key, value in CONFIG.items()
                   if key not in {"SEEDS", "EXPERIMENTS_TO_RUN", "STUDENT_INIT_MODES"}}
    CONFIG_HASH = hashlib.sha256(json.dumps(HASH_CONFIG, sort_keys=True).encode()).hexdigest()[:12]
    RUN_TIMESTAMP = RESUME_TIMESTAMP or datetime.now(ZoneInfo("America/Sao_Paulo")).strftime("%Y%m%d_%H%M%S")
    PROTOCOL_TAG = (f"{RUN_TIMESTAMP}_QW_V8_mobilenet_K{WHERE_TOP_K}_b{WHERE_BETA:g}"
                    f"_A{AMOUNT_APPLY_TO}_W{WHERE_APPLY_TO}"
                    f"_C{WHERE_CORRECT_MODE}-{WHERE_CORRECT_MASK_MODE}"
                    f"_E{WHERE_ERROR_MODE}-{WHERE_ERROR_MASK_MODE}_{CONFIG_HASH}")
    RESULTS_DIR = os.path.join(PROJECT_ROOT, "results", "QW_V8_MobileNet")
    RUNS_ROOT = os.path.join(RESULTS_DIR, PROTOCOL_TAG)
    RESULTS_XLSX_PATH = os.path.join(RUNS_ROOT, "complete_results.xlsx")
    SUMMARY_XLSX_PATH = os.path.join(RUNS_ROOT, "summary_diagnostics.xlsx")
    os.makedirs(RUNS_ROOT, exist_ok=True)
    with open(os.path.join(RUNS_ROOT, "config.json"), "w", encoding="utf-8") as file:
        json.dump(CONFIG, file, ensure_ascii=False, indent=2, allow_nan=False)
    print("Device:", DEVICE, "| seeds:", SEEDS, "| experiments:", EXPERIMENTS_TO_RUN)
    print("V8_mobilenet | correct:", WHERE_CORRECT_MODE, WHERE_CORRECT_MASK_MODE,
          "| error:", WHERE_ERROR_MODE, WHERE_ERROR_MASK_MODE,
          "| alpha:", CAM_ALPHA, "| beta:", WHERE_BETA)
    print("Apply QUANTO:", AMOUNT_APPLY_TO, "| apply ONDE:", WHERE_APPLY_TO,
          "| global s_T: all QW0/QW1/QW2/QW3")
    print("Config hash:", CONFIG_HASH, "| output:", RUNS_ROOT)


    def validate_v8_mobilenet_configuration(_config_keys=tuple(CONFIG_KEYS)):
        """Reject mixed notebook versions, stale globals and mismatched output folders."""
        expected_method = "QW_V8_mobilenet_signed_spatial_cam_sign_scaled_target_outcome_controls"
        if CONFIG.get("method_version") != expected_method:
            raise RuntimeError("Configuração incompatível com a V8_mobilenet. Reinicie o ambiente e execute este notebook em ordem.")
        changed = [key for key in _config_keys
                   if key not in CONFIG or key not in globals()
                   or json.dumps(CONFIG[key], sort_keys=True) != json.dumps(config_value(globals()[key]), sort_keys=True)]
        if changed:
            raise RuntimeError("Parâmetros diferem da configuração registrada: " + ", ".join(changed)
                               + ". Edite a configuração e execute desde o início.")
        hash_config = {key: value for key, value in CONFIG.items()
                       if key not in {"SEEDS", "EXPERIMENTS_TO_RUN", "STUDENT_INIT_MODES"}}
        expected_hash = hashlib.sha256(json.dumps(hash_config, sort_keys=True).encode()).hexdigest()[:12]
        expected_tag = (f"{RUN_TIMESTAMP}_QW_V8_mobilenet_K{WHERE_TOP_K}_b{WHERE_BETA:g}"
                        f"_A{AMOUNT_APPLY_TO}_W{WHERE_APPLY_TO}"
                        f"_C{WHERE_CORRECT_MODE}-{WHERE_CORRECT_MASK_MODE}"
                        f"_E{WHERE_ERROR_MODE}-{WHERE_ERROR_MASK_MODE}_{expected_hash}")
        expected_dir = os.path.join(PROJECT_ROOT, "results", "QW_V8_MobileNet")
        expected_root = os.path.join(expected_dir, expected_tag)
        expected_paths = {"RESULTS_DIR": expected_dir, "RUNS_ROOT": expected_root,
            "RESULTS_XLSX_PATH": os.path.join(expected_root, "complete_results.xlsx"),
            "SUMMARY_XLSX_PATH": os.path.join(expected_root, "summary_diagnostics.xlsx")}
        if CONFIG_HASH != expected_hash or PROTOCOL_TAG != expected_tag or any(
                os.path.normpath(str(globals()[key])) != os.path.normpath(path)
                for key, path in expected_paths.items()):
            raise RuntimeError("Hash, pasta ou tag não corresponde à configuração ativa da V8_mobilenet.")
        if getattr(globals().get("qw_cam_loss"), "qw_method_version", None) != expected_method:
            raise RuntimeError("A função qw_cam_loss ativa não é a V8_mobilenet. Execute as células deste notebook em ordem.")
        print(f"CONFIGURAÇÃO VALIDADA | V8_mobilenet | alpha={CAM_ALPHA:g} | beta={WHERE_BETA:g} | "
              f"amount_apply_to={AMOUNT_APPLY_TO} | where_apply_to={WHERE_APPLY_TO} | "
              f"correct={WHERE_CORRECT_MODE}/{WHERE_CORRECT_MASK_MODE} | "
              f"error={WHERE_ERROR_MODE}/{WHERE_ERROR_MASK_MODE}")

    print("MobileNetV2 / ImageNet-LT | AMP extrator:", USE_AMP, "| CAM/cabeça/loss: float32")


    # Execution/output controls only: intentionally outside CONFIG/CONFIG_HASH.
    ESSENTIAL_HISTORY_WRITE_EVERY = 10
    SAVE_RECOVERY_CHECKPOINTS = False
    EXECUTION_PROFILE = {'name': 'mobilenet_v8_essential_execution_v1', 'source_notebook_sha256': '7ba65eefc114d1197fa275e522d5825d62c79e5a72c4569fdc36940455de36d7', 'training_core_unchanged': True, 'changes': ['one detached essential accumulator', 'no gradient diagnostics or final CAM probes', 'epoch validation without unused probabilities', 'one final compact export', 'top5 groups saved directly; no prediction probability archives'], 'diagnostic_averaging': 'image means only for observed training views; static QUANTO calibration remains class-balanced as originally implemented', 'helpers_sha256': '1a6728bf83bf4ad3d3683bb2e53af114ecd43b901ecb4f2f1a7716b6d43be8f0', 'selfcheck_sha256': 'e8fccb31870138fcd0221f4707c3319e9b91008976472346fd00cb1b66e7e259'}


<a id="dados"></a>
## Dados, arquitetura e professores

| Aspecto | ResNet32 / CIFAR-100-LT | MobileNetV2 / ImageNet-LT |
|---|---|---|
| Construção dos dados | Pool desbalanceado e divisão estratificada 80/20 por seed | Splits oficiais, fixos entre seeds, sem nova subamostragem |
| Seed do pool | `LT_POOL_SEED=0` por padrão | Não há reconstrução de pool por seed |
| Dimensão de entrada | CIFAR, 32×32 | `IMG_SIZE=224`, avaliação com resize 256 e center crop |
| Professor | ResNet32 definido neste notebook, correspondente à divisão | MobileNetV2 baseline do ImageNet-LT, correspondente aos manifestos |
| Estudante padrão | Inicialização aleatória | Inicialização aleatória, mantendo Dropout e classificador originais |
| Precisão e execução | Política original do anexo ResNet | AMP bf16 opcional no extrator; GAP, cabeça, CAM e loss em float32 |

Os grupos usam contagens de treinamento: frequentes com mais de 100 imagens por classe, intermediárias de 20 a 100 e raras com menos de 20. Professor e estudante recebem a mesma view aumentada em cada batch; avaliação e calibração de Quanto são determinísticas conforme o perfil.

A preparação MobileNet valida todos os manifestos e arquivos antes do treinamento. Essas verificações são executadas somente quando esse perfil está selecionado. O carregamento estrito verifica a arquitetura dos pesos, mas a correspondência histórica entre professor e protocolo precisa ser garantida pelos arquivos fornecidos.


In [ ]:
assert_active_profile()

if EXPERIMENT_PROFILE == "resnet32_cifar100_lt":
    CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
    CIFAR_STD = (0.2023, 0.1994, 0.2010)
    to_normalized_tensor = transforms.Compose([
        transforms.ToTensor(), transforms.Normalize(CIFAR_MEAN, CIFAR_STD)])
    transform_eval = to_normalized_tensor


    def train_geometry(image):
        if not USE_TRAIN_AUG:
            return image
        image = TF.pad(image, padding=TRAIN_CROP_PADDING, fill=0)
        i, j, h, w = transforms.RandomCrop.get_params(image, output_size=(32, 32))
        image = TF.crop(image, i, j, h, w)
        if random.random() < TRAIN_HORIZONTAL_FLIP_P:
            image = TF.hflip(image)
        return image


    def transform_train(image):
        return to_normalized_tensor(train_geometry(image))


    class SingleViewCIFARArrayDataset(Dataset):
        def __init__(self, data, targets, transform=None, return_index=False):
            self.data, self.targets = data, list(targets)
            self.transform, self.return_index = transform, return_index

        def __len__(self):
            return len(self.targets)

        def __getitem__(self, index):
            image = Image.fromarray(self.data[index])
            if self.transform is not None:
                image = self.transform(image)
            pair = (image, int(self.targets[index]))
            return (*pair, int(index)) if self.return_index else pair


    _BASE_LT_DATA = _BASE_LT_TARGETS = _CLASS_NAMES = None


    def initialize_long_tail_pool():
        global _BASE_LT_DATA, _BASE_LT_TARGETS, _CLASS_NAMES
        if _BASE_LT_DATA is not None:
            return
        expected = Path(IMBALANCE_CIFAR_CODE_ROOT, "imbalance_cifar.py").resolve()
        if not expected.is_file():
            raise FileNotFoundError(f"Missing source sampler: {expected}")
        sys.path.insert(0, str(expected.parent))
        module = importlib.import_module("imbalance_cifar")
        if Path(module.__file__).resolve() != expected:
            raise RuntimeError("A different imbalance_cifar module is already imported; restart runtime.")
        previous_state = np.random.get_state()
        try:
            np.random.seed(LT_POOL_SEED)
            base = module.IMBALANCECIFAR100(root=DATA_ROOT, imb_type=IMB_TYPE,
                imb_factor=IMB_FACTOR, train=True, download=True, transform=None)
            _BASE_LT_DATA = np.asarray(base.data)
            _BASE_LT_TARGETS = np.asarray(base.targets, dtype=np.int64)
            _CLASS_NAMES = list(base.classes)
        finally:
            np.random.set_state(previous_state)


    def _get_targets(dataset):
        if isinstance(dataset, Subset):
            return np.asarray(dataset.dataset.targets)[np.asarray(dataset.indices)]
        return np.asarray(dataset.targets)


    def get_class_counts(dataset, num_classes=NUM_CLASSES):
        return np.bincount(_get_targets(dataset), minlength=num_classes).astype(int)


    def stratified_train_val_indices(targets, seed):
        train_idx, val_idx = train_test_split(np.arange(len(targets)), test_size=VAL_RATIO,
            random_state=seed, shuffle=True, stratify=targets)
        return train_idx.tolist(), val_idx.tolist()


    def build_datasets(seed):
        initialize_long_tail_pool()
        train_idx, val_idx = stratified_train_val_indices(_BASE_LT_TARGETS, seed)
        train = Subset(SingleViewCIFARArrayDataset(_BASE_LT_DATA, _BASE_LT_TARGETS,
                        transform_train, return_index=True), train_idx)
        val = Subset(SingleViewCIFARArrayDataset(_BASE_LT_DATA, _BASE_LT_TARGETS,
                        transform_eval), val_idx)
        test = datasets.CIFAR100(root=DATA_ROOT, train=False, download=True, transform=transform_eval)
        return train, val, test


    def build_loaders(seed):
        train, val, test = build_datasets(seed)
        common = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
        if NUM_WORKERS:
            common.update(worker_init_fn=seed_worker, persistent_workers=True, prefetch_factor=2)
        # Independent generators also for evaluation/statistics: no consumption of training RNG.
        train_loader = DataLoader(train, shuffle=True, generator=make_generator(seed), **common)
        val_loader = DataLoader(val, shuffle=False, generator=make_generator(seed + 1001), **common)
        test_loader = DataLoader(test, shuffle=False, generator=make_generator(seed + 1002), **common)
        return train, val, test, train_loader, val_loader, test_loader


    def build_deterministic_train_loader(train_dataset):
        base = SingleViewCIFARArrayDataset(_BASE_LT_DATA, _BASE_LT_TARGETS, transform_eval)
        dataset = Subset(base, list(train_dataset.indices))
        return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                          pin_memory=PIN_MEMORY, generator=make_generator(77123))
else:  # mobilenetv2_imagenet_lt
    # =====================
    # Copia e extrai os dados para o disco local da sessão
    # =====================
    # Só roda se os dados ainda não estiverem em /content (cada sessão nova começa vazia).

    def prepare_data():
        if (DATA_ROOT / 'train').is_dir() and (DATA_ROOT / 'val').is_dir() and all(path.exists() for path in (TRAIN_TXT, VAL_TXT, TEST_TXT)):
            print('Dados já estão em', DATA_ROOT)
            return
        if not DATA_ARCHIVE.exists():
            raise FileNotFoundError(f'Não encontrei o arquivo de dados no Drive: {DATA_ARCHIVE}')
        local_archive = LOCAL_DIR / DATA_ARCHIVE.name
        t0 = time.time()
        print(f'Copiando {DATA_ARCHIVE} -> {local_archive} ...')
        shutil.copy(DATA_ARCHIVE, local_archive)
        print(f'Cópia: {(time.time() - t0) / 60:.1f} min')
        t0 = time.time()
        print('Extraindo ...')
        if local_archive.suffix == '.zip':
            subprocess.run(['unzip', '-q', '-o', str(local_archive), '-d', str(LOCAL_DIR)], check=True)
        else:
            subprocess.run(['tar', '-xf', str(local_archive), '-C', str(LOCAL_DIR)], check=True)
        print(f'Extração: {(time.time() - t0) / 60:.1f} min')
        local_archive.unlink()
        if not (DATA_ROOT / 'train').is_dir():
            raise RuntimeError(f'Depois de extrair, não encontrei {DATA_ROOT / "train"}. '
                               'O arquivo precisa conter a pasta imagenet_lt/ com train/, val/ e os .txt.')


    prepare_data()

    # ImageNet-LT: splits oficiais; nenhuma subamostragem nem divisão 80/20.
    IMAGENET_MEAN = (0.485, 0.456, 0.406)
    IMAGENET_STD = (0.229, 0.224, 0.225)
    _CLASS_NAMES = None  # identificadores das pastas (WNIDs), sem inferir nomes semânticos.


    def read_split(txt_path):
        paths, labels = [], []
        with open(txt_path, encoding="utf-8") as handle:
            for line_number, line in enumerate(handle, 1):
                if not line.strip():
                    continue
                try:
                    relative, label = line.strip().rsplit(maxsplit=1)
                    label = int(label)
                except ValueError as error:
                    raise ValueError(f"Split inválido em {txt_path}:{line_number}") from error
                relative = relative.replace("\\", "/")
                if Path(relative).is_absolute() or ".." in Path(relative).parts:
                    raise ValueError(f"O split deve conter caminhos relativos ao DATA_ROOT: {relative}")
                if not 0 <= label < NUM_CLASSES:
                    raise ValueError(f"Rótulo {label} fora de [0, {NUM_CLASSES - 1}].")
                paths.append(relative)
                labels.append(label)
        if not paths or len(set(paths)) != len(paths):
            raise ValueError(f"Split vazio ou com caminhos duplicados: {txt_path}")
        return paths, labels


    class ImageNetLT(Dataset):
        """IDs são as linhas não vazias do respectivo manifesto, sempre junto ao split."""
        def __init__(self, root, txt_path, transform=None, return_index=False):
            self.root, self.txt_path = str(root), str(txt_path)
            self.rel_paths, self.targets = read_split(txt_path)
            self.paths = [os.path.join(self.root, *path.split("/")) for path in self.rel_paths]
            self.transform, self.return_index = transform, return_index
            self.indices = list(range(len(self.targets)))
            self.manifest_sha256 = hashlib.sha256(Path(txt_path).read_bytes()).hexdigest()

        def __len__(self):
            return len(self.targets)

        def __getitem__(self, index):
            with open(self.paths[index], "rb") as handle:
                image = Image.open(handle).convert("RGB")
            if self.transform is not None:
                image = self.transform(image)
            result = (image, int(self.targets[index]))
            return (*result, int(index)) if self.return_index else result


    _train_tf = [transforms.RandomResizedCrop(IMG_SIZE),
                 transforms.RandomHorizontalFlip(p=TRAIN_HORIZONTAL_FLIP_P)]
    if USE_COLOR_JITTER:
        _train_tf.append(transforms.ColorJitter(0.4, 0.4, 0.4, 0.0))
    transform_eval = transforms.Compose([
        transforms.Resize(RESIZE_TEST), transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
    transform_train = (transforms.Compose(_train_tf + [transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)]) if USE_TRAIN_AUG else transform_eval)


    def make_loader(dataset, shuffle=False, generator=None, persistent=True):
        common = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                      pin_memory=PIN_MEMORY, shuffle=shuffle,
                      generator=generator if generator is not None else make_generator(0))
        if NUM_WORKERS > 0:
            common.update(worker_init_fn=seed_worker, persistent_workers=persistent,
                          prefetch_factor=PREFETCH_FACTOR)
        return DataLoader(dataset, **common)


    def _get_targets(dataset):
        if isinstance(dataset, Subset):
            return _get_targets(dataset.dataset)[np.asarray(dataset.indices)]
        return np.asarray(dataset.targets, dtype=np.int64)


    def get_class_counts(dataset, num_classes=NUM_CLASSES):
        return np.bincount(_get_targets(dataset), minlength=num_classes).astype(int)


    def build_datasets(seed=None):
        # seed não altera os splits oficiais; permanece na interface da V8.
        return (ImageNetLT(DATA_ROOT, TRAIN_TXT, transform_train, return_index=True),
                ImageNetLT(DATA_ROOT, VAL_TXT, transform_eval),
                ImageNetLT(DATA_ROOT, TEST_TXT, transform_eval))


    def build_loaders(seed):
        train, val, test = build_datasets(seed)
        return (train, val, test,
            make_loader(train, True, make_generator(seed)),
            make_loader(val, False, make_generator(seed + 1001)),
            make_loader(test, False, make_generator(seed + 1002)))


    def build_deterministic_train_loader(train_dataset):
        # Apenas TRAIN, com o mesmo Resize/CenterCrop da avaliação; sem aumento.
        dataset = ImageNetLT(DATA_ROOT, TRAIN_TXT, transform_eval)
        return make_loader(dataset, False, make_generator(77123), persistent=False)


    def current_split_metadata():
        metadata = {}
        for split, txt in (("train", TRAIN_TXT), ("val", VAL_TXT), ("test", TEST_TXT)):
            metadata[f"{split}_manifest_path"] = str(txt)
            metadata[f"{split}_manifest_sha256"] = hashlib.sha256(Path(txt).read_bytes()).hexdigest()
        hashes = {key: value for key, value in metadata.items() if key.endswith("_sha256")}
        metadata["split_manifests_sha256"] = hashlib.sha256(
            json.dumps(hashes, sort_keys=True).encode()).hexdigest()
        metadata["sample_id_definition"] = "zero_based_nonempty_line_in_each_official_split_manifest"
        metadata["split_policy"] = "official_ImageNet_LT_train_val_test_no_resampling"
        return metadata


    def validate_imagenet_lt_data():
        global _CLASS_NAMES
        train, val, test = build_datasets()
        names, rows = [None] * NUM_CLASSES, []
        for relative, label in zip(train.rel_paths, train.targets):
            folder = Path(relative).parent.name
            if names[label] is None:
                names[label] = folder or str(label)
            elif names[label] != folder:
                names[label] = str(label)  # um identificador seguro se a organização variar.
        _CLASS_NAMES = [name if name is not None else str(label) for label, name in enumerate(names)]
        path_sets = {}
        for split, dataset in (("train", train), ("val", val), ("test", test)):
            counts = get_class_counts(dataset)
            missing = [path for path in dataset.paths if not os.path.isfile(path)]
            rows.append(dict(split=split, n_images=len(dataset),
                expected=EXPECTED_SPLIT_SIZES[split], n_classes=int((counts > 0).sum()),
                min_per_class=int(counts.min()), max_per_class=int(counts.max()),
                missing_files=len(missing)))
            if missing:
                raise FileNotFoundError(f"{split}: {len(missing)} imagens ausentes. Exemplos: {missing[:3]}")
            if len(dataset) != EXPECTED_SPLIT_SIZES[split] or not (counts > 0).all():
                raise RuntimeError(f"Split {split} incompatível com o ImageNet-LT oficial: {rows[-1]}")
            path_sets[split] = set(dataset.paths)
        for first, second in (("train", "val"), ("train", "test"), ("val", "test")):
            if path_sets[first] & path_sets[second]:
                raise RuntimeError(f"Há imagens compartilhadas entre {first} e {second}.")
        report = pd.DataFrame(rows)
        report.to_csv(Path(RUNS_ROOT, "dataset_splits.csv"), index=False)
        pd.DataFrame({"class_id": np.arange(NUM_CLASSES), "class_name": _CLASS_NAMES,
            "n_train": get_class_counts(train), "n_val": get_class_counts(val),
            "n_test": get_class_counts(test)}).to_csv(Path(RUNS_ROOT, "dataset_classes.csv"), index=False)
        Path(RUNS_ROOT, "data_protocol.json").write_text(
            json.dumps(current_split_metadata(), indent=2, ensure_ascii=False), encoding="utf-8")
        display(report)
        print("Dados validados: splits oficiais disjuntos, 1.000 classes e imagens disponíveis.")


    validate_imagenet_lt_data()


In [ ]:
assert_active_profile()

if EXPERIMENT_PROFILE == "resnet32_cifar100_lt":
    # CIFAR ResNet architecture used by the teacher and student.
    class LambdaLayer(nn.Module):
        def __init__(self, lambd):
            super().__init__()
            self.lambd = lambd

        def forward(self, x):
            return self.lambd(x)


    class BasicBlockCIFAR(nn.Module):
        expansion = 1

        def __init__(self, in_planes, planes, stride=1, option="A"):
            super().__init__()

            self.conv1 = nn.Conv2d(
                in_planes,
                planes,
                kernel_size=3,
                stride=stride,
                padding=1,
                bias=False,
            )
            self.bn1 = nn.BatchNorm2d(planes)
            self.relu = nn.ReLU(inplace=True)

            self.conv2 = nn.Conv2d(
                planes,
                planes,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False,
            )
            self.bn2 = nn.BatchNorm2d(planes)

            self.shortcut = nn.Sequential()

            if stride != 1 or in_planes != planes:
                if option == "A":
                    # CIFAR ResNet option A: spatial slicing plus channel zero-padding.
                    self.shortcut = LambdaLayer(
                        lambda x: F.pad(
                            x[:, :, ::2, ::2],
                            (0, 0, 0, 0, planes // 4, planes // 4),
                            "constant",
                            0,
                        )
                    )
                elif option == "B":
                    self.shortcut = nn.Sequential(
                        nn.Conv2d(
                            in_planes,
                            planes,
                            kernel_size=1,
                            stride=stride,
                            bias=False,
                        ),
                        nn.BatchNorm2d(planes),
                    )
                else:
                    raise ValueError("option must be 'A' or 'B'")

        def forward(self, x):
            out = self.relu(self.bn1(self.conv1(x)))
            out = self.bn2(self.conv2(out))
            out += self.shortcut(x)
            out = self.relu(out)
            return out


    class ResNetCIFAR(nn.Module):
        def __init__(self, block, num_blocks, num_classes=100, option="A"):
            super().__init__()

            self.in_planes = 16
            self.option = option

            self.conv1 = nn.Conv2d(
                3, 16, kernel_size=3, stride=1, padding=1, bias=False
            )
            self.bn1 = nn.BatchNorm2d(16)
            self.relu = nn.ReLU(inplace=True)

            self.layer1 = self._make_layer(block, 16, num_blocks[0], stride=1)
            self.layer2 = self._make_layer(block, 32, num_blocks[1], stride=2)
            self.layer3 = self._make_layer(block, 64, num_blocks[2], stride=2)

            self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
            self.fc = nn.Linear(64 * block.expansion, num_classes)

            self._init_weights()

        def _make_layer(self, block, planes, num_blocks, stride):
            strides = [stride] + [1] * (num_blocks - 1)
            layers = []

            for current_stride in strides:
                layers.append(
                    block(
                        self.in_planes,
                        planes,
                        current_stride,
                        option=self.option,
                    )
                )
                self.in_planes = planes * block.expansion

            return nn.Sequential(*layers)

        def _init_weights(self):
            for module in self.modules():
                if isinstance(module, nn.Conv2d):
                    nn.init.kaiming_normal_(
                        module.weight,
                        mode="fan_out",
                        nonlinearity="relu",
                    )
                elif isinstance(module, nn.BatchNorm2d):
                    nn.init.constant_(module.weight, 1)
                    nn.init.constant_(module.bias, 0)
                elif isinstance(module, nn.Linear):
                    nn.init.normal_(module.weight, 0, 0.01)
                    nn.init.constant_(module.bias, 0)

        def forward(self, x):
            out = self.relu(self.bn1(self.conv1(x)))
            f0 = out

            out = self.layer1(out)
            f1 = out

            out = self.layer2(out)
            f2 = out

            out = self.layer3(out)
            f3 = out

            out = self.avgpool(out)
            pooled = torch.flatten(out, 1)
            logits = self.fc(pooled)

            features = {
                "feats": [f0, f1, f2, f3],
                "pooled_feat": pooled,
            }

            return logits, features


    def resnet32_cifar(num_classes=100):
        # CIFAR ResNet depth: 6n + 2 = 32 -> n = 5.
        return ResNetCIFAR(
            BasicBlockCIFAR,
            [5, 5, 5],
            num_classes=num_classes,
            option="A",
        )
else:  # mobilenetv2_imagenet_lt
    # MobileNetV2 torchvision padrão (width 1.0), com acesso às últimas features.
    # As chaves features.* / classifier.* continuam idênticas às do checkpoint original.
    class MobileNetV2CAM(nn.Module):
        def __init__(self, num_classes=NUM_CLASSES):
            super().__init__()
            backbone = mobilenet_v2(weights=None, num_classes=num_classes)
            self.features = backbone.features
            self.classifier = backbone.classifier  # preserva Dropout(0.2) + Linear.
            self.last_channel = backbone.last_channel

        @property
        def fc(self):
            # Alias por propriedade: NÃO registra uma segunda cópia da camada.
            return self.classifier[-1]

        def forward(self, images):
            if CHANNELS_LAST and images.device.type == "cuda":
                images = images.contiguous(memory_format=torch.channels_last)
            # bf16 acelera o extrator; CAMs, GAP, classificador e loss permanecem float32.
            # A conversão ocorre ANTES do GAP: CE e CAM compartilham o mesmo tensor,
            # preservando os diagnósticos de gradientes e a identidade CAM-logit.
            with torch.autocast(device_type=images.device.type, dtype=torch.bfloat16,
                                enabled=bool(USE_AMP and images.device.type == "cuda")):
                feature_map = self.features(images)
            with torch.autocast(device_type=images.device.type, enabled=False):
                feature_map = feature_map.float()
                pooled = F.adaptive_avg_pool2d(feature_map, (1, 1)).flatten(1)
                logits = self.classifier(pooled)
            return logits, {"feats": [feature_map], "pooled_feat": pooled}


    def mobilenet_v2_cam(num_classes=NUM_CLASSES):
        return MobileNetV2CAM(num_classes=num_classes)


In [ ]:
assert_active_profile()

if EXPERIMENT_PROFILE == "resnet32_cifar100_lt":
    # Teacher and student builders.
    def load_teacher(seed):
        """Load and freeze the harmonized CE teacher for the current seed."""
        teacher_ckpt_path = TEACHER_CKPT_TEMPLATE.format(seed=seed)

        if not os.path.isfile(teacher_ckpt_path):
            raise FileNotFoundError(
                f"Teacher checkpoint not found: {teacher_ckpt_path}. "
                "Adjust TEACHER_CKPT_TEMPLATE to the folder produced by the "
                "harmonized CE baseline notebook."
            )

        teacher = resnet32_cifar(num_classes=NUM_CLASSES).to(DEVICE)
        checkpoint = torch.load(
            teacher_ckpt_path,
            map_location=DEVICE,
            weights_only=False,
        )

        if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
            state_dict = checkpoint["model_state_dict"]
        else:
            state_dict = checkpoint

        teacher.load_state_dict(state_dict, strict=True)
        teacher.eval()

        for parameter in teacher.parameters():
            parameter.requires_grad_(False)

        return teacher, teacher_ckpt_path


    def build_student(student_init_mode, teacher=None):
        """Build a fully trainable ResNet32 student."""
        student = resnet32_cifar(num_classes=NUM_CLASSES).to(DEVICE)

        if student_init_mode == "random":
            pass
        elif student_init_mode == "teacher_init":
            if teacher is None:
                raise ValueError("teacher must be provided when using teacher_init")
            student.load_state_dict(teacher.state_dict(), strict=True)
        else:
            raise ValueError(f"Unknown student initialization mode: {student_init_mode}")

        for parameter in student.parameters():
            parameter.requires_grad_(True)

        return student
else:  # mobilenetv2_imagenet_lt

    def load_teacher(seed):
        """Carrega o baseline CE já treinado na mesma seed; congela todos os pesos."""
        path = TEACHER_CKPT_TEMPLATE.format(seed=seed)
        if not os.path.isfile(path):
            raise FileNotFoundError(f"Checkpoint do professor não encontrado: {path}")
        checkpoint = torch.load(path, map_location="cpu", weights_only=False)
        state = (checkpoint["model_state_dict"] if isinstance(checkpoint, dict)
                 and "model_state_dict" in checkpoint else checkpoint)
        teacher = mobilenet_v2_cam(NUM_CLASSES)
        teacher.load_state_dict(state, strict=True)
        teacher = teacher.to(DEVICE)
        if CHANNELS_LAST and DEVICE.type == "cuda":
            teacher = teacher.to(memory_format=torch.channels_last)
        teacher.eval()
        teacher.requires_grad_(False)
        teacher._qw_checkpoint_sha256 = file_sha256(path)
        return teacher, path


    def build_student(student_init_mode, teacher=None):
        """A inicialização é sempre aleatória; run_experiment fixa a seed antes daqui."""
        if student_init_mode != "random":
            raise ValueError("V8_mobilenet exige STUDENT_INIT_MODES=['random'].")
        student = mobilenet_v2_cam(NUM_CLASSES).to(DEVICE)
        if CHANNELS_LAST and DEVICE.type == "cuda":
            student = student.to(memory_format=torch.channels_last)
        student.requires_grad_(True)
        return student


    @torch.no_grad()
    def check_teacher_baseline(teacher, val_loader, class_counts, seed):
        """Verifica somente validação; não consulta o teste antes da seleção do estudante."""
        if not RUN_TEACHER_CHECK:
            return {"enabled": False}
        path = TEACHER_METRICS_TEMPLATE.format(seed=seed)
        if not os.path.isfile(path):
            raise FileNotFoundError(f"Métricas do baseline ausentes: {path}. "
                "Ajuste TEACHER_METRICS_TEMPLATE ou desative explicitamente RUN_TEACHER_CHECK.")
        stored = json.loads(Path(path).read_text(encoding="utf-8"))
        expected = float(stored["best_monitor_metric"])
        cache_key = (seed, teacher._qw_checkpoint_sha256, current_split_metadata()["val_manifest_sha256"],
                     bool(USE_AMP), bool(CHANNELS_LAST), IMG_SIZE, RESIZE_TEST)
        if cache_key in TEACHER_VALIDATION_CACHE:
            return dict(TEACHER_VALIDATION_CACHE[cache_key])
        metrics, _, _, _ = evaluate_with_arrays(teacher, val_loader, class_counts)
        observed = float(metrics["f1_macro"])
        difference = abs(observed - expected)
        result = {"enabled": True, "registered_val_f1_macro": expected,
            "observed_val_f1_macro": observed, "absolute_difference": difference,
            "tolerance": TEACHER_CHECK_TOL, "ok": bool(np.isfinite(difference) and difference <= TEACHER_CHECK_TOL),
            "metrics_path": path,
            "precision_note": "bf16 optional on extractor; GAP, classifier and signed CAM arithmetic float32"}
        if not result["ok"]:
            raise RuntimeError(f"Professor seed={seed}: F1 de validação {observed:.6f}, "
                f"registrado {expected:.6f}; diferença {difference:.6f} > {TEACHER_CHECK_TOL}. "
                "Confira checkpoint, splits e pré-processamento. USE_AMP=False permite conferir em float32.")
        TEACHER_VALIDATION_CACHE[cache_key] = result
        print(f"Professor validado | seed={seed} | F1={observed:.6f} | diferença={difference:.3g}")
        return dict(result)


    TEACHER_VALIDATION_CACHE = {}


<a id="metricas"></a>
## Métricas de classificação

Top-1 (`acc`) e top-5 global (`top5_acc`) são salvos em porcentagem. F1 e acurácia balanceada são salvos em [0, 1]. O F1 por grupo utiliza o subconjunto de imagens da classe verdadeira e os rótulos das classes do grupo; não é substituído silenciosamente por uma média de F1 por classe no conjunto completo.

O perfil MobileNet essencial salva também top-5 por grupo diretamente na avaliação final, mantendo todas as classes no ranking. O perfil ResNet mantém seu conjunto original de métricas e arquivos de predição.

O checkpoint é selecionado pelo F1 macro de validação com o conjunto fixo de classes. Recuperação é a proporção de erros do professor corrigidos pelo estudante; regressão é a proporção de acertos do professor perdidos pelo estudante. Os denominadores são distintos. Deltas entre estudante e professor preservam as unidades da métrica correspondente.


In [ ]:
assert_active_profile()

# Código compartilhado, idêntico nos dois anexos.
# Metrics for imbalanced classification.
def get_shot_groups(
        class_counts,
        many_thr=MANY_SHOT_THRESHOLD,
        few_thr=FEW_SHOT_THRESHOLD):
    """Split classes into many-, medium-, and few-shot groups."""
    class_counts = np.asarray(class_counts)

    many_shot = np.where(class_counts > many_thr)[0]
    medium_shot = np.where(
        (class_counts <= many_thr) &
        (class_counts >= few_thr)
    )[0]
    few_shot = np.where(class_counts < few_thr)[0]

    return {
        "many": many_shot,
        "medium": medium_shot,
        "few": few_shot,
    }


def compute_group_metrics(all_labels, all_preds, groups):
    """Compute accuracy and Macro-F1 for each shot group."""
    metrics = {}

    for group_name, class_ids in groups.items():
        mask = np.isin(all_labels, class_ids)

        if mask.sum() == 0:
            metrics[f"{group_name}_shot_acc"] = np.nan
            metrics[f"{group_name}_shot_f1"] = np.nan
            metrics[f"{group_name}_shot_support"] = 0
            metrics[f"{group_name}_shot_num_classes"] = len(class_ids)
            continue

        y_true = all_labels[mask]
        y_pred = all_preds[mask]

        metrics[f"{group_name}_shot_acc"] = (
            (y_true == y_pred).mean() * 100
        )

        metrics[f"{group_name}_shot_f1"] = f1_score(
            y_true,
            y_pred,
            labels=class_ids,
            average="macro",
            zero_division=0,
        )

        metrics[f"{group_name}_shot_support"] = int(mask.sum())
        metrics[f"{group_name}_shot_num_classes"] = len(class_ids)

    return metrics


def evaluate_model(
        model,
        val_loader,
        device,
        num_classes,
        class_counts=None):
    """Evaluate global and many/medium/few-shot performance."""
    model.eval()

    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            logits, _ = model(images)
            probs = torch.softmax(logits, dim=1)
            preds = logits.argmax(dim=1)

            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
            all_probs.append(probs.cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    all_probs = np.vstack(all_probs)

    metrics = {
        "acc": (all_preds == all_labels).mean() * 100,
        "top5_acc": top_k_accuracy_score(
            all_labels,
            all_probs,
            k=5,
            labels=np.arange(num_classes),
        ) * 100,
        "f1_macro": f1_score(
            all_labels,
            all_preds,
            average="macro",
            zero_division=0,
        ),
        "f1_weighted": f1_score(
            all_labels,
            all_preds,
            average="weighted",
            zero_division=0,
        ),
        "balanced_acc": balanced_accuracy_score(
            all_labels,
            all_preds,
        ),
    }

    if class_counts is not None:
        groups = get_shot_groups(class_counts)
        metrics.update(
            compute_group_metrics(
                all_labels,
                all_preds,
                groups,
            )
        )

    return metrics


In [ ]:
assert_active_profile()

# Código compartilhado, idêntico nos dois anexos.
GROUP_NAME_TO_ID = {"many":0, "medium":1, "few":2}
GROUP_ID_TO_NAME = {value:key for key,value in GROUP_NAME_TO_ID.items()}
def build_class_to_group(
        class_counts):

    groups = get_shot_groups(
        class_counts,
        many_thr=MANY_SHOT_THRESHOLD,
        few_thr=FEW_SHOT_THRESHOLD,
    )

    class_to_group = np.full(
        len(
            class_counts
        ),
        -1,
        dtype=np.int64,
    )

    for name, ids in groups.items():
        class_to_group[
            np.asarray(
                ids,
                dtype=np.int64,
            )
        ] = GROUP_NAME_TO_ID[
            name
        ]

    if np.any(
        class_to_group <
        0
    ):
        raise RuntimeError(
            "Unassigned classes."
        )

    return (
        torch.as_tensor(
            class_to_group,
            dtype=torch.long,
            device=DEVICE,
        ),
        groups,
    )


<a id="quanto"></a>
## Quanto — compensação por grupo e sinal

As CAMs alvo do professor no treino determinístico fornecem as massas positivas e negativas. A estimação agrega imagens por classe e depois classes por grupo. A potência `AMOUNT_POWER` (**Beta Quanto**) e a calibração conjunta definem fatores $q_g^+$ e $q_g^-$ fixos por grupo, configuração e seed.

Nas imagens selecionadas, a referência é $T_{\mathrm{ret}}=q_g^+\max(T,0)-q_g^-\max(-T,0)$. Fora da seleção de `AMOUNT_APPLY_TO`, ou com Quanto desativado, a referência é exatamente $T$. A seleção por desfecho não recalcula os fatores nem produz outra normalização.

O valor resumido `q`/`amount_weight` é a média espacial do mapa de fatores aplicado, não um multiplicador escalar de toda a CAM. Massas negativas são magnitudes não negativas. A escala global $s_T$ é estimada nas CAMs originais do professor e permanece ativa em QW0, QW1, QW2 e QW3.


In [ ]:
assert_active_profile()

# Código compartilhado, idêntico nos dois anexos.
def extract_feature_tensor(
        features):

    if torch.is_tensor(
        features
    ):
        return features

    if isinstance(
        features,
        dict,
    ):
        features = features[
            "feats"
        ]

    if torch.is_tensor(
        features
    ):
        return features

    if isinstance(
        features,
        (list, tuple),
    ):
        for feature in reversed(
            features
        ):
            if (
                torch.is_tensor(
                    feature
                ) and
                feature.dim() == 4
            ):
                return feature

    raise TypeError(
        f"Could not extract feature tensor "
        f"from {type(features)}"
    )

def raw_target_cam(
        features,
        fc_layer,
        labels):

    feature_map = (
        extract_feature_tensor(
            features
        )
    )

    labels = labels.long()

    weights = fc_layer.weight[
        labels
    ]

    return (
        feature_map *
        weights[
            :,
            :,
            None,
            None,
        ]
    ).sum(
        dim=1
    )

def _normalize_group_factors(
        factors,
        sample_proportions=None):
    """
    Normalize group factors without changing their relative ratios.

    For QUANTO, sample_proportions is supplied and normalization enforces:

        E[q] = sum_G pi_G q_G = 1

    sample_proportions=None retains the generic equal-group normalization.
    New ONDE does not use inverse positive/negative group factors.
    """
    factors = np.asarray(
        factors,
        dtype=np.float64,
    )

    if sample_proportions is None:
        normalizer = float(
            factors.mean()
        )
    else:
        sample_proportions = np.asarray(
            sample_proportions,
            dtype=np.float64,
        )

        sample_proportions = (
            sample_proportions /
            max(
                float(
                    sample_proportions.sum()
                ),
                1e-12,
            )
        )

        normalizer = float(
            np.sum(
                sample_proportions *
                factors
            )
        )

    return (
        factors /
        max(
            normalizer,
            1e-12,
        )
    )

def _apply_power_and_optional_clip(
        raw_factors,
        power,
        min_value=None,
        max_value=None,
        sample_proportions=None):

    factors = np.power(
        np.asarray(
            raw_factors,
            dtype=np.float64,
        ),
        power,
    )

    factors = (
        _normalize_group_factors(
            factors,
            sample_proportions=sample_proportions,
        )
    )

    if (
        min_value is not None or
        max_value is not None
    ):
        lower = (
            -np.inf
            if min_value is None
            else min_value
        )

        upper = (
            np.inf
            if max_value is None
            else max_value
        )

        factors = np.clip(
            factors,
            lower,
            upper,
        )

        factors = (
            _normalize_group_factors(
                factors,
                sample_proportions=sample_proportions,
            )
        )

    return factors


In [ ]:
assert_active_profile()

if EXPERIMENT_PROFILE == "resnet32_cifar100_lt":
    # Frozen-teacher group statistics; named to avoid shadowing Python's statistics module.
    TEACHER_GROUP_STATS_CACHE = {}


    def compute_signed_amount_factors(group_sign_means, group_sign_pixel_counts,
                                      power, min_value=None, max_value=None):
        """Positive/negative group factors with ONE joint TRAIN-pixel calibration.

        Rows of group_sign_pixel_counts are positive, negative and zero; columns
        are many/medium/few. A missing sign/group has raw factor one because no
        training position supports estimating its inverse magnitude. Zero targets
        remain exactly zero. The old power/clipping/renormalization helper is reused.
        """
        means=np.asarray(group_sign_means,dtype=np.float64)
        counts=np.asarray(group_sign_pixel_counts,dtype=np.float64)
        if means.shape!=(2,3) or counts.shape!=(3,3) or (counts<0).any() or not (counts.sum(0)>0).all():
            raise ValueError("Signed QUANTO requires 2x3 group means and nonempty 3x3 sign/group pixel counts.")
        raw=np.ones((3,3),dtype=np.float64)
        raw[:2]=np.where(counts[:2]>0,means.mean(1,keepdims=True)/np.maximum(means,1e-12),1.)
        joint=counts/counts.sum()
        factors=_apply_power_and_optional_clip(raw.ravel(),power=power,
            min_value=min_value,max_value=max_value,sample_proportions=joint.ravel()).reshape(3,3)
        if not np.isfinite(factors).all():
            raise FloatingPointError("Nonfinite signed intensity factors; inspect AMOUNT_POWER and group statistics.")
        fractions=counts/counts.sum(0,keepdims=True)
        nominal=(factors*fractions).sum(0)
        return {"raw_amount_factors":(raw*fractions).sum(0),"amount_factors":nominal,
            "raw_amount_factors_positive":raw[0],"raw_amount_factors_negative":raw[1],
            "amount_factors_positive":factors[0],"amount_factors_negative":factors[1],
            "amount_factors_zero":factors[2],"group_sign_pixel_counts":counts.astype(np.int64),
            "group_sign_pixel_proportions":joint,"group_sign_pixel_fractions":fractions,
            "amount_expected_weight":float((joint*factors).sum())}


    @torch.no_grad()
    def estimate_teacher_group_cam_statistics(seed, teacher, deterministic_train_loader,
                                              class_counts, class_to_group):
        """V8 QUANTO: deterministic TRAIN; signed class/group means; joint factor calibration."""
        split_hash = hashlib.sha256(np.asarray(deterministic_train_loader.dataset.indices,
                                              dtype=np.int64).tobytes()).hexdigest()
        cache_key = (seed, id(teacher), split_hash, AMOUNT_POWER,
                     AMOUNT_FACTOR_MIN, AMOUNT_FACTOR_MAX)
        if cache_key in TEACHER_GROUP_STATS_CACHE:
            return TEACHER_GROUP_STATS_CACHE[cache_key]
        teacher.eval()
        totals = np.zeros((3, NUM_CLASSES), dtype=np.float64)
        numbers = np.zeros(NUM_CLASSES, dtype=np.int64)
        squares = np.zeros(3, dtype=np.float64)
        sign_pixels = np.zeros((3, NUM_CLASSES), dtype=np.int64)
        n_pixels = 0
        for images, labels in deterministic_train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            logits, features = teacher(images)
            cam = raw_target_cam(features, teacher.fc, labels)
            torch.testing.assert_close(cam.mean((-2,-1)) + teacher.fc.bias[labels],
                logits.gather(1, labels[:,None]).squeeze(1), rtol=1e-4, atol=1e-5)
            components = (cam.abs(), cam.clamp_min(0), (-cam).clamp_min(0))
            y = labels.cpu().numpy()
            np.add.at(numbers, y, 1)
            for j, values in enumerate(components):
                np.add.at(totals[j], y, values.mean((-2,-1)).cpu().numpy())
                squares[j] += float(values.square().sum().item())
            for j, mask in enumerate((cam.gt(0),cam.lt(0),cam.eq(0))):
                np.add.at(sign_pixels[j],y,mask.sum((-2,-1)).cpu().numpy())
            n_pixels += cam.numel()
        if (numbers == 0).any():
            raise RuntimeError("At least one class has no training samples.")
        class_means = totals / numbers[None, :]
        group_np = class_to_group.cpu().numpy()
        group_means = np.stack([class_means[:, group_np == g].mean(1) for g in range(3)], 1)
        group_counts = np.array([np.asarray(class_counts)[group_np == g].sum()
                                 for g in range(3)], dtype=np.float64)
        proportions = group_counts / group_counts.sum()
        group_sign_counts=np.stack([sign_pixels[:,group_np==g].sum(1) for g in range(3)],1)
        signed_factors=compute_signed_amount_factors(group_means[1:3],group_sign_counts,
            power=AMOUNT_POWER,min_value=AMOUNT_FACTOR_MIN,max_value=AMOUNT_FACTOR_MAX)
        stats = {
            "class_abs_mean": class_means[0], "class_pos_mean": class_means[1],
            "class_neg_mean": class_means[2], "group_abs_mean": group_means[0],
            "group_pos_mean": group_means[1], "group_neg_mean": group_means[2],
            "group_num_classes": np.array([(group_np == g).sum() for g in range(3)]),
            "group_sample_counts": group_counts, "group_sample_proportions": proportions,
            **signed_factors,
            "raw_cam_mean_sq": squares[0] / max(n_pixels, 1),
            "positive_cam_mean_sq": squares[1] / max(n_pixels, 1),
            "negative_cam_mean_sq": squares[2] / max(n_pixels, 1),
        }
        np.testing.assert_allclose(stats["amount_expected_weight"], 1., atol=1e-10)
        TEACHER_GROUP_STATS_CACHE[cache_key] = stats
        return stats


    def print_group_statistics(stats):
        table = pd.DataFrame([{
            "group": GROUP_ID_TO_NAME[g], "n_classes": int(stats["group_num_classes"][g]),
            "n_train_samples": int(stats["group_sample_counts"][g]),
            "sample_proportion": stats["group_sample_proportions"][g],
            "raw_abs_mass_mean": stats["group_abs_mean"][g],
            "raw_pos_mass_mean": stats["group_pos_mean"][g],
            "raw_neg_mass_mean": stats["group_neg_mean"][g],
            "amount_factor_raw": stats["raw_amount_factors"][g],
            "amount_factor_if_enabled": stats["amount_factors"][g],
            "E_q_global": stats["amount_expected_weight"],
            "amount_factor_positive_raw":stats["raw_amount_factors_positive"][g],
            "amount_factor_negative_raw":stats["raw_amount_factors_negative"][g],
            "amount_factor_positive":stats["amount_factors_positive"][g],
            "amount_factor_negative":stats["amount_factors_negative"][g],
            "amount_factor_zero":stats["amount_factors_zero"][g],
            "positive_pixel_fraction":stats["group_sign_pixel_fractions"][0,g],
            "negative_pixel_fraction":stats["group_sign_pixel_fractions"][1,g],
            "zero_pixel_fraction":stats["group_sign_pixel_fractions"][2,g],
            "q_times_raw_mass_diagnostic":stats["amount_factors_positive"][g]*stats["group_pos_mean"][g]
                +stats["amount_factors_negative"][g]*stats["group_neg_mean"][g],
        } for g in range(3)])
        display(table.round(6))
        return table
else:  # mobilenetv2_imagenet_lt
    # Frozen-teacher group statistics; named to avoid shadowing Python's statistics module.
    TEACHER_GROUP_STATS_CACHE = {}


    def compute_signed_amount_factors(group_sign_means, group_sign_pixel_counts,
                                      power, min_value=None, max_value=None):
        """Positive/negative group factors with ONE joint TRAIN-pixel calibration.

        Rows of group_sign_pixel_counts are positive, negative and zero; columns
        are many/medium/few. A missing sign/group has raw factor one because no
        training position supports estimating its inverse magnitude. Zero targets
        remain exactly zero. The old power/clipping/renormalization helper is reused.
        """
        means=np.asarray(group_sign_means,dtype=np.float64)
        counts=np.asarray(group_sign_pixel_counts,dtype=np.float64)
        if means.shape!=(2,3) or counts.shape!=(3,3) or (counts<0).any() or not (counts.sum(0)>0).all():
            raise ValueError("Signed QUANTO requires 2x3 group means and nonempty 3x3 sign/group pixel counts.")
        raw=np.ones((3,3),dtype=np.float64)
        raw[:2]=np.where(counts[:2]>0,means.mean(1,keepdims=True)/np.maximum(means,1e-12),1.)
        joint=counts/counts.sum()
        factors=_apply_power_and_optional_clip(raw.ravel(),power=power,
            min_value=min_value,max_value=max_value,sample_proportions=joint.ravel()).reshape(3,3)
        if not np.isfinite(factors).all():
            raise FloatingPointError("Nonfinite signed intensity factors; inspect AMOUNT_POWER and group statistics.")
        fractions=counts/counts.sum(0,keepdims=True)
        nominal=(factors*fractions).sum(0)
        return {"raw_amount_factors":(raw*fractions).sum(0),"amount_factors":nominal,
            "raw_amount_factors_positive":raw[0],"raw_amount_factors_negative":raw[1],
            "amount_factors_positive":factors[0],"amount_factors_negative":factors[1],
            "amount_factors_zero":factors[2],"group_sign_pixel_counts":counts.astype(np.int64),
            "group_sign_pixel_proportions":joint,"group_sign_pixel_fractions":fractions,
            "amount_expected_weight":float((joint*factors).sum())}


    @torch.no_grad()
    def estimate_teacher_group_cam_statistics(seed, teacher, deterministic_train_loader,
                                              class_counts, class_to_group):
        """V8 QUANTO: deterministic TRAIN; signed class/group means; joint factor calibration."""
        split_hash = deterministic_train_loader.dataset.manifest_sha256
        cache_key = (seed, getattr(teacher, "_qw_checkpoint_sha256", id(teacher)), split_hash,
                     AMOUNT_POWER, AMOUNT_FACTOR_MIN, AMOUNT_FACTOR_MAX,
                     bool(USE_AMP), bool(CHANNELS_LAST), IMG_SIZE, RESIZE_TEST)
        if cache_key in TEACHER_GROUP_STATS_CACHE:
            return TEACHER_GROUP_STATS_CACHE[cache_key]
        teacher.eval()
        totals = np.zeros((3, NUM_CLASSES), dtype=np.float64)
        numbers = np.zeros(NUM_CLASSES, dtype=np.int64)
        squares = np.zeros(3, dtype=np.float64)
        sign_pixels = np.zeros((3, NUM_CLASSES), dtype=np.int64)
        n_pixels = 0
        for images, labels in deterministic_train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            logits, features = teacher(images)
            cam = raw_target_cam(features, teacher.fc, labels)
            torch.testing.assert_close(cam.mean((-2,-1)) + teacher.fc.bias[labels],
                logits.gather(1, labels[:,None]).squeeze(1), rtol=1e-4, atol=1e-5)
            components = (cam.abs(), cam.clamp_min(0), (-cam).clamp_min(0))
            y = labels.cpu().numpy()
            np.add.at(numbers, y, 1)
            for j, values in enumerate(components):
                np.add.at(totals[j], y, values.mean((-2,-1)).cpu().numpy())
                squares[j] += float(values.square().sum().item())
            for j, mask in enumerate((cam.gt(0),cam.lt(0),cam.eq(0))):
                np.add.at(sign_pixels[j],y,mask.sum((-2,-1)).cpu().numpy())
            n_pixels += cam.numel()
        if (numbers == 0).any():
            raise RuntimeError("At least one class has no training samples.")
        class_means = totals / numbers[None, :]
        group_np = class_to_group.cpu().numpy()
        group_means = np.stack([class_means[:, group_np == g].mean(1) for g in range(3)], 1)
        group_counts = np.array([np.asarray(class_counts)[group_np == g].sum()
                                 for g in range(3)], dtype=np.float64)
        proportions = group_counts / group_counts.sum()
        group_sign_counts=np.stack([sign_pixels[:,group_np==g].sum(1) for g in range(3)],1)
        signed_factors=compute_signed_amount_factors(group_means[1:3],group_sign_counts,
            power=AMOUNT_POWER,min_value=AMOUNT_FACTOR_MIN,max_value=AMOUNT_FACTOR_MAX)
        stats = {
            "class_abs_mean": class_means[0], "class_pos_mean": class_means[1],
            "class_neg_mean": class_means[2], "group_abs_mean": group_means[0],
            "group_pos_mean": group_means[1], "group_neg_mean": group_means[2],
            "group_num_classes": np.array([(group_np == g).sum() for g in range(3)]),
            "group_sample_counts": group_counts, "group_sample_proportions": proportions,
            **signed_factors,
            "raw_cam_mean_sq": squares[0] / max(n_pixels, 1),
            "positive_cam_mean_sq": squares[1] / max(n_pixels, 1),
            "negative_cam_mean_sq": squares[2] / max(n_pixels, 1),
        }
        np.testing.assert_allclose(stats["amount_expected_weight"], 1., atol=1e-10)
        TEACHER_GROUP_STATS_CACHE[cache_key] = stats
        return stats


    def print_group_statistics(stats):
        table = pd.DataFrame([{
            "group": GROUP_ID_TO_NAME[g], "n_classes": int(stats["group_num_classes"][g]),
            "n_train_samples": int(stats["group_sample_counts"][g]),
            "sample_proportion": stats["group_sample_proportions"][g],
            "raw_abs_mass_mean": stats["group_abs_mean"][g],
            "raw_pos_mass_mean": stats["group_pos_mean"][g],
            "raw_neg_mass_mean": stats["group_neg_mean"][g],
            "amount_factor_raw": stats["raw_amount_factors"][g],
            "amount_factor_if_enabled": stats["amount_factors"][g],
            "E_q_global": stats["amount_expected_weight"],
            "amount_factor_positive_raw":stats["raw_amount_factors_positive"][g],
            "amount_factor_negative_raw":stats["raw_amount_factors_negative"][g],
            "amount_factor_positive":stats["amount_factors_positive"][g],
            "amount_factor_negative":stats["amount_factors_negative"][g],
            "amount_factor_zero":stats["amount_factors_zero"][g],
            "positive_pixel_fraction":stats["group_sign_pixel_fractions"][0,g],
            "negative_pixel_fraction":stats["group_sign_pixel_fractions"][1,g],
            "zero_pixel_fraction":stats["group_sign_pixel_fractions"][2,g],
            "q_times_raw_mass_diagnostic":stats["amount_factors_positive"][g]*stats["group_pos_mean"][g]
                +stats["amount_factors_negative"][g]*stats["group_neg_mean"][g],
        } for g in range(3)])
        display(table.round(6))
        return table


<a id="onde"></a>
## Onde — ponderação espacial e seleção de desfechos

As CAMs assinadas da classe verdadeira e de K−1 concorrentes, escolhidas pelos logits originais do professor, produzem o guia espacial. A normalização por classe é utilizada somente na construção das evidências.

- `raw`: pesos uniformes.
- `difference`: média das diferenças absolutas entre a CAM alvo normalizada e cada concorrente.
- `coincidence`: sobreposição de magnitudes em posições de mesmo sinal; sinais opostos não contam como coincidência.

`WHERE_BETA` (**Beta Onde**) mistura atenção uniforme e orientada pelas evidências. Os pesos finais têm média espacial 1. `aligned`, `uniform` e `shuffled` permitem controlar a organização espacial do guia; o embaralhamento utiliza um gerador local determinístico.

`WHERE_APPLY_TO` seleciona `all`, `correct` ou `error`, independentemente de `AMOUNT_APPLY_TO`. Imagens não elegíveis ou com modo `raw` mantêm pesos uniformes, enquanto Quanto conserva sua seleção própria. O desfecho é calculado na mesma view utilizada na perda. A compensação de Quanto modifica a referência depois da construção do guia espacial.


In [ ]:
assert_active_profile()

if EXPERIMENT_PROFILE == "resnet32_cifar100_lt":
    def raw_class_cams(features, fc_layer, class_ids):
        """Raw signed CAMs [B,K,H,W], selected by teacher class identities."""
        feature_map = extract_feature_tensor(features)
        weights = fc_layer.weight[class_ids.long()]
        return torch.einsum("bdhw,bkd->bkhw", feature_map, weights)


    def teacher_outcome_mask(correct, apply_to):
        """Select teacher outcomes without changing ordering, RNG or CAM values."""
        if apply_to not in {"all", "correct", "error"}:
            raise ValueError("Outcome selection must be all, correct or error.")
        if not torch.is_tensor(correct) or correct.ndim != 1:
            raise ValueError("Teacher correctness must be a one-dimensional tensor [B].")
        correct = correct.bool()
        if apply_to == "all":
            return torch.ones_like(correct, dtype=torch.bool)
        return correct if apply_to == "correct" else ~correct


    def normalize_signed_cams(cams, eps):
        """Original V3 mean-absolute normalization, used for attention ONLY.

        Raw target CAMs and QUANTO retain their original definitions. Both signs
        of a class share a divisor; no ReLU or independent student normalization
        enters the raw-CAM imitation error.
        """
        mass = cams.abs().mean(dim=(-2, -1), keepdim=True)
        normalized = torch.where(mass > eps, cams / mass.clamp_min(eps), torch.zeros_like(cams))
        return normalized, mass.squeeze(-1).squeeze(-1)


    @torch.no_grad()
    def signed_spatial_evidence(normalized_cams):
        """Teacher-only difference and signed-overlap evidence [B,H,W].

        Difference is MEAN OF ABSOLUTE pairwise differences. Taking an absolute
        value after averaging rivals would allow their disagreements to cancel.
        Coincidence is shared magnitude with the SAME strict sign. In particular,
        negative/negative overlap counts, opposite signs do not, and zero/zero
        contributes zero. No complement-of-difference similarity is used.
        """
        if normalized_cams.ndim != 4 or normalized_cams.shape[1] < 2:
            raise ValueError("Expected teacher CAMs [B,K,H,W] with K >= 2.")
        target, rivals = normalized_cams[:, :1], normalized_cams[:, 1:]
        difference = (target-rivals).abs().mean(dim=1)
        same_sign = (target*rivals).gt(0)
        overlap = torch.minimum(target.abs(), rivals.abs())
        coincidence = torch.where(same_sign, overlap, torch.zeros_like(overlap)).mean(dim=1)
        return {"difference": difference.detach(), "coincidence": coincidence.detach()}


    @torch.no_grad()
    def spatial_attention_from_evidence(evidence, beta, eps):
        """Sum-one attention, mean-one weights, and per-image zero fallback.

        With N spatial positions, weights=(1-beta)+beta*N*E/sum(E). Zero evidence
        falls back to exactly one at every position. beta=0 is also exactly raw.
        """
        if not 0 <= beta <= 1:
            raise ValueError("WHERE_BETA must be in [0,1].")
        if evidence.ndim != 3 or not torch.isfinite(evidence).all():
            raise ValueError("Evidence must be a finite tensor [B,H,W].")
        n = evidence.shape[-2]*evidence.shape[-1]
        nonnegative = evidence.clamp_min(0)
        total = nonnegative.sum((-2, -1), keepdim=True)
        fallback = total.squeeze(-1).squeeze(-1).le(eps)
        uniform = torch.full_like(nonnegative, 1./n)
        normalized = torch.where(total > eps, nonnegative/total.clamp_min(eps), uniform)
        attention = (1.-beta)*uniform + beta*normalized
        weights = attention*n
        # Exact identities matter for raw, zero-evidence and uniform controls.
        weights = torch.where(fallback[:,None,None], torch.ones_like(weights), weights)
        if beta == 0:
            weights = torch.ones_like(weights)
        return attention.detach(), weights.detach(), fallback.detach()


    @torch.no_grad()
    def shuffle_spatial_attention(first, second, sample_ids, seed, epoch):
        """One deterministic spatial permutation per image, shared by both maps.

        A local CPU generator preserves the global training RNG. The seed formula
        matches the original V3 spatial shuffle control.
        """
        if first.shape != second.shape or first.ndim != 3 or len(sample_ids) != first.shape[0]:
            raise ValueError("Matching [B,H,W] attention maps and B sample ids are required.")
        batch, height, width = first.shape
        permutations = []
        for image_id in sample_ids.detach().cpu().tolist():
            local_seed = (int(seed)*1000003 + int(epoch)*9176 + int(image_id)) % (2**63-1)
            generator = torch.Generator().manual_seed(local_seed)
            permutations.append(torch.randperm(height*width, generator=generator))
        perm = torch.stack(permutations).to(first.device)
        return tuple(value.flatten(1).gather(1,perm).reshape(batch,height,width)
                     for value in (first, second))


    @torch.no_grad()
    def build_spatial_cam_guidance(logits_t, features_t, teacher_fc, labels,
                                   class_to_group, use_where, sample_ids, seed, epoch):
        """Detached teacher spatial weights, chosen independently for each outcome.

        Teacher mistakes change WHICH weighting mode is selected; they do not
        rectify the raw target CAM or make a spatial explanation reliable. All
        ineligible images, including many by default, and outcomes outside
        WHERE_APPLY_TO retain raw signed imitation.
        """
        modes = {"raw", "difference", "coincidence"}
        masks = {"aligned", "uniform", "shuffled"}
        if WHERE_CORRECT_MODE not in modes or WHERE_ERROR_MODE not in modes:
            raise ValueError("Correct/error CAM modes must be raw, difference or coincidence.")
        if WHERE_CORRECT_MASK_MODE not in masks or WHERE_ERROR_MASK_MODE not in masks:
            raise ValueError("Correct/error mask modes must be aligned, uniform or shuffled.")
        _, n_classes = logits_t.shape
        if not 2 <= WHERE_TOP_K <= n_classes:
            raise ValueError(f"WHERE_TOP_K={WHERE_TOP_K}; valid range is [2,{n_classes}].")
        excluded_logits = logits_t.clone()
        excluded_logits.scatter_(1, labels[:,None], -torch.inf)
        rival_ids = excluded_logits.topk(WHERE_TOP_K-1, dim=1).indices
        selected_ids = torch.cat((labels[:,None],rival_ids),dim=1)
        teacher_cams = raw_class_cams(features_t, teacher_fc, selected_ids)
        # Keep the exact historical target-CAM arithmetic in slot zero.
        teacher_cams[:,0] = raw_target_cam(features_t, teacher_fc, labels)
        raw_t = teacher_cams[:,0]
        normalized, class_mass = normalize_signed_cams(teacher_cams, WHERE_NORM_EPS)
        evidence = signed_spatial_evidence(normalized)
        attention, weights, fallback = {}, {}, {}
        for mode in ("difference","coincidence"):
            attention[mode], weights[mode], fallback[mode] = spatial_attention_from_evidence(
                evidence[mode], WHERE_BETA, WHERE_NORM_EPS)
        shuffled_weights = {}
        if "shuffled" in {WHERE_CORRECT_MASK_MODE,WHERE_ERROR_MASK_MODE}:
            shuffled_difference, shuffled_coincidence = shuffle_spatial_attention(
                weights["difference"],weights["coincidence"],sample_ids,seed,epoch)
            shuffled_weights = {"difference":shuffled_difference,"coincidence":shuffled_coincidence}
        uniform_weights = torch.ones_like(raw_t)
        def outcome_weights(mode,mask):
            if mode == "raw" or mask == "uniform":
                return uniform_weights
            return shuffled_weights[mode] if mask == "shuffled" else weights[mode]
        correct = logits_t.argmax(1).eq(labels)
        group_ids = class_to_group[labels]
        eligible = torch.zeros_like(labels,dtype=torch.bool)
        for name in WHERE_GROUPS:
            eligible |= group_ids.eq(GROUP_NAME_TO_ID[name])
        where_selected_outcome = teacher_outcome_mask(correct, WHERE_APPLY_TO)
        enabled = eligible & where_selected_outcome & bool(use_where)
        difference_mask = enabled & ((correct & (WHERE_CORRECT_MODE == "difference"))
                                     | (~correct & (WHERE_ERROR_MODE == "difference")))
        coincidence_mask = enabled & ((correct & (WHERE_CORRECT_MODE == "coincidence"))
                                      | (~correct & (WHERE_ERROR_MODE == "coincidence")))
        applied = difference_mask | coincidence_mask
        raw_mask = ~applied
        selected_weights = torch.where(correct[:,None,None],
            outcome_weights(WHERE_CORRECT_MODE,WHERE_CORRECT_MASK_MODE),
            outcome_weights(WHERE_ERROR_MODE,WHERE_ERROR_MASK_MODE))
        spatial_weight = torch.where(applied[:,None,None], selected_weights, uniform_weights)
        applied_attention = spatial_weight/raw_t[0].numel()
        selected_fallback = ((difference_mask & fallback["difference"])
                             | (coincidence_mask & fallback["coincidence"]))
        n_pixels = raw_t[0].numel()
        if n_pixels > 1:
            attention_entropy = -(applied_attention*applied_attention.clamp_min(1e-30).log()).sum((-2,-1))/np.log(n_pixels)
        else:
            attention_entropy = torch.zeros_like(labels,dtype=raw_t.dtype)
        target_logit = logits_t.gather(1,labels[:,None]).squeeze(1)
        probability = logits_t.softmax(1)
        positive,negative = raw_t.clamp_min(0),(-raw_t).clamp_min(0)
        rival_groups = class_to_group[rival_ids]
        fields = {
            "teacher_correct":correct.float(),"teacher_error":(~correct).float(),
            "where_eligible":eligible.float(),"where_applied":applied.float(),
            "where_selected_outcome":where_selected_outcome.float(),
            "raw_branch":raw_mask.float(),"difference_branch":difference_mask.float(),
            "coincidence_branch":coincidence_mask.float(),
            "teacher_target_probability":probability.gather(1,labels[:,None]).squeeze(1),
            "teacher_target_rank":(logits_t>target_logit[:,None]).sum(1).float()+1,
            "teacher_target_margin":target_logit-excluded_logits.max(1).values,
            "target_in_natural_topk":logits_t.topk(WHERE_TOP_K,1).indices.eq(labels[:,None]).any(1).float(),
            "raw_abs_mass":raw_t.abs().mean((-2,-1)),
            "raw_pos_mass":positive.mean((-2,-1)),"raw_neg_mass":negative.mean((-2,-1)),
            "attention_entropy":attention_entropy,
            "attention_effective_area_fraction":1./(n_pixels*applied_attention.square().sum((-2,-1))),
            "attention_mass_target_positive":(applied_attention*raw_t.gt(0)).sum((-2,-1)),
            "attention_mass_target_negative":(applied_attention*raw_t.lt(0)).sum((-2,-1)),
            "evidence_zero_fallback":selected_fallback.float(),
            "difference_zero_fallback":fallback["difference"].float(),
            "coincidence_zero_fallback":fallback["coincidence"].float(),
            "difference_evidence_mean":evidence["difference"].mean((-2,-1)),
            "coincidence_evidence_mean":evidence["coincidence"].mean((-2,-1)),
            "weight_min":spatial_weight.flatten(1).amin(1),
            "weight_max":spatial_weight.flatten(1).amax(1),
            "weight_mean":spatial_weight.mean((-2,-1)),
            "weight_max_abs_deviation":(spatial_weight-1).abs().flatten(1).amax(1),
            "rival_many_fraction":rival_groups.eq(GROUP_NAME_TO_ID["many"]).float().mean(1),
            "rival_medium_fraction":rival_groups.eq(GROUP_NAME_TO_ID["medium"]).float().mean(1),
            "rival_few_fraction":rival_groups.eq(GROUP_NAME_TO_ID["few"]).float().mean(1),
            "rival_mass_mean":class_mass[:,1:].mean(1),
        }
        maps = {
            "teacher_raw":raw_t,"teacher_selected_cams":teacher_cams,
            "normalized_teacher_cams":normalized,"selected_ids":selected_ids,
            "rival_mean":normalized[:,1:].mean(1),
            "difference_evidence":evidence["difference"],"coincidence_evidence":evidence["coincidence"],
            "difference_attention":attention["difference"],"coincidence_attention":attention["coincidence"],
            "applied_attention":applied_attention,"applied_spatial_weight":spatial_weight,
            "raw_mask":raw_mask,"difference_mask":difference_mask,"coincidence_mask":coincidence_mask,
            "where_selected_outcome_mask":where_selected_outcome,
        }
        return {key:value.detach() for key,value in fields.items()}, {key:value.detach() for key,value in maps.items()}


    def qw_cam_loss(experiment, logits_s, features_s, student_fc, teacher, labels,
                    reference, class_to_group, teacher_stats, sample_ids, seed=0, epoch=0):
        """Global CE plus signed target-CAM imitation with QUANTO inside the MSE.

        Raw/difference/coincidence keep their original teacher-only spatial weights.
        The teacher reference is r_applied * raw_t, with separate group factors for teacher signs.
        AMOUNT_APPLY_TO is unchanged; positive/negative factors share one joint TRAIN normalization. Unselected outcomes use exactly q=1. The same original global
        teacher scale is used by ALL four methods. No other objective is added.
        """
        cfg = EXPERIMENT_DEFINITIONS[experiment]
        cam_s = raw_target_cam(features_s,student_fc,labels)
        with torch.no_grad():
            logits_t,features_t = teacher(reference)
            diag,maps = build_spatial_cam_guidance(logits_t,features_t,teacher.fc,labels,
                class_to_group,cfg["use_where"],sample_ids,seed,epoch)
            # Preserve the exact raw_target_cam call used by QW0/QW1 historically.
            raw_t = raw_target_cam(features_t,teacher.fc,labels)
            maps["teacher_raw"] = raw_t
        if cam_s.shape != raw_t.shape:
            raise ValueError("Student and teacher target CAM shapes must match.")
        scale = max(float(teacher_stats["raw_cam_mean_sq"]),CAM_SCALE_EPS)
        squared_error = (cam_s-raw_t).square()
        raw_per_image = squared_error.mean((-2,-1))/scale
        spatial_per_image = (squared_error*maps["applied_spatial_weight"]).mean((-2,-1))/scale
        raw_mask,difference_mask,coincidence_mask = maps["raw_mask"],maps["difference_mask"],maps["coincidence_mask"]
        if not cfg["use_where"] or (WHERE_CORRECT_MODE == "raw" and WHERE_ERROR_MODE == "raw"):
            per_image = raw_per_image
        else:
            per_image = torch.where(raw_mask,raw_per_image,spatial_per_image)
        group_ids = class_to_group[labels]
        factors = torch.as_tensor(teacher_stats["amount_factors"],device=cam_s.device,dtype=cam_s.dtype)
        q_base = factors[group_ids]
        amount_selected_outcome = teacher_outcome_mask(diag["teacher_correct"].bool(), AMOUNT_APPLY_TO)
        amount_applied = amount_selected_outcome & bool(cfg["use_amount"])
        signed_bases={name:torch.as_tensor(teacher_stats["amount_factors_"+name],
            device=cam_s.device,dtype=cam_s.dtype)[group_ids] for name in ("positive","negative","zero")}
        signed_applied={name:torch.where(amount_applied,value,torch.ones_like(value))
            for name,value in signed_bases.items()}
        positive_fraction=raw_t.gt(0).to(cam_s.dtype).mean((-2,-1))
        negative_fraction=raw_t.lt(0).to(cam_s.dtype).mean((-2,-1))
        zero_fraction=raw_t.eq(0).to(cam_s.dtype).mean((-2,-1))
        amount_factor_map=torch.where(raw_t.gt(0),signed_applied["positive"][:,None,None],
            torch.where(raw_t.lt(0),signed_applied["negative"][:,None,None],signed_applied["zero"][:,None,None]))
        # q is a summary (mean spatial factor), never the actual scalar target multiplier.
        q=signed_applied["zero"]+(signed_applied["positive"]-signed_applied["zero"])*positive_fraction \
            +(signed_applied["negative"]-signed_applied["zero"])*negative_fraction
        rectified_target=amount_factor_map*raw_t
        maps["amount_factor_map"]=amount_factor_map.detach()
        maps["teacher_rectified"]=rectified_target.detach()
        rectified_squared_error = (cam_s-rectified_target).square()
        raw_after_q_per_image = rectified_squared_error.mean((-2,-1))/scale
        spatial_after_q_per_image = (rectified_squared_error*maps["applied_spatial_weight"]).mean((-2,-1))/scale
        if not cfg["use_where"] or (WHERE_CORRECT_MODE == "raw" and WHERE_ERROR_MODE == "raw"):
            weighted_per_image = raw_after_q_per_image
        else:
            weighted_per_image = torch.where(raw_mask,raw_after_q_per_image,spatial_after_q_per_image)
        loss_ce = F.cross_entropy(logits_s,labels)
        ce_per_image = F.cross_entropy(logits_s,labels,reduction="none")
        loss_guidance = weighted_per_image.mean()
        total_loss = CE_WEIGHT*loss_ce + CAM_ALPHA*loss_guidance
        correct_mask = diag["teacher_correct"].bool()
        error_mask = ~correct_mask
        per_components = {
            "correct":torch.where(correct_mask,weighted_per_image,torch.zeros_like(weighted_per_image)),
            "error":torch.where(error_mask,weighted_per_image,torch.zeros_like(weighted_per_image)),
            "raw":torch.where(raw_mask,weighted_per_image,torch.zeros_like(weighted_per_image)),
            "difference":torch.where(difference_mask,weighted_per_image,torch.zeros_like(weighted_per_image)),
            "coincidence":torch.where(coincidence_mask,weighted_per_image,torch.zeros_like(weighted_per_image)),
        }
        with torch.no_grad():
            global_prob = logits_s.softmax(1)
            excluded_student = logits_s.clone()
            excluded_student.scatter_(1,labels[:,None],-torch.inf)
            weighted_error = rectified_squared_error*maps["applied_spatial_weight"]/scale
            diag.update({
                "amount_factor_base":q_base,
                "amount_selected_outcome":amount_selected_outcome.float(),
                "amount_applied":amount_applied.float(),
                "amount_weight":q,"ce_loss":ce_per_image,
                "cam_scale_reference":torch.full_like(per_image, scale),
                "cam_scale_applied":torch.full_like(per_image, scale),
                "cam_global_coefficient":torch.full_like(per_image, CAM_ALPHA/scale),
                "cam_effective_coefficient":torch.full_like(q, CAM_ALPHA/scale),
                "guidance_before_q":per_image,"guidance_after_q":weighted_per_image,
                "guidance_alpha_weighted":CAM_ALPHA*weighted_per_image,
                "raw_loss":raw_per_image,
                "target_abs_mass_before_q":diag["raw_abs_mass"],
                "target_abs_mass_after_q":rectified_target.abs().mean((-2,-1)),
                "target_pos_mass_after_q":rectified_target.clamp_min(0).mean((-2,-1)),
                "target_neg_mass_after_q":(-rectified_target).clamp_min(0).mean((-2,-1)),
                "spatial_weighted_raw_abs_mass":(maps["applied_spatial_weight"]*raw_t.abs()).mean((-2,-1)),
                "q_spatial_weighted_raw_abs_mass":(maps["applied_spatial_weight"]*rectified_target.abs()).mean((-2,-1)),
                "student_abs_mass":cam_s.abs().mean((-2,-1)),
                "student_pos_mass":cam_s.clamp_min(0).mean((-2,-1)),
                "student_neg_mass":(-cam_s).clamp_min(0).mean((-2,-1)),
                "student_teacher_sign_agreement":cam_s.sign().eq(raw_t.sign()).float().mean((-2,-1)),
                "positive_region_loss":(weighted_error*raw_t.gt(0)).mean((-2,-1)),
                "negative_region_loss":(weighted_error*raw_t.lt(0)).mean((-2,-1)),
                "zero_region_loss":(weighted_error*raw_t.eq(0)).mean((-2,-1)),
                "student_target_probability":global_prob.gather(1,labels[:,None]).squeeze(1),
                "student_target_margin":logits_s.gather(1,labels[:,None]).squeeze(1)-excluded_student.max(1).values,
                "student_correct":logits_s.argmax(1).eq(labels).float(),
            })
            for name in ("positive","negative","zero"):
                diag["amount_factor_"+name+"_base"]=signed_bases[name]
                diag["amount_factor_"+name+"_applied"]=signed_applied[name]
            diag.update(amount_positive_fraction=positive_fraction,amount_negative_fraction=negative_fraction,
                        amount_zero_fraction=zero_fraction)
            for name,value in per_components.items():
                diag[name+"_loss"] = value
        maps["student_cam"] = cam_s.detach()
        losses = {
            "ce":loss_ce,"ce_per_image":ce_per_image,
            "guidance_before_q":per_image.mean(),"guidance":loss_guidance,
            "guidance_weighted":CAM_ALPHA*loss_guidance,
            "raw_baseline":raw_after_q_per_image.mean(),
            "guidance_per_image":per_image,"raw_per_image":raw_per_image,"amount_per_image":q,
            "guidance_after_q_per_image":weighted_per_image,"raw_after_q_per_image":raw_after_q_per_image,
            "amount_base_per_image":q_base,
            "amount_selected_outcome_mask":amount_selected_outcome,
            "amount_applied_mask":amount_applied,
            "where_selected_outcome_mask":maps["where_selected_outcome_mask"],
            "cam_scale_reference":cam_s.new_tensor(scale),
            "cam_scale_applied":cam_s.new_tensor(scale),
            "cam_global_coefficient":cam_s.new_tensor(CAM_ALPHA/scale),
            "cam_effective_coefficient_per_image":torch.full_like(q, CAM_ALPHA/scale),
            "teacher_correct_mask":correct_mask,"teacher_error_mask":error_mask,
            "raw_mask":raw_mask,"difference_mask":difference_mask,"coincidence_mask":coincidence_mask,
        }
        for name,value in per_components.items():
            losses[name+"_component"] = value.mean()
            losses[name+"_component_per_image"] = value
        return total_loss,{key:value.detach() for key,value in diag.items()},maps,losses


    qw_cam_loss.qw_method_version = "QW_V8_signed_spatial_cam_sign_scaled_target_outcome_controls"
else:  # mobilenetv2_imagenet_lt
    def raw_class_cams(features, fc_layer, class_ids):
        """Raw signed CAMs [B,K,H,W], selected by teacher class identities."""
        feature_map = extract_feature_tensor(features)
        weights = fc_layer.weight[class_ids.long()]
        return torch.einsum("bdhw,bkd->bkhw", feature_map, weights)


    def teacher_outcome_mask(correct, apply_to):
        """Select teacher outcomes without changing ordering, RNG or CAM values."""
        if apply_to not in {"all", "correct", "error"}:
            raise ValueError("Outcome selection must be all, correct or error.")
        if not torch.is_tensor(correct) or correct.ndim != 1:
            raise ValueError("Teacher correctness must be a one-dimensional tensor [B].")
        correct = correct.bool()
        if apply_to == "all":
            return torch.ones_like(correct, dtype=torch.bool)
        return correct if apply_to == "correct" else ~correct


    def normalize_signed_cams(cams, eps):
        """Original V3 mean-absolute normalization, used for attention ONLY.

        Raw target CAMs and QUANTO retain their original definitions. Both signs
        of a class share a divisor; no ReLU or independent student normalization
        enters the raw-CAM imitation error.
        """
        mass = cams.abs().mean(dim=(-2, -1), keepdim=True)
        normalized = torch.where(mass > eps, cams / mass.clamp_min(eps), torch.zeros_like(cams))
        return normalized, mass.squeeze(-1).squeeze(-1)


    @torch.no_grad()
    def signed_spatial_evidence(normalized_cams):
        """Teacher-only difference and signed-overlap evidence [B,H,W].

        Difference is MEAN OF ABSOLUTE pairwise differences. Taking an absolute
        value after averaging rivals would allow their disagreements to cancel.
        Coincidence is shared magnitude with the SAME strict sign. In particular,
        negative/negative overlap counts, opposite signs do not, and zero/zero
        contributes zero. No complement-of-difference similarity is used.
        """
        if normalized_cams.ndim != 4 or normalized_cams.shape[1] < 2:
            raise ValueError("Expected teacher CAMs [B,K,H,W] with K >= 2.")
        target, rivals = normalized_cams[:, :1], normalized_cams[:, 1:]
        difference = (target-rivals).abs().mean(dim=1)
        same_sign = (target*rivals).gt(0)
        overlap = torch.minimum(target.abs(), rivals.abs())
        coincidence = torch.where(same_sign, overlap, torch.zeros_like(overlap)).mean(dim=1)
        return {"difference": difference.detach(), "coincidence": coincidence.detach()}


    @torch.no_grad()
    def spatial_attention_from_evidence(evidence, beta, eps):
        """Sum-one attention, mean-one weights, and per-image zero fallback.

        With N spatial positions, weights=(1-beta)+beta*N*E/sum(E). Zero evidence
        falls back to exactly one at every position. beta=0 is also exactly raw.
        """
        if not 0 <= beta <= 1:
            raise ValueError("WHERE_BETA must be in [0,1].")
        if evidence.ndim != 3 or not torch.isfinite(evidence).all():
            raise ValueError("Evidence must be a finite tensor [B,H,W].")
        n = evidence.shape[-2]*evidence.shape[-1]
        nonnegative = evidence.clamp_min(0)
        total = nonnegative.sum((-2, -1), keepdim=True)
        fallback = total.squeeze(-1).squeeze(-1).le(eps)
        uniform = torch.full_like(nonnegative, 1./n)
        normalized = torch.where(total > eps, nonnegative/total.clamp_min(eps), uniform)
        attention = (1.-beta)*uniform + beta*normalized
        weights = attention*n
        # Exact identities matter for raw, zero-evidence and uniform controls.
        weights = torch.where(fallback[:,None,None], torch.ones_like(weights), weights)
        if beta == 0:
            weights = torch.ones_like(weights)
        return attention.detach(), weights.detach(), fallback.detach()


    @torch.no_grad()
    def shuffle_spatial_attention(first, second, sample_ids, seed, epoch):
        """One deterministic spatial permutation per image, shared by both maps.

        A local CPU generator preserves the global training RNG. The seed formula
        matches the original V3 spatial shuffle control.
        """
        if first.shape != second.shape or first.ndim != 3 or len(sample_ids) != first.shape[0]:
            raise ValueError("Matching [B,H,W] attention maps and B sample ids are required.")
        batch, height, width = first.shape
        permutations = []
        for image_id in sample_ids.detach().cpu().tolist():
            local_seed = (int(seed)*1000003 + int(epoch)*9176 + int(image_id)) % (2**63-1)
            generator = torch.Generator().manual_seed(local_seed)
            permutations.append(torch.randperm(height*width, generator=generator))
        perm = torch.stack(permutations).to(first.device)
        return tuple(value.flatten(1).gather(1,perm).reshape(batch,height,width)
                     for value in (first, second))


    @torch.no_grad()
    def build_spatial_cam_guidance(logits_t, features_t, teacher_fc, labels,
                                   class_to_group, use_where, sample_ids, seed, epoch):
        """Detached teacher spatial weights, chosen independently for each outcome.

        Teacher mistakes change WHICH weighting mode is selected; they do not
        rectify the raw target CAM or make a spatial explanation reliable. All
        ineligible images, including many by default, and outcomes outside
        WHERE_APPLY_TO retain raw signed imitation.
        """
        modes = {"raw", "difference", "coincidence"}
        masks = {"aligned", "uniform", "shuffled"}
        if WHERE_CORRECT_MODE not in modes or WHERE_ERROR_MODE not in modes:
            raise ValueError("Correct/error CAM modes must be raw, difference or coincidence.")
        if WHERE_CORRECT_MASK_MODE not in masks or WHERE_ERROR_MASK_MODE not in masks:
            raise ValueError("Correct/error mask modes must be aligned, uniform or shuffled.")
        _, n_classes = logits_t.shape
        if not 2 <= WHERE_TOP_K <= n_classes:
            raise ValueError(f"WHERE_TOP_K={WHERE_TOP_K}; valid range is [2,{n_classes}].")
        excluded_logits = logits_t.clone()
        excluded_logits.scatter_(1, labels[:,None], -torch.inf)
        rival_ids = excluded_logits.topk(WHERE_TOP_K-1, dim=1).indices
        selected_ids = torch.cat((labels[:,None],rival_ids),dim=1)
        teacher_cams = raw_class_cams(features_t, teacher_fc, selected_ids)
        # Keep the exact historical target-CAM arithmetic in slot zero.
        teacher_cams[:,0] = raw_target_cam(features_t, teacher_fc, labels)
        raw_t = teacher_cams[:,0]
        normalized, class_mass = normalize_signed_cams(teacher_cams, WHERE_NORM_EPS)
        evidence = signed_spatial_evidence(normalized)
        attention, weights, fallback = {}, {}, {}
        for mode in ("difference","coincidence"):
            attention[mode], weights[mode], fallback[mode] = spatial_attention_from_evidence(
                evidence[mode], WHERE_BETA, WHERE_NORM_EPS)
        shuffled_weights = {}
        if "shuffled" in {WHERE_CORRECT_MASK_MODE,WHERE_ERROR_MASK_MODE}:
            shuffled_difference, shuffled_coincidence = shuffle_spatial_attention(
                weights["difference"],weights["coincidence"],sample_ids,seed,epoch)
            shuffled_weights = {"difference":shuffled_difference,"coincidence":shuffled_coincidence}
        uniform_weights = torch.ones_like(raw_t)
        def outcome_weights(mode,mask):
            if mode == "raw" or mask == "uniform":
                return uniform_weights
            return shuffled_weights[mode] if mask == "shuffled" else weights[mode]
        correct = logits_t.argmax(1).eq(labels)
        group_ids = class_to_group[labels]
        eligible = torch.zeros_like(labels,dtype=torch.bool)
        for name in WHERE_GROUPS:
            eligible |= group_ids.eq(GROUP_NAME_TO_ID[name])
        where_selected_outcome = teacher_outcome_mask(correct, WHERE_APPLY_TO)
        enabled = eligible & where_selected_outcome & bool(use_where)
        difference_mask = enabled & ((correct & (WHERE_CORRECT_MODE == "difference"))
                                     | (~correct & (WHERE_ERROR_MODE == "difference")))
        coincidence_mask = enabled & ((correct & (WHERE_CORRECT_MODE == "coincidence"))
                                      | (~correct & (WHERE_ERROR_MODE == "coincidence")))
        applied = difference_mask | coincidence_mask
        raw_mask = ~applied
        selected_weights = torch.where(correct[:,None,None],
            outcome_weights(WHERE_CORRECT_MODE,WHERE_CORRECT_MASK_MODE),
            outcome_weights(WHERE_ERROR_MODE,WHERE_ERROR_MASK_MODE))
        spatial_weight = torch.where(applied[:,None,None], selected_weights, uniform_weights)
        applied_attention = spatial_weight/raw_t[0].numel()
        selected_fallback = ((difference_mask & fallback["difference"])
                             | (coincidence_mask & fallback["coincidence"]))
        n_pixels = raw_t[0].numel()
        if n_pixels > 1:
            attention_entropy = -(applied_attention*applied_attention.clamp_min(1e-30).log()).sum((-2,-1))/np.log(n_pixels)
        else:
            attention_entropy = torch.zeros_like(labels,dtype=raw_t.dtype)
        target_logit = logits_t.gather(1,labels[:,None]).squeeze(1)
        probability = logits_t.softmax(1)
        positive,negative = raw_t.clamp_min(0),(-raw_t).clamp_min(0)
        rival_groups = class_to_group[rival_ids]
        fields = {
            "teacher_correct":correct.float(),"teacher_error":(~correct).float(),
            "where_eligible":eligible.float(),"where_applied":applied.float(),
            "where_selected_outcome":where_selected_outcome.float(),
            "raw_branch":raw_mask.float(),"difference_branch":difference_mask.float(),
            "coincidence_branch":coincidence_mask.float(),
            "teacher_target_probability":probability.gather(1,labels[:,None]).squeeze(1),
            "teacher_target_rank":(logits_t>target_logit[:,None]).sum(1).float()+1,
            "teacher_target_margin":target_logit-excluded_logits.max(1).values,
            "target_in_natural_topk":logits_t.topk(WHERE_TOP_K,1).indices.eq(labels[:,None]).any(1).float(),
            "raw_abs_mass":raw_t.abs().mean((-2,-1)),
            "raw_pos_mass":positive.mean((-2,-1)),"raw_neg_mass":negative.mean((-2,-1)),
            "attention_entropy":attention_entropy,
            "attention_effective_area_fraction":1./(n_pixels*applied_attention.square().sum((-2,-1))),
            "attention_mass_target_positive":(applied_attention*raw_t.gt(0)).sum((-2,-1)),
            "attention_mass_target_negative":(applied_attention*raw_t.lt(0)).sum((-2,-1)),
            "evidence_zero_fallback":selected_fallback.float(),
            "difference_zero_fallback":fallback["difference"].float(),
            "coincidence_zero_fallback":fallback["coincidence"].float(),
            "difference_evidence_mean":evidence["difference"].mean((-2,-1)),
            "coincidence_evidence_mean":evidence["coincidence"].mean((-2,-1)),
            "weight_min":spatial_weight.flatten(1).amin(1),
            "weight_max":spatial_weight.flatten(1).amax(1),
            "weight_mean":spatial_weight.mean((-2,-1)),
            "weight_max_abs_deviation":(spatial_weight-1).abs().flatten(1).amax(1),
            "rival_many_fraction":rival_groups.eq(GROUP_NAME_TO_ID["many"]).float().mean(1),
            "rival_medium_fraction":rival_groups.eq(GROUP_NAME_TO_ID["medium"]).float().mean(1),
            "rival_few_fraction":rival_groups.eq(GROUP_NAME_TO_ID["few"]).float().mean(1),
            "rival_mass_mean":class_mass[:,1:].mean(1),
        }
        maps = {
            "teacher_raw":raw_t,"teacher_selected_cams":teacher_cams,
            "normalized_teacher_cams":normalized,"selected_ids":selected_ids,
            "rival_mean":normalized[:,1:].mean(1),
            "difference_evidence":evidence["difference"],"coincidence_evidence":evidence["coincidence"],
            "difference_attention":attention["difference"],"coincidence_attention":attention["coincidence"],
            "applied_attention":applied_attention,"applied_spatial_weight":spatial_weight,
            "raw_mask":raw_mask,"difference_mask":difference_mask,"coincidence_mask":coincidence_mask,
            "where_selected_outcome_mask":where_selected_outcome,
        }
        return {key:value.detach() for key,value in fields.items()}, {key:value.detach() for key,value in maps.items()}


    def qw_cam_loss(experiment, logits_s, features_s, student_fc, teacher, labels,
                    reference, class_to_group, teacher_stats, sample_ids, seed=0, epoch=0):
        """Global CE plus signed target-CAM imitation with QUANTO inside the MSE.

        Raw/difference/coincidence keep their original teacher-only spatial weights.
        The teacher reference is r_applied * raw_t, with separate group factors for teacher signs.
        AMOUNT_APPLY_TO is unchanged; positive/negative factors share one joint TRAIN normalization. Unselected outcomes use exactly q=1. The same original global
        teacher scale is used by ALL four methods. No other objective is added.
        """
        cfg = EXPERIMENT_DEFINITIONS[experiment]
        cam_s = raw_target_cam(features_s,student_fc,labels)
        with torch.no_grad():
            logits_t,features_t = teacher(reference)
            diag,maps = build_spatial_cam_guidance(logits_t,features_t,teacher.fc,labels,
                class_to_group,cfg["use_where"],sample_ids,seed,epoch)
            # Preserve the exact raw_target_cam call used by QW0/QW1 historically.
            raw_t = raw_target_cam(features_t,teacher.fc,labels)
            maps["teacher_raw"] = raw_t
        if cam_s.shape != raw_t.shape:
            raise ValueError("Student and teacher target CAM shapes must match.")
        scale = max(float(teacher_stats["raw_cam_mean_sq"]),CAM_SCALE_EPS)
        squared_error = (cam_s-raw_t).square()
        raw_per_image = squared_error.mean((-2,-1))/scale
        spatial_per_image = (squared_error*maps["applied_spatial_weight"]).mean((-2,-1))/scale
        raw_mask,difference_mask,coincidence_mask = maps["raw_mask"],maps["difference_mask"],maps["coincidence_mask"]
        if not cfg["use_where"] or (WHERE_CORRECT_MODE == "raw" and WHERE_ERROR_MODE == "raw"):
            per_image = raw_per_image
        else:
            per_image = torch.where(raw_mask,raw_per_image,spatial_per_image)
        group_ids = class_to_group[labels]
        factors = torch.as_tensor(teacher_stats["amount_factors"],device=cam_s.device,dtype=cam_s.dtype)
        q_base = factors[group_ids]
        amount_selected_outcome = teacher_outcome_mask(diag["teacher_correct"].bool(), AMOUNT_APPLY_TO)
        amount_applied = amount_selected_outcome & bool(cfg["use_amount"])
        signed_bases={name:torch.as_tensor(teacher_stats["amount_factors_"+name],
            device=cam_s.device,dtype=cam_s.dtype)[group_ids] for name in ("positive","negative","zero")}
        signed_applied={name:torch.where(amount_applied,value,torch.ones_like(value))
            for name,value in signed_bases.items()}
        positive_fraction=raw_t.gt(0).to(cam_s.dtype).mean((-2,-1))
        negative_fraction=raw_t.lt(0).to(cam_s.dtype).mean((-2,-1))
        zero_fraction=raw_t.eq(0).to(cam_s.dtype).mean((-2,-1))
        amount_factor_map=torch.where(raw_t.gt(0),signed_applied["positive"][:,None,None],
            torch.where(raw_t.lt(0),signed_applied["negative"][:,None,None],signed_applied["zero"][:,None,None]))
        # q is a summary (mean spatial factor), never the actual scalar target multiplier.
        q=signed_applied["zero"]+(signed_applied["positive"]-signed_applied["zero"])*positive_fraction \
            +(signed_applied["negative"]-signed_applied["zero"])*negative_fraction
        rectified_target=amount_factor_map*raw_t
        maps["amount_factor_map"]=amount_factor_map.detach()
        maps["teacher_rectified"]=rectified_target.detach()
        rectified_squared_error = (cam_s-rectified_target).square()
        raw_after_q_per_image = rectified_squared_error.mean((-2,-1))/scale
        spatial_after_q_per_image = (rectified_squared_error*maps["applied_spatial_weight"]).mean((-2,-1))/scale
        if not cfg["use_where"] or (WHERE_CORRECT_MODE == "raw" and WHERE_ERROR_MODE == "raw"):
            weighted_per_image = raw_after_q_per_image
        else:
            weighted_per_image = torch.where(raw_mask,raw_after_q_per_image,spatial_after_q_per_image)
        loss_ce = F.cross_entropy(logits_s,labels)
        ce_per_image = F.cross_entropy(logits_s,labels,reduction="none")
        loss_guidance = weighted_per_image.mean()
        total_loss = CE_WEIGHT*loss_ce + CAM_ALPHA*loss_guidance
        correct_mask = diag["teacher_correct"].bool()
        error_mask = ~correct_mask
        per_components = {
            "correct":torch.where(correct_mask,weighted_per_image,torch.zeros_like(weighted_per_image)),
            "error":torch.where(error_mask,weighted_per_image,torch.zeros_like(weighted_per_image)),
            "raw":torch.where(raw_mask,weighted_per_image,torch.zeros_like(weighted_per_image)),
            "difference":torch.where(difference_mask,weighted_per_image,torch.zeros_like(weighted_per_image)),
            "coincidence":torch.where(coincidence_mask,weighted_per_image,torch.zeros_like(weighted_per_image)),
        }
        with torch.no_grad():
            global_prob = logits_s.softmax(1)
            excluded_student = logits_s.clone()
            excluded_student.scatter_(1,labels[:,None],-torch.inf)
            weighted_error = rectified_squared_error*maps["applied_spatial_weight"]/scale
            diag.update({
                "amount_factor_base":q_base,
                "amount_selected_outcome":amount_selected_outcome.float(),
                "amount_applied":amount_applied.float(),
                "amount_weight":q,"ce_loss":ce_per_image,
                "cam_scale_reference":torch.full_like(per_image, scale),
                "cam_scale_applied":torch.full_like(per_image, scale),
                "cam_global_coefficient":torch.full_like(per_image, CAM_ALPHA/scale),
                "cam_effective_coefficient":torch.full_like(q, CAM_ALPHA/scale),
                "guidance_before_q":per_image,"guidance_after_q":weighted_per_image,
                "guidance_alpha_weighted":CAM_ALPHA*weighted_per_image,
                "raw_loss":raw_per_image,
                "target_abs_mass_before_q":diag["raw_abs_mass"],
                "target_abs_mass_after_q":rectified_target.abs().mean((-2,-1)),
                "target_pos_mass_after_q":rectified_target.clamp_min(0).mean((-2,-1)),
                "target_neg_mass_after_q":(-rectified_target).clamp_min(0).mean((-2,-1)),
                "spatial_weighted_raw_abs_mass":(maps["applied_spatial_weight"]*raw_t.abs()).mean((-2,-1)),
                "q_spatial_weighted_raw_abs_mass":(maps["applied_spatial_weight"]*rectified_target.abs()).mean((-2,-1)),
                "student_abs_mass":cam_s.abs().mean((-2,-1)),
                "student_pos_mass":cam_s.clamp_min(0).mean((-2,-1)),
                "student_neg_mass":(-cam_s).clamp_min(0).mean((-2,-1)),
                "student_teacher_sign_agreement":cam_s.sign().eq(raw_t.sign()).float().mean((-2,-1)),
                "positive_region_loss":(weighted_error*raw_t.gt(0)).mean((-2,-1)),
                "negative_region_loss":(weighted_error*raw_t.lt(0)).mean((-2,-1)),
                "zero_region_loss":(weighted_error*raw_t.eq(0)).mean((-2,-1)),
                "student_target_probability":global_prob.gather(1,labels[:,None]).squeeze(1),
                "student_target_margin":logits_s.gather(1,labels[:,None]).squeeze(1)-excluded_student.max(1).values,
                "student_correct":logits_s.argmax(1).eq(labels).float(),
            })
            for name in ("positive","negative","zero"):
                diag["amount_factor_"+name+"_base"]=signed_bases[name]
                diag["amount_factor_"+name+"_applied"]=signed_applied[name]
            diag.update(amount_positive_fraction=positive_fraction,amount_negative_fraction=negative_fraction,
                        amount_zero_fraction=zero_fraction)
            for name,value in per_components.items():
                diag[name+"_loss"] = value
        maps["student_cam"] = cam_s.detach()
        losses = {
            "ce":loss_ce,"ce_per_image":ce_per_image,
            "guidance_before_q":per_image.mean(),"guidance":loss_guidance,
            "guidance_weighted":CAM_ALPHA*loss_guidance,
            "raw_baseline":raw_after_q_per_image.mean(),
            "guidance_per_image":per_image,"raw_per_image":raw_per_image,"amount_per_image":q,
            "guidance_after_q_per_image":weighted_per_image,"raw_after_q_per_image":raw_after_q_per_image,
            "amount_base_per_image":q_base,
            "amount_selected_outcome_mask":amount_selected_outcome,
            "amount_applied_mask":amount_applied,
            "where_selected_outcome_mask":maps["where_selected_outcome_mask"],
            "cam_scale_reference":cam_s.new_tensor(scale),
            "cam_scale_applied":cam_s.new_tensor(scale),
            "cam_global_coefficient":cam_s.new_tensor(CAM_ALPHA/scale),
            "cam_effective_coefficient_per_image":torch.full_like(q, CAM_ALPHA/scale),
            "teacher_correct_mask":correct_mask,"teacher_error_mask":error_mask,
            "raw_mask":raw_mask,"difference_mask":difference_mask,"coincidence_mask":coincidence_mask,
        }
        for name,value in per_components.items():
            losses[name+"_component"] = value.mean()
            losses[name+"_component_per_image"] = value
        return total_loss,{key:value.detach() for key,value in diag.items()},maps,losses


    qw_cam_loss.qw_method_version = "QW_V8_mobilenet_signed_spatial_cam_sign_scaled_target_outcome_controls"


<a id="diagnosticos"></a>
## Diagnósticos e saídas por perfil

As perdas, a calibração por sinal e os mecanismos de Quanto/Onde são preservados. O formato de observação/exportação acompanha a implementação de origem:

- **ResNet32:** diagnósticos completos, predições em CSV/NPZ, probes e figuras conforme as flags; consolidação CSV/XLSX original.
- **MobileNetV2:** observador essencial por imagem, sem diagnósticos extras de gradientes ou probes finais; validação por época sem probabilidades não utilizadas; avaliação final completa e consolidação compacta. As flags históricas de probes/gradientes permanecem na configuração para compatibilidade, mas não são executadas nesse perfil.

Ambos preservam fatores calibrados e aplicados, massas, balanço de perdas e diagnósticos de classificação. A calibração estática de Quanto continua balanceada por classe; as médias dinâmicas do observador essencial MobileNet são por imagem. Entropia e área efetiva descrevem a atenção aplicada, não a qualidade semântica da localização.

A razão entre perda de destilação e classificação considera os termos ponderados; não é uma razão de gradientes. Os erros antes/após Quanto comparam referências no mesmo estado do estudante em cada batch, não o começo e o fim do treinamento.


In [ ]:
assert_active_profile()

if EXPERIMENT_PROFILE == "resnet32_cifar100_lt":
    def hash_state_dict(model):
        digest = hashlib.sha256()
        for name, tensor in sorted(model.state_dict().items()):
            digest.update(name.encode())
            digest.update(tensor.detach().cpu().contiguous().numpy().tobytes())
        return digest.hexdigest()


    def file_sha256(path):
        digest = hashlib.sha256()
        with open(path, "rb") as file:
            for chunk in iter(lambda: file.read(1024 * 1024), b""):
                digest.update(chunk)
        return digest.hexdigest()


    def metrics_from_arrays(labels, preds, probs, class_counts):
        # Same definitions as the historical evaluate_model / compute_group_metrics.
        metrics = {
            "acc": float((preds == labels).mean() * 100),
            "top5_acc": float(top_k_accuracy_score(labels, probs, k=5,
                                  labels=np.arange(NUM_CLASSES)) * 100),
            "f1_macro": float(f1_score(labels, preds, average="macro", zero_division=0)),
            "f1_weighted": float(f1_score(labels, preds, average="weighted", zero_division=0)),
            "balanced_acc": float(balanced_accuracy_score(labels, preds)),
        }
        metrics.update(compute_group_metrics(labels, preds, get_shot_groups(class_counts)))
        return metrics


    @torch.no_grad()
    def evaluate_with_arrays(model, loader, class_counts):
        model.eval()
        ys, ps, probabilities = [], [], []
        ce_sum, n = 0., 0
        for batch in loader:
            images, labels = batch[:2]
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            logits, _ = model(images)
            ce_sum += F.cross_entropy(logits, labels, reduction="sum").item()
            n += labels.numel()
            ys.append(labels.cpu().numpy())
            ps.append(logits.argmax(1).cpu().numpy())
            probabilities.append(logits.softmax(1).cpu().numpy())
        y, pred, prob = np.concatenate(ys), np.concatenate(ps), np.concatenate(probabilities)
        metrics = metrics_from_arrays(y, pred, prob, class_counts)
        metrics["ce_loss"] = ce_sum / n
        return metrics, y, pred, prob


    class ClassDiagnosticAccumulator:
        """Finite-aware class/image means, including base/applied q and outcome masks.

        Per-image outcome selectors, application flags, scale and coefficients use
        the same generic aggregation path. Missing observations remain NaN.
        """
        def __init__(self):
            self.n = np.zeros(NUM_CLASSES, dtype=np.int64)
            self.sums, self.valid = {}, {}

        def update(self, labels, diagnostics):
            y = labels.detach().cpu().numpy()
            np.add.at(self.n, y, 1)
            names = [name for name, tensor in diagnostics.items()
                     if torch.is_tensor(tensor) and tensor.ndim == 1 and len(tensor) == len(y)]
            if not names:
                return
            matrix = torch.stack([diagnostics[name].detach().float() for name in names], dim=1).cpu().numpy()
            for col, name in enumerate(names):
                values = matrix[:, col].astype(np.float64)
                finite = np.isfinite(values)
                if name not in self.sums:
                    self.sums[name] = np.zeros(NUM_CLASSES, dtype=np.float64)
                    self.valid[name] = np.zeros(NUM_CLASSES, dtype=np.int64)
                np.add.at(self.sums[name], y[finite], values[finite])
                np.add.at(self.valid[name], y[finite], 1)

        def frames(self, class_to_group, epoch):
            group_ids = class_to_group.detach().cpu().numpy()
            # Build all diagnostic columns together to avoid repeated DataFrame
            # fragmentation when every epoch/outcome has hundreds of columns.
            class_columns = {"epoch": np.full(NUM_CLASSES, epoch), "class_id": np.arange(NUM_CLASSES),
                "class_name": _CLASS_NAMES or [str(c) for c in range(NUM_CLASSES)],
                "group": [GROUP_ID_TO_NAME[int(x)] for x in group_ids], "n_samples": self.n}
            for name in self.sums:
                count = self.valid[name]
                class_columns[name] = np.divide(self.sums[name], count,
                    out=np.full(NUM_CLASSES, np.nan), where=count > 0)
                class_columns[name + "_valid_n"] = count
            # Class-level sums/contributions use the same observed full-epoch
            # denominator as group rows. Means remain NaN for unobserved classes.
            for source, prefix in (("guidance_after_q", "guidance"),
                                   ("raw_abs_mass", "raw_abs_mass"),
                                   ("target_abs_mass_after_q", "q_times_raw_abs_mass"),
                                   ("spatial_weighted_raw_abs_mass", "spatial_weighted_raw_abs_mass"),
                                   ("q_spatial_weighted_raw_abs_mass", "q_spatial_weighted_raw_abs_mass")):
                if source in self.sums:
                    values = self.sums[source]
                    class_columns[prefix + "_sum"] = values
                    class_columns[prefix + "_epoch_contribution_to_mean"] = values / max(self.n.sum(), 1)
                    class_columns[prefix + "_epoch_contribution_fraction"] = (values / values.sum()
                        if values.sum() > 0 else np.full(NUM_CLASSES, np.nan))
            class_table = pd.DataFrame(class_columns)
            rows = []
            guidance = self.sums.get("guidance_after_q", np.zeros(NUM_CLASSES))
            total_guidance = guidance.sum()
            for group_id in range(3):
                mask = group_ids == group_id
                row = {"epoch": epoch, "group": GROUP_ID_TO_NAME[group_id],
                    "n_samples": int(self.n[mask].sum()), "n_classes": int(mask.sum()),
                    "n_classes_observed": int(np.count_nonzero(self.n[mask]))}
                for name in self.sums:
                    count = self.valid[name][mask].sum()
                    row["image_" + name] = self.sums[name][mask].sum() / count if count else np.nan
                    row["class_" + name] = class_table.loc[mask, name].mean()
                    row[name + "_valid_n"] = int(count)
                    row[name + "_classes_valid_n"] = int(class_table.loc[mask, name].notna().sum())
                row["epoch_sample_fraction"] = row["n_samples"] / max(self.n.sum(), 1)
                contribution = guidance[mask].sum()
                row["guidance_sum"] = float(contribution)
                row["guidance_epoch_contribution_fraction"] = contribution / total_guidance if total_guidance > 0 else np.nan
                row["guidance_epoch_contribution_to_mean"] = contribution / max(self.n.sum(), 1)
                # These mass statistics use q actually applied to each image. The
                # fixed q_base is logged separately; neither factor changes the CAM.
                # L1 mass statistics omit alpha / the shared global denominator.
                row["raw_abs_mass_image_mean"] = row.get("image_raw_abs_mass", np.nan)
                row["raw_abs_mass_class_mean"] = row.get("class_raw_abs_mass", np.nan)
                row["q_image_mean"] = row.get("image_amount_weight", np.nan)
                row["q_class_mean"] = row.get("class_amount_weight", np.nan)
                row["q_applied_image_mean"] = row["q_image_mean"]
                row["q_applied_class_mean"] = row["q_class_mean"]
                row["q_base_image_mean"] = row.get("image_amount_factor_base", np.nan)
                row["q_base_class_mean"] = row.get("class_amount_factor_base", np.nan)
                row["q_times_raw_abs_mass_image_mean"] = row.get("image_target_abs_mass_after_q", np.nan)
                row["q_times_raw_abs_mass_class_mean"] = row.get("class_target_abs_mass_after_q", np.nan)
                row["raw_signed_cam_image_mean"] = row.get("image_raw_pos_mass", np.nan) - row.get("image_raw_neg_mass", np.nan)
                row["raw_signed_cam_class_mean"] = row.get("class_raw_pos_mass", np.nan) - row.get("class_raw_neg_mass", np.nan)
                row["cam_mass_definition"] = "mean_spatial_abs_of_signed_target_CAM"
                row["cam_mass_sample_scope"] = "observed_epoch_views; static_QUANTO_uses_deterministic_training_class_means"
                for source, prefix in (("raw_abs_mass", "raw_abs_mass"),
                                       ("target_abs_mass_after_q", "q_times_raw_abs_mass"),
                                       ("spatial_weighted_raw_abs_mass", "spatial_weighted_raw_abs_mass"),
                                       ("q_spatial_weighted_raw_abs_mass", "q_spatial_weighted_raw_abs_mass")):
                    row[prefix + "_image_mean"] = row.get("image_" + source, np.nan)
                    row[prefix + "_class_mean"] = row.get("class_" + source, np.nan)
                    values = self.sums.get(source)
                    observed = self.valid.get(source)
                    mass_sum = float(values[mask].sum()) if values is not None else np.nan
                    all_mass = float(values.sum()) if values is not None else np.nan
                    row[prefix + "_sum"] = mass_sum
                    row[prefix + "_valid_n"] = int(observed[mask].sum()) if observed is not None else 0
                    row[prefix + "_epoch_contribution_to_mean"] = mass_sum / max(self.n.sum(), 1)
                    row[prefix + "_epoch_contribution_fraction"] = mass_sum / all_mass if all_mass > 0 else np.nan
                rows.append(row)
            return class_table, pd.DataFrame(rows)


    class BranchDiagnosticAccumulator:
        """Training diagnostics stratified by teacher correctness within each true class.

        Class means use only OBSERVED classes in that outcome; the two outcomes can
        contain different classes and must not be interpreted as a paired comparison.
        """
        def __init__(self):
            self.n = np.zeros((2, NUM_CLASSES), dtype=np.int64)
            self.sums, self.valid = {}, {}

        def update(self, labels, diagnostics):
            y = labels.detach().cpu().numpy()
            outcome = diagnostics["teacher_correct"].detach().cpu().numpy().astype(np.int64)
            np.add.at(self.n, (outcome, y), 1)
            names = [name for name, tensor in diagnostics.items()
                     if torch.is_tensor(tensor) and tensor.ndim == 1 and len(tensor) == len(y)]
            if not names:
                return
            matrix = torch.stack([diagnostics[name].detach().float() for name in names], dim=1).cpu().numpy()
            for col, name in enumerate(names):
                values = matrix[:, col].astype(np.float64)
                finite = np.isfinite(values)
                if name not in self.sums:
                    self.sums[name] = np.zeros((2, NUM_CLASSES), dtype=np.float64)
                    self.valid[name] = np.zeros((2, NUM_CLASSES), dtype=np.int64)
                np.add.at(self.sums[name], (outcome[finite], y[finite]), values[finite])
                np.add.at(self.valid[name], (outcome[finite], y[finite]), 1)

        def frames(self, class_to_group, epoch):
            classes, groups = [], []
            all_n = int(self.n.sum())
            guidance = self.sums.get("guidance_after_q", np.zeros_like(self.n, dtype=float))
            total_guidance = guidance.sum()
            group_ids = class_to_group.detach().cpu().numpy()
            for outcome in (0, 1):
                if not self.n[outcome].sum():
                    continue
                acc = ClassDiagnosticAccumulator()
                acc.n = self.n[outcome]
                acc.sums = {name: values[outcome] for name, values in self.sums.items()}
                acc.valid = {name: values[outcome] for name, values in self.valid.items()}
                class_frame, group_frame = acc.frames(class_to_group, epoch)
                class_frame = class_frame[class_frame.n_samples > 0].copy()
                group_frame = group_frame[group_frame.n_samples > 0].copy()
                for frame in (class_frame, group_frame):
                    frame["teacher_correct"] = outcome
                    frame["teacher_outcome"] = "correct" if outcome else "error"
                # Class contributions also refer to the WHOLE epoch, including
                # both teacher outcomes, rather than each outcome separately.
                for source, prefix in (("guidance_after_q", "guidance"),
                                       ("raw_abs_mass", "raw_abs_mass"),
                                       ("target_abs_mass_after_q", "q_times_raw_abs_mass"),
                                       ("spatial_weighted_raw_abs_mass", "spatial_weighted_raw_abs_mass"),
                                       ("q_spatial_weighted_raw_abs_mass", "q_spatial_weighted_raw_abs_mass")):
                    if source in self.sums:
                        observed_classes = class_frame.class_id.to_numpy(dtype=int)
                        values = self.sums[source][outcome, observed_classes]
                        total = self.sums[source].sum()
                        class_frame[prefix + "_sum"] = values
                        class_frame[prefix + "_epoch_contribution_to_mean"] = values / max(all_n, 1)
                        class_frame[prefix + "_epoch_contribution_fraction"] = (values / total
                            if total > 0 else np.full(len(class_frame), np.nan))
                # Each row's contribution is measured against the WHOLE epoch, not
                # independently renormalized within the correct/error branch.
                for index, row in group_frame.iterrows():
                    group_id = GROUP_NAME_TO_ID[row["group"]]
                    contribution = guidance[outcome, group_ids == group_id].sum()
                    group_frame.loc[index, "guidance_within_outcome_contribution_fraction"] = row["guidance_epoch_contribution_fraction"]
                    group_frame.loc[index, "guidance_epoch_contribution_fraction"] = contribution / total_guidance if total_guidance > 0 else np.nan
                    group_frame.loc[index, "guidance_epoch_contribution_to_mean"] = contribution / max(all_n, 1)
                    group_frame.loc[index, "epoch_sample_fraction"] = row["n_samples"] / max(all_n, 1)
                    for source, prefix in (("raw_abs_mass", "raw_abs_mass"),
                                           ("target_abs_mass_after_q", "q_times_raw_abs_mass"),
                                       ("spatial_weighted_raw_abs_mass", "spatial_weighted_raw_abs_mass"),
                                       ("q_spatial_weighted_raw_abs_mass", "q_spatial_weighted_raw_abs_mass")):
                        values = self.sums.get(source)
                        observed = self.valid.get(source)
                        group_mask = group_ids == group_id
                        mass_sum = float(values[outcome, group_mask].sum()) if values is not None else np.nan
                        all_mass = float(values.sum()) if values is not None else np.nan
                        group_frame.loc[index, prefix + "_sum"] = mass_sum
                        group_frame.loc[index, prefix + "_valid_n"] = int(observed[outcome, group_mask].sum()) if observed is not None else 0
                        group_frame.loc[index, prefix + "_epoch_contribution_to_mean"] = mass_sum / max(all_n, 1)
                        group_frame.loc[index, prefix + "_epoch_contribution_fraction"] = mass_sum / all_mass if all_mass > 0 else np.nan
                classes.append(class_frame)
                groups.append(group_frame)
            return (pd.concat(classes, ignore_index=True) if classes else pd.DataFrame(),
                    pd.concat(groups, ignore_index=True) if groups else pd.DataFrame())


    def feature_gradient_diagnostics(losses, features_s, class_to_group, labels, epoch, batch_id, diagnostics=None):
        """Weighted CE/method gradients in sampled last-feature tensors.

        The method includes alpha, q inside the teacher target, and shared global s_T.
        Subsets retain the original full-batch denominator; norms are not additive
        shares of losses or model parameters.
        """
        feature_map = extract_feature_tensor(features_s)
        def gradient(value):
            if not torch.is_tensor(value) or not value.requires_grad:
                return torch.zeros_like(feature_map)
            grad = torch.autograd.grad(value, feature_map, retain_graph=True, allow_unused=True)[0]
            return torch.zeros_like(feature_map) if grad is None else grad.detach()
        zero = feature_map.sum() * 0.
        ce_grad = gradient(CE_WEIGHT * losses["ce"])
        components = {"guidance": gradient(losses["guidance_weighted"])}
        for name in ("raw", "difference", "coincidence", "correct", "error"):
            components[name] = gradient(CAM_ALPHA * losses.get(name + "_component", zero))
        all_samples = torch.ones_like(labels, dtype=torch.bool)
        outcomes, branches = {"all": all_samples}, {"all": all_samples}
        if diagnostics is not None:
            teacher_correct = diagnostics["teacher_correct"].bool()
            outcomes.update(correct=teacher_correct, error=~teacher_correct)
            branches.update({name: diagnostics[name + "_branch"].bool()
                             for name in ("raw", "difference", "coincidence")})
        rows = []
        for group in ("all", "many", "medium", "few"):
            group_mask = all_samples if group == "all" else class_to_group[labels].eq(GROUP_NAME_TO_ID[group])
            for outcome, outcome_mask in outcomes.items():
                for branch, branch_mask in branches.items():
                    mask = group_mask & outcome_mask & branch_mask
                    if not mask.any():
                        continue
                    ce = ce_grad[mask].flatten()
                    ce_norm = ce.norm().item()
                    row = {"epoch": epoch, "batch": batch_id, "group": group,
                        "teacher_outcome": outcome, "branch": branch,
                        "n_samples": int(mask.sum()), "n_total_batch": len(labels),
                        "subset_fraction": float(mask.float().mean()), "ce_feature_grad_norm": ce_norm,
                        "diagnostic_scope": "sampled_batches_last_feature_tensor",
                        "loss_denominator": "original_full_batch",
                        "grad_diag_max_batches_configured": GRAD_DIAG_MAX_BATCHES,
                        "ce_weight": CE_WEIGHT, "cam_alpha": CAM_ALPHA,
                        "amount_apply_to": AMOUNT_APPLY_TO, "where_apply_to": WHERE_APPLY_TO}
                    if diagnostics is not None:
                        row["sample_diagnostic_aggregation"] = "finite_sample_mean_in_row_subset"
                        for name in ("amount_factor_base", "amount_weight", "amount_selected_outcome",
                                     "amount_applied", "where_selected_outcome", "where_eligible",
                                     "where_applied", "cam_scale_reference", "cam_scale_applied",
                                     "cam_global_coefficient", "cam_effective_coefficient"):
                            if name in diagnostics:
                                values = diagnostics[name][mask].detach()
                                finite = torch.isfinite(values)
                                row[name] = float(values[finite].mean()) if finite.any() else np.nan
                                row[name + "_valid_n"] = int(finite.sum())
                    for name, grad in components.items():
                        values = grad[mask].flatten()
                        norm = values.norm().item()
                        row[name + "_feature_grad_norm"] = norm
                        row[name + "_to_ce_feature_grad_ratio"] = norm / ce_norm if ce_norm > 0 else np.nan
                        row[name + "_feature_grad_cosine"] = (float(torch.dot(ce, values) / (ce_norm * norm))
                            if ce_norm > 0 and norm > 0 else np.nan)
                    rows.append(row)
        return rows


    def classification_outcome_diagnostics(labels, teacher_pred, student_pred, class_to_group, split):
        """Recovery/regression on fixed final predictions; no alternative F1 convention."""
        labels = np.asarray(labels)
        teacher_correct = np.asarray(teacher_pred) == labels
        student_correct = np.asarray(student_pred) == labels
        mapping = class_to_group.detach().cpu().numpy() if torch.is_tensor(class_to_group) else np.asarray(class_to_group)
        rows = []
        for group in ("all", "many", "medium", "few"):
            group_mask = np.ones(len(labels), dtype=bool) if group == "all" else mapping[labels] == GROUP_NAME_TO_ID[group]
            for outcome in (0, 1):
                mask = group_mask & (teacher_correct == bool(outcome))
                count = int(mask.sum())
                if not count:
                    continue
                successes = int(student_correct[mask].sum())
                failures = count - successes
                per_class_rate = pd.DataFrame({"class_id": labels[mask],
                    "student_correct": student_correct[mask].astype(float)}).groupby("class_id").student_correct.mean()
                class_balanced_rate = float(per_class_rate.mean())
                rows.append({"split": split, "group": group, "teacher_correct": outcome,
                    "teacher_outcome": "correct" if outcome else "error",
                    "n_samples": count, "n_classes_observed": int(np.unique(labels[mask]).size),
                    "student_correct_n": successes, "student_correct_rate": successes / count,
                    "student_correct_rate_class_balanced": class_balanced_rate,
                    "recovered_n": successes if not outcome else 0,
                    "recovery_rate": successes / count if not outcome else np.nan,
                    "recovery_rate_class_balanced": class_balanced_rate if not outcome else np.nan,
                    "regressed_n": failures if outcome else 0,
                    "regression_rate": failures / count if outcome else np.nan,
                    "regression_rate_class_balanced": 1. - class_balanced_rate if outcome else np.nan})
        return pd.DataFrame(rows)


    def fixed_probe_ids(pool_ids, seed):
        rng = np.random.default_rng(seed + 48721)
        selected = []
        pool_ids = np.asarray(pool_ids, dtype=np.int64)
        for c in range(NUM_CLASSES):
            available = pool_ids[_BASE_LT_TARGETS[pool_ids] == c]
            count = len(available) if DIAGNOSTIC_MAX_PER_CLASS is None else min(len(available), DIAGNOSTIC_MAX_PER_CLASS)
            selected.extend(rng.choice(available, count, replace=False).tolist())
        return sorted(selected)


    def plot_training_history(history, group_history, best_epoch, save_dir):
        """Weighted epoch losses, branch/outcome terms, and observed raw-CAM mass."""
        fig, axes = plt.subplots(2, 3, figsize=(16, 8))
        axes[0,0].plot(history.epoch, history.val_f1_macro, label="Macro-F1 validação")
        axes[0,0].axvline(best_epoch, color="gray", linestyle="--", label="checkpoint escolhido")
        axes[0,0].set_title("Seleção por validação")
        ce_weighted = history.train_ce_weighted.to_numpy()
        method_weighted = history.train_method_weighted.to_numpy()
        axes[0,2].plot(history.epoch, ce_weighted, label="CE_WEIGHT × CE")
        axes[0,2].plot(history.epoch, method_weighted, label="alpha × mean(D_retificado)")
        axes[0,2].set_title("Termos da loss: D_retificado inclui s_T e q no alvo")
        for name in ("many", "medium", "few"):
            axes[0,1].plot(history.epoch, history[f"val_{name}_shot_f1"], label=name)
        axes[0,1].set_title("F1 por grupo: definição histórica")
        for name in ("raw", "difference", "coincidence"):
            column = "train_" + name + "_weighted"
            if column in history:
                axes[1,0].plot(history.epoch, history[column], label=name)
        axes[1,0].set_title("Método por ramo: alpha × mean(D_retificado × ramo)")
        ratio = np.divide(method_weighted, ce_weighted,
            out=np.full(len(history), np.nan), where=ce_weighted > 0)
        axes[1,1].plot(history.epoch, ratio, label="Método ponderado / CE ponderada")
        axes[1,1].set_title("Razão das somas ponderadas da época")
        for name in ("correct", "error"):
            column = "train_" + name + "_weighted"
            if column in history:
                axes[1,2].plot(history.epoch, history[column], label="professor " + name)
        axes[1,2].set_title("Método por acerto/erro: contribuição à média total")
        for ax in axes.flat:
            ax.set_xlabel("Época"); ax.grid(alpha=.2)
            if ax.lines:
                ax.legend(fontsize=8)
        fig.tight_layout()
        fig.savefig(os.path.join(save_dir, "learning_and_guidance_curves.png"), dpi=130)
        plt.close(fig)

        if "raw_abs_mass_class_mean" in group_history:
            fig, axes = plt.subplots(2, 2, figsize=(12, 8))
            for name in ("many", "medium", "few"):
                part = group_history[group_history.group == name]
                axes[0,0].plot(part.epoch, part.raw_abs_mass_class_mean, label=name)
                axes[0,1].plot(part.epoch, part.q_times_raw_abs_mass_class_mean, label=name)
                axes[1,0].plot(part.epoch, part.raw_abs_mass_image_mean, label=name + " raw")
                axes[1,0].plot(part.epoch, part.q_times_raw_abs_mass_image_mean, linestyle="--", label=name + " q×raw")
                axes[1,1].plot(part.epoch, part.raw_abs_mass_epoch_contribution_fraction, label=name + " raw")
                axes[1,1].plot(part.epoch, part.q_times_raw_abs_mass_epoch_contribution_fraction,
                               linestyle="--", label=name + " q×raw")
            axes[0,0].set_title("Massa L1 raw do professor: média entre classes")
            axes[0,1].set_title("Estatística q × massa L1: média entre classes")
            axes[1,0].set_title("Médias por imagem: antes/depois de q")
            axes[1,1].set_title("Participação na massa total da época")
            for ax in axes.flat:
                ax.set_xlabel("Época"); ax.grid(alpha=.2); ax.legend(fontsize=8)
            fig.suptitle("CAM alvo assinada: massa = média espacial de |CAM|; r é o mapa de fatores por sinal, não altera o mapa\n"
                         "Estatísticas sem alpha/s_T; views de treino; q_base é fixo e estimado no treino determinístico", fontsize=10)
            fig.tight_layout(rect=(0, 0, 1, .92))
            fig.savefig(os.path.join(save_dir, "learning_and_mass_curves.png"), dpi=130)
            plt.close(fig)


    def group_probe_summary(frame, by_outcome=False):
        """Image means and equal observed-class means, with finite denominators."""
        if frame.empty:
            return pd.DataFrame()
        keys = ["split", "group"] + (["teacher_correct"] if by_outcome else [])
        excluded = {"pool_id", "class_id", "teacher_pred", "student_pred", "teacher_correct"}
        numeric = [c for c in frame.select_dtypes(include=[np.number]).columns if c not in excluded]
        if not by_outcome:
            numeric.append("teacher_correct")
        rows = []
        for values, part in frame.groupby(keys, dropna=False):
            if not isinstance(values, tuple):
                values = (values,)
            row = dict(zip(keys, values))
            row.update(n_samples=len(part), n_classes=part.class_id.nunique())
            finite_part = part[numeric].replace([np.inf, -np.inf], np.nan)
            class_means = finite_part.assign(class_id=part.class_id).groupby("class_id")[numeric].mean()
            for col in numeric:
                row["image_" + col] = finite_part[col].mean()
                row["class_" + col] = class_means[col].mean()
                row[col + "_valid_n"] = int(finite_part[col].notna().sum())
                row[col + "_classes_valid_n"] = int(class_means[col].notna().sum())
            rows.append(row)
        return pd.DataFrame(rows)


    def plot_cam_example(image, maps, row, destination):
        """Raw signed target CAMs and actual spatial evidence/weights, on one sample."""
        def arr(name):
            value = maps.get(name)
            return value.detach().cpu().numpy() if torch.is_tensor(value) else None
        raw, student = arr("teacher_raw"), arr("student_cam")
        rectified=arr("teacher_rectified")
        squared_error = (student - rectified) ** 2 if rectified is not None and student is not None else None
        image_np = image.detach().cpu().permute(1, 2, 0).numpy()
        image_np = np.clip(image_np * np.array(CIFAR_STD) + np.array(CIFAR_MEAN), 0, 1)
        branch = next((name for name in ("raw", "difference", "coincidence")
                       if row.get(name + "_branch", 0)), "raw")
        panels = [("Imagem", image_np, "image"),
            ("Professor: CAM alvo raw assinada", raw, "signed"),
            ("Aluno: CAM alvo raw assinada", student, "signed"),
            ("Erro pontual: (aluno − alvo retificado por sinal)²", squared_error, "positive"),
            ("Evidência de diferença\n(mean por rival de |norm alvo − norm rival|)", arr("difference_evidence"), "positive"),
            ("Evidência de coincidência\n(mínimo ativo com mesmo sinal, por rival)", arr("coincidence_evidence"), "positive"),
            ("Atenção efetiva p\n(soma espacial = 1)", arr("applied_attention"), "positive"),
            ("Peso efetivo w da MSE\n(média espacial = 1)", arr("applied_spatial_weight"), "positive")]
        finite_values = [np.asarray(value)[np.isfinite(value)] for _, value, kind in panels
                         if kind == "signed" and value is not None]
        limit = max([float(np.abs(value).max()) for value in finite_values if value.size] + [1e-8])
        fig, axes = plt.subplots(2, 4, figsize=(17, 8))
        for ax, (title, values, kind) in zip(axes.flat, panels):
            if values is None or not np.isfinite(np.asarray(values)).any():
                ax.text(.5, .5, "Indisponível", ha="center", va="center", transform=ax.transAxes)
            elif kind == "image":
                ax.imshow(values)
            else:
                kwargs = dict(cmap="coolwarm", vmin=-limit, vmax=limit) if kind == "signed" else dict(cmap="viridis", vmin=0)
                fig.colorbar(ax.imshow(values, **kwargs), ax=ax, fraction=.046, pad=.04)
            ax.set_title(title, fontsize=9); ax.axis("off")
        correctness = bool(row["teacher_correct"])
        configured_mode = WHERE_CORRECT_MODE if correctness else WHERE_ERROR_MODE
        mask_mode = WHERE_CORRECT_MASK_MODE if correctness else WHERE_ERROR_MASK_MODE
        fig.suptitle(f"{row['split']} | {row['class_name']} ({row['group']}) | professor correto={int(correctness)} | ramo selecionado={branch}\n"
            f"QUANTO: filtro={AMOUNT_APPLY_TO}; selecionado={row.get('amount_selected_outcome', float('nan')):g}; "
            f"aplicado={row.get('amount_applied', float('nan')):g}; q_base={row.get('amount_factor_base', float('nan')):.3f}; "
            f"q médio={row.get('amount_weight', float('nan')):.3f}; "
            f"q+={row.get('amount_factor_positive_applied', float('nan')):.3f}; "
            f"q−={row.get('amount_factor_negative_applied', float('nan')):.3f}\n"
            f"ONDE: filtro={WHERE_APPLY_TO}; selecionado={row.get('where_selected_outcome', float('nan')):g}; "
            f"modo={configured_mode}; máscara={mask_mode}; beta={WHERE_BETA:g} | D={row.get('guidance_before_q', float('nan')):.4g}\n"
            f"s_T={row.get('cam_scale_applied', float('nan')):.4g}; alpha/s_T={row.get('cam_global_coefficient', float('nan')):.4g}; "
            f"coeficiente do erro={row.get('cam_effective_coefficient', float('nan')):.4g}\n"
            f"classes selecionadas: {row['selected_classes']}", fontsize=10)
        fig.subplots_adjust(top=.77, bottom=.05, hspace=.35, wspace=.32)
        fig.savefig(destination, dpi=130, bbox_inches="tight")
        plt.close(fig)


    @torch.no_grad()
    def run_fixed_probe(student, teacher, experiment, seed, class_to_group, stats,
                        split, pool_ids, save_dir):
        if split not in {"train", "val"}:
            raise ValueError("CAM probes are restricted to train/val; official test is classification-only.")
        ids = fixed_probe_ids(pool_ids, seed)
        base = SingleViewCIFARArrayDataset(_BASE_LT_DATA, _BASE_LT_TARGETS,
                                          transform_eval, return_index=True)
        loader = DataLoader(Subset(base, ids), batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=0, generator=make_generator(seed+9127))
        student.eval(); teacher.eval()
        rows, figure_counts, map_examples = [], {}, []
        for images, labels, sample_ids in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            logits_s, features_s = student(images)
            _, diag, maps, _ = qw_cam_loss(experiment, logits_s, features_s, student.fc,
                teacher, labels, images, class_to_group, stats, sample_ids, seed, epoch=0)
            logits_t, _ = teacher(images)
            y = labels.cpu().tolist()
            selected = maps["selected_ids"].cpu().tolist()
            numeric = {k: v.detach().cpu().numpy() for k, v in diag.items()
                       if torch.is_tensor(v) and v.ndim == 1 and len(v) == len(labels)}
            names = _CLASS_NAMES or [str(c) for c in range(NUM_CLASSES)]
            for j, image_id in enumerate(sample_ids.tolist()):
                group = GROUP_ID_TO_NAME[int(class_to_group[labels[j]])]
                row = {"split": split, "experiment": experiment,
                    "amount_apply_to": AMOUNT_APPLY_TO, "where_apply_to": WHERE_APPLY_TO,
                    "pool_id": image_id, "class_id": y[j],
                    "class_name": names[y[j]], "group": group,
                    "teacher_pred": int(logits_t[j].argmax()), "student_pred": int(logits_s[j].argmax()),
                    "student_correct": int(logits_s[j].argmax() == labels[j]),
                    "student_target_probability": float(logits_s[j].softmax(0)[labels[j]]),
                    "selected_class_ids": json.dumps(selected[j]),
                    "selected_classes": ", ".join(names[c] for c in selected[j])}
                row.update({k: float(v[j]) for k,v in numeric.items()})
                rows.append(row)
                key = (split, group, int(row["teacher_correct"]))
                if SAVE_CAM_FIGURES and figure_counts.get(key,0) < FIGURES_PER_GROUP_OUTCOME:
                    figure_counts[key] = figure_counts.get(key,0) + 1
                    figure_dir = os.path.join(save_dir, "cam_figures")
                    os.makedirs(figure_dir, exist_ok=True)
                    sample_maps = {k:v[j] for k,v in maps.items()
                                   if torch.is_tensor(v) and v.ndim > 0 and len(v) == len(labels)}
                    path = os.path.join(figure_dir, f"{split}_{group}_correct{key[-1]}_id{image_id}.png")
                    plot_cam_example(images[j], sample_maps, row, path)
                    map_examples.append({"pool_id": image_id,
                        **{k:v.detach().cpu().numpy() for k,v in sample_maps.items()}})
        frame = pd.DataFrame(rows)
        frame.to_csv(os.path.join(save_dir, f"probe_{split}_per_image.csv"), index=False)
        if map_examples:
            arrays = {name: np.stack([item[name] for item in map_examples])
                      for name in map_examples[0]}
            np.savez_compressed(os.path.join(save_dir, f"probe_{split}_map_examples.npz"), **arrays)
        return frame
else:  # mobilenetv2_imagenet_lt
    def hash_state_dict(model):
        digest = hashlib.sha256()
        for name, tensor in sorted(model.state_dict().items()):
            digest.update(name.encode())
            digest.update(tensor.detach().cpu().contiguous().numpy().tobytes())
        return digest.hexdigest()


    def file_sha256(path):
        digest = hashlib.sha256()
        with open(path, "rb") as file:
            for chunk in iter(lambda: file.read(1024 * 1024), b""):
                digest.update(chunk)
        return digest.hexdigest()


    def metrics_from_arrays(labels, preds, probs, class_counts):
        # Same definitions as the historical evaluate_model / compute_group_metrics.
        metrics = {
            "acc": float((preds == labels).mean() * 100),
            "top5_acc": float(top_k_accuracy_score(labels, probs, k=5,
                                  labels=np.arange(NUM_CLASSES)) * 100),
            "f1_macro": float(f1_score(labels, preds, average="macro", zero_division=0)),
            "f1_weighted": float(f1_score(labels, preds, average="weighted", zero_division=0)),
            "balanced_acc": float(balanced_accuracy_score(labels, preds)),
        }
        metrics.update(compute_group_metrics(labels, preds, get_shot_groups(class_counts)))
        return metrics


    @torch.no_grad()
    def evaluate_with_arrays(model, loader, class_counts):
        model.eval()
        ys, ps, probabilities = [], [], []
        ce_sum, n = 0., 0
        for batch in loader:
            images, labels = batch[:2]
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            logits, _ = model(images)
            ce_sum += F.cross_entropy(logits, labels, reduction="sum").item()
            n += labels.numel()
            ys.append(labels.cpu().numpy())
            ps.append(logits.argmax(1).cpu().numpy())
            probabilities.append(logits.softmax(1).cpu().numpy())
        y, pred, prob = np.concatenate(ys), np.concatenate(ps), np.concatenate(probabilities)
        metrics = metrics_from_arrays(y, pred, prob, class_counts)
        metrics["ce_loss"] = ce_sum / n
        return metrics, y, pred, prob


    class ClassDiagnosticAccumulator:
        """Finite-aware class/image means, including base/applied q and outcome masks.

        Per-image outcome selectors, application flags, scale and coefficients use
        the same generic aggregation path. Missing observations remain NaN.
        """
        def __init__(self):
            self.n = np.zeros(NUM_CLASSES, dtype=np.int64)
            self.sums, self.valid = {}, {}

        def update(self, labels, diagnostics):
            y = labels.detach().cpu().numpy()
            np.add.at(self.n, y, 1)
            names = [name for name, tensor in diagnostics.items()
                     if torch.is_tensor(tensor) and tensor.ndim == 1 and len(tensor) == len(y)]
            if not names:
                return
            matrix = torch.stack([diagnostics[name].detach().float() for name in names], dim=1).cpu().numpy()
            for col, name in enumerate(names):
                values = matrix[:, col].astype(np.float64)
                finite = np.isfinite(values)
                if name not in self.sums:
                    self.sums[name] = np.zeros(NUM_CLASSES, dtype=np.float64)
                    self.valid[name] = np.zeros(NUM_CLASSES, dtype=np.int64)
                np.add.at(self.sums[name], y[finite], values[finite])
                np.add.at(self.valid[name], y[finite], 1)

        def frames(self, class_to_group, epoch):
            group_ids = class_to_group.detach().cpu().numpy()
            # Build all diagnostic columns together to avoid repeated DataFrame
            # fragmentation when every epoch/outcome has hundreds of columns.
            class_columns = {"epoch": np.full(NUM_CLASSES, epoch), "class_id": np.arange(NUM_CLASSES),
                "class_name": _CLASS_NAMES or [str(c) for c in range(NUM_CLASSES)],
                "group": [GROUP_ID_TO_NAME[int(x)] for x in group_ids], "n_samples": self.n}
            for name in self.sums:
                count = self.valid[name]
                class_columns[name] = np.divide(self.sums[name], count,
                    out=np.full(NUM_CLASSES, np.nan), where=count > 0)
                class_columns[name + "_valid_n"] = count
            # Class-level sums/contributions use the same observed full-epoch
            # denominator as group rows. Means remain NaN for unobserved classes.
            for source, prefix in (("guidance_after_q", "guidance"),
                                   ("raw_abs_mass", "raw_abs_mass"),
                                   ("target_abs_mass_after_q", "q_times_raw_abs_mass"),
                                   ("spatial_weighted_raw_abs_mass", "spatial_weighted_raw_abs_mass"),
                                   ("q_spatial_weighted_raw_abs_mass", "q_spatial_weighted_raw_abs_mass")):
                if source in self.sums:
                    values = self.sums[source]
                    class_columns[prefix + "_sum"] = values
                    class_columns[prefix + "_epoch_contribution_to_mean"] = values / max(self.n.sum(), 1)
                    class_columns[prefix + "_epoch_contribution_fraction"] = (values / values.sum()
                        if values.sum() > 0 else np.full(NUM_CLASSES, np.nan))
            class_table = pd.DataFrame(class_columns)
            rows = []
            guidance = self.sums.get("guidance_after_q", np.zeros(NUM_CLASSES))
            total_guidance = guidance.sum()
            for group_id in range(3):
                mask = group_ids == group_id
                row = {"epoch": epoch, "group": GROUP_ID_TO_NAME[group_id],
                    "n_samples": int(self.n[mask].sum()), "n_classes": int(mask.sum()),
                    "n_classes_observed": int(np.count_nonzero(self.n[mask]))}
                for name in self.sums:
                    count = self.valid[name][mask].sum()
                    row["image_" + name] = self.sums[name][mask].sum() / count if count else np.nan
                    row["class_" + name] = class_table.loc[mask, name].mean()
                    row[name + "_valid_n"] = int(count)
                    row[name + "_classes_valid_n"] = int(class_table.loc[mask, name].notna().sum())
                row["epoch_sample_fraction"] = row["n_samples"] / max(self.n.sum(), 1)
                contribution = guidance[mask].sum()
                row["guidance_sum"] = float(contribution)
                row["guidance_epoch_contribution_fraction"] = contribution / total_guidance if total_guidance > 0 else np.nan
                row["guidance_epoch_contribution_to_mean"] = contribution / max(self.n.sum(), 1)
                # These mass statistics use q actually applied to each image. The
                # fixed q_base is logged separately; neither factor changes the CAM.
                # L1 mass statistics omit alpha / the shared global denominator.
                row["raw_abs_mass_image_mean"] = row.get("image_raw_abs_mass", np.nan)
                row["raw_abs_mass_class_mean"] = row.get("class_raw_abs_mass", np.nan)
                row["q_image_mean"] = row.get("image_amount_weight", np.nan)
                row["q_class_mean"] = row.get("class_amount_weight", np.nan)
                row["q_applied_image_mean"] = row["q_image_mean"]
                row["q_applied_class_mean"] = row["q_class_mean"]
                row["q_base_image_mean"] = row.get("image_amount_factor_base", np.nan)
                row["q_base_class_mean"] = row.get("class_amount_factor_base", np.nan)
                row["q_times_raw_abs_mass_image_mean"] = row.get("image_target_abs_mass_after_q", np.nan)
                row["q_times_raw_abs_mass_class_mean"] = row.get("class_target_abs_mass_after_q", np.nan)
                row["raw_signed_cam_image_mean"] = row.get("image_raw_pos_mass", np.nan) - row.get("image_raw_neg_mass", np.nan)
                row["raw_signed_cam_class_mean"] = row.get("class_raw_pos_mass", np.nan) - row.get("class_raw_neg_mass", np.nan)
                row["cam_mass_definition"] = "mean_spatial_abs_of_signed_target_CAM"
                row["cam_mass_sample_scope"] = "observed_epoch_views; static_QUANTO_uses_deterministic_training_class_means"
                for source, prefix in (("raw_abs_mass", "raw_abs_mass"),
                                       ("target_abs_mass_after_q", "q_times_raw_abs_mass"),
                                       ("spatial_weighted_raw_abs_mass", "spatial_weighted_raw_abs_mass"),
                                       ("q_spatial_weighted_raw_abs_mass", "q_spatial_weighted_raw_abs_mass")):
                    row[prefix + "_image_mean"] = row.get("image_" + source, np.nan)
                    row[prefix + "_class_mean"] = row.get("class_" + source, np.nan)
                    values = self.sums.get(source)
                    observed = self.valid.get(source)
                    mass_sum = float(values[mask].sum()) if values is not None else np.nan
                    all_mass = float(values.sum()) if values is not None else np.nan
                    row[prefix + "_sum"] = mass_sum
                    row[prefix + "_valid_n"] = int(observed[mask].sum()) if observed is not None else 0
                    row[prefix + "_epoch_contribution_to_mean"] = mass_sum / max(self.n.sum(), 1)
                    row[prefix + "_epoch_contribution_fraction"] = mass_sum / all_mass if all_mass > 0 else np.nan
                rows.append(row)
            return class_table, pd.DataFrame(rows)


    class BranchDiagnosticAccumulator:
        """Training diagnostics stratified by teacher correctness within each true class.

        Class means use only OBSERVED classes in that outcome; the two outcomes can
        contain different classes and must not be interpreted as a paired comparison.
        """
        def __init__(self):
            self.n = np.zeros((2, NUM_CLASSES), dtype=np.int64)
            self.sums, self.valid = {}, {}

        def update(self, labels, diagnostics):
            y = labels.detach().cpu().numpy()
            outcome = diagnostics["teacher_correct"].detach().cpu().numpy().astype(np.int64)
            np.add.at(self.n, (outcome, y), 1)
            names = [name for name, tensor in diagnostics.items()
                     if torch.is_tensor(tensor) and tensor.ndim == 1 and len(tensor) == len(y)]
            if not names:
                return
            matrix = torch.stack([diagnostics[name].detach().float() for name in names], dim=1).cpu().numpy()
            for col, name in enumerate(names):
                values = matrix[:, col].astype(np.float64)
                finite = np.isfinite(values)
                if name not in self.sums:
                    self.sums[name] = np.zeros((2, NUM_CLASSES), dtype=np.float64)
                    self.valid[name] = np.zeros((2, NUM_CLASSES), dtype=np.int64)
                np.add.at(self.sums[name], (outcome[finite], y[finite]), values[finite])
                np.add.at(self.valid[name], (outcome[finite], y[finite]), 1)

        def frames(self, class_to_group, epoch):
            classes, groups = [], []
            all_n = int(self.n.sum())
            guidance = self.sums.get("guidance_after_q", np.zeros_like(self.n, dtype=float))
            total_guidance = guidance.sum()
            group_ids = class_to_group.detach().cpu().numpy()
            for outcome in (0, 1):
                if not self.n[outcome].sum():
                    continue
                acc = ClassDiagnosticAccumulator()
                acc.n = self.n[outcome]
                acc.sums = {name: values[outcome] for name, values in self.sums.items()}
                acc.valid = {name: values[outcome] for name, values in self.valid.items()}
                class_frame, group_frame = acc.frames(class_to_group, epoch)
                class_frame = class_frame[class_frame.n_samples > 0].copy()
                group_frame = group_frame[group_frame.n_samples > 0].copy()
                for frame in (class_frame, group_frame):
                    frame["teacher_correct"] = outcome
                    frame["teacher_outcome"] = "correct" if outcome else "error"
                # Class contributions also refer to the WHOLE epoch, including
                # both teacher outcomes, rather than each outcome separately.
                for source, prefix in (("guidance_after_q", "guidance"),
                                       ("raw_abs_mass", "raw_abs_mass"),
                                       ("target_abs_mass_after_q", "q_times_raw_abs_mass"),
                                       ("spatial_weighted_raw_abs_mass", "spatial_weighted_raw_abs_mass"),
                                       ("q_spatial_weighted_raw_abs_mass", "q_spatial_weighted_raw_abs_mass")):
                    if source in self.sums:
                        observed_classes = class_frame.class_id.to_numpy(dtype=int)
                        values = self.sums[source][outcome, observed_classes]
                        total = self.sums[source].sum()
                        class_frame[prefix + "_sum"] = values
                        class_frame[prefix + "_epoch_contribution_to_mean"] = values / max(all_n, 1)
                        class_frame[prefix + "_epoch_contribution_fraction"] = (values / total
                            if total > 0 else np.full(len(class_frame), np.nan))
                # Each row's contribution is measured against the WHOLE epoch, not
                # independently renormalized within the correct/error branch.
                for index, row in group_frame.iterrows():
                    group_id = GROUP_NAME_TO_ID[row["group"]]
                    contribution = guidance[outcome, group_ids == group_id].sum()
                    group_frame.loc[index, "guidance_within_outcome_contribution_fraction"] = row["guidance_epoch_contribution_fraction"]
                    group_frame.loc[index, "guidance_epoch_contribution_fraction"] = contribution / total_guidance if total_guidance > 0 else np.nan
                    group_frame.loc[index, "guidance_epoch_contribution_to_mean"] = contribution / max(all_n, 1)
                    group_frame.loc[index, "epoch_sample_fraction"] = row["n_samples"] / max(all_n, 1)
                    for source, prefix in (("raw_abs_mass", "raw_abs_mass"),
                                           ("target_abs_mass_after_q", "q_times_raw_abs_mass"),
                                       ("spatial_weighted_raw_abs_mass", "spatial_weighted_raw_abs_mass"),
                                       ("q_spatial_weighted_raw_abs_mass", "q_spatial_weighted_raw_abs_mass")):
                        values = self.sums.get(source)
                        observed = self.valid.get(source)
                        group_mask = group_ids == group_id
                        mass_sum = float(values[outcome, group_mask].sum()) if values is not None else np.nan
                        all_mass = float(values.sum()) if values is not None else np.nan
                        group_frame.loc[index, prefix + "_sum"] = mass_sum
                        group_frame.loc[index, prefix + "_valid_n"] = int(observed[outcome, group_mask].sum()) if observed is not None else 0
                        group_frame.loc[index, prefix + "_epoch_contribution_to_mean"] = mass_sum / max(all_n, 1)
                        group_frame.loc[index, prefix + "_epoch_contribution_fraction"] = mass_sum / all_mass if all_mass > 0 else np.nan
                classes.append(class_frame)
                groups.append(group_frame)
            return (pd.concat(classes, ignore_index=True) if classes else pd.DataFrame(),
                    pd.concat(groups, ignore_index=True) if groups else pd.DataFrame())


    def feature_gradient_diagnostics(losses, features_s, class_to_group, labels, epoch, batch_id, diagnostics=None):
        """Weighted CE/method gradients in sampled last-feature tensors.

        The method includes alpha, q inside the teacher target, and shared global s_T.
        Subsets retain the original full-batch denominator; norms are not additive
        shares of losses or model parameters.
        """
        feature_map = extract_feature_tensor(features_s)
        def gradient(value):
            if not torch.is_tensor(value) or not value.requires_grad:
                return torch.zeros_like(feature_map)
            grad = torch.autograd.grad(value, feature_map, retain_graph=True, allow_unused=True)[0]
            return torch.zeros_like(feature_map) if grad is None else grad.detach()
        zero = feature_map.sum() * 0.
        ce_grad = gradient(CE_WEIGHT * losses["ce"])
        components = {"guidance": gradient(losses["guidance_weighted"])}
        for name in ("raw", "difference", "coincidence", "correct", "error"):
            components[name] = gradient(CAM_ALPHA * losses.get(name + "_component", zero))
        all_samples = torch.ones_like(labels, dtype=torch.bool)
        outcomes, branches = {"all": all_samples}, {"all": all_samples}
        if diagnostics is not None:
            teacher_correct = diagnostics["teacher_correct"].bool()
            outcomes.update(correct=teacher_correct, error=~teacher_correct)
            branches.update({name: diagnostics[name + "_branch"].bool()
                             for name in ("raw", "difference", "coincidence")})
        rows = []
        for group in ("all", "many", "medium", "few"):
            group_mask = all_samples if group == "all" else class_to_group[labels].eq(GROUP_NAME_TO_ID[group])
            for outcome, outcome_mask in outcomes.items():
                for branch, branch_mask in branches.items():
                    mask = group_mask & outcome_mask & branch_mask
                    if not mask.any():
                        continue
                    ce = ce_grad[mask].flatten()
                    ce_norm = ce.norm().item()
                    row = {"epoch": epoch, "batch": batch_id, "group": group,
                        "teacher_outcome": outcome, "branch": branch,
                        "n_samples": int(mask.sum()), "n_total_batch": len(labels),
                        "subset_fraction": float(mask.float().mean()), "ce_feature_grad_norm": ce_norm,
                        "diagnostic_scope": "sampled_batches_last_feature_tensor",
                        "loss_denominator": "original_full_batch",
                        "grad_diag_max_batches_configured": GRAD_DIAG_MAX_BATCHES,
                        "ce_weight": CE_WEIGHT, "cam_alpha": CAM_ALPHA,
                        "amount_apply_to": AMOUNT_APPLY_TO, "where_apply_to": WHERE_APPLY_TO}
                    if diagnostics is not None:
                        row["sample_diagnostic_aggregation"] = "finite_sample_mean_in_row_subset"
                        for name in ("amount_factor_base", "amount_weight", "amount_selected_outcome",
                                     "amount_applied", "where_selected_outcome", "where_eligible",
                                     "where_applied", "cam_scale_reference", "cam_scale_applied",
                                     "cam_global_coefficient", "cam_effective_coefficient"):
                            if name in diagnostics:
                                values = diagnostics[name][mask].detach()
                                finite = torch.isfinite(values)
                                row[name] = float(values[finite].mean()) if finite.any() else np.nan
                                row[name + "_valid_n"] = int(finite.sum())
                    for name, grad in components.items():
                        values = grad[mask].flatten()
                        norm = values.norm().item()
                        row[name + "_feature_grad_norm"] = norm
                        row[name + "_to_ce_feature_grad_ratio"] = norm / ce_norm if ce_norm > 0 else np.nan
                        row[name + "_feature_grad_cosine"] = (float(torch.dot(ce, values) / (ce_norm * norm))
                            if ce_norm > 0 and norm > 0 else np.nan)
                    rows.append(row)
        return rows


    def classification_outcome_diagnostics(labels, teacher_pred, student_pred, class_to_group, split):
        """Recovery/regression on fixed final predictions; no alternative F1 convention."""
        labels = np.asarray(labels)
        teacher_correct = np.asarray(teacher_pred) == labels
        student_correct = np.asarray(student_pred) == labels
        mapping = class_to_group.detach().cpu().numpy() if torch.is_tensor(class_to_group) else np.asarray(class_to_group)
        rows = []
        for group in ("all", "many", "medium", "few"):
            group_mask = np.ones(len(labels), dtype=bool) if group == "all" else mapping[labels] == GROUP_NAME_TO_ID[group]
            for outcome in (0, 1):
                mask = group_mask & (teacher_correct == bool(outcome))
                count = int(mask.sum())
                if not count:
                    continue
                successes = int(student_correct[mask].sum())
                failures = count - successes
                per_class_rate = pd.DataFrame({"class_id": labels[mask],
                    "student_correct": student_correct[mask].astype(float)}).groupby("class_id").student_correct.mean()
                class_balanced_rate = float(per_class_rate.mean())
                rows.append({"split": split, "group": group, "teacher_correct": outcome,
                    "teacher_outcome": "correct" if outcome else "error",
                    "n_samples": count, "n_classes_observed": int(np.unique(labels[mask]).size),
                    "student_correct_n": successes, "student_correct_rate": successes / count,
                    "student_correct_rate_class_balanced": class_balanced_rate,
                    "recovered_n": successes if not outcome else 0,
                    "recovery_rate": successes / count if not outcome else np.nan,
                    "recovery_rate_class_balanced": class_balanced_rate if not outcome else np.nan,
                    "regressed_n": failures if outcome else 0,
                    "regression_rate": failures / count if outcome else np.nan,
                    "regression_rate_class_balanced": 1. - class_balanced_rate if outcome else np.nan})
        return pd.DataFrame(rows)


    def fixed_probe_ids(pool_ids, seed, targets):
        rng = np.random.default_rng(seed + 48721)
        selected = []
        pool_ids = np.asarray(pool_ids, dtype=np.int64)
        targets = np.asarray(targets, dtype=np.int64)
        for class_id in range(NUM_CLASSES):
            available = pool_ids[targets[pool_ids] == class_id]
            count = len(available) if DIAGNOSTIC_MAX_PER_CLASS is None else min(len(available), DIAGNOSTIC_MAX_PER_CLASS)
            selected.extend(rng.choice(available, count, replace=False).tolist())
        return sorted(selected)


    def plot_training_history(history, group_history, best_epoch, save_dir):
        """Weighted epoch losses, branch/outcome terms, and observed raw-CAM mass."""
        fig, axes = plt.subplots(2, 3, figsize=(16, 8))
        axes[0,0].plot(history.epoch, history.val_f1_macro, label="Macro-F1 validação")
        axes[0,0].axvline(best_epoch, color="gray", linestyle="--", label="checkpoint escolhido")
        axes[0,0].set_title("Seleção por validação")
        ce_weighted = history.train_ce_weighted.to_numpy()
        method_weighted = history.train_method_weighted.to_numpy()
        axes[0,2].plot(history.epoch, ce_weighted, label="CE_WEIGHT × CE")
        axes[0,2].plot(history.epoch, method_weighted, label="alpha × mean(D_retificado)")
        axes[0,2].set_title("Termos da loss: D_retificado inclui s_T e q no alvo")
        for name in ("many", "medium", "few"):
            axes[0,1].plot(history.epoch, history[f"val_{name}_shot_f1"], label=name)
        axes[0,1].set_title("F1 por grupo: definição histórica")
        for name in ("raw", "difference", "coincidence"):
            column = "train_" + name + "_weighted"
            if column in history:
                axes[1,0].plot(history.epoch, history[column], label=name)
        axes[1,0].set_title("Método por ramo: alpha × mean(D_retificado × ramo)")
        ratio = np.divide(method_weighted, ce_weighted,
            out=np.full(len(history), np.nan), where=ce_weighted > 0)
        axes[1,1].plot(history.epoch, ratio, label="Método ponderado / CE ponderada")
        axes[1,1].set_title("Razão das somas ponderadas da época")
        for name in ("correct", "error"):
            column = "train_" + name + "_weighted"
            if column in history:
                axes[1,2].plot(history.epoch, history[column], label="professor " + name)
        axes[1,2].set_title("Método por acerto/erro: contribuição à média total")
        for ax in axes.flat:
            ax.set_xlabel("Época"); ax.grid(alpha=.2)
            if ax.lines:
                ax.legend(fontsize=8)
        fig.tight_layout()
        fig.savefig(os.path.join(save_dir, "learning_and_guidance_curves.png"), dpi=130)
        plt.close(fig)

        if "raw_abs_mass_class_mean" in group_history:
            fig, axes = plt.subplots(2, 2, figsize=(12, 8))
            for name in ("many", "medium", "few"):
                part = group_history[group_history.group == name]
                axes[0,0].plot(part.epoch, part.raw_abs_mass_class_mean, label=name)
                axes[0,1].plot(part.epoch, part.q_times_raw_abs_mass_class_mean, label=name)
                axes[1,0].plot(part.epoch, part.raw_abs_mass_image_mean, label=name + " raw")
                axes[1,0].plot(part.epoch, part.q_times_raw_abs_mass_image_mean, linestyle="--", label=name + " q×raw")
                axes[1,1].plot(part.epoch, part.raw_abs_mass_epoch_contribution_fraction, label=name + " raw")
                axes[1,1].plot(part.epoch, part.q_times_raw_abs_mass_epoch_contribution_fraction,
                               linestyle="--", label=name + " q×raw")
            axes[0,0].set_title("Massa L1 raw do professor: média entre classes")
            axes[0,1].set_title("Estatística q × massa L1: média entre classes")
            axes[1,0].set_title("Médias por imagem: antes/depois de q")
            axes[1,1].set_title("Participação na massa total da época")
            for ax in axes.flat:
                ax.set_xlabel("Época"); ax.grid(alpha=.2); ax.legend(fontsize=8)
            fig.suptitle("CAM alvo assinada: massa = média espacial de |CAM|; r é o mapa de fatores por sinal, não altera o mapa\n"
                         "Estatísticas sem alpha/s_T; views de treino; q_base é fixo e estimado no treino determinístico", fontsize=10)
            fig.tight_layout(rect=(0, 0, 1, .92))
            fig.savefig(os.path.join(save_dir, "learning_and_mass_curves.png"), dpi=130)
            plt.close(fig)


    def group_probe_summary(frame, by_outcome=False):
        """Image means and equal observed-class means, with finite denominators."""
        if frame.empty:
            return pd.DataFrame()
        keys = ["split", "group"] + (["teacher_correct"] if by_outcome else [])
        excluded = {"pool_id", "class_id", "teacher_pred", "student_pred", "teacher_correct"}
        numeric = [c for c in frame.select_dtypes(include=[np.number]).columns if c not in excluded]
        if not by_outcome:
            numeric.append("teacher_correct")
        rows = []
        for values, part in frame.groupby(keys, dropna=False):
            if not isinstance(values, tuple):
                values = (values,)
            row = dict(zip(keys, values))
            row.update(n_samples=len(part), n_classes=part.class_id.nunique())
            finite_part = part[numeric].replace([np.inf, -np.inf], np.nan)
            class_means = finite_part.assign(class_id=part.class_id).groupby("class_id")[numeric].mean()
            for col in numeric:
                row["image_" + col] = finite_part[col].mean()
                row["class_" + col] = class_means[col].mean()
                row[col + "_valid_n"] = int(finite_part[col].notna().sum())
                row[col + "_classes_valid_n"] = int(class_means[col].notna().sum())
            rows.append(row)
        return pd.DataFrame(rows)


    def plot_cam_example(image, maps, row, destination):
        """Raw signed target CAMs and actual spatial evidence/weights, on one sample."""
        def arr(name):
            value = maps.get(name)
            return value.detach().cpu().numpy() if torch.is_tensor(value) else None
        raw, student = arr("teacher_raw"), arr("student_cam")
        rectified=arr("teacher_rectified")
        squared_error = (student - rectified) ** 2 if rectified is not None and student is not None else None
        image_np = image.detach().cpu().permute(1, 2, 0).numpy()
        image_np = np.clip(image_np * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN), 0, 1)
        branch = next((name for name in ("raw", "difference", "coincidence")
                       if row.get(name + "_branch", 0)), "raw")
        panels = [("Imagem", image_np, "image"),
            ("Professor: CAM alvo raw assinada", raw, "signed"),
            ("Aluno: CAM alvo raw assinada", student, "signed"),
            ("Erro pontual: (aluno − alvo retificado por sinal)²", squared_error, "positive"),
            ("Evidência de diferença\n(mean por rival de |norm alvo − norm rival|)", arr("difference_evidence"), "positive"),
            ("Evidência de coincidência\n(mínimo ativo com mesmo sinal, por rival)", arr("coincidence_evidence"), "positive"),
            ("Atenção efetiva p\n(soma espacial = 1)", arr("applied_attention"), "positive"),
            ("Peso efetivo w da MSE\n(média espacial = 1)", arr("applied_spatial_weight"), "positive")]
        finite_values = [np.asarray(value)[np.isfinite(value)] for _, value, kind in panels
                         if kind == "signed" and value is not None]
        limit = max([float(np.abs(value).max()) for value in finite_values if value.size] + [1e-8])
        fig, axes = plt.subplots(2, 4, figsize=(17, 8))
        for ax, (title, values, kind) in zip(axes.flat, panels):
            if values is None or not np.isfinite(np.asarray(values)).any():
                ax.text(.5, .5, "Indisponível", ha="center", va="center", transform=ax.transAxes)
            elif kind == "image":
                ax.imshow(values)
            else:
                kwargs = dict(cmap="coolwarm", vmin=-limit, vmax=limit) if kind == "signed" else dict(cmap="viridis", vmin=0)
                fig.colorbar(ax.imshow(values, **kwargs), ax=ax, fraction=.046, pad=.04)
            ax.set_title(title, fontsize=9); ax.axis("off")
        correctness = bool(row["teacher_correct"])
        configured_mode = WHERE_CORRECT_MODE if correctness else WHERE_ERROR_MODE
        mask_mode = WHERE_CORRECT_MASK_MODE if correctness else WHERE_ERROR_MASK_MODE
        fig.suptitle(f"{row['split']} | {row['class_name']} ({row['group']}) | professor correto={int(correctness)} | ramo selecionado={branch}\n"
            f"QUANTO: filtro={AMOUNT_APPLY_TO}; selecionado={row.get('amount_selected_outcome', float('nan')):g}; "
            f"aplicado={row.get('amount_applied', float('nan')):g}; q_base={row.get('amount_factor_base', float('nan')):.3f}; "
            f"q médio={row.get('amount_weight', float('nan')):.3f}; "
            f"q+={row.get('amount_factor_positive_applied', float('nan')):.3f}; "
            f"q−={row.get('amount_factor_negative_applied', float('nan')):.3f}\n"
            f"ONDE: filtro={WHERE_APPLY_TO}; selecionado={row.get('where_selected_outcome', float('nan')):g}; "
            f"modo={configured_mode}; máscara={mask_mode}; beta={WHERE_BETA:g} | D={row.get('guidance_before_q', float('nan')):.4g}\n"
            f"s_T={row.get('cam_scale_applied', float('nan')):.4g}; alpha/s_T={row.get('cam_global_coefficient', float('nan')):.4g}; "
            f"coeficiente do erro={row.get('cam_effective_coefficient', float('nan')):.4g}\n"
            f"classes selecionadas: {row['selected_classes']}", fontsize=10)
        fig.subplots_adjust(top=.77, bottom=.05, hspace=.35, wspace=.32)
        fig.savefig(destination, dpi=130, bbox_inches="tight")
        plt.close(fig)


    @torch.no_grad()
    def run_fixed_probe(student, teacher, experiment, seed, class_to_group, stats,
                        split, pool_ids, save_dir):
        if split not in {"train", "val"}:
            raise ValueError("CAM probes are restricted to train/val; official test is classification-only.")
        base = ImageNetLT(DATA_ROOT, TRAIN_TXT if split == "train" else VAL_TXT,
                          transform_eval, return_index=True)
        ids = fixed_probe_ids(pool_ids, seed, base.targets)
        loader = DataLoader(Subset(base, ids), batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=0, generator=make_generator(seed+9127))
        student.eval(); teacher.eval()
        rows, figure_counts, map_examples = [], {}, []
        for images, labels, sample_ids in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            logits_s, features_s = student(images)
            _, diag, maps, _ = qw_cam_loss(experiment, logits_s, features_s, student.fc,
                teacher, labels, images, class_to_group, stats, sample_ids, seed, epoch=0)
            logits_t, _ = teacher(images)
            y = labels.cpu().tolist()
            selected = maps["selected_ids"].cpu().tolist()
            numeric = {k: v.detach().cpu().numpy() for k, v in diag.items()
                       if torch.is_tensor(v) and v.ndim == 1 and len(v) == len(labels)}
            names = _CLASS_NAMES or [str(c) for c in range(NUM_CLASSES)]
            for j, image_id in enumerate(sample_ids.tolist()):
                group = GROUP_ID_TO_NAME[int(class_to_group[labels[j]])]
                row = {"split": split, "experiment": experiment,
                    "amount_apply_to": AMOUNT_APPLY_TO, "where_apply_to": WHERE_APPLY_TO,
                    "pool_id": image_id, "class_id": y[j],
                    "class_name": names[y[j]], "group": group,
                    "teacher_pred": int(logits_t[j].argmax()), "student_pred": int(logits_s[j].argmax()),
                    "student_correct": int(logits_s[j].argmax() == labels[j]),
                    "student_target_probability": float(logits_s[j].softmax(0)[labels[j]]),
                    "selected_class_ids": json.dumps(selected[j]),
                    "selected_classes": ", ".join(names[c] for c in selected[j])}
                row.update({k: float(v[j]) for k,v in numeric.items()})
                rows.append(row)
                key = (split, group, int(row["teacher_correct"]))
                if SAVE_CAM_FIGURES and figure_counts.get(key,0) < FIGURES_PER_GROUP_OUTCOME:
                    figure_counts[key] = figure_counts.get(key,0) + 1
                    figure_dir = os.path.join(save_dir, "cam_figures")
                    os.makedirs(figure_dir, exist_ok=True)
                    sample_maps = {k:v[j] for k,v in maps.items()
                                   if torch.is_tensor(v) and v.ndim > 0 and len(v) == len(labels)}
                    path = os.path.join(figure_dir, f"{split}_{group}_correct{key[-1]}_id{image_id}.png")
                    plot_cam_example(images[j], sample_maps, row, path)
                    map_examples.append({"pool_id": image_id,
                        **{k:v.detach().cpu().numpy() for k,v in sample_maps.items()}})
        frame = pd.DataFrame(rows)
        frame.to_csv(os.path.join(save_dir, f"probe_{split}_per_image.csv"), index=False)
        if map_examples:
            arrays = {name: np.stack([item[name] for item in map_examples])
                      for name in map_examples[0]}
            np.savez_compressed(os.path.join(save_dir, f"probe_{split}_map_examples.npz"), **arrays)
        return frame


    # Essential detached training diagnostics for the unchanged signed QW objective.
    # This fragment expects torch, np, pd, CE_WEIGHT, CAM_ALPHA and group mappings
    # in its notebook namespace. update_arrays can be tested using NumPy alone.


    class EssentialEpochAccumulator:
        """One packed device-to-host transfer per batch; image-weighted diagnostics.

        This observer never changes the loss, gradients, optimizer or random state.
        It preserves all 24 group/partition/stratum rows needed by the original
        outcome-control validation. Class-balanced mass means are intentionally not
        constructed: only image means, sums, coverage and actual coefficients are
        reported. q remains the spatial mean of the applied sign-factor map.
        """

        def __init__(self, class_to_group):
            if hasattr(class_to_group, "detach"):
                class_to_group = class_to_group.detach().cpu().numpy()
            self.class_to_group = np.asarray(class_to_group, dtype=np.int64)
            self.groups = ("all", "many", "medium", "few")
            self.strata = (("all", "all"), ("teacher_outcome", "correct"),
                ("teacher_outcome", "error"), ("guidance_branch", "raw"),
                ("guidance_branch", "difference"), ("guidance_branch", "coincidence"))
            self.identities = [(group, partition, stratum) for group in self.groups
                               for partition, stratum in self.strata]
            self.row_lookup = {identity: i for i, identity in enumerate(self.identities)}
            self.n = np.zeros(len(self.identities), dtype=np.int64)
            self.class_seen = np.zeros((len(self.identities), len(self.class_to_group)), dtype=bool)
            self.names = None
            self.sums = self.valid = None

        def update(self, logits, labels, diagnostics, maps, losses):
            # All observations below come from the ORIGINAL forward/loss call.
            # Reuse the existing per-image CE instead of evaluating CE a third time.
            with torch.no_grad():
                ce = losses["ce_per_image"].detach()
                guidance = losses["guidance_per_image"].detach()
                after = losses["guidance_after_q_per_image"].detach()
                raw = losses["raw_per_image"].detach()
                raw_after = losses["raw_after_q_per_image"].detach()
                q = losses["amount_per_image"].detach()
                positive = diagnostics["raw_pos_mass"].detach()
                negative = diagnostics["raw_neg_mass"].detach()
                values = {
                    "ce_unweighted": ce,
                    "guidance_before_q": guidance,
                    "guidance_after_q": after,
                    "raw_baseline_before_q": raw,
                    "raw_baseline_after_q": raw_after,
                    "ce_objective": CE_WEIGHT * ce,
                    "method_objective": CAM_ALPHA * after,
                    "total_objective": CE_WEIGHT * ce + CAM_ALPHA * after,
                    "amount_factor_base": diagnostics["amount_factor_base"].detach(),
                    "amount_selected_outcome": diagnostics["amount_selected_outcome"].detach(),
                    "amount_applied": diagnostics["amount_applied"].detach(),
                    "where_selected_outcome": diagnostics["where_selected_outcome"].detach(),
                    "where_applied": diagnostics["where_applied"].detach(),
                    "cam_scale_reference": diagnostics["cam_scale_reference"].detach(),
                    "cam_scale_applied": diagnostics["cam_scale_applied"].detach(),
                    "cam_global_coefficient": diagnostics["cam_global_coefficient"].detach(),
                    "cam_effective_coefficient": diagnostics["cam_effective_coefficient"].detach(),
                    "q": q,
                    "teacher_raw_target_cam_l1": diagnostics["raw_abs_mass"].detach(),
                    "q_times_teacher_raw_target_cam_l1": diagnostics["target_abs_mass_after_q"].detach(),
                    "teacher_raw_target_cam_positive_mass": positive,
                    "teacher_raw_target_cam_negative_mass": negative,
                    "teacher_raw_target_cam_signed_mean": positive - negative,
                    "teacher_rectified_target_cam_positive_mass": diagnostics["target_pos_mass_after_q"].detach(),
                    "teacher_rectified_target_cam_negative_mass": diagnostics["target_neg_mass_after_q"].detach(),
                    "attention_entropy": diagnostics["attention_entropy"].detach(),
                    "attention_effective_area_fraction": diagnostics["attention_effective_area_fraction"].detach(),
                }
                for sign in ("positive", "negative", "zero"):
                    values["amount_factor_" + sign + "_applied"] = diagnostics["amount_factor_" + sign + "_applied"].detach()
                    values["amount_" + sign + "_fraction"] = diagnostics["amount_" + sign + "_fraction"].detach()
                    base_name = "amount_factor_" + sign + "_base"
                    if base_name in diagnostics:
                        values[base_name] = diagnostics[base_name].detach()
                for name in ("spatial_weighted_raw_abs_mass", "q_spatial_weighted_raw_abs_mass"):
                    if name in diagnostics:
                        target = ("teacher_spatial_weighted_target_cam_l1" if name == "spatial_weighted_raw_abs_mass"
                                  else "q_times_teacher_spatial_weighted_target_cam_l1")
                        values[target] = diagnostics[name].detach()
                names = list(values)
                # Labels/masks travel in the same packed transfer as all values.
                # Label indices are exactly representable in float32 for these datasets.
                packed = torch.stack([values[name] for name in names] + [labels.detach(),
                    diagnostics["teacher_correct"].detach(), diagnostics["raw_branch"].detach(),
                    diagnostics["difference_branch"].detach(), diagnostics["coincidence_branch"].detach()], dim=1)
                packet = packed.detach().cpu().numpy().astype(np.float64, copy=False)
            width = len(names)
            self.update_arrays(packet[:, :width], names, packet[:, width].astype(np.int64),
                               packet[:, width + 1].astype(bool), packet[:, width + 2:width + 5].astype(bool))

        def update_arrays(self, matrix, names, y, teacher_correct, branches):
            """Observe NumPy [B,M] values and [B,3] raw/difference/coincidence flags."""
            values = np.asarray(matrix, dtype=np.float64)
            y = np.asarray(y, dtype=np.int64)
            teacher_correct = np.asarray(teacher_correct, dtype=bool)
            if isinstance(branches, dict):
                branches = np.column_stack([branches[name] for name in ("raw", "difference", "coincidence")])
            branches = np.asarray(branches, dtype=bool)
            names = list(names)
            if values.shape != (len(y), len(names)) or branches.shape != (len(y), 3) or teacher_correct.shape != y.shape:
                raise ValueError("Essential diagnostic packet has inconsistent dimensions.")
            if len(names) != len(set(names)) or (y < 0).any() or (y >= len(self.class_to_group)).any():
                raise ValueError("Invalid diagnostic names or true-class indices.")
            if not np.equal(branches.sum(1), 1).all():
                raise RuntimeError("Spatial-guidance branches do not partition the batch.")
            if not np.isfinite(values).all():
                raise FloatingPointError("Nonfinite essential per-image training diagnostic.")
            if self.names is None:
                self.names = names
                self.sums = np.zeros((len(self.identities), len(names)), dtype=np.float64)
                self.valid = np.zeros_like(self.sums, dtype=np.int64)
            elif names != self.names:
                raise ValueError("Essential diagnostic columns changed within an epoch.")
            group_ids = self.class_to_group[y]
            ids = globals().get("GROUP_NAME_TO_ID", {"many": 0, "medium": 1, "few": 2})
            group_masks = np.stack([np.ones(len(y), dtype=bool)] + [group_ids == ids[group] for group in self.groups[1:]])
            stratum_masks = np.stack([np.ones(len(y), dtype=bool), teacher_correct, ~teacher_correct,
                                     branches[:, 0], branches[:, 1], branches[:, 2]])
            masks = (group_masks[:, None, :] & stratum_masks[None, :, :]).reshape(len(self.identities), len(y))
            self.n += masks.sum(1, dtype=np.int64)
            self.sums += masks.astype(np.float64) @ values
            # The strict finite check preserves the original accumulator's behavior.
            # Store explicit finite counts nevertheless; empty strata retain n=0.
            self.valid += masks.astype(np.int64) @ np.isfinite(values).astype(np.int64)
            stratum_rows, sample_columns = np.nonzero(masks)
            self.class_seen[stratum_rows, y[sample_columns]] = True

        def frame(self, epoch):
            if self.names is None:
                raise ValueError("Cannot summarize an epoch without observed training images.")
            total_id = self.row_lookup[("all", "all", "all")]
            population_n = int(self.n[total_id])
            column_ids = {name: i for i, name in enumerate(self.names)}

            def ratio(numerator, denominator):
                return float(numerator) / float(denominator) if denominator > 0 else np.nan

            def sum_at(row, name):
                return float(self.sums[row, column_ids[name]])

            records = []
            for row_id, (group, partition, stratum) in enumerate(self.identities):
                group_id = self.row_lookup[(group, "all", "all")]
                row = {"epoch": int(epoch), "group": group, "partition": partition, "stratum": stratum,
                    "n_samples": int(self.n[row_id]), "n_classes_observed": int(self.class_seen[row_id].sum()),
                    "n_epoch_samples": population_n, "n_group_samples": int(self.n[group_id]),
                    "sample_fraction_of_epoch": ratio(self.n[row_id], population_n),
                    "sample_fraction_within_group": ratio(self.n[row_id], self.n[group_id])}
                for col, name in enumerate(self.names):
                    value, count = float(self.sums[row_id, col]), int(self.valid[row_id, col])
                    row[name + "_sum"] = value
                    row[name + "_valid_n"] = count
                    row[name + "_conditional_mean"] = ratio(value, count)
                    row[name + "_population_mean_contribution"] = ratio(value, population_n)
                for suffix in ("before_q", "after_q"):
                    name = "guidance_" + suffix
                    row["guidance_share_of_epoch_" + suffix] = ratio(sum_at(row_id, name), sum_at(total_id, name))
                    row["guidance_share_within_group_" + suffix] = ratio(sum_at(row_id, name), sum_at(group_id, name))
                for suffix, name in (("before_q", "teacher_raw_target_cam_l1"), ("after_q", "q_times_teacher_raw_target_cam_l1")):
                    row["raw_target_cam_l1_share_of_epoch_" + suffix] = ratio(sum_at(row_id, name), sum_at(total_id, name))
                row["method_to_ce_ratio_of_sums"] = ratio(sum_at(row_id, "method_objective"), sum_at(row_id, "ce_objective"))
                row["raw_mass_reweighting_ratio_of_sums"] = ratio(sum_at(row_id, "q_times_teacher_raw_target_cam_l1"), sum_at(row_id, "teacher_raw_target_cam_l1"))
                records.append(row)
            return pd.DataFrame(records)

        def group_frame(self, epoch):
            """Three group rows with historical image_* names; no class means."""
            balance = self.frame(epoch)
            selected = balance[(balance.partition == "all") & balance.group.ne("all")]
            aliases = {
                "raw_abs_mass": "teacher_raw_target_cam_l1", "raw_pos_mass": "teacher_raw_target_cam_positive_mass",
                "raw_neg_mass": "teacher_raw_target_cam_negative_mass", "target_abs_mass_before_q": "teacher_raw_target_cam_l1",
                "target_abs_mass_after_q": "q_times_teacher_raw_target_cam_l1",
                "target_pos_mass_after_q": "teacher_rectified_target_cam_positive_mass",
                "target_neg_mass_after_q": "teacher_rectified_target_cam_negative_mass",
                "amount_weight": "q", "amount_factor_base": "amount_factor_base",
                "attention_entropy": "attention_entropy", "attention_effective_area_fraction": "attention_effective_area_fraction",
                "where_applied": "where_applied", "amount_applied": "amount_applied",
                "amount_selected_outcome": "amount_selected_outcome", "where_selected_outcome": "where_selected_outcome",
                "cam_effective_coefficient": "cam_effective_coefficient", "cam_scale_reference": "cam_scale_reference",
                "cam_scale_applied": "cam_scale_applied", "cam_global_coefficient": "cam_global_coefficient",
            }
            for sign in ("positive", "negative", "zero"):
                for suffix in ("applied", "base"):
                    aliases["amount_factor_" + sign + "_" + suffix] = "amount_factor_" + sign + "_" + suffix
                aliases["amount_" + sign + "_fraction"] = "amount_" + sign + "_fraction"
            for name, original in (("spatial_weighted_raw_abs_mass", "teacher_spatial_weighted_target_cam_l1"),
                                   ("q_spatial_weighted_raw_abs_mass", "q_times_teacher_spatial_weighted_target_cam_l1")):
                aliases[name] = original
            ids = globals().get("GROUP_NAME_TO_ID", {"many": 0, "medium": 1, "few": 2})
            rows = []
            for source in selected.to_dict("records"):
                group = source["group"]
                row = {"epoch": int(epoch), "group": group, "n_samples": int(source["n_samples"]),
                    "n_classes": int((self.class_to_group == ids[group]).sum()),
                    "n_classes_observed": int(source["n_classes_observed"])}
                for name, original in aliases.items():
                    if original + "_conditional_mean" in source:
                        row["image_" + name] = source[original + "_conditional_mean"]
                        row[name + "_valid_n"] = source[original + "_valid_n"]
                row["raw_abs_mass_image_mean"] = row.get("image_raw_abs_mass", np.nan)
                row["q_times_raw_abs_mass_image_mean"] = row.get("image_target_abs_mass_after_q", np.nan)
                row["q_image_mean"] = row["q_applied_image_mean"] = row.get("image_amount_weight", np.nan)
                row["q_base_image_mean"] = row.get("image_amount_factor_base", np.nan)
                row["raw_signed_cam_image_mean"] = row.get("image_raw_pos_mass", np.nan) - row.get("image_raw_neg_mass", np.nan)
                row["guidance_sum"] = source["guidance_after_q_sum"]
                row["guidance_epoch_contribution_fraction"] = source["guidance_share_of_epoch_after_q"]
                row["guidance_epoch_contribution_to_mean"] = source["guidance_after_q_population_mean_contribution"]
                row["epoch_sample_fraction"] = source["sample_fraction_of_epoch"]
                row["cam_mass_definition"] = "mean_spatial_abs_of_signed_target_CAM"
                row["cam_mass_sample_scope"] = "observed_epoch_views; image_weighted_only"
                rows.append(row)
            return pd.DataFrame(rows)


    # Essential validation and reporting helpers for QW V8 MobileNetV2.
    # These definitions are inserted after the original notebook helpers; no work
    # executes at import. Training, loss, checkpoint and RNG policies are unchanged.


    @torch.no_grad()
    def evaluate_epoch_essential(model, loader, class_counts):
        """Original epoch evaluation minus probabilities/softmax and unused top-5.

        Forward passes, model.eval(), input order, device transfers and CE summation
        keep the original arithmetic and order. The checkpoint's fixed-class F1 is
        still computed by the caller exactly as before. No test data are evaluated.
        """
        model.eval()
        ys, ps = [], []
        ce_sum, n = 0., 0
        for batch in loader:
            images, labels = batch[:2]
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            logits, _ = model(images)
            ce_sum += F.cross_entropy(logits, labels, reduction="sum").item()
            n += labels.numel()
            ys.append(labels.cpu().numpy())
            ps.append(logits.argmax(1).cpu().numpy())
        y, pred = np.concatenate(ys), np.concatenate(ps)
        metrics = {
            "acc": float((pred == y).mean() * 100),
            "f1_macro": float(f1_score(y, pred, average="macro", zero_division=0)),
            "f1_weighted": float(f1_score(y, pred, average="weighted", zero_division=0)),
            "balanced_acc": float(balanced_accuracy_score(y, pred)),
        }
        metrics.update(compute_group_metrics(y, pred, get_shot_groups(class_counts)))
        metrics["ce_loss"] = ce_sum / n
        return metrics, y, pred, None


    def add_group_top5(metrics, labels, probs, class_counts):
        """Add final group top-5 with exactly sklearn's ranking/tie convention.

        Select images by their TRUE class group, retaining every class probability
        column in the ranking. Call only for final evaluations; not every epoch.
        The dictionary is updated in place and returned for convenient assignment.
        """
        labels, probs = np.asarray(labels), np.asarray(probs)
        if labels.ndim != 1 or probs.ndim != 2 or len(labels) != len(probs):
            raise ValueError("Final labels/probabilities have incompatible shapes.")
        if probs.shape[1] != NUM_CLASSES:
            raise ValueError("Final probability columns differ from NUM_CLASSES.")
        full_class_ids = np.arange(NUM_CLASSES)
        for group, class_ids in get_shot_groups(class_counts).items():
            mask = np.isin(labels, class_ids)
            metrics[group + "_shot_top5_acc"] = float(top_k_accuracy_score(
                labels[mask], probs[mask], k=5, labels=full_class_ids) * 100) if mask.any() else np.nan
        return metrics


    def _essential_summary_metric_names(results):
        """Curated scalar metrics only; complete_results retains all scalar fields."""
        suffixes = ["acc", "top5_acc", "f1_macro", "balanced_acc"]
        suffixes += [group + "_shot_" + metric for group in ("many", "medium", "few")
                     for metric in ("acc", "top5_acc", "f1")]
        names = [prefix + "_" + split + "_" + metric
                 for prefix in ("student", "teacher", "gain_vs_teacher")
                 for split in ("val", "test") for metric in suffixes]
        names += ["best_epoch", "epochs_ran", "best_val_f1_macro", "mean_train_epoch_seconds",
                  "total_seconds_including_final_diagnostics", "train_method_to_ce_ratio_at_best",
                  "train_guidance_before_q_at_best", "train_guidance_after_q_at_best"]
        for group in ("all", "many", "medium", "few"):
            for sign in ("positive", "negative"):
                names.append("amount_factor_" + sign + "_base_" + group)
                names.append("amount_factor_" + sign + "_applied_" + group)
                for outcome in ("correct", "error"):
                    names.append("amount_factor_" + sign + "_applied_teacher_" + outcome + "_" + group)
            names.append("amount_applied_fraction_" + group)
            names.append("where_applied_fraction_" + group)
            for diagnostic in ("raw_pos_mass", "raw_neg_mass", "raw_abs_mass",
                    "target_pos_mass_after_q", "target_neg_mass_after_q", "target_abs_mass_after_q",
                    "amount_factor_positive_applied", "amount_factor_negative_applied",
                    "attention_entropy", "attention_effective_area_fraction", "where_applied"):
                names.append(group + "_image_" + diagnostic + "_at_best")
        return [name for name in dict.fromkeys(names) if name in results]


    def _essential_scalar(value):
        return value is None or np.isscalar(value) or isinstance(value, Path)


    def export_summary_essential(results):
        """Write only two compact CSVs; do not reload any run diagnostic files.

        Complete results keep every scalar result field. The long summary gives
        equal seed weights, sample std, median, linear Q1/Q3 and IQR. Exact metadata
        validation remains delegated to the original completed-run validator.
        """
        validate_v8_mobilenet_configuration()
        if results.empty:
            return
        results = results.drop_duplicates("run_id", keep="last").sort_values(
            ["experiment", "student_init_mode", "seed"]).reset_index(drop=True)
        for record in results.to_dict("records"):
            validate_completed_result(record, "essential in-memory consolidation")
        columns = [name for name in results if all(_essential_scalar(v) for v in results[name])]
        complete = results.loc[:, columns].copy()
        keys = [name for name in ("experiment", "student_init_mode", "config_hash") if name in complete]
        if complete.duplicated(keys + ["seed"]).any():
            raise ValueError("More than one execution per seed in an essential summary.")
        metrics = _essential_summary_metric_names(complete)
        rows = []
        for values, part in complete.groupby(keys, dropna=False, sort=True):
            if len(keys) == 1:
                values = (values,)
            identity = dict(zip(keys, values))
            for metric in metrics:
                data = pd.to_numeric(part[metric], errors="coerce").replace([np.inf, -np.inf], np.nan)
                valid = data.notna()
                numbers = data[valid].to_numpy(dtype=float)
                n = len(numbers)
                q1, median, q3 = np.quantile(numbers, [0.25, 0.5, 0.75], method="linear") if n else (np.nan,) * 3
                rows.append({**identity, "metric": metric, "n": n,
                    "seeds": ",".join(str(int(s)) for s in sorted(part.loc[valid, "seed"].unique())),
                    "mean": float(numbers.mean()) if n else np.nan,
                    "std": float(numbers.std(ddof=1)) if n > 1 else np.nan,
                    "median": median, "q1": q1, "q3": q3, "iqr": q3-q1})
        summary = pd.DataFrame(rows, columns=keys + ["metric", "n", "seeds", "mean", "std", "median", "q1", "q3", "iqr"])
        atomic_csv(Path(RUNS_ROOT, "complete_results.csv"), complete)
        atomic_csv(Path(RUNS_ROOT, "summary_metrics.csv"), summary)
        print("Essential per-seed results:", Path(RUNS_ROOT, "complete_results.csv"))
        print("Essential seed summary:", Path(RUNS_ROOT, "summary_metrics.csv"))


In [ ]:
assert_active_profile()

if EXPERIMENT_PROFILE == "resnet32_cifar100_lt":
    parameter_table_df = pd.DataFrame([
        ("AMOUNT_APPLY_TO", AMOUNT_APPLY_TO, "all; correct; error", "seleciona outcomes para aplicar q_base em QW1/QW3; fora do filtro q=1; não filtra a escala global"),
        ("WHERE_APPLY_TO", WHERE_APPLY_TO, "all; correct; error", "seleciona outcomes para ONDE em QW2/QW3, independentemente de QUANTO; fora do filtro a MSE permanece raw"),
        ("WHERE_TOP_K", WHERE_TOP_K, "2; 5; 10", "classe verdadeira y + K−1 rivais com maiores logits do professor, excluindo y"),
        ("WHERE_CORRECT_MODE", WHERE_CORRECT_MODE, "raw; difference; coincidence", "modo nos acertos, se WHERE_APPLY_TO os seleciona e o grupo é elegível; raw preserva MSE uniforme"),
        ("WHERE_ERROR_MODE", WHERE_ERROR_MODE, "raw; difference; coincidence", "modo nos erros, se WHERE_APPLY_TO os seleciona e o grupo é elegível; independente de AMOUNT_APPLY_TO"),
        ("WHERE_CORRECT_MASK_MODE", WHERE_CORRECT_MASK_MODE, "aligned; uniform; shuffled", "controle dos pesos espaciais nos acertos; raw ignora esta opção"),
        ("WHERE_ERROR_MASK_MODE", WHERE_ERROR_MASK_MODE, "aligned; uniform; shuffled", "controle dos pesos espaciais nos erros; raw ignora esta opção"),
        ("WHERE_BETA", WHERE_BETA, "0; 0.5; 1", "Beta Onde: mistura uniforme/evidência, comum aos dois desfechos; beta=0 dá peso espacial 1"),
        ("WHERE_GROUPS", str(WHERE_GROUPS), "medium/few inicialmente", "fora destes grupos a MSE é raw; many fica raw na configuração inicial"),
        ("WHERE_NORM_EPS", WHERE_NORM_EPS, "manter fixo", "CAM de massa quase nula vira zero só na evidência; evidência total quase nula usa pesos uniformes"),
        ("CE_WEIGHT", CE_WEIGHT, "manter fixo", "peso da CE global; incluído na comparação das losses e dos gradientes"),
        ("CAM_ALPHA", CAM_ALPHA, "manter o valor da configuração", "força global: coeficiente alpha / s_T antes do peso espacial; r_aplicado, selecionado pelo sinal do professor, escala o alvo; alpha=0 deixa somente CE"),
        ("AMOUNT_POWER", AMOUNT_POWER, "manter o valor da configuração", "Beta Quanto: expoente comum das razões positivas/negativas por grupo; p=0 dá fatores unitários; calibração conjunta fixa antes do filtro"),
        ("CAM_SCALE_EPS", CAM_SCALE_EPS, "manter fixo", "s_T=max(raw_cam_mean_sq, eps) é aplicado globalmente em QW0/QW1/QW2/QW3, em todos os outcomes"),
        ("DIAGNOSTIC_MAX_PER_CLASS", DIAGNOSTIC_MAX_PER_CLASS, "8 ou None", "probes train/val fixos, sem aumento; não estimam q nem escolhem o checkpoint"),
        ("GRAD_DIAG_MAX_BATCHES", GRAD_DIAG_MAX_BATCHES, "1 ou 0", "primeiros batches da época para gradientes nas últimas features; 0 desliga o diagnóstico"),
    ], columns=["parameter", "initial", "small_ablation", "meaning"])

    spatial_formula_df = pd.DataFrame([
        ("CAMs raw", "T_y(u), S_y(u)", "CAMs assinadas da classe verdadeira; a MSE compara S_y com r_i(u)*T_y"),
        ("Normalização só para evidência", "t_c(u) = T_c(u) / mean_u |T_c(u)|", "uma escala por classe; se massa ≤ eps, t_c=0; preservar ambos os sinais"),
        ("difference", "E_diff(u) = mean_r |t_y(u) − t_r(u)|", "média das diferenças absolutas por rival; não |alvo − média dos rivais|"),
        ("coincidence", "E_coinc(u) = mean_r [min(|t_y(u)|, |t_r(u)|) × 1(t_y(u)t_r(u)>0)]", "overlap ativo com mesmo sinal; positivo/positivo e negativo/negativo contam; zero/zero não conta"),
        ("Atenção e peso", "p(u) = (1−beta)/N + beta × E(u)/sum_u E(u); w(u)=N p(u)", "sum p=1 e mean w=1; evidência total ≤ eps usa w=1"),
        ("Controles espaciais", "aligned: w; uniform: 1; shuffled: permutação de w", "shuffled preserva a distribuição dos pesos e usa RNG local determinístico por seed/época/id"),
        ("Seletores independentes", "a_i=1(outcome_i em AMOUNT_APPLY_TO); h_i=1(outcome_i em WHERE_APPLY_TO)", "all seleciona todos, correct seleciona acertos do professor, error seleciona erros; filtros definidos na configuração principal"),
        ("Fator aplicado", "r_i(u)=q_G+ se T_y(u)>0, q_G− se T_y(u)<0 e q_G0 em zero; fora da seleção r_i(u)=1", "fatores por sinal fixos e conjuntamente calibrados no treino; aplicar o seletor sem segunda normalização"),
        ("Escala global compartilhada", "s_T=max(raw_cam_mean_sq, CAM_SCALE_EPS)", "denominador global de todos os quatro métodos e todos os outcomes; AMOUNT_APPLY_TO filtra somente q"),
        ("Erro por imagem", "D_i = mean_u [w_i(u)(S_y(u)−r_i(u)*T_y(u))²] / s_T", "fora de WHERE_APPLY_TO, dos grupos elegíveis ou com modo raw, w=1 e a MSE raw continua ativa"),
        ("Coeficientes antes do peso espacial", "coef_global=coef_i=alpha/s_T; alvo_i=r_i(u)*T_y", "r_i(u) varia por grupo, sinal do professor e seleção; atua somente na referência dentro da MSE"),
        ("Loss total", "L = CE_WEIGHT × mean_i CE_i + CAM_ALPHA × mean_i D_i", "QW1/QW3 habilitam q condicionado; QW2/QW3 habilitam ONDE condicionado; os dois filtros são independentes"),
    ], columns=["quantity", "formula", "interpretation"])

    diagnostic_guide_df = pd.DataFrame([
        ("train_ce / train_ce_weighted", "média da CE crua e CE_WEIGHT × média da CE; a segunda entra na loss total"),
        ("train_method_weighted", "alpha × mean(D_retificado); D_retificado usa S_y−r_aplicado*T_y e inclui s_T compartilhado"),
        ("train_method_to_ce_ratio", "razão das somas ponderadas da época: método / CE; não média de razões por batch"),
        ("train_raw_weighted / train_difference_weighted / train_coincidence_weighted", "alpha × mean(D_retificado × máscara do ramo), com denominador de toda a época; os três somam o termo do método"),
        ("train_correct_weighted / train_error_weighted", "contribuições do método por acerto/erro do professor à média de toda a época; somam o termo do método"),
        ("train_loss_balance.csv", "todas as imagens da época: grupo/estrato, contagens, somas, médias condicionais e contribuições à média da população"),
        ("guidance_before_q / guidance_after_q", "por imagem: MSE contra T_y e MSE contra r_aplicado*T_y, ambas com os mesmos pesos espaciais e s_T"),
        ("amount_factor_base", "média estática dos fatores por sinal no grupo, calibrada no treino determinístico; disponível mesmo quando o método ou outcome não aplica QUANTO"),
        ("amount_selected_outcome / amount_applied", "seleção por AMOUNT_APPLY_TO / seleção e use_amount; amount_applied pode ser 1 mesmo se q_base=1, portanto não prova alteração numérica do fator"),
        ("amount_weight", "média espacial do mapa r aplicado por sinal; fora de amount_applied vale exatamente um; sem segunda normalização"),
        ("E[q_base] / E[q_aplicado]", "E_sample[q_base]=1 na população determinística usada para a normalização original; após selecionar outcomes, E[q_aplicado] não precisa ser 1"),
        ("where_selected_outcome", "seleção por WHERE_APPLY_TO, independente do filtro de QUANTO, da habilitação de ONDE e da elegibilidade de grupo"),
        ("cam_scale_reference / cam_scale_applied", "ambos são s_T=max(raw_cam_mean_sq, CAM_SCALE_EPS) em todos os métodos e outcomes; nenhum dos filtros desliga a escala global"),
        ("cam_global_coefficient / cam_effective_coefficient", "ambos alpha/s_T, antes do peso espacial w; r_aplicado, selecionado pelo sinal do professor, escala o alvo dentro da MSE; logs por imagem/classe/grupo/outcome e gradientes amostrados"),
        ("raw_branch / difference_branch / coincidence_branch", "modos selecionados efetivamente, exclusivos e somando 1; difference/coincidence podem produzir w=1 com beta=0, uniform ou fallback"),
        ("where_eligible / where_applied", "pertence a WHERE_GROUPS / modo não raw selecionado com use_where, grupo elegível e outcome selecionado; não provam pesos não uniformes"),
        ("weight_min / max / mean / max_abs_deviation", "pesos reais usados na MSE; mean=1, max_abs_deviation=0 quando todos os pesos são 1"),
        ("attention_entropy / attention_effective_area_fraction", "concentração espacial da atenção efetivamente aplicada; não valida a localização semântica"),
        ("evidence_zero_fallback", "evidência total quase nula no modo selecionado; a MSE usa w=1"),
        ("attention_mass_target_positive / negative", "atenção onde o CAM alvo raw é positivo/negativo; o erro quadrático preserva os dois sinais"),
        ("rival_many_fraction / medium_fraction / few_fraction", "composição do top K−1 do professor; a normalização de amplitude não muda este ranking"),
        ("teacher_group_cam_statistics.csv", "estatísticas estáticas originais de massa L1 e q_base por many/medium/few; o filtro de outcome não reestima nem renormaliza estes valores"),
        ("raw_abs_mass_image_mean / class_mean", "massa L1 raw nas views observadas da época: média entre imagens versus média igual entre classes observadas"),
        ("q_base_image_mean / q_base_class_mean", "médias do q_base fixo observado no estrato; pesos por imagem e iguais entre classes, respectivamente"),
        ("q_image_mean / q_class_mean; q_applied_image_mean / class_mean", "q efetivamente aplicado, com aliases explícitos q_applied; q=1 em QW0/QW2 e fora de AMOUNT_APPLY_TO; não confundir com q_base"),
        ("q_times_raw_abs_mass_image_mean / class_mean", "média de massa L1 da referência retificada por sinal, sem alpha/s_T; magnitude da CAM alvo retificada; não é uma contribuição à loss"),
        ("raw_abs_mass_sum / q_times_raw_abs_mass_sum", "somas sobre as imagens do estrato; *_epoch_contribution_to_mean divide pelo N total da época"),
        ("*_epoch_contribution_fraction", "participação do grupo/estrato na soma total da época, incluindo ambos os outcomes; somam 1 em partições completas"),
        ("spatial_weighted_raw_abs_mass / q_spatial_weighted_raw_abs_mass", "diagnósticos mean(w|T_y|) e mean(w|T_ret|), sem alpha/s_T; descrevem as massas original e retificada ponderadas espacialmente; não são loss ou gradiente"),
        ("raw_signed_cam_image_mean / class_mean", "média assinada: massa positiva menos magnitude negativa; distinta da massa L1 usada em QUANTO"),
        ("feature_gradient_diagnostics.csv", "CE_WEIGHT×CE versus alpha×mean(D_retificado), com s_T; últimas features e denominador full batch. Fatores, seletores, flags e coeficientes são médias finitas do subconjunto de cada linha"),
        ("gradient norms by group", "normas não são shares aditivas de loss ou parâmetros; em um mesmo batch, grupos disjuntos particionam a norma ao quadrado do tensor de features"),
        ("probe train / val", "mesmos cálculos diagnósticos em imagens fixas sem aumento; nenhuma atualização; teste oficial somente classificação após selecionar o checkpoint"),
        ("CAM figures / NPZ", "CAMs alvo originais, referência retificada por sinal, mapa de fatores, evidências de diferença/coincidência, atenção e peso da MSE; mapas assinados de professor/aluno compartilham a escala de cor"),
        ("classification_outcome_diagnostics", "recuperação de erros e regressão de acertos do professor com contagens e médias entre classes observadas; os outcomes podem conter classes diferentes"),
        ("historical group F1", "filtro pelo grupo verdadeiro e métricas históricas preservados literalmente"),
        ("raw/raw, beta=0 ou uniform nos ramos não raw", "controle de identidade da loss espacial: QW3 recupera QW1; QW2 recupera QW0, mantendo os demais parâmetros e a mesma inicialização"),
        ("AMOUNT_POWER=0", "q_base=1 e q_aplicado=1 em todos os outcomes; o denominador s_T continua global nos quatro métodos"),
        ("fora de WHERE_APPLY_TO ou modo raw", "permanece CE global + MSE raw com q_aplicado e s_T; não vira CE somente; AMOUNT_APPLY_TO pode selecionar outro outcome independentemente"),
        ("comparar aligned / uniform / shuffled", "fixar o modo difference ou coincidence no outcome estudado e variar somente sua máscara; raw ignora a máscara"),
        ("*_valid_n / *_classes_valid_n", "número de valores finitos por imagem / número de médias de classe válidas; ausência permanece NaN"),
        ("paired_deltas", "comparações por seed/inicialização com hashes; médias e DP não são testes de significância"),
    ], columns=["diagnostic", "interpretation"])
    display(parameter_table_df)
    display(spatial_formula_df)
    display(diagnostic_guide_df)
else:  # mobilenetv2_imagenet_lt
    # Active controls and essential diagnostics only.
    parameter_table_df = pd.DataFrame([
        ("SEEDS", str(SEEDS), "mesmas seeds do anexo"),
        ("EXPERIMENTS_TO_RUN", str(EXPERIMENTS_TO_RUN), "mesmas variantes selecionadas no anexo"),
        ("CAM_ALPHA", CAM_ALPHA, "peso original da loss CAM"),
        ("AMOUNT_POWER", AMOUNT_POWER, "potência original da compensação por sinal/grupo"),
        ("AMOUNT_APPLY_TO", AMOUNT_APPLY_TO, "filtro de acertos/erros original"),
        ("WHERE_APPLY_TO", WHERE_APPLY_TO, "filtro espacial original"),
        ("WHERE_CORRECT_MODE", WHERE_CORRECT_MODE, "raw/difference/coincidence"),
        ("WHERE_ERROR_MODE", WHERE_ERROR_MODE, "raw/difference/coincidence"),
        ("WHERE_TOP_K", WHERE_TOP_K, "alvo e K-1 concorrentes"),
        ("ESSENTIAL_HISTORY_WRITE_EVERY", ESSENTIAL_HISTORY_WRITE_EVERY, "somente frequência de gravação do histórico"),
        ("SAVE_RECOVERY_CHECKPOINTS", SAVE_RECOVERY_CHECKPOINTS, "somente checkpoint adicional de recuperação"),
    ], columns=["parameter", "value", "interpretation"])
    diagnostic_guide_df = pd.DataFrame([
        ("teacher_group_cam_statistics.csv", "q_g positivo/negativo fixos e estatísticas de TRAIN determinístico; calibração original por classe/grupo"),
        ("train_loss_balance_best.csv", "massas, fatores aplicados e razões de perdas por imagem; partições sobrepostas não devem ser somadas"),
        ("train_diagnostics_per_group.csv", "melhor época: entropia/área efetiva da atenção aplicada, não entropia da CAM"),
        ("where_applied", "acionamento do ramo; beta=0/máscara uniforme/fallback podem manter os pesos uniformes"),
        ("result.json", "métricas finais, top-5 por grupo e marcador de conclusão"),
    ], columns=["diagnostic", "interpretation"])
    display(parameter_table_df)
    display(diagnostic_guide_df)


## Verificações internas

O teste matemático original em CPU é comum aos anexos e mantém a ordem de execução. No perfil MobileNet, a verificação adicional da execução essencial ocorre depois de definir seu observador e antes dos experimentos.

Os testes internos restauram os estados aleatórios e as configurações temporárias. Eles não executam um treinamento completo nem comprovam sozinhos a compatibilidade histórica dos checkpoints e dados fornecidos.


In [ ]:
assert_active_profile()

# Código compartilhado, idêntico nos dois anexos.
def run_core_self_tests():
    """Small CPU behavioral checks; preserve configuration and every global RNG."""
    import random as _test_random

    settings = {"DEVICE": torch.device("cpu"), "NUM_CLASSES": 4, "WHERE_TOP_K": 3,
        "WHERE_GROUPS": ("medium", "few"), "WHERE_BETA": .5,
        "WHERE_CORRECT_MODE": "difference", "WHERE_ERROR_MODE": "coincidence",
        "WHERE_CORRECT_MASK_MODE": "aligned", "WHERE_ERROR_MASK_MODE": "aligned",
        "AMOUNT_APPLY_TO": "all", "WHERE_APPLY_TO": "all",
        "WHERE_NORM_EPS": 1e-8, "CAM_SCALE_EPS": 1e-8,
        "CE_WEIGHT": 1.7, "CAM_ALPHA": .3}
    absent = object()
    original = {key: globals().get(key, absent) for key in settings}
    np_state, python_state = np.random.get_state(), _test_random.getstate()
    checks = []

    def check(name):
        checks.append(name)

    def assert_equal_gradients(actual, expected, variables, exact=False):
        first = torch.autograd.grad(actual, variables, retain_graph=True)
        second = torch.autograd.grad(expected, variables, retain_graph=True)
        for aa, bb in zip(first, second):
            torch.testing.assert_close(aa, bb, rtol=0 if exact else 2e-6,
                                       atol=0 if exact else 2e-7)

    try:
        globals().update(settings)
        cuda_devices = list(range(torch.cuda.device_count())) if torch.cuda.is_available() else []
        with torch.random.fork_rng(devices=cuda_devices):
            torch.manual_seed(80531)
            # Hand-worked maps already have mean absolute value one per class.
            known = torch.tensor([[[[2., -2., 0., 0.]], [[1., -1., 2., 0.]],
                                    [[3., 1., 0., 0.]]]])
            evidence = signed_spatial_evidence(known)
            torch.testing.assert_close(evidence["difference"], torch.tensor([[[1., 2., 1., 0.]]]), rtol=0, atol=0)
            torch.testing.assert_close(evidence["coincidence"], torch.tensor([[[1.5, .5, 0., 0.]]]), rtol=0, atol=0)
            assert evidence["coincidence"][0, 0, 1] > 0, "Matching negative signs lost their coincidence"
            assert evidence["coincidence"][0, 0, 2:].eq(0).all(), "Zeros introduced false coincidence"
            check("hand-calculated difference/coincidence include negative evidence, opposite signs and zero/zero")

            cancellation = torch.tensor([[[[0.]], [[1.]], [[-1.]]]])
            cancel = signed_spatial_evidence(cancellation)
            torch.testing.assert_close(cancel["difference"], torch.ones(1, 1, 1), rtol=0, atol=0)
            assert (cancellation[:, 0] - cancellation[:, 1:].mean(1)).abs().eq(0).all()
            assert cancel["coincidence"].eq(0).all()
            zeros = signed_spatial_evidence(torch.zeros_like(known))
            assert all(value.eq(0).all() for value in zeros.values())
            check("difference averages per-rival absolute differences without cancellation")

            class TestTeacher(nn.Module):
                def __init__(self):
                    super().__init__()
                    self.fc = nn.Linear(4, 4)
                    with torch.no_grad():
                        self.fc.weight.copy_(torch.eye(4))
                        self.fc.bias.copy_(torch.tensor([.2, -.2, .1, -.1]))
                    for parameter in self.parameters():
                        parameter.requires_grad_(False)

                def forward(self, image):
                    return self.fc(image.mean((-2, -1))), {"feats": [image]}

            teacher = TestTeacher().eval()
            means = torch.tensor([[3., .1, .2, .4], [.1, 3., .2, .4], [.1, .2, 3., .4],
                                  [2., 3., .1, 1.], [3., .1, 2., 1.], [.1, 3., 1., 2.]])
            variation = torch.randn(6, 4, 2, 3)
            variation -= variation.mean((-2, -1), keepdim=True)
            images = means[:, :, None, None] + .6 * variation
            labels = torch.tensor([0, 1, 2, 2, 1, 0])
            class_to_group = torch.tensor([0, 1, 2, 2])
            sample_ids = torch.tensor([13, 21, 34, 55, 89, 144])
            features = (images + .17 * torch.randn_like(images)).requires_grad_()
            student_fc = nn.Linear(4, 4)
            logits_s = student_fc(features.mean((-2, -1)))
            stats = {"raw_cam_mean_sq": 2.3, "amount_factors": np.array([.8, 1.1, 1.3])}
            for sign in ("positive","negative","zero"):
                stats["amount_factors_"+sign]=stats["amount_factors"].copy()
            q = torch.tensor([.8, 1.1, 1.3, 1.3, 1.1, .8])
            teacher_before = {key: value.clone() for key, value in teacher.state_dict().items()}
            with torch.no_grad():
                logits_t, _ = teacher(images)
            correct = logits_t.argmax(1).eq(labels)
            eligible = class_to_group[labels].ne(0)
            assert correct.tolist() == [True, True, True, False, False, False]
            assert not logits_t.topk(WHERE_TOP_K, 1).indices[3].eq(labels[3]).any()

            # Independent CAM/evidence arithmetic: no production CAM,
            # normalization or spatial-attention helper is reused here.
            excluded = logits_t.clone()
            excluded.scatter_(1, labels[:, None], -torch.inf)
            rivals = excluded.topk(WHERE_TOP_K - 1, 1).indices
            selected = torch.cat((labels[:, None], rivals), 1)
            teacher_cams = (images[:, None] * teacher.fc.weight[selected, :, None, None]).sum(2)
            raw_t = (images * teacher.fc.weight[labels, :, None, None]).sum(1)
            teacher_cams[:, 0] = raw_t
            cam_s = (features * student_fc.weight[labels, :, None, None]).sum(1)
            scale = teacher_cams.abs().mean((-2, -1), keepdim=True)
            normalized = torch.where(scale > WHERE_NORM_EPS,
                teacher_cams / scale.clamp_min(WHERE_NORM_EPS), torch.zeros_like(teacher_cams))
            target, rivals_maps = normalized[:, :1], normalized[:, 1:]
            oracle_evidence = {
                "difference": (target - rivals_maps).abs().mean(1),
                "coincidence": torch.where(target * rivals_maps > 0,
                    torch.minimum(target.abs(), rivals_maps.abs()), torch.zeros_like(rivals_maps)).mean(1)}
            raw_per_image = (cam_s - raw_t).square().mean((-2, -1)) / stats["raw_cam_mean_sq"]
            variables = (features, student_fc.weight, student_fc.bias)

            def call(method="QW3_AMOUNT_WHERE", subset=None):
                if subset is None:
                    return qw_cam_loss(method, logits_s, features, student_fc, teacher, labels,
                        images, class_to_group, stats, sample_ids, 42, 7)
                return qw_cam_loss(method, logits_s[subset], features[subset], student_fc, teacher,
                    labels[subset], images[subset], class_to_group, stats, sample_ids[subset], 42, 7)

            for correct_mode in ("raw", "difference", "coincidence"):
                for error_mode in ("raw", "difference", "coincidence"):
                    globals().update(WHERE_CORRECT_MODE=correct_mode, WHERE_ERROR_MODE=error_mode)
                    weights = torch.ones_like(raw_t)
                    expected_branches = {mode: torch.zeros(6, dtype=torch.bool)
                                         for mode in ("raw", "difference", "coincidence")}
                    expected_branches["raw"] = ~eligible
                    for outcome, mode in ((correct, correct_mode), (~correct, error_mode)):
                        mask = eligible & outcome
                        expected_branches[mode] |= mask
                        if mode == "raw":
                            continue
                        value = oracle_evidence[mode]
                        total = value.sum((-2, -1), keepdim=True)
                        pixels = raw_t[0].numel()
                        uniform = torch.full_like(value, 1. / pixels)
                        attention = (1. - WHERE_BETA) * uniform + WHERE_BETA * torch.where(
                            total > WHERE_NORM_EPS, value / total.clamp_min(WHERE_NORM_EPS), uniform)
                        weights[mask] = (pixels * attention)[mask]
                    expected_per_image = ((cam_s - raw_t).square() * weights).mean((-2, -1)) / stats["raw_cam_mean_sq"]
                    expected_guidance = (((cam_s-q[:,None,None]*raw_t).square()*weights).mean((-2,-1))/stats["raw_cam_mean_sq"]).mean()
                    expected_total = CE_WEIGHT * F.cross_entropy(logits_s, labels) + CAM_ALPHA * expected_guidance
                    total, diag, maps, losses = call()
                    torch.testing.assert_close(maps["selected_ids"], selected, rtol=0, atol=0)
                    torch.testing.assert_close(maps["teacher_raw"], raw_t, rtol=0, atol=0)
                    torch.testing.assert_close(maps["applied_spatial_weight"], weights, rtol=2e-6, atol=2e-7)
                    torch.testing.assert_close(weights.mean((-2, -1)), torch.ones(6), rtol=2e-6, atol=2e-7)
                    torch.testing.assert_close(losses["guidance_per_image"], expected_per_image, rtol=2e-6, atol=2e-7)
                    torch.testing.assert_close(losses["guidance"], expected_guidance, rtol=2e-6, atol=2e-7)
                    torch.testing.assert_close(total, expected_total, rtol=2e-6, atol=2e-7)
                    torch.testing.assert_close(losses["amount_per_image"], q, rtol=0, atol=0)
                    partition = sum(diag[mode + "_branch"] for mode in expected_branches)
                    torch.testing.assert_close(partition, torch.ones(6), rtol=0, atol=0)
                    for mode, mask in expected_branches.items():
                        assert torch.equal(diag[mode + "_branch"].bool(), mask)
                    assert_equal_gradients(total, expected_total, variables)
                    assert all(not value.requires_grad for value in maps.values() if torch.is_tensor(value))
            check("all nine correct/error mode pairs match independent values, gradients and exclusive routing")
            check("teacher errors retain the true-label target before q scaling, including labels outside natural Top-K")
            check("spatial weights average one, many remains raw, and original q is unchanged")

            # beta=0 and uniform are arithmetic identities even if the branch
            # remains named difference/coincidence for diagnostic purposes.
            baseline_total = CE_WEIGHT * F.cross_entropy(logits_s, labels) + CAM_ALPHA * ((cam_s-q[:,None,None]*raw_t).square().mean((-2,-1))/stats["raw_cam_mean_sq"]).mean()
            for correct_mode in ("raw", "difference", "coincidence"):
                for error_mode in ("raw", "difference", "coincidence"):
                    globals().update(WHERE_CORRECT_MODE=correct_mode, WHERE_ERROR_MODE=error_mode)
                    for beta, correct_mask, error_mask in ((0., "aligned", "aligned"), (.5, "uniform", "uniform")):
                        globals().update(WHERE_BETA=beta, WHERE_CORRECT_MASK_MODE=correct_mask,
                                         WHERE_ERROR_MASK_MODE=error_mask)
                        total, _, maps, losses = call()
                        torch.testing.assert_close(maps["applied_spatial_weight"], torch.ones_like(raw_t), rtol=0, atol=0)
                        torch.testing.assert_close(losses["guidance_per_image"], raw_per_image, rtol=0, atol=0)
                        torch.testing.assert_close(total, baseline_total, rtol=0, atol=0)
                        assert_equal_gradients(total, baseline_total, variables, exact=True)
            check("beta zero and uniform controls exactly recover raw values and gradients for all nine pairs")

            globals().update(WHERE_BETA=.5, WHERE_CORRECT_MODE="difference", WHERE_ERROR_MODE="coincidence",
                WHERE_CORRECT_MASK_MODE="aligned", WHERE_ERROR_MASK_MODE="aligned")
            _, _, aligned, _ = call()
            globals()["WHERE_CORRECT_MASK_MODE"] = "uniform"
            _, _, correct_uniform, _ = call()
            torch.testing.assert_close(correct_uniform["applied_spatial_weight"][~correct],
                aligned["applied_spatial_weight"][~correct], rtol=0, atol=0)
            globals().update(WHERE_CORRECT_MASK_MODE="aligned", WHERE_ERROR_MASK_MODE="uniform")
            _, _, error_uniform, _ = call()
            torch.testing.assert_close(error_uniform["applied_spatial_weight"][correct],
                aligned["applied_spatial_weight"][correct], rtol=0, atol=0)
            check("correct and error spatial controls operate independently")

            globals().update(WHERE_CORRECT_MASK_MODE="shuffled", WHERE_ERROR_MASK_MODE="shuffled")
            torch_state = torch.get_rng_state().clone()
            _, _, first, _ = call()
            _, _, second, _ = call()
            assert torch.equal(torch_state, torch.get_rng_state()), "Shuffling consumed global Torch RNG"
            torch.testing.assert_close(first["applied_spatial_weight"], second["applied_spatial_weight"], rtol=0, atol=0)
            torch.testing.assert_close(first["applied_spatial_weight"].flatten(1).sort(1).values,
                aligned["applied_spatial_weight"].flatten(1).sort(1).values, rtol=0, atol=0)
            for index in range(6):
                _, _, single, _ = call(subset=slice(index, index + 1))
                torch.testing.assert_close(single["applied_spatial_weight"][0],
                    first["applied_spatial_weight"][index], rtol=0, atol=0)
            check("local shuffles preserve RNG, map values, determinism and per-image batch independence")

            # Zero evidence must give finite uniform weights, without hiding
            # signed CAM values or inventing positive coincidences.
            for mode in ("difference", "coincidence"):
                zero_attention, zero_weights, fallback = spatial_attention_from_evidence(torch.zeros(2, 2, 3), .5, WHERE_NORM_EPS)
                torch.testing.assert_close(zero_attention, torch.full((2, 2, 3), 1. / 6.), rtol=0, atol=0)
                torch.testing.assert_close(zero_weights, torch.ones(2, 2, 3), rtol=0, atol=0)
                assert fallback.all() and torch.isfinite(zero_attention).all()
            check("zero evidence falls back to finite uniform attention")

            # Independent outcome routing: the original q table is frozen.
            # Selecting one outcome replaces q by exactly one elsewhere;
            # it never renormalizes the resulting per-image weights.
            routes = ("all", "correct", "error")
            method_flags = {
                "QW0_BASE_RAW": (False, False),
                "QW1_AMOUNT_ONLY": (True, False),
                "QW2_WHERE_ONLY": (False, True),
                "QW3_AMOUNT_WHERE": (True, True),
            }

            def selected_outcome(route):
                return torch.ones_like(correct) if route == "all" else (
                    correct if route == "correct" else ~correct)

            def oracle_weights(correct_mode, error_mode, where_route, use_where):
                enabled = eligible & selected_outcome(where_route) & use_where
                result = torch.ones_like(raw_t)
                branches = {name: torch.zeros_like(correct) for name in ("raw", "difference", "coincidence")}
                branches["raw"] = ~enabled
                for outcome, mode in ((correct, correct_mode), (~correct, error_mode)):
                    mask = enabled & outcome
                    branches[mode] |= mask
                    if mode == "raw":
                        continue
                    value = oracle_evidence[mode]
                    evidence_sum = value.sum((-2, -1), keepdim=True)
                    pixels = raw_t[0].numel()
                    uniform = torch.full_like(value, 1. / pixels)
                    attention = (1. - WHERE_BETA) * uniform + WHERE_BETA * torch.where(
                        evidence_sum > WHERE_NORM_EPS,
                        value / evidence_sum.clamp_min(WHERE_NORM_EPS), uniform)
                    result[mask] = (pixels * attention)[mask]
                return result, branches

            def assert_routed_case(method, amount_route, where_route):
                use_amount, use_where = method_flags[method]
                globals().update(AMOUNT_APPLY_TO=amount_route, WHERE_APPLY_TO=where_route)
                base_q = torch.as_tensor(stats["amount_factors"], dtype=cam_s.dtype)[class_to_group[labels]]
                selected_amount = selected_outcome(amount_route)
                applied_amount = selected_amount & use_amount
                expected_q = torch.where(applied_amount, base_q, torch.ones_like(base_q))
                expected_weight, expected_branches = oracle_weights(
                    WHERE_CORRECT_MODE, WHERE_ERROR_MODE, where_route, use_where)
                denominator = max(float(stats["raw_cam_mean_sq"]), CAM_SCALE_EPS)
                squared_error = (cam_s - raw_t).square()
                expected_raw = squared_error.mean((-2, -1)) / denominator
                expected_spatial = (squared_error * expected_weight).mean((-2, -1)) / denominator
                expected_per_image = (expected_raw if not use_where or
                    (WHERE_CORRECT_MODE == "raw" and WHERE_ERROR_MODE == "raw") else
                    torch.where(expected_branches["raw"], expected_raw, expected_spatial))
                rectified_error = (cam_s-expected_q[:,None,None]*raw_t).square()
                rectified_raw = rectified_error.mean((-2,-1))/denominator
                rectified_spatial = (rectified_error*expected_weight).mean((-2,-1))/denominator
                rectified_per_image = (rectified_raw if not use_where or
                    (WHERE_CORRECT_MODE == "raw" and WHERE_ERROR_MODE == "raw") else
                    torch.where(expected_branches["raw"],rectified_raw,rectified_spatial))
                expected_guidance = rectified_per_image.mean()
                expected_total = CE_WEIGHT * F.cross_entropy(logits_s, labels) + CAM_ALPHA * expected_guidance
                actual_total, actual_diag, actual_maps, actual_losses = call(method)
                torch.testing.assert_close(actual_total, expected_total, rtol=2e-6, atol=2e-7)
                torch.testing.assert_close(actual_losses["guidance_per_image"], expected_per_image, rtol=2e-6, atol=2e-7)
                torch.testing.assert_close(actual_losses["raw_per_image"], expected_raw, rtol=0, atol=0)
                torch.testing.assert_close(actual_losses["guidance"], expected_guidance, rtol=2e-6, atol=2e-7)
                assert_equal_gradients(actual_total, expected_total, variables)
                torch.testing.assert_close(actual_maps["applied_spatial_weight"], expected_weight, rtol=2e-6, atol=2e-7)
                torch.testing.assert_close(actual_losses["amount_base_per_image"], base_q, rtol=0, atol=0)
                torch.testing.assert_close(actual_diag["amount_factor_base"], base_q, rtol=0, atol=0)
                torch.testing.assert_close(actual_losses["amount_per_image"], expected_q, rtol=0, atol=0)
                torch.testing.assert_close(actual_diag["amount_weight"], expected_q, rtol=0, atol=0)
                assert torch.equal(actual_losses["amount_selected_outcome_mask"], selected_amount)
                assert torch.equal(actual_losses["amount_applied_mask"], applied_amount)
                assert torch.equal(actual_losses["where_selected_outcome_mask"], selected_outcome(where_route))
                torch.testing.assert_close(actual_losses["amount_per_image"][~applied_amount],
                    torch.ones_like(expected_q[~applied_amount]), rtol=0, atol=0)
                where_enabled = eligible & selected_outcome(where_route) & use_where
                torch.testing.assert_close(actual_maps["applied_spatial_weight"][~where_enabled],
                    torch.ones_like(raw_t[~where_enabled]), rtol=0, atol=0)
                torch.testing.assert_close(actual_losses["guidance_per_image"][~where_enabled],
                    expected_raw[~where_enabled], rtol=0, atol=0)
                for name, mask in expected_branches.items():
                    assert torch.equal(actual_diag[name + "_branch"].bool(), mask)
                torch.testing.assert_close(sum(actual_diag[name + "_branch"] for name in expected_branches),
                    torch.ones_like(q), rtol=0, atol=0)
                torch.testing.assert_close(sum(actual_losses[name + "_component"] for name in expected_branches),
                    actual_losses["guidance"], rtol=2e-6, atol=2e-7)
                torch.testing.assert_close(actual_losses["correct_component"] + actual_losses["error_component"],
                    actual_losses["guidance"], rtol=2e-6, atol=2e-7)
                torch.testing.assert_close(actual_maps["selected_ids"], selected, rtol=0, atol=0)
                torch.testing.assert_close(actual_maps["teacher_raw"], raw_t, rtol=0, atol=0)
                # Rebuild the direct V8 all/all formula, using the already verified
                # detached weights to require bit-exact all/all compatibility.
                if amount_route == where_route == "all":
                    previous_q = base_q if use_amount else torch.ones_like(base_q)
                    previous_error = (cam_s-previous_q[:,None,None]*raw_t).square()
                    previous_raw = previous_error.mean((-2, -1)) / denominator
                    previous_spatial = (previous_error * actual_maps["applied_spatial_weight"]).mean((-2, -1)) / denominator
                    previous_per_image = (previous_raw if not use_where or
                        (WHERE_CORRECT_MODE == "raw" and WHERE_ERROR_MODE == "raw") else
                        torch.where(expected_branches["raw"], previous_raw, previous_spatial))
                    previous_total = CE_WEIGHT * F.cross_entropy(logits_s, labels) + CAM_ALPHA * previous_per_image.mean()
                    torch.testing.assert_close(actual_total, previous_total, rtol=0, atol=0)
                    assert_equal_gradients(actual_total, previous_total, variables, exact=True)
                return actual_total, actual_diag, actual_maps, actual_losses

            globals().update(WHERE_BETA=.5, WHERE_CORRECT_MASK_MODE="aligned", WHERE_ERROR_MASK_MODE="aligned")
            original_factors = stats["amount_factors"].copy()
            for correct_mode in ("raw", "difference", "coincidence"):
                for error_mode in ("raw", "difference", "coincidence"):
                    globals().update(WHERE_CORRECT_MODE=correct_mode, WHERE_ERROR_MODE=error_mode)
                    for amount_route in routes:
                        for where_route in routes:
                            assert_routed_case("QW3_AMOUNT_WHERE", amount_route, where_route)
            np.testing.assert_array_equal(stats["amount_factors"], original_factors)
            check("all 81 mode/outcome combinations match independent values, gradients, q routing and exclusive spatial branches")
            check("all/all matches direct V8 objective values and gradients with q-scaled true-label teacher CAMs")

            globals().update(WHERE_CORRECT_MODE="difference", WHERE_ERROR_MODE="coincidence")
            for method in ("QW0_BASE_RAW", "QW1_AMOUNT_ONLY", "QW2_WHERE_ONLY"):
                for amount_route in routes:
                    for where_route in routes:
                        assert_routed_case(method, amount_route, where_route)
            check("QW0/QW1/QW2 flags independently disable amount or spatial controls without disabling global teacher scale")

            # Every method keeps the same teacher reference scale, including
            # QW0/QW2 and the epsilon floor, regardless of outcome selection.
            for teacher_mean_sq in (11., .37, 0., CAM_SCALE_EPS * .1):
                stats["raw_cam_mean_sq"] = teacher_mean_sq
                for method in method_flags:
                    assert_routed_case(method, "error", "error")
            stats["raw_cam_mean_sq"] = 2.3
            check("teacher scale remains global in all four methods above and below one and at the epsilon floor")

            # A normalized base table need not have a normalized conditional
            # application. No second normalization may perturb correct images.
            normalized_factors = original_factors / np.mean(q[:4].numpy())
            stats["amount_factors"] = normalized_factors
            for sign in ("positive","negative","zero"):
                stats["amount_factors_"+sign]=stats["amount_factors"].copy()
            globals().update(AMOUNT_APPLY_TO="error", WHERE_APPLY_TO="error")
            _, _, _, conditional_losses = call(subset=slice(0, 4))
            conditional_base = torch.as_tensor(normalized_factors, dtype=q.dtype)[class_to_group[labels[:4]]]
            expected_conditional = torch.where(correct[:4], torch.ones_like(conditional_base), conditional_base)
            torch.testing.assert_close(conditional_base.mean(), torch.tensor(1.), rtol=2e-6, atol=2e-7)
            assert not torch.isclose(expected_conditional.mean(), torch.tensor(1.))
            torch.testing.assert_close(conditional_losses["amount_per_image"], expected_conditional, rtol=0, atol=0)
            np.testing.assert_array_equal(stats["amount_factors"], normalized_factors)
            stats["amount_factors"] = original_factors
            for sign in ("positive","negative","zero"):
                stats["amount_factors_"+sign]=stats["amount_factors"].copy()
            check("conditional q is not renormalized: unselected images stay exactly one even when the applied mean differs from one")

            # beta=0/uniform remove only spatial weighting; the amount route
            # stays active. raw/raw must give the same identity for every route.
            for amount_route in routes:
                for where_route in routes:
                    globals().update(AMOUNT_APPLY_TO=amount_route, WHERE_APPLY_TO=where_route,
                        WHERE_CORRECT_MODE="difference", WHERE_ERROR_MODE="coincidence")
                    expected_q = torch.where(selected_outcome(amount_route), q, torch.ones_like(q))
                    expected_baseline = CE_WEIGHT * F.cross_entropy(logits_s, labels) + CAM_ALPHA * ((cam_s-expected_q[:,None,None]*raw_t).square().mean((-2,-1))/stats["raw_cam_mean_sq"]).mean()
                    for beta, mask in ((0., "aligned"), (.5, "uniform")):
                        globals().update(WHERE_BETA=beta, WHERE_CORRECT_MASK_MODE=mask, WHERE_ERROR_MASK_MODE=mask)
                        identity_total, _, identity_maps, identity_losses = call()
                        torch.testing.assert_close(identity_maps["applied_spatial_weight"], torch.ones_like(raw_t), rtol=0, atol=0)
                        torch.testing.assert_close(identity_losses["amount_per_image"], expected_q, rtol=0, atol=0)
                        torch.testing.assert_close(identity_total, expected_baseline, rtol=0, atol=0)
                        assert_equal_gradients(identity_total, expected_baseline, variables, exact=True)
                    globals().update(WHERE_BETA=.5, WHERE_CORRECT_MASK_MODE="aligned", WHERE_ERROR_MASK_MODE="aligned",
                        WHERE_CORRECT_MODE="raw", WHERE_ERROR_MODE="raw")
                    identity_total, _, _, _ = call()
                    torch.testing.assert_close(identity_total, expected_baseline, rtol=0, atol=0)
                    assert_equal_gradients(identity_total, expected_baseline, variables, exact=True)
            check("beta zero, uniform masks and raw/raw preserve routed q and recover exact raw-MSE values and gradients")

            stats["amount_factors"] = _apply_power_and_optional_clip(np.array([.6, 1.4, 2.9]),
                power=0., sample_proportions=np.array([.5, .25, .25]))
            np.testing.assert_array_equal(stats["amount_factors"], np.ones(3))
            for sign in ("positive","negative","zero"):
                stats["amount_factors_"+sign]=stats["amount_factors"].copy()
            globals().update(WHERE_CORRECT_MODE="difference", WHERE_ERROR_MODE="coincidence")
            for amount_route in routes:
                for where_route in routes:
                    globals().update(AMOUNT_APPLY_TO=amount_route, WHERE_APPLY_TO=where_route)
                    for off, on in (("QW0_BASE_RAW", "QW1_AMOUNT_ONLY"), ("QW2_WHERE_ONLY", "QW3_AMOUNT_WHERE")):
                        off_total, _, off_maps, off_losses = call(off)
                        on_total, _, on_maps, on_losses = call(on)
                        torch.testing.assert_close(on_losses["amount_per_image"], torch.ones_like(q), rtol=0, atol=0)
                        torch.testing.assert_close(on_maps["applied_spatial_weight"], off_maps["applied_spatial_weight"], rtol=0, atol=0)
                        torch.testing.assert_close(on_losses["guidance_per_image"], off_losses["guidance_per_image"], rtol=0, atol=0)
                        torch.testing.assert_close(on_total, off_total, rtol=0, atol=0)
                        assert_equal_gradients(on_total, off_total, variables, exact=True)
            stats["amount_factors"] = original_factors
            for sign in ("positive","negative","zero"):
                stats["amount_factors_"+sign]=stats["amount_factors"].copy()
            check("power zero recovers QW0=QW1 and QW2=QW3 exactly for every outcome route with global teacher scale retained")

            # New V8 arithmetic, independent from the production reference code.
            special_images=images.clone()
            special_images[0,labels[0],0]=torch.tensor([-2.,0.,2.])
            signed_stats={**stats,"amount_factors_positive":np.array([1.6,.9,1.4]),
                "amount_factors_negative":np.array([.4,1.3,.7]),"amount_factors_zero":np.array([1.1,1.1,1.1])}
            true_raw=(special_images*teacher.fc.weight[labels,:,None,None]).sum(1)
            special_correct=teacher(special_images)[0].argmax(1).eq(labels)
            for method in method_flags:
                for route in routes:
                    globals().update(AMOUNT_APPLY_TO=route,WHERE_APPLY_TO="all")
                    selected_sign=(torch.ones_like(special_correct) if route=="all" else
                        special_correct if route=="correct" else ~special_correct)&method_flags[method][0]
                    expected_factor=torch.ones_like(true_raw)
                    for sign,mask in (("positive",true_raw>0),("negative",true_raw<0),("zero",true_raw==0)):
                        values=torch.as_tensor(signed_stats["amount_factors_"+sign],dtype=cam_s.dtype)[class_to_group[labels]]
                        expected_factor=torch.where(mask&selected_sign[:,None,None],values[:,None,None],expected_factor)
                    expected_target=expected_factor*true_raw
                    actual,dd,mm,ll=qw_cam_loss(method,logits_s,features,student_fc,teacher,labels,
                        special_images,class_to_group,signed_stats,sample_ids,42,7)
                    torch.testing.assert_close(mm["teacher_rectified"],expected_target,rtol=0,atol=0)
                    expected_error=((cam_s-expected_target).square()*mm["applied_spatial_weight"]).mean((-2,-1))/stats["raw_cam_mean_sq"]
                    expected=CE_WEIGHT*F.cross_entropy(logits_s,labels)+CAM_ALPHA*expected_error.mean()
                    torch.testing.assert_close(actual,expected,rtol=2e-6,atol=2e-7)
                    assert_equal_gradients(actual,expected,variables)
                    torch.testing.assert_close(dd["amount_weight"],expected_factor.mean((-2,-1)),rtol=2e-6,atol=2e-7)
                    torch.testing.assert_close(dd["target_abs_mass_after_q"],expected_target.abs().mean((-2,-1)),rtol=0,atol=0)
                    assert mm["teacher_rectified"][true_raw==0].eq(0).all()
            check("separate positive/negative teacher factors match values and gradients in all methods/outcome routes; zeros stay zero")
            signed_counts=np.array([[4,3,2],[1,2,4],[1,1,0]])
            signed_means=np.array([[3.,2.,1.],[1.,2.,3.]])
            for power in (0.,.5,1.,2.):
                ff=compute_signed_amount_factors(signed_means,signed_counts,power)
                np.testing.assert_allclose(ff["amount_expected_weight"],1.,rtol=0,atol=1e-12)
                assert all(np.isfinite(ff["amount_factors_"+s]).all() for s in ("positive","negative","zero"))
                if power==0.:
                    for s in ("positive","negative","zero"):
                        np.testing.assert_allclose(ff["amount_factors_"+s],np.ones(3),rtol=0,atol=1e-12)
            ff=compute_signed_amount_factors(signed_means,signed_counts,1.)
            np.testing.assert_allclose(ff["amount_factors_positive"][2]/ff["amount_factors_positive"][0],3.)
            np.testing.assert_allclose(ff["amount_factors_negative"][2]/ff["amount_factors_negative"][0],1./3.)
            for all_sign in (0,1,2):
                counts=np.zeros((3,3));counts[all_sign]=[5,7,9]
                means=np.zeros((2,3))
                if all_sign<2:means[all_sign]=[1.,2.,3.]
                ff=compute_signed_amount_factors(means,counts,2.)
                np.testing.assert_allclose(ff["amount_expected_weight"],1.,rtol=0,atol=1e-12)
                assert np.isfinite(ff["amount_factors"]).all()
            check("joint TRAIN calibration, opposite sign compensations, zero power and absent-sign/all-zero statistics are finite")

            gradients = torch.autograd.grad(total, variables, retain_graph=True)
            assert all(torch.isfinite(value).all() for value in gradients)
            assert any(value.abs().sum() > 0 for value in gradients)
            assert all(parameter.grad is None for parameter in teacher.parameters())
            for key, value in teacher.state_dict().items():
                torch.testing.assert_close(value, teacher_before[key], rtol=0, atol=0)
            check("teacher and attention stay detached while student gradients remain finite")
    finally:
        for key, value in original.items():
            if value is absent:
                globals().pop(key, None)
            else:
                globals()[key] = value
        np.random.set_state(np_state)
        _test_random.setstate(python_state)
    print("V8 CPU self-tests passed (" + str(len(checks)) + " checks): " + "; ".join(checks) + ".")
    return checks


if RUN_SELF_TESTS:
    run_core_self_tests()


<a id="treinamento"></a>
## Treinamento e avaliação final

Somente a implementação do perfil selecionado é definida e executada. O professor permanece congelado, e o estudante utiliza a mesma perda, otimizador, scheduler e early stopping do respectivo anexo. O melhor estudante é recarregado do checkpoint escolhido pela validação antes da avaliação final.

Nenhuma avaliação de teste escolhe o checkpoint. A reutilização considera somente execuções concluídas e compatíveis; não há retomada automática de épocas interrompidas.


In [ ]:
assert_active_profile()

if EXPERIMENT_PROFILE == "resnet32_cifar100_lt":
    def json_compatible(value):
        if isinstance(value, dict):
            return {str(key): json_compatible(item) for key, item in value.items()}
        if isinstance(value, (list, tuple)):
            return [json_compatible(item) for item in value]
        if isinstance(value, np.ndarray):
            return json_compatible(value.tolist())
        if isinstance(value, np.generic):
            return json_compatible(value.item())
        if isinstance(value, float) and not np.isfinite(value):
            return None
        if isinstance(value, Path):
            return str(value)
        return value


    def atomic_json(path, value):
        target = Path(path)
        temporary = target.with_name(target.name + ".writing")
        try:
            temporary.write_text(json.dumps(json_compatible(value), ensure_ascii=False,
                indent=2, allow_nan=False), encoding="utf-8")
            os.replace(temporary, target)
        finally:
            if temporary.exists():
                temporary.unlink()


    INITIAL_STATE_HASH_CACHE = {}


    def save_predictions(path, labels, preds, probs, ids=None):
        labels, preds, probs = np.asarray(labels), np.asarray(preds), np.asarray(probs)
        sample_ids = np.arange(len(labels)) if ids is None else np.asarray(ids)
        if not (len(labels) == len(preds) == len(probs) == len(sample_ids)):
            raise ValueError("Prediction arrays and sample IDs must have identical lengths.")
        frame = pd.DataFrame({"sample_id": sample_ids, "true_class": labels,
            "predicted_class": preds, "correct": labels == preds,
            "target_probability": probs[np.arange(len(labels)), labels],
            "predicted_probability": probs.max(1)})
        frame.to_csv(path, index=False)
        np.savez_compressed(str(Path(path).with_suffix(".npz")), labels=labels,
            predictions=preds, probabilities=probs, sample_ids=sample_ids)


    def classification_diagnostics(labels, teacher_pred, student_pred, class_counts, class_to_group):
        # Complementary class diagnostics do not replace historical group F1.
        from sklearn.metrics import precision_recall_fscore_support
        table = pd.DataFrame({"class_id": np.arange(NUM_CLASSES), "class_name": _CLASS_NAMES,
            "n_train": class_counts, "group": [GROUP_ID_TO_NAME[int(group)] for group in class_to_group.cpu()]})
        for prefix, predictions in (("teacher", teacher_pred), ("student", student_pred)):
            precision, recall, f1, support = precision_recall_fscore_support(
                labels, predictions, labels=np.arange(NUM_CLASSES), zero_division=0)
            table[prefix + "_precision_diagnostic"] = precision
            table[prefix + "_recall"] = recall
            table[prefix + "_fullset_class_f1_diagnostic"] = f1
            table[prefix + "_support"] = support
        table["delta_recall"] = table.student_recall - table.teacher_recall
        table["delta_fullset_class_f1_diagnostic"] = table.student_fullset_class_f1_diagnostic - table.teacher_fullset_class_f1_diagnostic
        return table


    def outcome_control_metadata(experiment, teacher_stats):
        """Static policy and potential coefficients, never an empirical applied-q mean.

        The historical global CAM-squared scale is applied to EVERY experiment.
        Only q_G and the spatial weighting are selected by their independent
        teacher-outcome filters. No renormalization follows that selection.
        """
        cfg = EXPERIMENT_DEFINITIONS[experiment]
        estimate = float(teacher_stats["raw_cam_mean_sq"])
        scale = max(estimate, float(CAM_SCALE_EPS))
        coefficient = float(CAM_ALPHA) / scale
        metadata = {"amount_apply_to": AMOUNT_APPLY_TO, "where_apply_to": WHERE_APPLY_TO,
            "amount_enabled": bool(cfg["use_amount"]), "where_enabled": bool(cfg["use_where"]),
            "amount_scope_definition": "signed_group_target_factors_on_selected_teacher_outcome_without_renormalization",
            "amount_expected_weight_base": float(teacher_stats["amount_expected_weight"]),
            "amount_expected_weight_base_definition": "Original deterministic-training expectation of the spatial mean of sign/group factors before teacher-outcome selection. Applied q is not renormalized and its observed expectation need not equal one.",
            "teacher_raw_cam_mean_sq": estimate, "cam_scale_eps": float(CAM_SCALE_EPS),
            "cam_scale_reference": scale, "cam_scale_applied": scale, "cam_scale_enabled": True,
            "cam_scale_scope": "all_experiments", "cam_global_coefficient": coefficient,
            "amount_factor_outside_selection": 1.,
            "cam_effective_coefficient_outside_amount_selection": coefficient}
        for group_id in range(3):
            group = GROUP_ID_TO_NAME[group_id]
            base = float(teacher_stats["amount_factors"][group_id])
            selected = base if cfg["use_amount"] else 1.
            metadata["amount_factor_base_" + group] = base
            metadata["amount_factor_if_selected_" + group] = selected
            metadata["cam_effective_coefficient_if_selected_" + group] = coefficient
            for sign in ("positive","negative","zero"):
                value=float(teacher_stats["amount_factors_"+sign][group_id])
                metadata["amount_factor_"+sign+"_base_"+group]=value
                metadata["amount_factor_"+sign+"_if_selected_"+group]=value if cfg["use_amount"] else 1.
        return metadata


    def observed_outcome_control_metadata(balance_frame):
        """Observed image means in one training epoch, with conditional finite counts."""
        observed = {}
        for group in ("all", "many", "medium", "few"):
            group_rows = balance_frame[balance_frame.group == group]
            for outcome in ("all", "correct", "error"):
                partition = "all" if outcome == "all" else "teacher_outcome"
                row = group_rows[(group_rows.partition == partition) & (group_rows.stratum == outcome)].iloc[0]
                suffix = group if outcome == "all" else "teacher_" + outcome + "_" + group
                observed["amount_factor_applied_" + suffix] = float(row.q_conditional_mean)
                observed["amount_factor_applied_valid_n_" + suffix] = int(row.q_valid_n)
                observed["cam_effective_coefficient_" + suffix] = float(row.cam_effective_coefficient_conditional_mean)
                observed["cam_effective_coefficient_valid_n_" + suffix] = int(row.cam_effective_coefficient_valid_n)
                observed["amount_applied_fraction_" + suffix] = float(row.amount_applied_conditional_mean)
                observed["amount_applied_n_" + suffix] = int(row.amount_applied_sum)
                observed["amount_selected_outcome_fraction_" + suffix] = float(row.amount_selected_outcome_conditional_mean)
                observed["amount_selected_outcome_n_" + suffix] = int(row.amount_selected_outcome_sum)
                observed["where_selected_outcome_fraction_" + suffix] = float(row.where_selected_outcome_conditional_mean)
                observed["where_selected_outcome_n_" + suffix] = int(row.where_selected_outcome_sum)
                observed["where_applied_fraction_" + suffix] = float(row.where_applied_conditional_mean)
                observed["where_applied_n_" + suffix] = int(row.where_applied_sum)
                for sign in ("positive","negative","zero"):
                    observed["amount_factor_"+sign+"_applied_"+suffix]=float(row["amount_factor_"+sign+"_applied_conditional_mean"])
                    observed["amount_"+sign+"_fraction_"+suffix]=float(row["amount_"+sign+"_fraction_conditional_mean"])
        observed["amount_applied_weight_mean_at_best"] = observed["amount_factor_applied_all"]
        observed["amount_applied_weight_valid_n_at_best"] = observed["amount_factor_applied_valid_n_all"]
        observed["observed_training_epoch"] = int(balance_frame.epoch.iloc[0])
        observed["applied_factor_mean_definition"] = "Observed image mean in the selected training epoch, with teacher-correct/error conditional means reported separately; empty strata have null means and zero valid_n. The applied q is a spatial factor mean, distinct from the static group mean and the separate sign coefficients."
        return observed


    class EpochLossBalanceAccumulator:
        """Full-epoch sums and explicit denominators for signed spatial CAM guidance.

        Partitions overlap with one another; strata within each partition are
        disjoint and complete. Objective contributions use ALL epoch images as
        denominator. Conditional means use only the selected group's/stratum's
        images. Raw CAM masses and q-scaled reference masses are recorded separately.
        guidance_before_q already includes the global CAM-squared denominator.
        q is the spatial mean of the ACTUAL outcome-selected sign-factor map; no second normalization is used.
        """
        def __init__(self, class_to_group):
            self.class_to_group = class_to_group.detach().cpu().numpy()
            self.groups = ("all", "many", "medium", "few")
            self.strata = (("all", "all"), ("teacher_outcome", "correct"),
                ("teacher_outcome", "error"), ("guidance_branch", "raw"),
                ("guidance_branch", "difference"), ("guidance_branch", "coincidence"))
            self.mass_names = ("teacher_raw_target_cam_l1", "q_times_teacher_raw_target_cam_l1",
                "teacher_raw_target_cam_positive_mass", "teacher_raw_target_cam_negative_mass",
                "teacher_raw_target_cam_signed_mean", "teacher_spatial_weighted_target_cam_l1",
                "q_times_teacher_spatial_weighted_target_cam_l1",
                "teacher_rectified_target_cam_positive_mass","teacher_rectified_target_cam_negative_mass")
            self.rows = {(group, partition, stratum): {"n": 0, "classes": set(),
                "sum": {}, "valid": {}, "class_sum": {}, "class_valid": {}}
                for group in self.groups for partition, stratum in self.strata}

        def update(self, logits, labels, diagnostics, maps, losses):
            with torch.no_grad():
                ce = F.cross_entropy(logits.detach(), labels, reduction="none")
                q = losses["amount_per_image"].detach()
                guidance = losses["guidance_per_image"].detach()
                guidance_after_q = losses["guidance_after_q_per_image"].detach()
                raw_after_q = losses["raw_after_q_per_image"].detach()
                raw_loss = losses["raw_per_image"].detach()
                raw_mass = diagnostics["raw_abs_mass"].detach()
                positive = diagnostics["raw_pos_mass"].detach()
                negative = diagnostics["raw_neg_mass"].detach()
                correct = diagnostics["teacher_correct"].bool()
                branch_masks = {name: diagnostics[name + "_branch"].bool()
                                for name in ("raw", "difference", "coincidence")}
                if not sum(mask.long() for mask in branch_masks.values()).eq(1).all():
                    raise RuntimeError("Spatial-guidance branches do not partition the batch.")
                values = {"ce_unweighted": ce, "guidance_before_q": guidance,
                    "guidance_after_q": guidance_after_q, "raw_baseline_before_q": raw_loss,
                    "raw_baseline_after_q": raw_after_q, "ce_objective": CE_WEIGHT * ce,
                    "method_objective": CAM_ALPHA * guidance_after_q,
                    "total_objective": CE_WEIGHT * ce + CAM_ALPHA * guidance_after_q,
                    "amount_factor_base": diagnostics["amount_factor_base"].detach(),
                    "amount_selected_outcome": diagnostics["amount_selected_outcome"].detach(),
                    "amount_applied": diagnostics["amount_applied"].detach(),
                    "where_selected_outcome": diagnostics["where_selected_outcome"].detach(),
                    "where_applied": diagnostics["where_applied"].detach(),
                    "cam_scale_reference": diagnostics["cam_scale_reference"].detach(),
                    "cam_scale_applied": diagnostics["cam_scale_applied"].detach(),
                    "cam_global_coefficient": diagnostics["cam_global_coefficient"].detach(),
                    "cam_effective_coefficient": diagnostics["cam_effective_coefficient"].detach(),
                    "q": q, "teacher_raw_target_cam_l1": raw_mass,
                    "q_times_teacher_raw_target_cam_l1": diagnostics["target_abs_mass_after_q"].detach(),
                    "teacher_spatial_weighted_target_cam_l1": diagnostics["spatial_weighted_raw_abs_mass"].detach(),
                    "q_times_teacher_spatial_weighted_target_cam_l1": diagnostics["q_spatial_weighted_raw_abs_mass"].detach(),
                    "teacher_raw_target_cam_positive_mass": positive,
                    "teacher_raw_target_cam_negative_mass": negative,
                    "teacher_raw_target_cam_signed_mean": positive - negative,
                    "teacher_rectified_target_cam_positive_mass":diagnostics["target_pos_mass_after_q"].detach(),
                    "teacher_rectified_target_cam_negative_mass":diagnostics["target_neg_mass_after_q"].detach()}
                for sign in ("positive","negative","zero"):
                    values["amount_factor_"+sign+"_applied"]=diagnostics["amount_factor_"+sign+"_applied"].detach()
                    values["amount_"+sign+"_fraction"]=diagnostics["amount_"+sign+"_fraction"].detach()
                for name, mask in {"correct": correct, "error": ~correct, **branch_masks}.items():
                    component = guidance * mask.to(guidance.dtype)
                    values[name + "_before_q"] = component
                    values[name + "_after_q"] = guidance_after_q * mask.to(guidance_after_q.dtype)
                    values[name + "_objective"] = CAM_ALPHA * values[name + "_after_q"]
                names = list(values)
                matrix = torch.stack([values[name] for name in names], 1).double().cpu().numpy()
                y = labels.detach().cpu().numpy()
                masks = {("all", "all"): np.ones(len(y), dtype=bool),
                    ("teacher_outcome", "correct"): correct.cpu().numpy(),
                    ("teacher_outcome", "error"): (~correct).cpu().numpy()}
                masks.update({("guidance_branch", name): mask.cpu().numpy() for name, mask in branch_masks.items()})
            if not np.isfinite(matrix).all():
                raise FloatingPointError("Nonfinite per-image objective or raw teacher CAM mass.")
            for group in self.groups:
                group_mask = np.ones(len(y), dtype=bool) if group == "all" else (
                    self.class_to_group[y] == GROUP_NAME_TO_ID[group])
                for (partition, stratum), stratum_mask in masks.items():
                    mask = group_mask & stratum_mask
                    source = self.rows[(group, partition, stratum)]
                    source["n"] += int(mask.sum())
                    source["classes"].update(y[mask].tolist())
                    selected = matrix[mask]
                    for index, name in enumerate(names):
                        source["sum"][name] = source["sum"].get(name, 0.) + float(selected[:, index].sum())
                        source["valid"][name] = source["valid"].get(name, 0) + int(mask.sum())
                        if name in self.mass_names:
                            sums = source["class_sum"].setdefault(name, {})
                            counts = source["class_valid"].setdefault(name, {})
                            for class_id in np.unique(y[mask]):
                                class_mask = y[mask] == class_id
                                sums[int(class_id)] = sums.get(int(class_id), 0.) + float(selected[class_mask, index].sum())
                                counts[int(class_id)] = counts.get(int(class_id), 0) + int(class_mask.sum())

        def frame(self, epoch):
            total = self.rows[("all", "all", "all")]
            population_n = total["n"]
            def ratio(numerator, denominator):
                return numerator / denominator if denominator > 0 else np.nan
            records = []
            for (group, partition, stratum), source in self.rows.items():
                group_all = self.rows[(group, "all", "all")]
                row = {"epoch": epoch, "group": group, "partition": partition, "stratum": stratum,
                    "n_samples": source["n"], "n_classes_observed": len(source["classes"]),
                    "n_epoch_samples": population_n, "n_group_samples": group_all["n"],
                    "sample_fraction_of_epoch": ratio(source["n"], population_n),
                    "sample_fraction_within_group": ratio(source["n"], group_all["n"])}
                for name, value in source["sum"].items():
                    valid = source["valid"][name]
                    row[name + "_sum"] = value
                    row[name + "_valid_n"] = valid
                    row[name + "_conditional_mean"] = value / valid if valid else np.nan
                    row[name + "_population_mean_contribution"] = ratio(value, population_n)
                    if name in self.mass_names:
                        means = [value / source["class_valid"][name][class_id]
                                 for class_id, value in source["class_sum"][name].items()]
                        row[name + "_class_balanced_mean"] = float(np.mean(means)) if means else np.nan
                        row[name + "_classes_valid_n"] = len(means)
                for suffix in ("before_q", "after_q"):
                    metric = "guidance_" + suffix
                    row["guidance_share_of_epoch_" + suffix] = ratio(source["sum"][metric], total["sum"][metric])
                    row["guidance_share_within_group_" + suffix] = ratio(source["sum"][metric], group_all["sum"][metric])
                for suffix, metric in (("before_q", "teacher_raw_target_cam_l1"),
                        ("after_q", "q_times_teacher_raw_target_cam_l1")):
                    row["raw_target_cam_l1_share_of_epoch_" + suffix] = ratio(source["sum"][metric], total["sum"][metric])
                row["method_to_ce_ratio_of_sums"] = ratio(source["sum"]["method_objective"], source["sum"]["ce_objective"])
                row["raw_mass_reweighting_ratio_of_sums"] = ratio(
                    source["sum"]["q_times_teacher_raw_target_cam_l1"], source["sum"]["teacher_raw_target_cam_l1"])
                records.append(row)
            frame = pd.DataFrame(records)
            check_metrics = ("ce_objective_sum", "method_objective_sum", "guidance_before_q_sum",
                "guidance_after_q_sum", "teacher_raw_target_cam_l1_sum", "q_times_teacher_raw_target_cam_l1_sum")
            for group in self.groups:
                reference = frame[(frame.group == group) & (frame.partition == "all")].iloc[0]
                for partition in ("teacher_outcome", "guidance_branch"):
                    part = frame[(frame.group == group) & (frame.partition == partition)]
                    if int(part.n_samples.sum()) != int(reference.n_samples):
                        raise RuntimeError("Loss-balance sample counts do not close.")
                    for metric in check_metrics:
                        np.testing.assert_allclose(part[metric].sum(), reference[metric], rtol=1e-10, atol=1e-10)
            selected_groups = frame[(frame.group != "all") & (frame.partition == "all")]
            for metric in check_metrics:
                np.testing.assert_allclose(selected_groups[metric].sum(),
                    frame[(frame.group == "all") & (frame.partition == "all")].iloc[0][metric],
                    rtol=1e-10, atol=1e-10)
            return frame


    def run_experiment(experiment, seed, student_init_mode="random"):
        validate_v8_configuration()
        if experiment not in EXPERIMENT_DEFINITIONS:
            raise ValueError(experiment)
        set_seed(seed)
        train_ds, val_ds, test_ds, train_loader, val_loader, test_loader = build_loaders(seed)
        class_counts = get_class_counts(train_ds)
        class_to_group, groups = build_class_to_group(class_counts)
        if any(len(groups[name]) == 0 for name in ("many", "medium", "few")):
            raise ValueError("This QUANTO protocol requires nonempty many/medium/few groups.")
        teacher, teacher_path = load_teacher(seed)
        stats = estimate_teacher_group_cam_statistics(seed, teacher,
            build_deterministic_train_loader(train_ds), class_counts, class_to_group)
        group_stats = print_group_statistics(stats)
        control_metadata = outcome_control_metadata(experiment, stats)
        # Reference statistics stay unchanged. Potential factors are separate from
        # the actual outcome-conditioned means, which are available after training.
        for name, value in control_metadata.items():
            if not name.startswith(("amount_factor_base_", "amount_factor_if_selected_",
                                    "cam_effective_coefficient_if_selected_")):
                group_stats[name] = value
        group_stats["amount_factor_base"] = stats["amount_factors"]
        group_stats["amount_factor_if_selected"] = [control_metadata["amount_factor_if_selected_" +
            GROUP_ID_TO_NAME[group_id]] for group_id in range(3)]
        group_stats["cam_effective_coefficient_if_selected"] = [control_metadata[
            "cam_effective_coefficient_if_selected_" + GROUP_ID_TO_NAME[group_id]] for group_id in range(3)]
        group_stats["base_q_times_reference_raw_mass_diagnostic"] = group_stats.q_times_raw_mass_diagnostic
        group_stats["q_times_raw_mass_diagnostic"] = np.nan
        group_stats["q_times_raw_mass_definition"] = "Observed mean of ACTUAL applied q times raw CAM L1 at the best training epoch; excludes alpha and the global denominator. Base/reference product is stored separately."

        # Teacher statistics cannot perturb the paired student initialization or augmentation stream.
        set_seed(seed)
        student = build_student(student_init_mode, teacher=teacher)
        initial_hash = hash_state_dict(student)
        pairing_key = (seed, student_init_mode)
        if pairing_key in INITIAL_STATE_HASH_CACHE and INITIAL_STATE_HASH_CACHE[pairing_key] != initial_hash:
            raise RuntimeError("Paired experiments have different student initialization hashes.")
        INITIAL_STATE_HASH_CACHE[pairing_key] = initial_hash
        set_seed(seed + 50000)
        train_loader.generator.manual_seed(seed)
        save_dir = Path(RUNS_ROOT, experiment, f"seed_{seed}_init_{student_init_mode}")
        save_dir.mkdir(parents=True, exist_ok=True)
        for filename in ("result.json", "feature_gradient_diagnostics.csv"):
            stale = save_dir / filename
            if stale.exists():
                stale.unlink()
        run_id = f"{CONFIG_HASH}|{experiment}|seed={seed}|init={student_init_mode}"
        atomic_json(save_dir / "config.json", {"run_id": run_id, "config_hash": CONFIG_HASH,
            "method_version": CONFIG["method_version"], "run_seed": seed, "run_experiment": experiment,
            "run_student_init_mode": student_init_mode, "config": CONFIG, **control_metadata})
        group_stats.to_csv(save_dir / "teacher_group_cam_statistics.csv", index=False)
        np.savez_compressed(save_dir / "split_indices.npz", train=np.asarray(train_ds.indices),
                            validation=np.asarray(val_ds.indices))
        split_hash = hashlib.sha256(np.asarray(train_ds.indices, dtype=np.int64).tobytes()).hexdigest()
        provenance = {"torch": torch.__version__, "cuda": torch.version.cuda, "device": str(DEVICE),
            "gpu": torch.cuda.get_device_name(DEVICE) if DEVICE.type == "cuda" else "CPU",
            "student_initial_sha256": initial_hash, "teacher_checkpoint_sha256": file_sha256(teacher_path),
            "train_indices_sha256": split_hash, "config_hash": CONFIG_HASH,
            "method_version": CONFIG["method_version"], "group_f1_definition": CONFIG["group_f1_definition"],
            **control_metadata}
        atomic_json(save_dir / "provenance.json", provenance)

        optimizer = optim.Adam(student.parameters(), lr=LR, betas=ADAM_BETAS, weight_decay=WEIGHT_DECAY)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max",
            factor=PLATEAU_FACTOR, patience=PLATEAU_PATIENCE)
        history, gradient_rows, group_history, outcome_history, balance_history = [], [], [], [], []
        best_f1, best_epoch, no_improvement = -np.inf, 0, 0
        early_stopped = False
        started = time.perf_counter()
        components = ("correct", "error", "raw", "difference", "coincidence")
        loss_names = ("ce", "guidance_before_q", "guidance", "guidance_weighted", "raw_baseline") + tuple(
            name + "_component" for name in components)
        print(f"{experiment} | seed={seed} | alpha={CAM_ALPHA:g} | K={WHERE_TOP_K} | beta={WHERE_BETA:g}")
        print(f"Correct: {WHERE_CORRECT_MODE}/{WHERE_CORRECT_MASK_MODE} | error: {WHERE_ERROR_MODE}/{WHERE_ERROR_MASK_MODE}")
        print(f"Outcome controls: QUANTO={AMOUNT_APPLY_TO} | WHERE={WHERE_APPLY_TO} | "
            f"global CAM scale={control_metadata['cam_scale_applied']:.6g} for every method")
        print("Initial student:", initial_hash[:16], "| test used only after checkpoint selection")
        for epoch in range(1, EPOCHS + 1):
            student.train()
            teacher.eval()
            if DEVICE.type == "cuda":
                torch.cuda.reset_peak_memory_stats(DEVICE)
                torch.cuda.synchronize(DEVICE)
            epoch_started = time.perf_counter()
            sums = {name: 0. for name in ("total",) + loss_names}
            n_train = 0
            accumulator, outcome_accumulator = ClassDiagnosticAccumulator(), BranchDiagnosticAccumulator()
            balance_accumulator = EpochLossBalanceAccumulator(class_to_group)
            for batch_id, (images, labels, sample_ids) in enumerate(train_loader):
                images, labels = images.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                logits_s, features_s = student(images)
                total, diagnostics, maps, losses = qw_cam_loss(experiment, logits_s, features_s,
                    student.fc, teacher, labels, images, class_to_group, stats, sample_ids, seed, epoch)
                if not torch.isfinite(total):
                    raise FloatingPointError(f"Nonfinite loss at epoch {epoch}, batch {batch_id}")
                if batch_id < GRAD_DIAG_MAX_BATCHES:
                    gradient_rows.extend(feature_gradient_diagnostics(losses, features_s, class_to_group,
                        labels, epoch, batch_id, diagnostics=diagnostics))
                balance_accumulator.update(logits_s, labels, diagnostics, maps, losses)
                total.backward()
                optimizer.step()
                batch_size = labels.numel()
                n_train += batch_size
                sums["total"] += total.item() * batch_size
                for name in loss_names:
                    sums[name] += losses[name].item() * batch_size
                accumulator.update(labels, diagnostics)
                outcome_accumulator.update(labels, diagnostics)
            if DEVICE.type == "cuda":
                torch.cuda.synchronize(DEVICE)
            train_seconds = time.perf_counter() - epoch_started
            peak_mb = torch.cuda.max_memory_allocated(DEVICE) / 2**20 if DEVICE.type == "cuda" else np.nan
            val_metrics, y_val, pred_val, prob_val = evaluate_with_arrays(student, val_loader, class_counts)
            # Historical fixed-class Macro-F1 checkpoint selection is unchanged.
            val_f1 = float(f1_score(y_val, pred_val, labels=np.arange(NUM_CLASSES), average="macro", zero_division=0))
            lr_now = optimizer.param_groups[0]["lr"]
            scheduler.step(val_f1)
            class_frame, group_frame = accumulator.frames(class_to_group, epoch)
            class_outcome, group_outcome = outcome_accumulator.frames(class_to_group, epoch)
            balance_frame = balance_accumulator.frame(epoch)
            group_history.append(group_frame)
            outcome_history.append(group_outcome)
            balance_history.append(balance_frame)
            all_balance = balance_frame.query("group == 'all' and partition == 'all'").iloc[0]
            for actual, expected in ((all_balance.ce_objective_sum, CE_WEIGHT * sums["ce"]),
                    (all_balance.method_objective_sum, sums["guidance_weighted"]),
                    (all_balance.guidance_before_q_sum, sums["guidance_before_q"]),
                    (all_balance.guidance_after_q_sum, sums["guidance"])):
                np.testing.assert_allclose(actual, expected, rtol=5e-6, atol=1e-8)
            ce_objective_sum = CE_WEIGHT * sums["ce"]
            ratio = sums["guidance_weighted"] / ce_objective_sum if ce_objective_sum > 0 else np.nan
            record = {"epoch": epoch, **control_metadata, "train_n_samples": n_train, "train_total": sums["total"] / n_train,
                "train_total_sum": sums["total"], "train_ce": sums["ce"] / n_train,
                "train_ce_weighted": ce_objective_sum / n_train, "train_ce_unweighted_sum": sums["ce"],
                "train_ce_weighted_sum": ce_objective_sum,
                "train_guidance_before_q": sums["guidance_before_q"] / n_train,
                "train_guidance_after_q": sums["guidance"] / n_train,
                "train_guidance_before_q_sum": sums["guidance_before_q"],
                "train_guidance_after_q_sum": sums["guidance"],
                "train_method_weighted": sums["guidance_weighted"] / n_train,
                "train_method_weighted_sum": sums["guidance_weighted"], "train_method_to_ce_ratio": ratio,
                "train_guidance_to_ce_ratio": ratio, "train_raw_baseline": sums["raw_baseline"] / n_train,
                "train_amount_applied_weight_mean": float(all_balance.q_conditional_mean),
                "train_amount_applied_weight_valid_n": int(all_balance.q_valid_n),
                "train_amount_applied_fraction": float(all_balance.amount_applied_conditional_mean),
                "train_amount_applied_n": int(all_balance.amount_applied_sum),
                "train_amount_selected_outcome_fraction": float(all_balance.amount_selected_outcome_conditional_mean),
                "train_where_selected_outcome_fraction": float(all_balance.where_selected_outcome_conditional_mean),
                "train_where_applied_fraction": float(all_balance.where_applied_conditional_mean),
                "train_cam_effective_coefficient_mean": float(all_balance.cam_effective_coefficient_conditional_mean),
                "val_ce_loss": val_metrics["ce_loss"], "val_acc": val_metrics["acc"],
                "val_f1_macro": val_f1, "val_balanced_acc": val_metrics["balanced_acc"],
                "val_many_shot_f1": val_metrics["many_shot_f1"],
                "val_medium_shot_f1": val_metrics["medium_shot_f1"], "val_few_shot_f1": val_metrics["few_shot_f1"],
                "lr": lr_now, "train_seconds": train_seconds,
                "images_per_second": n_train / max(train_seconds, 1e-12), "peak_cuda_mb": peak_mb}
            for name in components:
                record["train_" + name + "_component"] = sums[name + "_component"] / n_train
                record["train_" + name + "_weighted"] = CAM_ALPHA * sums[name + "_component"] / n_train
                record["train_" + name + "_weighted_sum"] = CAM_ALPHA * sums[name + "_component"]
            for _, row in balance_frame.query("group == 'all' and partition == 'guidance_branch'").iterrows():
                record["train_" + row.stratum + "_n"] = int(row.n_samples)
                record["train_" + row.stratum + "_fraction"] = row.sample_fraction_of_epoch
            history.append(record)
            if val_f1 > best_f1 + ES_MIN_DELTA:
                best_f1, best_epoch, no_improvement = val_f1, epoch, 0
                torch.save(student.state_dict(), save_dir / "best_model_f1.pth")
            else:
                no_improvement += 1
            should_stop = USE_EARLY_STOPPING and no_improvement >= ES_PATIENCE
            pd.DataFrame(history).to_csv(save_dir / "history.csv", index=False)
            for filename, frame in (("train_diagnostics_per_class.csv", class_frame),
                    ("train_diagnostics_per_group.csv", group_frame),
                    ("train_diagnostics_class_outcome.csv", class_outcome),
                    ("train_diagnostics_group_outcome.csv", group_outcome), ("train_loss_balance.csv", balance_frame)):
                frame.to_csv(save_dir / filename, index=False, mode="w" if epoch == 1 else "a", header=epoch == 1)
            current_gradients = [row for row in gradient_rows if row["epoch"] == epoch]
            if current_gradients:
                path = save_dir / "feature_gradient_diagnostics.csv"
                first_write = not path.exists() or path.stat().st_size == 0
                pd.DataFrame(current_gradients).to_csv(path, index=False,
                    mode="w" if first_write else "a", header=first_write)
            if epoch % CKPT_INTERVAL == 0 or should_stop:
                torch.save({"model_state_dict": student.state_dict(), "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict(), "epoch": epoch, "history": history,
                    "best_f1": best_f1, "best_epoch": best_epoch, "config": CONFIG,
                    "torch_rng": torch.get_rng_state(), "numpy_rng": np.random.get_state(),
                    "python_rng": random.getstate()}, save_dir / f"checkpoint_epoch{epoch}.pth")
            if epoch == 1 or epoch % 10 == 0 or should_stop:
                print(f"epoch {epoch:3d} | CE={record['train_ce_weighted']:.4f} | method/CE={ratio:.3e} | "
                    f"val F1={val_f1:.4f} | few F1={record['val_few_shot_f1']:.4f} | {train_seconds:.1f}s")
            if should_stop:
                early_stopped = True
                break

        student.load_state_dict(torch.load(save_dir / "best_model_f1.pth", map_location=DEVICE, weights_only=True))
        plot_training_history(pd.DataFrame(history), pd.concat(group_history, ignore_index=True), best_epoch, save_dir)
        full_balance = pd.concat(balance_history, ignore_index=True)
        best_balance = full_balance[full_balance.epoch == best_epoch].copy()
        observed_controls = observed_outcome_control_metadata(best_balance)
        for index, row in group_stats.iterrows():
            group = row["group"]
            selected = best_balance[(best_balance.group == group) & (best_balance.partition == "all")].iloc[0]
            group_stats.loc[index, "amount_factor_applied_observed_mean_at_best"] = observed_controls["amount_factor_applied_" + group]
            group_stats.loc[index, "amount_factor_applied_observed_valid_n_at_best"] = observed_controls["amount_factor_applied_valid_n_" + group]
            group_stats.loc[index, "cam_effective_coefficient_observed_mean_at_best"] = observed_controls["cam_effective_coefficient_" + group]
            group_stats.loc[index, "q_times_raw_mass_diagnostic"] = selected.q_times_teacher_raw_target_cam_l1_conditional_mean
        group_stats.to_csv(save_dir / "teacher_group_cam_statistics.csv", index=False)
        provenance["observed_training_at_best"] = observed_controls
        atomic_json(save_dir / "provenance.json", provenance)
        saved_config = json.loads((save_dir / "config.json").read_text(encoding="utf-8"))
        saved_config["observed_training_at_best"] = observed_controls
        atomic_json(save_dir / "config.json", saved_config)
        best_balance.to_csv(save_dir / "train_loss_balance_best.csv", index=False)
        atomic_json(save_dir / "train_loss_balance.json", full_balance.to_dict("split"))
        atomic_json(save_dir / "train_loss_balance_best.json", best_balance.to_dict("records"))
        atomic_json(save_dir / "loss_balance_metadata.json", {
            "method_version": CONFIG["method_version"], "ce_weight": CE_WEIGHT, "cam_alpha": CAM_ALPHA,
            **control_metadata, "observed_training_at_best": observed_controls,
            "partitions": "all, teacher_outcome and guidance_branch are separate overlapping views; each partition's strata are disjoint.",
            "conditional_mean": "sum divided by finite valid_n for that group/stratum.",
            "population_mean_contribution": "sum divided by every processed training image in the epoch.",
            "ratio": "Ratio of population sums, never the mean of batch ratios.",
            "guidance_before_q": "Signed target-CAM spatial MSE D divided by the historical global sT for ALL methods; includes unit-mean spatial weights and excludes actual q and alpha.",
            "guidance_after_q": "Spatial MSE against r_applied*teacher_raw, divided by sT, before alpha.", "ce_objective": "CE_WEIGHT times per-image CE.",
            "method_objective": "CAM_ALPHA times the spatial MSE against r_applied*teacher_raw, divided by sT.",
            "amount_selection": "The teacher-sign factor map is applied only when QUANTO is enabled and AMOUNT_APPLY_TO selects the original teacher outcome; otherwise every factor is exactly one. The q column is its spatial mean. No normalization follows this selection.",
            "where_selection": "WHERE_APPLY_TO independently selects the original teacher outcome; outside that selection the auxiliary objective remains raw signed CAM MSE.",
            "global_cam_scale": "cam_scale_reference=max(teacher_raw_cam_mean_sq,CAM_SCALE_EPS), applied to QW0/QW1/QW2/QW3 without outcome gating.",
            "effective_coefficient": "CAM_ALPHA/sT multiplies the spatial squared error; sign/group r_applied scales only the teacher target inside that error. Actual group means and conditional teacher-outcome means use observed training images.",
            "raw_target_cam_mass": "Unchanged signed teacher target CAM; L1 is the spatial mean of its absolute value.",
            "class_balanced_mean": "Equal mean over observed per-class image means in the selected group/stratum.",
            "static_quanto_statistics": "teacher_group_cam_statistics.csv uses deterministic training images and class-balanced group means; dynamic logging uses augmented images.",
            "mass_vs_objective": "the directly measured signed-rectified CAM mass is the teacher target mass, not a loss or gradient contribution.",
            "spatial_weighted_mass": "Mean of abs(original target CAM) times the applied spatial weight, with and without q; a diagnostic of the original and q-scaled teacher targets, not a loss or gradient.",
            "missing_values": "Undefined means/ratios are null in JSON and empty numeric values in CSV/XLSX."})

        final_metrics, per_class_frames, classification_frames = {}, [], []
        for split, loader, sample_ids in (("val", val_loader, val_ds.indices), ("test", test_loader, None)):
            teacher_metrics, ty, tp, tprob = evaluate_with_arrays(teacher, loader, class_counts)
            student_metrics, sy, sp, sprob = evaluate_with_arrays(student, loader, class_counts)
            np.testing.assert_array_equal(ty, sy)
            save_predictions(save_dir / f"teacher_{split}_predictions.csv", ty, tp, tprob, sample_ids)
            save_predictions(save_dir / f"student_{split}_predictions.csv", sy, sp, sprob, sample_ids)
            for name in teacher_metrics:
                final_metrics[f"teacher_{split}_{name}"] = teacher_metrics[name]
                final_metrics[f"student_{split}_{name}"] = student_metrics[name]
                final_metrics[f"gain_vs_teacher_{split}_{name}"] = student_metrics[name] - teacher_metrics[name]
            per_class = classification_diagnostics(sy, tp, sp, class_counts, class_to_group)
            per_class.insert(0, "split", split)
            per_class_frames.append(per_class)
            classification_frames.append(classification_outcome_diagnostics(sy, tp, sp, class_to_group, split))
        pd.concat(per_class_frames, ignore_index=True).to_csv(save_dir / "per_class_classification_diagnostics.csv", index=False)
        classification_outcomes = pd.concat(classification_frames, ignore_index=True)
        classification_outcomes.to_csv(save_dir / "classification_outcome_diagnostics.csv", index=False)
        probes = pd.concat([run_fixed_probe(student, teacher, experiment, seed, class_to_group, stats,
            split, indices, str(save_dir)) for split, indices in (("train", train_ds.indices), ("val", val_ds.indices))], ignore_index=True)
        group_probe_summary(probes).to_csv(save_dir / "probe_group_summary.csv", index=False)
        group_probe_summary(probes, by_outcome=True).to_csv(save_dir / "probe_group_correct_wrong.csv", index=False)
        probes.groupby(["split", "class_id", "group"], as_index=False).mean(numeric_only=True).to_csv(
            save_dir / "probe_class_summary.csv", index=False)

        best_history = history[best_epoch - 1]
        result = {"run_id": run_id, "config_hash": CONFIG_HASH, "method_version": CONFIG["method_version"],
            "experiment": experiment, "description": EXPERIMENT_DEFINITIONS[experiment]["description"],
            "use_amount": EXPERIMENT_DEFINITIONS[experiment]["use_amount"],
            "use_where": EXPERIMENT_DEFINITIONS[experiment]["use_where"], "seed": seed,
            "student_init_mode": student_init_mode, "best_epoch": best_epoch, "best_val_f1_macro": best_f1,
            "epochs_ran": len(history), "early_stopped": early_stopped, "cam_alpha": CAM_ALPHA,
            "amount_power": AMOUNT_POWER, "amount_normalization": AMOUNT_NORMALIZATION,
            "amount_compensation": "positive_negative_teacher_target",
            "amount_expected_weight": stats["amount_expected_weight"], "teacher_ckpt_path": teacher_path,
            **control_metadata, **observed_controls,
            "where_k": WHERE_TOP_K, "where_beta": WHERE_BETA, "where_correct_mode": WHERE_CORRECT_MODE,
            "where_error_mode": WHERE_ERROR_MODE, "where_correct_mask_mode": WHERE_CORRECT_MASK_MODE,
            "where_error_mask_mode": WHERE_ERROR_MASK_MODE, "where_groups": ",".join(WHERE_GROUPS),
            "guidance_definition": "Signed target-CAM spatial MSE against sign/group-rectified teacher reference, divided by original global sT for every method. QUANTO and WHERE have independent teacher-outcome filters; unselected QUANTO uses q=1 and unselected WHERE uses raw CAM MSE. No second q normalization. Objective is CE_WEIGHT*CE + CAM_ALPHA*mean(mean_spatial(weight*(student_raw-r_applied*teacher_raw)^2)/sT).",
            "student_initial_sha256": initial_hash, "teacher_checkpoint_sha256": provenance["teacher_checkpoint_sha256"],
            "train_indices_sha256": split_hash, "save_dir": str(save_dir),
            "mean_train_epoch_seconds": float(np.mean([row["train_seconds"] for row in history])),
            "total_seconds_including_final_diagnostics": time.perf_counter() - started, **final_metrics}
        result.update({name + "_at_best": value for name, value in best_history.items() if name.startswith("train_")})
        for group_id in range(3):
            group = GROUP_ID_TO_NAME[group_id]
            result[f"amount_factor_computed_{group}"] = float(stats["amount_factors"][group_id])
            # Actual applied factors/coefficient means already came from observed
            # best-epoch group/outcome rows, not a static group-only assignment.
            result[f"teacher_group_abs_{group}"] = float(stats["group_abs_mean"][group_id])
            result[f"sample_proportion_{group}"] = float(stats["group_sample_proportions"][group_id])
            selected = group_history[best_epoch - 1].query("group == @group").iloc[0]
            for name, value in selected.items():
                if name not in {"epoch", "group"} and isinstance(value, (int, float, np.number)):
                    result[f"{group}_{name}_at_best"] = float(value)
        for _, row in best_balance[best_balance.partition == "all"].iterrows():
            for name in ("n_samples", "guidance_before_q_sum", "guidance_after_q_sum",
                    "guidance_share_of_epoch_before_q", "guidance_share_of_epoch_after_q",
                    "method_objective_population_mean_contribution", "method_to_ce_ratio_of_sums",
                    "teacher_raw_target_cam_l1_conditional_mean", "q_times_teacher_raw_target_cam_l1_conditional_mean",
                    "teacher_raw_target_cam_l1_class_balanced_mean", "q_times_teacher_raw_target_cam_l1_class_balanced_mean"):
                result[f"{row['group']}_loss_balance_{name}_at_best"] = float(row[name])
        # Written last: this file is the authoritative completed-run marker.
        atomic_json(save_dir / "result.json", result)
        return result, group_stats
else:  # mobilenetv2_imagenet_lt
    def json_compatible(value):
        if isinstance(value, dict):
            return {str(key): json_compatible(item) for key, item in value.items()}
        if isinstance(value, (list, tuple)):
            return [json_compatible(item) for item in value]
        if isinstance(value, np.ndarray):
            return json_compatible(value.tolist())
        if isinstance(value, np.generic):
            return json_compatible(value.item())
        if isinstance(value, float) and not np.isfinite(value):
            return None
        if isinstance(value, Path):
            return str(value)
        return value


    def atomic_json(path, value):
        target = Path(path)
        temporary = target.with_name(target.name + ".writing")
        try:
            temporary.write_text(json.dumps(json_compatible(value), ensure_ascii=False,
                indent=2, allow_nan=False), encoding="utf-8")
            os.replace(temporary, target)
        finally:
            if temporary.exists():
                temporary.unlink()


    INITIAL_STATE_HASH_CACHE = {}


    def save_predictions(path, labels, preds, probs, ids=None):
        labels, preds, probs = np.asarray(labels), np.asarray(preds), np.asarray(probs)
        sample_ids = np.arange(len(labels)) if ids is None else np.asarray(ids)
        if not (len(labels) == len(preds) == len(probs) == len(sample_ids)):
            raise ValueError("Prediction arrays and sample IDs must have identical lengths.")
        frame = pd.DataFrame({"sample_id": sample_ids, "true_class": labels,
            "predicted_class": preds, "correct": labels == preds,
            "target_probability": probs[np.arange(len(labels)), labels],
            "predicted_probability": probs.max(1)})
        frame.to_csv(path, index=False)
        np.savez_compressed(str(Path(path).with_suffix(".npz")), labels=labels,
            predictions=preds, probabilities=probs, sample_ids=sample_ids)


    def classification_diagnostics(labels, teacher_pred, student_pred, class_counts, class_to_group):
        # Complementary class diagnostics do not replace historical group F1.
        from sklearn.metrics import precision_recall_fscore_support
        table = pd.DataFrame({"class_id": np.arange(NUM_CLASSES), "class_name": _CLASS_NAMES,
            "n_train": class_counts, "group": [GROUP_ID_TO_NAME[int(group)] for group in class_to_group.cpu()]})
        for prefix, predictions in (("teacher", teacher_pred), ("student", student_pred)):
            precision, recall, f1, support = precision_recall_fscore_support(
                labels, predictions, labels=np.arange(NUM_CLASSES), zero_division=0)
            table[prefix + "_precision_diagnostic"] = precision
            table[prefix + "_recall"] = recall
            table[prefix + "_fullset_class_f1_diagnostic"] = f1
            table[prefix + "_support"] = support
        table["delta_recall"] = table.student_recall - table.teacher_recall
        table["delta_fullset_class_f1_diagnostic"] = table.student_fullset_class_f1_diagnostic - table.teacher_fullset_class_f1_diagnostic
        return table


    def outcome_control_metadata(experiment, teacher_stats):
        """Static policy and potential coefficients, never an empirical applied-q mean.

        The historical global CAM-squared scale is applied to EVERY experiment.
        Only q_G and the spatial weighting are selected by their independent
        teacher-outcome filters. No renormalization follows that selection.
        """
        cfg = EXPERIMENT_DEFINITIONS[experiment]
        estimate = float(teacher_stats["raw_cam_mean_sq"])
        scale = max(estimate, float(CAM_SCALE_EPS))
        coefficient = float(CAM_ALPHA) / scale
        metadata = {"amount_apply_to": AMOUNT_APPLY_TO, "where_apply_to": WHERE_APPLY_TO,
            "amount_enabled": bool(cfg["use_amount"]), "where_enabled": bool(cfg["use_where"]),
            "amount_scope_definition": "signed_group_target_factors_on_selected_teacher_outcome_without_renormalization",
            "amount_expected_weight_base": float(teacher_stats["amount_expected_weight"]),
            "amount_expected_weight_base_definition": "Original deterministic-training expectation of the spatial mean of sign/group factors before teacher-outcome selection. Applied q is not renormalized and its observed expectation need not equal one.",
            "teacher_raw_cam_mean_sq": estimate, "cam_scale_eps": float(CAM_SCALE_EPS),
            "cam_scale_reference": scale, "cam_scale_applied": scale, "cam_scale_enabled": True,
            "cam_scale_scope": "all_experiments", "cam_global_coefficient": coefficient,
            "amount_factor_outside_selection": 1.,
            "cam_effective_coefficient_outside_amount_selection": coefficient}
        for group_id in range(3):
            group = GROUP_ID_TO_NAME[group_id]
            base = float(teacher_stats["amount_factors"][group_id])
            selected = base if cfg["use_amount"] else 1.
            metadata["amount_factor_base_" + group] = base
            metadata["amount_factor_if_selected_" + group] = selected
            metadata["cam_effective_coefficient_if_selected_" + group] = coefficient
            for sign in ("positive","negative","zero"):
                value=float(teacher_stats["amount_factors_"+sign][group_id])
                metadata["amount_factor_"+sign+"_base_"+group]=value
                metadata["amount_factor_"+sign+"_if_selected_"+group]=value if cfg["use_amount"] else 1.
        return metadata


    def observed_outcome_control_metadata(balance_frame):
        """Observed image means in one training epoch, with conditional finite counts."""
        observed = {}
        for group in ("all", "many", "medium", "few"):
            group_rows = balance_frame[balance_frame.group == group]
            for outcome in ("all", "correct", "error"):
                partition = "all" if outcome == "all" else "teacher_outcome"
                row = group_rows[(group_rows.partition == partition) & (group_rows.stratum == outcome)].iloc[0]
                suffix = group if outcome == "all" else "teacher_" + outcome + "_" + group
                observed["amount_factor_applied_" + suffix] = float(row.q_conditional_mean)
                observed["amount_factor_applied_valid_n_" + suffix] = int(row.q_valid_n)
                observed["cam_effective_coefficient_" + suffix] = float(row.cam_effective_coefficient_conditional_mean)
                observed["cam_effective_coefficient_valid_n_" + suffix] = int(row.cam_effective_coefficient_valid_n)
                observed["amount_applied_fraction_" + suffix] = float(row.amount_applied_conditional_mean)
                observed["amount_applied_n_" + suffix] = int(row.amount_applied_sum)
                observed["amount_selected_outcome_fraction_" + suffix] = float(row.amount_selected_outcome_conditional_mean)
                observed["amount_selected_outcome_n_" + suffix] = int(row.amount_selected_outcome_sum)
                observed["where_selected_outcome_fraction_" + suffix] = float(row.where_selected_outcome_conditional_mean)
                observed["where_selected_outcome_n_" + suffix] = int(row.where_selected_outcome_sum)
                observed["where_applied_fraction_" + suffix] = float(row.where_applied_conditional_mean)
                observed["where_applied_n_" + suffix] = int(row.where_applied_sum)
                for sign in ("positive","negative","zero"):
                    observed["amount_factor_"+sign+"_applied_"+suffix]=float(row["amount_factor_"+sign+"_applied_conditional_mean"])
                    observed["amount_"+sign+"_fraction_"+suffix]=float(row["amount_"+sign+"_fraction_conditional_mean"])
        observed["amount_applied_weight_mean_at_best"] = observed["amount_factor_applied_all"]
        observed["amount_applied_weight_valid_n_at_best"] = observed["amount_factor_applied_valid_n_all"]
        observed["observed_training_epoch"] = int(balance_frame.epoch.iloc[0])
        observed["applied_factor_mean_definition"] = "Observed image mean in the selected training epoch, with teacher-correct/error conditional means reported separately; empty strata have null means and zero valid_n. The applied q is a spatial factor mean, distinct from the static group mean and the separate sign coefficients."
        return observed


    class EpochLossBalanceAccumulator:
        """Full-epoch sums and explicit denominators for signed spatial CAM guidance.

        Partitions overlap with one another; strata within each partition are
        disjoint and complete. Objective contributions use ALL epoch images as
        denominator. Conditional means use only the selected group's/stratum's
        images. Raw CAM masses and q-scaled reference masses are recorded separately.
        guidance_before_q already includes the global CAM-squared denominator.
        q is the spatial mean of the ACTUAL outcome-selected sign-factor map; no second normalization is used.
        """
        def __init__(self, class_to_group):
            self.class_to_group = class_to_group.detach().cpu().numpy()
            self.groups = ("all", "many", "medium", "few")
            self.strata = (("all", "all"), ("teacher_outcome", "correct"),
                ("teacher_outcome", "error"), ("guidance_branch", "raw"),
                ("guidance_branch", "difference"), ("guidance_branch", "coincidence"))
            self.mass_names = ("teacher_raw_target_cam_l1", "q_times_teacher_raw_target_cam_l1",
                "teacher_raw_target_cam_positive_mass", "teacher_raw_target_cam_negative_mass",
                "teacher_raw_target_cam_signed_mean", "teacher_spatial_weighted_target_cam_l1",
                "q_times_teacher_spatial_weighted_target_cam_l1",
                "teacher_rectified_target_cam_positive_mass","teacher_rectified_target_cam_negative_mass")
            self.rows = {(group, partition, stratum): {"n": 0, "classes": set(),
                "sum": {}, "valid": {}, "class_sum": {}, "class_valid": {}}
                for group in self.groups for partition, stratum in self.strata}

        def update(self, logits, labels, diagnostics, maps, losses):
            with torch.no_grad():
                ce = F.cross_entropy(logits.detach(), labels, reduction="none")
                q = losses["amount_per_image"].detach()
                guidance = losses["guidance_per_image"].detach()
                guidance_after_q = losses["guidance_after_q_per_image"].detach()
                raw_after_q = losses["raw_after_q_per_image"].detach()
                raw_loss = losses["raw_per_image"].detach()
                raw_mass = diagnostics["raw_abs_mass"].detach()
                positive = diagnostics["raw_pos_mass"].detach()
                negative = diagnostics["raw_neg_mass"].detach()
                correct = diagnostics["teacher_correct"].bool()
                branch_masks = {name: diagnostics[name + "_branch"].bool()
                                for name in ("raw", "difference", "coincidence")}
                if not sum(mask.long() for mask in branch_masks.values()).eq(1).all():
                    raise RuntimeError("Spatial-guidance branches do not partition the batch.")
                values = {"ce_unweighted": ce, "guidance_before_q": guidance,
                    "guidance_after_q": guidance_after_q, "raw_baseline_before_q": raw_loss,
                    "raw_baseline_after_q": raw_after_q, "ce_objective": CE_WEIGHT * ce,
                    "method_objective": CAM_ALPHA * guidance_after_q,
                    "total_objective": CE_WEIGHT * ce + CAM_ALPHA * guidance_after_q,
                    "amount_factor_base": diagnostics["amount_factor_base"].detach(),
                    "amount_selected_outcome": diagnostics["amount_selected_outcome"].detach(),
                    "amount_applied": diagnostics["amount_applied"].detach(),
                    "where_selected_outcome": diagnostics["where_selected_outcome"].detach(),
                    "where_applied": diagnostics["where_applied"].detach(),
                    "cam_scale_reference": diagnostics["cam_scale_reference"].detach(),
                    "cam_scale_applied": diagnostics["cam_scale_applied"].detach(),
                    "cam_global_coefficient": diagnostics["cam_global_coefficient"].detach(),
                    "cam_effective_coefficient": diagnostics["cam_effective_coefficient"].detach(),
                    "q": q, "teacher_raw_target_cam_l1": raw_mass,
                    "q_times_teacher_raw_target_cam_l1": diagnostics["target_abs_mass_after_q"].detach(),
                    "teacher_spatial_weighted_target_cam_l1": diagnostics["spatial_weighted_raw_abs_mass"].detach(),
                    "q_times_teacher_spatial_weighted_target_cam_l1": diagnostics["q_spatial_weighted_raw_abs_mass"].detach(),
                    "teacher_raw_target_cam_positive_mass": positive,
                    "teacher_raw_target_cam_negative_mass": negative,
                    "teacher_raw_target_cam_signed_mean": positive - negative,
                    "teacher_rectified_target_cam_positive_mass":diagnostics["target_pos_mass_after_q"].detach(),
                    "teacher_rectified_target_cam_negative_mass":diagnostics["target_neg_mass_after_q"].detach()}
                for sign in ("positive","negative","zero"):
                    values["amount_factor_"+sign+"_applied"]=diagnostics["amount_factor_"+sign+"_applied"].detach()
                    values["amount_"+sign+"_fraction"]=diagnostics["amount_"+sign+"_fraction"].detach()
                for name, mask in {"correct": correct, "error": ~correct, **branch_masks}.items():
                    component = guidance * mask.to(guidance.dtype)
                    values[name + "_before_q"] = component
                    values[name + "_after_q"] = guidance_after_q * mask.to(guidance_after_q.dtype)
                    values[name + "_objective"] = CAM_ALPHA * values[name + "_after_q"]
                names = list(values)
                matrix = torch.stack([values[name] for name in names], 1).double().cpu().numpy()
                y = labels.detach().cpu().numpy()
                masks = {("all", "all"): np.ones(len(y), dtype=bool),
                    ("teacher_outcome", "correct"): correct.cpu().numpy(),
                    ("teacher_outcome", "error"): (~correct).cpu().numpy()}
                masks.update({("guidance_branch", name): mask.cpu().numpy() for name, mask in branch_masks.items()})
            if not np.isfinite(matrix).all():
                raise FloatingPointError("Nonfinite per-image objective or raw teacher CAM mass.")
            for group in self.groups:
                group_mask = np.ones(len(y), dtype=bool) if group == "all" else (
                    self.class_to_group[y] == GROUP_NAME_TO_ID[group])
                for (partition, stratum), stratum_mask in masks.items():
                    mask = group_mask & stratum_mask
                    source = self.rows[(group, partition, stratum)]
                    source["n"] += int(mask.sum())
                    source["classes"].update(y[mask].tolist())
                    selected = matrix[mask]
                    for index, name in enumerate(names):
                        source["sum"][name] = source["sum"].get(name, 0.) + float(selected[:, index].sum())
                        source["valid"][name] = source["valid"].get(name, 0) + int(mask.sum())
                        if name in self.mass_names:
                            sums = source["class_sum"].setdefault(name, {})
                            counts = source["class_valid"].setdefault(name, {})
                            for class_id in np.unique(y[mask]):
                                class_mask = y[mask] == class_id
                                sums[int(class_id)] = sums.get(int(class_id), 0.) + float(selected[class_mask, index].sum())
                                counts[int(class_id)] = counts.get(int(class_id), 0) + int(class_mask.sum())

        def frame(self, epoch):
            total = self.rows[("all", "all", "all")]
            population_n = total["n"]
            def ratio(numerator, denominator):
                return numerator / denominator if denominator > 0 else np.nan
            records = []
            for (group, partition, stratum), source in self.rows.items():
                group_all = self.rows[(group, "all", "all")]
                row = {"epoch": epoch, "group": group, "partition": partition, "stratum": stratum,
                    "n_samples": source["n"], "n_classes_observed": len(source["classes"]),
                    "n_epoch_samples": population_n, "n_group_samples": group_all["n"],
                    "sample_fraction_of_epoch": ratio(source["n"], population_n),
                    "sample_fraction_within_group": ratio(source["n"], group_all["n"])}
                for name, value in source["sum"].items():
                    valid = source["valid"][name]
                    row[name + "_sum"] = value
                    row[name + "_valid_n"] = valid
                    row[name + "_conditional_mean"] = value / valid if valid else np.nan
                    row[name + "_population_mean_contribution"] = ratio(value, population_n)
                    if name in self.mass_names:
                        means = [value / source["class_valid"][name][class_id]
                                 for class_id, value in source["class_sum"][name].items()]
                        row[name + "_class_balanced_mean"] = float(np.mean(means)) if means else np.nan
                        row[name + "_classes_valid_n"] = len(means)
                for suffix in ("before_q", "after_q"):
                    metric = "guidance_" + suffix
                    row["guidance_share_of_epoch_" + suffix] = ratio(source["sum"][metric], total["sum"][metric])
                    row["guidance_share_within_group_" + suffix] = ratio(source["sum"][metric], group_all["sum"][metric])
                for suffix, metric in (("before_q", "teacher_raw_target_cam_l1"),
                        ("after_q", "q_times_teacher_raw_target_cam_l1")):
                    row["raw_target_cam_l1_share_of_epoch_" + suffix] = ratio(source["sum"][metric], total["sum"][metric])
                row["method_to_ce_ratio_of_sums"] = ratio(source["sum"]["method_objective"], source["sum"]["ce_objective"])
                row["raw_mass_reweighting_ratio_of_sums"] = ratio(
                    source["sum"]["q_times_teacher_raw_target_cam_l1"], source["sum"]["teacher_raw_target_cam_l1"])
                records.append(row)
            frame = pd.DataFrame(records)
            check_metrics = ("ce_objective_sum", "method_objective_sum", "guidance_before_q_sum",
                "guidance_after_q_sum", "teacher_raw_target_cam_l1_sum", "q_times_teacher_raw_target_cam_l1_sum")
            for group in self.groups:
                reference = frame[(frame.group == group) & (frame.partition == "all")].iloc[0]
                for partition in ("teacher_outcome", "guidance_branch"):
                    part = frame[(frame.group == group) & (frame.partition == partition)]
                    if int(part.n_samples.sum()) != int(reference.n_samples):
                        raise RuntimeError("Loss-balance sample counts do not close.")
                    for metric in check_metrics:
                        np.testing.assert_allclose(part[metric].sum(), reference[metric], rtol=1e-10, atol=1e-10)
            selected_groups = frame[(frame.group != "all") & (frame.partition == "all")]
            for metric in check_metrics:
                np.testing.assert_allclose(selected_groups[metric].sum(),
                    frame[(frame.group == "all") & (frame.partition == "all")].iloc[0][metric],
                    rtol=1e-10, atol=1e-10)
            return frame


    def run_experiment(experiment, seed, student_init_mode="random"):
        validate_v8_mobilenet_configuration()
        execution_metadata = dict(EXECUTION_PROFILE, history_write_every=ESSENTIAL_HISTORY_WRITE_EVERY,
            save_recovery_checkpoints=SAVE_RECOVERY_CHECKPOINTS)
        if experiment not in EXPERIMENT_DEFINITIONS:
            raise ValueError(experiment)
        set_seed(seed)
        train_ds, val_ds, test_ds, train_loader, val_loader, test_loader = build_loaders(seed)
        class_counts = get_class_counts(train_ds)
        class_to_group, groups = build_class_to_group(class_counts)
        if any(len(groups[name]) == 0 for name in ("many", "medium", "few")):
            raise ValueError("This QUANTO protocol requires nonempty many/medium/few groups.")
        teacher, teacher_path = load_teacher(seed)
        print("Validando o professor congelado...", flush=True)
        teacher_validation = check_teacher_baseline(teacher, val_loader, class_counts, seed)
        split_metadata = current_split_metadata()
        print("Calibrando q_g positivo/negativo e s_T no TRAIN determinístico (ou reutilizando cache)...", flush=True)
        stats = estimate_teacher_group_cam_statistics(seed, teacher,
            build_deterministic_train_loader(train_ds), class_counts, class_to_group)
        group_stats = print_group_statistics(stats)
        control_metadata = outcome_control_metadata(experiment, stats)
        # Reference statistics stay unchanged. Potential factors are separate from
        # the actual outcome-conditioned means, which are available after training.
        for name, value in control_metadata.items():
            if not name.startswith(("amount_factor_base_", "amount_factor_if_selected_",
                                    "cam_effective_coefficient_if_selected_")):
                group_stats[name] = value
        group_stats["amount_factor_base"] = stats["amount_factors"]
        group_stats["amount_factor_if_selected"] = [control_metadata["amount_factor_if_selected_" +
            GROUP_ID_TO_NAME[group_id]] for group_id in range(3)]
        group_stats["cam_effective_coefficient_if_selected"] = [control_metadata[
            "cam_effective_coefficient_if_selected_" + GROUP_ID_TO_NAME[group_id]] for group_id in range(3)]
        group_stats["base_q_times_reference_raw_mass_diagnostic"] = group_stats.q_times_raw_mass_diagnostic
        group_stats["q_times_raw_mass_diagnostic"] = np.nan
        group_stats["q_times_raw_mass_definition"] = "Observed mean of ACTUAL applied q times raw CAM L1 at the best training epoch; excludes alpha and the global denominator. Base/reference product is stored separately."

        # Teacher statistics cannot perturb the paired student initialization or augmentation stream.
        set_seed(seed)
        student = build_student(student_init_mode, teacher=teacher)
        initial_hash = hash_state_dict(student)
        pairing_key = (seed, student_init_mode)
        if pairing_key in INITIAL_STATE_HASH_CACHE and INITIAL_STATE_HASH_CACHE[pairing_key] != initial_hash:
            raise RuntimeError("Paired experiments have different student initialization hashes.")
        INITIAL_STATE_HASH_CACHE[pairing_key] = initial_hash
        set_seed(seed + 50000)
        train_loader.generator.manual_seed(seed)
        save_dir = Path(RUNS_ROOT, experiment, f"seed_{seed}_init_{student_init_mode}")
        save_dir.mkdir(parents=True, exist_ok=True)
        for filename in ("result.json", "feature_gradient_diagnostics.csv"):
            stale = save_dir / filename
            if stale.exists():
                stale.unlink()
        run_id = f"{CONFIG_HASH}|{experiment}|seed={seed}|init={student_init_mode}"
        atomic_json(save_dir / "config.json", {"run_id": run_id, "config_hash": CONFIG_HASH,
            "method_version": CONFIG["method_version"], "run_seed": seed, "run_experiment": experiment,
            "run_student_init_mode": student_init_mode, "config": CONFIG, "execution_profile": execution_metadata, **control_metadata})
        group_stats.to_csv(save_dir / "teacher_group_cam_statistics.csv", index=False)
        np.savez_compressed(save_dir / "split_indices.npz", train=np.asarray(train_ds.indices),
                            validation=np.asarray(val_ds.indices))
        split_hash = hashlib.sha256(np.asarray(train_ds.indices, dtype=np.int64).tobytes()).hexdigest()
        provenance = {"torch": torch.__version__, "cuda": torch.version.cuda, "device": str(DEVICE),
            "gpu": torch.cuda.get_device_name(DEVICE) if DEVICE.type == "cuda" else "CPU",
            "student_initial_sha256": initial_hash, "teacher_checkpoint_sha256": file_sha256(teacher_path),
            "train_indices_sha256": split_hash, "config_hash": CONFIG_HASH,
            "method_version": CONFIG["method_version"], "group_f1_definition": CONFIG["group_f1_definition"],
            "torchvision": torchvision.__version__, "teacher_validation_check": teacher_validation,
            "precision_policy": CONFIG["precision_policy"], "execution_profile": execution_metadata, **split_metadata, **control_metadata}
        atomic_json(save_dir / "provenance.json", provenance)
        atomic_json(save_dir / "teacher_baseline_validation.json", teacher_validation)

        optimizer = optim.Adam(student.parameters(), lr=LR, betas=ADAM_BETAS, weight_decay=WEIGHT_DECAY)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max",
            factor=PLATEAU_FACTOR, patience=PLATEAU_PATIENCE)
        history, gradient_rows, group_history, outcome_history, balance_history = [], [], [], [], []
        best_f1, best_epoch, no_improvement = -np.inf, 0, 0
        early_stopped = False
        started = time.perf_counter()
        components = ("correct", "error", "raw", "difference", "coincidence")
        loss_names = ("ce", "guidance_before_q", "guidance", "guidance_weighted", "raw_baseline") + tuple(
            name + "_component" for name in components)
        print(f"{experiment} | seed={seed} | alpha={CAM_ALPHA:g} | K={WHERE_TOP_K} | beta={WHERE_BETA:g}")
        print(f"Correct: {WHERE_CORRECT_MODE}/{WHERE_CORRECT_MASK_MODE} | error: {WHERE_ERROR_MODE}/{WHERE_ERROR_MASK_MODE}")
        print(f"Outcome controls: QUANTO={AMOUNT_APPLY_TO} | WHERE={WHERE_APPLY_TO} | "
            f"global CAM scale={control_metadata['cam_scale_applied']:.6g} for every method")
        print("Initial student:", initial_hash[:16], "| test used only after checkpoint selection")
        for epoch in range(1, EPOCHS + 1):
            student.train()
            teacher.eval()
            if DEVICE.type == "cuda":
                torch.cuda.reset_peak_memory_stats(DEVICE)
                torch.cuda.synchronize(DEVICE)
            epoch_started = time.perf_counter()
            sums = {name: 0. for name in ("total",) + loss_names}
            n_train = 0
            balance_accumulator = EssentialEpochAccumulator(class_to_group)
            scalar_names = list(sums)
            scalar_totals = torch.zeros(len(scalar_names), dtype=torch.float64, device=DEVICE)
            for batch_id, (images, labels, sample_ids) in enumerate(train_loader):
                images, labels = images.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                logits_s, features_s = student(images)
                total, diagnostics, maps, losses = qw_cam_loss(experiment, logits_s, features_s,
                    student.fc, teacher, labels, images, class_to_group, stats, sample_ids, seed, epoch)
                if not torch.isfinite(total):
                    raise FloatingPointError(f"Nonfinite loss at epoch {epoch}, batch {batch_id}")
                balance_accumulator.update(logits_s, labels, diagnostics, maps, losses)
                total.backward()
                optimizer.step()
                batch_size = labels.numel()
                n_train += batch_size
                # Detached logging only: no change to the total used by backward().
                scalar_values = [total.detach()] + [losses[name].detach() for name in loss_names]
                scalar_totals += torch.stack(scalar_values).double() * batch_size
            if DEVICE.type == "cuda":
                torch.cuda.synchronize(DEVICE)
            train_seconds = time.perf_counter() - epoch_started
            sums = dict(zip(scalar_names, scalar_totals.detach().cpu().tolist()))
            peak_mb = torch.cuda.max_memory_allocated(DEVICE) / 2**20 if DEVICE.type == "cuda" else np.nan
            validation_started = time.perf_counter()
            val_metrics, y_val, pred_val, prob_val = evaluate_epoch_essential(student, val_loader, class_counts)
            validation_seconds = time.perf_counter() - validation_started
            # Historical fixed-class Macro-F1 checkpoint selection is unchanged.
            val_f1 = float(f1_score(y_val, pred_val, labels=np.arange(NUM_CLASSES), average="macro", zero_division=0))
            lr_now = optimizer.param_groups[0]["lr"]
            scheduler.step(val_f1)
            group_frame = balance_accumulator.group_frame(epoch)
            balance_frame = balance_accumulator.frame(epoch)
            group_history.append(group_frame)
            balance_history.append(balance_frame)
            all_balance = balance_frame.query("group == 'all' and partition == 'all'").iloc[0]
            for actual, expected in ((all_balance.ce_objective_sum, CE_WEIGHT * sums["ce"]),
                    (all_balance.method_objective_sum, sums["guidance_weighted"]),
                    (all_balance.guidance_before_q_sum, sums["guidance_before_q"]),
                    (all_balance.guidance_after_q_sum, sums["guidance"])):
                np.testing.assert_allclose(actual, expected, rtol=5e-6, atol=1e-8)
            ce_objective_sum = CE_WEIGHT * sums["ce"]
            ratio = sums["guidance_weighted"] / ce_objective_sum if ce_objective_sum > 0 else np.nan
            record = {"epoch": epoch, **control_metadata, "train_n_samples": n_train, "train_total": sums["total"] / n_train,
                "train_total_sum": sums["total"], "train_ce": sums["ce"] / n_train,
                "train_ce_weighted": ce_objective_sum / n_train, "train_ce_unweighted_sum": sums["ce"],
                "train_ce_weighted_sum": ce_objective_sum,
                "train_guidance_before_q": sums["guidance_before_q"] / n_train,
                "train_guidance_after_q": sums["guidance"] / n_train,
                "train_guidance_before_q_sum": sums["guidance_before_q"],
                "train_guidance_after_q_sum": sums["guidance"],
                "train_method_weighted": sums["guidance_weighted"] / n_train,
                "train_method_weighted_sum": sums["guidance_weighted"], "train_method_to_ce_ratio": ratio,
                "train_guidance_to_ce_ratio": ratio, "train_raw_baseline": sums["raw_baseline"] / n_train,
                "train_amount_applied_weight_mean": float(all_balance.q_conditional_mean),
                "train_amount_applied_weight_valid_n": int(all_balance.q_valid_n),
                "train_amount_applied_fraction": float(all_balance.amount_applied_conditional_mean),
                "train_amount_applied_n": int(all_balance.amount_applied_sum),
                "train_amount_selected_outcome_fraction": float(all_balance.amount_selected_outcome_conditional_mean),
                "train_where_selected_outcome_fraction": float(all_balance.where_selected_outcome_conditional_mean),
                "train_where_applied_fraction": float(all_balance.where_applied_conditional_mean),
                "train_cam_effective_coefficient_mean": float(all_balance.cam_effective_coefficient_conditional_mean),
                "val_ce_loss": val_metrics["ce_loss"], "val_acc": val_metrics["acc"],
                "val_f1_macro": val_f1, "val_balanced_acc": val_metrics["balanced_acc"],
                "val_many_shot_f1": val_metrics["many_shot_f1"],
                "val_medium_shot_f1": val_metrics["medium_shot_f1"], "val_few_shot_f1": val_metrics["few_shot_f1"],
                "lr": lr_now, "train_seconds": train_seconds, "validation_seconds": validation_seconds,
                "images_per_second": n_train / max(train_seconds, 1e-12), "peak_cuda_mb": peak_mb}
            for name in components:
                record["train_" + name + "_component"] = sums[name + "_component"] / n_train
                record["train_" + name + "_weighted"] = CAM_ALPHA * sums[name + "_component"] / n_train
                record["train_" + name + "_weighted_sum"] = CAM_ALPHA * sums[name + "_component"]
            for _, row in balance_frame.query("group == 'all' and partition == 'guidance_branch'").iterrows():
                record["train_" + row.stratum + "_n"] = int(row.n_samples)
                record["train_" + row.stratum + "_fraction"] = row.sample_fraction_of_epoch
            history.append(record)
            if val_f1 > best_f1 + ES_MIN_DELTA:
                best_f1, best_epoch, no_improvement = val_f1, epoch, 0
                torch.save(student.state_dict(), save_dir / "best_model_f1.pth")
            else:
                no_improvement += 1
            should_stop = USE_EARLY_STOPPING and no_improvement >= ES_PATIENCE
            if epoch == 1 or epoch % ESSENTIAL_HISTORY_WRITE_EVERY == 0 or best_epoch == epoch or should_stop:
                pd.DataFrame(history).to_csv(save_dir / "history.csv", index=False)
            if SAVE_RECOVERY_CHECKPOINTS and (epoch % CKPT_INTERVAL == 0 or should_stop):
                torch.save({"model_state_dict": student.state_dict(), "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict(), "epoch": epoch, "history": history,
                    "best_f1": best_f1, "best_epoch": best_epoch, "config": CONFIG,
                    "torch_rng": torch.get_rng_state(), "numpy_rng": np.random.get_state(),
                    "python_rng": random.getstate()}, save_dir / "checkpoint_latest.pth")
            if epoch == 1 or epoch % 10 == 0 or should_stop:
                print(f"epoch {epoch:3d} | CE={record['train_ce_weighted']:.4f} | method/CE={ratio:.3e} | "
                    f"val F1={val_f1:.4f} | few F1={record['val_few_shot_f1']:.4f} | treino {train_seconds:.1f}s | val {validation_seconds:.1f}s")
            if should_stop:
                early_stopped = True
                break

        pd.DataFrame(history).to_csv(save_dir / "history.csv", index=False)
        print("Avaliando o melhor checkpoint em validação e teste...", flush=True)
        student.load_state_dict(torch.load(save_dir / "best_model_f1.pth", map_location=DEVICE, weights_only=True))
        full_balance = pd.concat(balance_history, ignore_index=True)
        best_balance = full_balance[full_balance.epoch == best_epoch].copy()
        best_group_frame = group_history[best_epoch - 1].copy()
        best_group_frame.to_csv(save_dir / "train_diagnostics_per_group.csv", index=False)
        observed_controls = observed_outcome_control_metadata(best_balance)
        for index, row in group_stats.iterrows():
            group = row["group"]
            selected = best_balance[(best_balance.group == group) & (best_balance.partition == "all")].iloc[0]
            group_stats.loc[index, "amount_factor_applied_observed_mean_at_best"] = observed_controls["amount_factor_applied_" + group]
            group_stats.loc[index, "amount_factor_applied_observed_valid_n_at_best"] = observed_controls["amount_factor_applied_valid_n_" + group]
            group_stats.loc[index, "cam_effective_coefficient_observed_mean_at_best"] = observed_controls["cam_effective_coefficient_" + group]
            group_stats.loc[index, "q_times_raw_mass_diagnostic"] = selected.q_times_teacher_raw_target_cam_l1_conditional_mean
        group_stats.to_csv(save_dir / "teacher_group_cam_statistics.csv", index=False)
        provenance["observed_training_at_best"] = observed_controls
        saved_config = json.loads((save_dir / "config.json").read_text(encoding="utf-8"))
        saved_config["observed_training_at_best"] = observed_controls
        atomic_json(save_dir / "config.json", saved_config)
        best_balance.to_csv(save_dir / "train_loss_balance_best.csv", index=False)
        essential_diagnostic_definitions = {
            "method_version": CONFIG["method_version"], "ce_weight": CE_WEIGHT, "cam_alpha": CAM_ALPHA,
            **control_metadata, "observed_training_at_best": observed_controls,
            "partitions": "all, teacher_outcome and guidance_branch are separate overlapping views; each partition's strata are disjoint.",
            "conditional_mean": "sum divided by finite valid_n for that group/stratum.",
            "population_mean_contribution": "sum divided by every processed training image in the epoch.",
            "ratio": "Ratio of population sums, never the mean of batch ratios.",
            "guidance_before_q": "Signed target-CAM spatial MSE D divided by the historical global sT for ALL methods; includes unit-mean spatial weights and excludes actual q and alpha.",
            "guidance_after_q": "Spatial MSE against r_applied*teacher_raw, divided by sT, before alpha.", "ce_objective": "CE_WEIGHT times per-image CE.",
            "method_objective": "CAM_ALPHA times the spatial MSE against r_applied*teacher_raw, divided by sT.",
            "amount_selection": "The teacher-sign factor map is applied only when QUANTO is enabled and AMOUNT_APPLY_TO selects the original teacher outcome; otherwise every factor is exactly one. The q column is its spatial mean. No normalization follows this selection.",
            "where_selection": "WHERE_APPLY_TO independently selects the original teacher outcome; outside that selection the auxiliary objective remains raw signed CAM MSE.",
            "global_cam_scale": "cam_scale_reference=max(teacher_raw_cam_mean_sq,CAM_SCALE_EPS), applied to QW0/QW1/QW2/QW3 without outcome gating.",
            "effective_coefficient": "CAM_ALPHA/sT multiplies the spatial squared error; sign/group r_applied scales only the teacher target inside that error. Actual group means and conditional teacher-outcome means use observed training images.",
            "raw_target_cam_mass": "Unchanged signed teacher target CAM; L1 is the spatial mean of its absolute value.",
            "static_quanto_statistics": "teacher_group_cam_statistics.csv uses deterministic training images and class-balanced group means; dynamic logging uses augmented images.",
            "mass_vs_objective": "the directly measured signed-rectified CAM mass is the teacher target mass, not a loss or gradient contribution.",
            "spatial_weighted_mass": "Mean of abs(original target CAM) times the applied spatial weight, with and without q; a diagnostic of the original and q-scaled teacher targets, not a loss or gradient.",
            "missing_values": "Undefined means/ratios are null in JSON and empty numeric values in CSV.",
            "class_balanced_mean": "Not emitted by the essential observer. Static deterministic TRAIN statistics used for QUANTO calibration remain unchanged."}
        provenance["essential_diagnostic_definitions"] = essential_diagnostic_definitions
        atomic_json(save_dir / "provenance.json", provenance)

        final_metrics, classification_frames = {}, []
        for split, loader, sample_ids in (("val", val_loader, val_ds.indices), ("test", test_loader, None)):
            teacher_metrics, ty, tp, tprob = evaluate_with_arrays(teacher, loader, class_counts)
            student_metrics, sy, sp, sprob = evaluate_with_arrays(student, loader, class_counts)
            np.testing.assert_array_equal(ty, sy)
            add_group_top5(teacher_metrics, ty, tprob, class_counts)
            add_group_top5(student_metrics, sy, sprob, class_counts)
            for name in teacher_metrics:
                final_metrics[f"teacher_{split}_{name}"] = teacher_metrics[name]
                final_metrics[f"student_{split}_{name}"] = student_metrics[name]
                final_metrics[f"gain_vs_teacher_{split}_{name}"] = student_metrics[name] - teacher_metrics[name]
            classification_frames.append(classification_outcome_diagnostics(sy, tp, sp, class_to_group, split))
        classification_outcomes = pd.concat(classification_frames, ignore_index=True)
        classification_outcomes.to_csv(save_dir / "classification_outcome_diagnostics.csv", index=False)
        best_history = history[best_epoch - 1]
        result = {"run_id": run_id, "config_hash": CONFIG_HASH, "method_version": CONFIG["method_version"],
            "execution_profile": execution_metadata,
            "experiment": experiment, "description": EXPERIMENT_DEFINITIONS[experiment]["description"],
            "use_amount": EXPERIMENT_DEFINITIONS[experiment]["use_amount"],
            "use_where": EXPERIMENT_DEFINITIONS[experiment]["use_where"], "seed": seed,
            "student_init_mode": student_init_mode, "best_epoch": best_epoch, "best_val_f1_macro": best_f1,
            "epochs_ran": len(history), "early_stopped": early_stopped, "cam_alpha": CAM_ALPHA,
            "amount_power": AMOUNT_POWER, "amount_normalization": AMOUNT_NORMALIZATION,
            "amount_compensation": "positive_negative_teacher_target",
            "amount_expected_weight": stats["amount_expected_weight"], "teacher_ckpt_path": teacher_path,
            **control_metadata, **observed_controls,
            "where_k": WHERE_TOP_K, "where_beta": WHERE_BETA, "where_correct_mode": WHERE_CORRECT_MODE,
            "where_error_mode": WHERE_ERROR_MODE, "where_correct_mask_mode": WHERE_CORRECT_MASK_MODE,
            "where_error_mask_mode": WHERE_ERROR_MASK_MODE, "where_groups": ",".join(WHERE_GROUPS),
            "guidance_definition": "Signed target-CAM spatial MSE against sign/group-rectified teacher reference, divided by original global sT for every method. QUANTO and WHERE have independent teacher-outcome filters; unselected QUANTO uses q=1 and unselected WHERE uses raw CAM MSE. No second q normalization. Objective is CE_WEIGHT*CE + CAM_ALPHA*mean(mean_spatial(weight*(student_raw-r_applied*teacher_raw)^2)/sT).",
            "student_initial_sha256": initial_hash, "teacher_checkpoint_sha256": provenance["teacher_checkpoint_sha256"],
            "train_indices_sha256": split_hash, **split_metadata, "save_dir": str(save_dir),
            "mean_train_epoch_seconds": float(np.mean([row["train_seconds"] for row in history])),
            "total_seconds_including_final_diagnostics": time.perf_counter() - started, **final_metrics}
        result.update({name + "_at_best": value for name, value in best_history.items() if name.startswith("train_")})
        for group_id in range(3):
            group = GROUP_ID_TO_NAME[group_id]
            result[f"amount_factor_computed_{group}"] = float(stats["amount_factors"][group_id])
            # Actual applied factors/coefficient means already came from observed
            # best-epoch group/outcome rows, not a static group-only assignment.
            result[f"teacher_group_abs_{group}"] = float(stats["group_abs_mean"][group_id])
            result[f"sample_proportion_{group}"] = float(stats["group_sample_proportions"][group_id])
            selected = group_history[best_epoch - 1].query("group == @group").iloc[0]
            for name, value in selected.items():
                if name not in {"epoch", "group"} and isinstance(value, (int, float, np.number)):
                    result[f"{group}_{name}_at_best"] = float(value)
        for _, row in best_balance[best_balance.partition == "all"].iterrows():
            for name in ("n_samples", "guidance_before_q_sum", "guidance_after_q_sum",
                    "guidance_share_of_epoch_before_q", "guidance_share_of_epoch_after_q",
                    "method_objective_population_mean_contribution", "method_to_ce_ratio_of_sums",
                    "teacher_raw_target_cam_l1_conditional_mean", "q_times_teacher_raw_target_cam_l1_conditional_mean"):
                result[f"{row['group']}_loss_balance_{name}_at_best"] = float(row[name])
        # Written last: this file is the authoritative completed-run marker.
        atomic_json(save_dir / "result.json", result)
        return result, group_stats


    def run_essential_execution_selfcheck():
        """CPU checks of detached observers, validation and two Adam steps.

        This test uses no ImageNet images or checkpoints. It restores the notebook
        globals and Python/NumPy/PyTorch random states, including every CUDA RNG.
        It does not change thread counts or deterministic/backend settings.
        """
        import copy as _copy
        import random as _random

        _absent = object()
        _overrides = {
            "DEVICE": torch.device("cpu"), "NUM_CLASSES": 6,
            "CE_WEIGHT": 1., "CAM_ALPHA": .8, "CAM_SCALE_EPS": 1e-8,
            "WHERE_TOP_K": 3, "WHERE_GROUPS": ("medium", "few"),
            "WHERE_APPLY_TO": "all", "AMOUNT_APPLY_TO": "error",
            "WHERE_CORRECT_MODE": "coincidence", "WHERE_ERROR_MODE": "difference",
            "WHERE_CORRECT_MASK_MODE": "aligned", "WHERE_ERROR_MASK_MODE": "aligned",
            "WHERE_BETA": .5, "WHERE_NORM_EPS": 1e-8, "GRAD_DIAG_MAX_BATCHES": 1,
            "MANY_SHOT_THRESHOLD": 100, "FEW_SHOT_THRESHOLD": 20,
            "GROUP_NAME_TO_ID": {"many": 0, "medium": 1, "few": 2},
            "GROUP_ID_TO_NAME": {0: "many", 1: "medium", 2: "few"},
        }
        _previous = {name: globals().get(name, _absent) for name in _overrides}
        _np_state = _copy.deepcopy(np.random.get_state())
        _python_state = _random.getstate()
        _cuda_devices = list(range(torch.cuda.device_count())) if torch.cuda.is_available() else []
        _checks = []

        def _same(a, b):
            if torch.is_tensor(a):
                return torch.equal(a, b)
            if isinstance(a, np.ndarray):
                return np.array_equal(a, b)
            if isinstance(a, dict):
                return a.keys() == b.keys() and all(_same(a[key], b[key]) for key in a)
            if isinstance(a, (list, tuple)):
                return len(a) == len(b) and all(_same(x, y) for x, y in zip(a, b))
            return a == b

        def _rng():
            return (torch.get_rng_state().clone(), _copy.deepcopy(np.random.get_state()), _random.getstate())

        def _assert_rng(before, description):
            assert _same(before, _rng()), description + " changed a global random state."

        class _ToyCAM(nn.Module):
            def __init__(self):
                super().__init__()
                self.features = nn.Sequential(nn.Conv2d(3, 6, 3, padding=1, bias=False),
                    nn.BatchNorm2d(6), nn.ReLU())
                self.classifier = nn.Sequential(nn.Dropout(.2), nn.Linear(6, 6))

            @property
            def fc(self):
                return self.classifier[-1]

            def forward(self, images):
                feature = self.features(images)
                pooled = F.adaptive_avg_pool2d(feature, (1, 1)).flatten(1)
                return self.classifier(pooled), {"feats": [feature], "pooled_feat": pooled}

        try:
            with torch.random.fork_rng(devices=_cuda_devices, enabled=True):
                globals().update(_overrides)
                torch.manual_seed(81672)
                generator = torch.Generator().manual_seed(81673)
                student_initial = _ToyCAM().to(device="cpu", dtype=torch.float32)
                teacher_initial = _ToyCAM().to(device="cpu", dtype=torch.float32).eval()
                teacher_initial.requires_grad_(False)
                group_ids = torch.tensor([0, 0, 1, 1, 2, 2], dtype=torch.long, device="cpu")
                # Each group has observations; some outcome/branch strata may be
                # empty, which also checks the observers' undefined means.
                batches = [(torch.randn(12, 3, 5, 5, generator=generator, device="cpu", dtype=torch.float32),
                            torch.arange(12, dtype=torch.long, device="cpu") % 6,
                            torch.arange(12, dtype=torch.long, device="cpu") + batch * 12)
                           for batch in range(2)]
                stats = {
                    "raw_cam_mean_sq": 2.3,
                    "amount_factors": np.array([1.1, .9, 1.2]),
                    "amount_factors_positive": np.array([1.4, .8, 1.2]),
                    "amount_factors_negative": np.array([.6, 1.3, .9]),
                    "amount_factors_zero": np.ones(3),
                }
                original_observer = globals().get("originalEpochLossBalanceAccumulator",
                    globals().get("EpochLossBalanceAccumulator"))
                if original_observer is None:
                    raise RuntimeError("Original EpochLossBalanceAccumulator is required by this self-check.")

                def _train_trace(observer_type, gradient_logging):
                    student = _copy.deepcopy(student_initial).train()
                    teacher = _copy.deepcopy(teacher_initial).eval()
                    observer = observer_type(group_ids)
                    optimizer = torch.optim.Adam(student.parameters(), lr=.001,
                        betas=(.9, .999), weight_decay=1e-4)
                    # Seed after optimizer construction so one-time lazy imports
                    # cannot make the first trace start from a different RNG state.
                    torch.manual_seed(81674)
                    np.random.seed(81674)
                    _random.seed(81674)
                    snapshots = []
                    for batch_id, (source, labels, sample_ids) in enumerate(batches):
                        # The RNG-sensitive augmentation and Dropout expose any
                        # accidental random consumption by the logging paths.
                        images = source + .01 * torch.randn_like(source)
                        if _random.random() < .5:
                            images = images.flip(-1)
                        optimizer.zero_grad(set_to_none=True)
                        logits, features = student(images)
                        total, diagnostics, maps, losses = qw_cam_loss("QW3_AMOUNT_WHERE", logits,
                            features, student.fc, teacher, labels, images, group_ids, stats,
                            sample_ids, seed=42, epoch=1)
                        rng_before = _rng()
                        model_before = _copy.deepcopy(student.state_dict())
                        observer.update(logits, labels, diagnostics, maps, losses)
                        _assert_rng(rng_before, observer_type.__name__)
                        assert _same(model_before, student.state_dict()), "Observer changed model parameters or buffers."
                        assert all(p.grad is None for p in student.parameters()), "Observer changed parameter.grad."
                        if gradient_logging and batch_id == 0:
                            feature_gradient_diagnostics(losses, features, group_ids, labels, 1,
                                batch_id, diagnostics=diagnostics)
                            _assert_rng(rng_before, "feature_gradient_diagnostics")
                            assert all(p.grad is None for p in student.parameters()), "Gradient diagnostics accumulated parameter.grad."
                        total.backward()
                        gradients = [parameter.grad.detach().clone() for parameter in student.parameters()]
                        optimizer.step()
                        snapshots.append({"loss": total.detach().clone(), "gradients": gradients,
                            "student": _copy.deepcopy(student.state_dict()),
                            "optimizer": _copy.deepcopy(optimizer.state_dict()), "rng": _rng()})
                    return student, observer.frame(1), snapshots

                old_student, old_frame, old_trace = _train_trace(original_observer, True)
                new_student, new_frame, new_trace = _train_trace(EssentialEpochAccumulator, False)
                assert _same(old_trace, new_trace), "Essential observation changed loss, gradients, Adam state, model/BN buffers or RNG."
                _checks.append("two Adam steps: exact loss, gradients, weights, BatchNorm buffers, optimizer state and RNG")
                _checks.append("both observers and optional gradient logging leave parameters, buffers and RNG unchanged")

                identity = ["group", "partition", "stratum"]
                old_frame = old_frame.set_index(identity).sort_index()
                new_frame = new_frame.set_index(identity).sort_index()
                assert old_frame.index.equals(new_frame.index), "Observer strata differ."
                common = [column for column in old_frame if column in new_frame]
                for column in common:
                    old_values = pd.to_numeric(old_frame[column], errors="coerce").to_numpy(dtype=float)
                    new_values = pd.to_numeric(new_frame[column], errors="coerce").to_numpy(dtype=float)
                    np.testing.assert_allclose(old_values, new_values, rtol=1e-12, atol=1e-12,
                        equal_nan=True, err_msg="Observer field: " + column)
                _checks.append(str(len(common)) + " shared observer fields: sums, means, factors, masses, ratios and coverage")

                class_counts = np.array([120, 110, 90, 70, 10, 5])
                val_images = torch.randn(12, 3, 5, 5, generator=generator, device="cpu", dtype=torch.float32)
                val_labels = torch.arange(12, dtype=torch.long, device="cpu") % 6

                def _loader():
                    return torch.utils.data.DataLoader(torch.utils.data.TensorDataset(val_images, val_labels),
                        batch_size=6, shuffle=False, num_workers=0,
                        generator=torch.Generator().manual_seed(81675))

                rng_before = _rng()
                old_metrics, old_y, old_pred, _ = evaluate_with_arrays(old_student, _loader(), class_counts)
                _assert_rng(rng_before, "Original validation with private loader generator")
                new_metrics, new_y, new_pred, new_probs = evaluate_epoch_essential(new_student, _loader(), class_counts)
                _assert_rng(rng_before, "Essential validation with private loader generator")
                np.testing.assert_array_equal(old_y, new_y)
                np.testing.assert_array_equal(old_pred, new_pred)
                assert new_probs is None
                common_metrics = sorted(set(old_metrics) & set(new_metrics))
                for metric in common_metrics:
                    a, b = old_metrics[metric], new_metrics[metric]
                    assert (np.isnan(a) and np.isnan(b)) or a == b, "Validation metric differs: " + metric
                _checks.append("validation: exact labels, predictions and " + str(len(common_metrics)) + " shared metrics")
        finally:
            for name, value in _previous.items():
                if value is _absent:
                    globals().pop(name, None)
                else:
                    globals()[name] = value
            np.random.set_state(_np_state)
            _random.setstate(_python_state)
        print("Essential execution CPU self-check passed: " + "; ".join(_checks) + ".")
        return {"checks": _checks, "scope": "CPU toy Conv/BatchNorm/Dropout; no ImageNet/GPU training executed"}


    if RUN_SELF_TESTS:
        run_essential_execution_selfcheck()


## Executar e consolidar

A próxima célula inicia as variantes e seeds ativas. Os resultados ficam em pastas separadas:

- ResNet32: `PROJECT_BASE/resnet32_cifar100_lt/results/QW_V8/<protocolo>/`.
- MobileNetV2: `PROJECT_BASE/mobilenetv2_imagenet_lt/results/QW_V8_MobileNet/<protocolo>/`.

`result.json` é o marcador de conclusão. Cada perfil mantém os nomes de arquivos e os validadores de sua implementação. No MobileNet essencial, o resumo final consiste em `complete_results.csv` e `summary_metrics.csv`; o ResNet conserva o exportador completo.

Os caminhos integram a assinatura da configuração; ajustá-los pode mudar o identificador do protocolo. Os hashes históricos do código de origem são preservados; os metadados deste notebook registram os dois anexos utilizados na integração.


In [ ]:
assert_active_profile()

if EXPERIMENT_PROFILE == "resnet32_cifar100_lt":
    # Consolidate only completed V8 runs with matching provenance and outcome controls.
    COMPARISON_METRICS = ["best_val_f1_macro", "student_val_few_shot_f1", "student_test_acc",
        "student_test_balanced_acc", "student_test_f1_macro", "student_test_few_shot_f1",
        "student_test_many_shot_f1", "student_test_medium_shot_f1", "train_method_to_ce_ratio_at_best"]


    def paired_comparisons(results):
        for record in results.to_dict("records"):
            validate_completed_result(record, "paired comparison")
        rows = []
        for (seed, init), part in results.groupby(["seed", "student_init_mode"], dropna=False):
            indexed = part.set_index("experiment")
            if not indexed.index.is_unique:
                raise ValueError("Duplicate method/seed/init in consolidated results.")
            metrics = [metric for metric in COMPARISON_METRICS if metric in indexed]
            def flags(selected):
                return {"paired_" + name: bool(selected[col].notna().all() and selected[col].nunique() == 1)
                    for name,col in (("initial_state", "student_initial_sha256"),
                        ("teacher_checkpoint", "teacher_checkpoint_sha256"), ("split", "train_indices_sha256"),
                        ("global_scale", "cam_scale_reference"))}
            scopes = {
                ("QW1_AMOUNT_ONLY", "QW0_BASE_RAW"): f"QUANTO group factors on {AMOUNT_APPLY_TO}; common global sT",
                ("QW2_WHERE_ONLY", "QW0_BASE_RAW"): f"WHERE on {WHERE_APPLY_TO}; common global sT",
                ("QW3_AMOUNT_WHERE", "QW0_BASE_RAW"): f"QUANTO on {AMOUNT_APPLY_TO} plus WHERE on {WHERE_APPLY_TO}; common global sT",
                ("QW3_AMOUNT_WHERE", "QW1_AMOUNT_ONLY"): f"WHERE on {WHERE_APPLY_TO}; same outcome-selected QUANTO and global sT",
                ("QW3_AMOUNT_WHERE", "QW2_WHERE_ONLY"): f"QUANTO group factors on {AMOUNT_APPLY_TO}; same WHERE and global sT"}
            for left, right in (("QW1_AMOUNT_ONLY", "QW0_BASE_RAW"),
                                ("QW2_WHERE_ONLY", "QW0_BASE_RAW"),
                                ("QW3_AMOUNT_WHERE", "QW0_BASE_RAW"),
                                ("QW3_AMOUNT_WHERE", "QW1_AMOUNT_ONLY"),
                                ("QW3_AMOUNT_WHERE", "QW2_WHERE_ONLY")):
                if left not in indexed.index or right not in indexed.index:
                    continue
                for metric in metrics:
                    rows.append({"seed": seed, "student_init_mode": init, "comparison": f"{left} - {right}",
                        "metric": metric, "delta": indexed.at[left, metric] - indexed.at[right, metric],
                        "amount_apply_to": AMOUNT_APPLY_TO, "where_apply_to": WHERE_APPLY_TO,
                        "comparison_scope": scopes[(left, right)],
                        **flags(indexed.loc[[left, right]])})
            if set(EXPERIMENT_DEFINITIONS) <= set(indexed.index):
                for metric in metrics:
                    interaction = (indexed.at["QW3_AMOUNT_WHERE", metric] - indexed.at["QW1_AMOUNT_ONLY", metric]
                        - indexed.at["QW2_WHERE_ONLY", metric] + indexed.at["QW0_BASE_RAW", metric])
                    rows.append({"seed": seed, "student_init_mode": init,
                        "comparison": "QW3-QW1-QW2+QW0 (interaction)", "metric": metric,
                        "amount_apply_to": AMOUNT_APPLY_TO, "where_apply_to": WHERE_APPLY_TO,
                        "comparison_scope": "Interaction of outcome-selected group factors and spatial weighting; common global sT",
                        "delta": interaction, **flags(indexed.loc[list(EXPERIMENT_DEFINITIONS)])})
        return pd.DataFrame(rows)


    def read_run_table(result, filename):
        path = Path(result["save_dir"], filename)
        if not path.exists() or path.stat().st_size < 2:
            return pd.DataFrame()
        try:
            table = pd.read_csv(path)
        except pd.errors.EmptyDataError:
            return pd.DataFrame()
        for key in ("run_id", "experiment", "seed", "student_init_mode", "config_hash", "method_version"):
            table[key] = result[key]
        return table


    def concatenate_tables(tables):
        tables = [table for table in tables if not table.empty]
        return pd.concat(tables, ignore_index=True) if tables else pd.DataFrame()


    def validate_completed_result(record, source):
        """A completed run may be reused only for the same method, configuration, seed and init."""
        if record.get("config_hash") != CONFIG_HASH or record.get("method_version") != CONFIG["method_version"]:
            raise RuntimeError(f"Result belongs to another configuration or method version: {source}")
        experiment = record.get("experiment")
        if experiment not in EXPERIMENT_DEFINITIONS:
            raise RuntimeError(f"Unknown experiment in completed result: {source}")
        seed = record.get("seed")
        if seed is None or not np.isfinite(float(seed)) or int(seed) != float(seed):
            raise RuntimeError(f"Invalid seed in completed result: {source}")
        init = record.get("student_init_mode")
        if init not in {"random", "teacher_init"}:
            # The builder remains authoritative; no silent reuse of an unknown init.
            if init not in STUDENT_INIT_MODES:
                raise RuntimeError(f"Unknown initialization in completed result: {source}")
        expected_id = f"{CONFIG_HASH}|{experiment}|seed={int(seed)}|init={init}"
        if record.get("run_id") != expected_id:
            raise RuntimeError(f"Completed run_id disagrees with its seed/experiment/init: {source}")
        required_metrics = ("best_epoch", "best_val_f1_macro", "student_test_f1_macro",
                            "student_test_balanced_acc", "student_test_few_shot_f1")
        if any(record.get(key) is None or not np.isfinite(float(record[key])) for key in required_metrics):
            raise RuntimeError(f"Completed run has missing final metrics: {source}")
        if record.get("amount_apply_to") != AMOUNT_APPLY_TO or record.get("where_apply_to") != WHERE_APPLY_TO:
            raise RuntimeError(f"Completed run has different teacher-outcome controls: {source}")
        if record.get("amount_scope_definition") != "signed_group_target_factors_on_selected_teacher_outcome_without_renormalization":
            raise RuntimeError(f"Completed run has another QUANTO selection policy: {source}")
        if (record.get("cam_scale_scope") != "all_experiments"
                or not isinstance(record.get("cam_scale_enabled"), (bool, np.bool_))
                or not bool(record["cam_scale_enabled"])):
            raise RuntimeError(f"Completed run does not use the common global CAM scale: {source}")
        cfg = EXPERIMENT_DEFINITIONS[experiment]
        if (not isinstance(record.get("amount_enabled"), (bool, np.bool_))
                or not isinstance(record.get("where_enabled"), (bool, np.bool_))
                or bool(record["amount_enabled"]) != bool(cfg["use_amount"])
                or bool(record["where_enabled"]) != bool(cfg["use_where"])):
            raise RuntimeError(f"Completed run has inconsistent QUANTO/WHERE enable flags: {source}")

        def number(name, allow_missing=False):
            value = record.get(name)
            if value is None or not np.isfinite(float(value)):
                if allow_missing:
                    return np.nan
                raise RuntimeError(f"Completed run has missing/nonfinite {name}: {source}")
            return float(value)

        def close(name, expected):
            if not np.isclose(number(name), expected, rtol=5e-6, atol=1e-10):
                raise RuntimeError(f"Completed run has inconsistent {name}: {source}")

        scale = max(number("teacher_raw_cam_mean_sq"), float(CAM_SCALE_EPS))
        close("cam_alpha", float(CAM_ALPHA))
        close("cam_scale_eps", float(CAM_SCALE_EPS))
        close("cam_scale_reference", scale)
        close("cam_scale_applied", scale)  # Never disabled by the QUANTO flag.
        coefficient = float(CAM_ALPHA) / scale
        close("cam_global_coefficient", coefficient)
        close("amount_factor_outside_selection", 1.)
        close("cam_effective_coefficient_outside_amount_selection", coefficient)
        close("amount_expected_weight", number("amount_expected_weight_base"))
        group_n, group_q_sum, group_coefficient_sum = 0, 0., 0.
        for group_id in range(3):
            group = GROUP_ID_TO_NAME[group_id]
            base = number("amount_factor_computed_" + group)
            close("amount_factor_base_" + group, base)
            potential = base if cfg["use_amount"] else 1.
            close("amount_factor_if_selected_" + group, potential)
            for sign in ("positive","negative","zero"):
                sign_base=number("amount_factor_"+sign+"_base_"+group)
                close("amount_factor_"+sign+"_if_selected_"+group,sign_base if cfg["use_amount"] else 1.)
            close("cam_effective_coefficient_if_selected_" + group, coefficient)
            n = number("amount_factor_applied_valid_n_" + group)
            if n <= 0 or n != int(n):
                raise RuntimeError(f"Completed run has no observed training images in {group}: {source}")
            count, weighted_q = 0, 0.
            for outcome in ("correct", "error"):
                suffix = "teacher_" + outcome + "_" + group
                valid = number("amount_factor_applied_valid_n_" + suffix)
                if valid < 0 or valid != int(valid):
                    raise RuntimeError(f"Completed run has invalid conditional counts: {source}")
                close("cam_effective_coefficient_valid_n_" + suffix, valid)
                selected = AMOUNT_APPLY_TO in {"all", outcome}
                expected_q = potential if selected else 1.
                if valid:
                    expected_sign_q={sign:(number("amount_factor_"+sign+"_if_selected_"+group)
                        if selected else 1.) for sign in ("positive","negative","zero")}
                    fractions={sign:number("amount_"+sign+"_fraction_"+suffix)
                        for sign in ("positive","negative","zero")}
                    if not np.isclose(sum(fractions.values()),1.,rtol=5e-6,atol=1e-7):
                        raise RuntimeError(f"Teacher sign fractions do not close: {source}")
                    for sign in expected_sign_q:
                        close("amount_factor_"+sign+"_applied_"+suffix,expected_sign_q[sign])
                    expected_q=sum(expected_sign_q[sign]*fractions[sign] for sign in expected_sign_q)
                    close("amount_factor_applied_" + suffix, expected_q)
                    close("cam_effective_coefficient_" + suffix, coefficient)
                    close("amount_applied_fraction_" + suffix, float(bool(cfg["use_amount"]) and selected))
                    close("amount_applied_n_" + suffix, valid * float(bool(cfg["use_amount"]) and selected))
                    close("amount_selected_outcome_n_" + suffix, valid * float(selected))
                    close("where_selected_outcome_fraction_" + suffix, float(WHERE_APPLY_TO in {"all", outcome}))
                else:
                    if np.isfinite(number("amount_factor_applied_" + suffix, True)) or np.isfinite(number("cam_effective_coefficient_" + suffix, True)):
                        raise RuntimeError(f"Completed run invents a mean for an empty outcome stratum: {source}")
                count += int(valid)
                weighted_q += valid * expected_q
            if count != int(n):
                raise RuntimeError(f"Conditional counts do not close in {group}: {source}")
            expected_mean = weighted_q / n
            close("amount_factor_applied_" + group, expected_mean)
            # The observed mean already closes across outcomes; sign composition can vary.
            close("cam_effective_coefficient_" + group, coefficient)
            close("cam_effective_coefficient_valid_n_" + group, n)
            group_n += int(n)
            group_q_sum += weighted_q
            group_coefficient_sum += n * coefficient
        close("amount_factor_applied_valid_n_all", group_n)
        close("amount_factor_applied_all", group_q_sum / group_n)
        close("amount_applied_weight_mean_at_best", group_q_sum / group_n)
        close("amount_applied_weight_valid_n_at_best", group_n)
        close("cam_effective_coefficient_all", group_coefficient_sum / group_n)


    def collect_completed_results():
        # JSON is the completion marker. A stale XLSX row without its completed JSON
        # never causes a run to be skipped. Re-export rebuilds both CSV and XLSX.
        records = []
        for path in sorted(Path(RUNS_ROOT).glob("*/seed_*_init_*/result.json")):
            with path.open(encoding="utf-8") as file:
                record = json.load(file)
            validate_completed_result(record, path)
            expected_folder = Path(RUNS_ROOT, record["experiment"],
                f"seed_{int(record['seed'])}_init_{record['student_init_mode']}")
            if path.parent.resolve() != expected_folder.resolve():
                raise RuntimeError(f"Completed result is in the wrong run folder: {path}")
            config_path = path.parent / "config.json"
            if not config_path.exists():
                raise RuntimeError(f"Missing per-run configuration: {config_path}")
            saved_config = json.loads(config_path.read_text(encoding="utf-8"))
            if (saved_config.get("run_id") != record["run_id"] or saved_config.get("config_hash") != CONFIG_HASH
                    or saved_config.get("method_version") != CONFIG["method_version"]):
                raise RuntimeError(f"Per-run configuration does not match completed result: {path}")
            saved_payload = saved_config.get("config", {})
            saved_hash_payload = {key: value for key, value in saved_payload.items()
                if key not in {"SEEDS", "EXPERIMENTS_TO_RUN", "STUDENT_INIT_MODES"}}
            saved_hash = hashlib.sha256(json.dumps(saved_hash_payload, sort_keys=True).encode()).hexdigest()[:12]
            if saved_hash != CONFIG_HASH:
                raise RuntimeError(f"Per-run configuration content has a different hash: {path}")
            record["save_dir"] = str(path.parent)
            records.append(record)
        if not records:
            return pd.DataFrame()
        results = pd.DataFrame(records)
        if results.run_id.duplicated().any():
            raise RuntimeError("Duplicate completed run IDs.")
        return results.sort_values(["experiment", "student_init_mode", "seed"]).reset_index(drop=True)


    def summarize_seed_metrics(table, keys, metrics):
        """Equal seed weights; sample std is undefined (NaN) with only one seed."""
        if table.empty:
            return pd.DataFrame()
        metrics = [metric for metric in dict.fromkeys(metrics) if metric in table]
        if table.duplicated(keys + ["seed"]).any():
            raise ValueError(f"More than one observation per seed for summary keys {keys}.")
        grouped = table.groupby(keys, dropna=False)
        seed_info = grouped["seed"].agg(n_seeds="nunique",
            seeds=lambda value: ", ".join(str(int(x)) for x in sorted(value.unique())))
        if not metrics:
            return seed_info.reset_index()
        summary = grouped[metrics].agg(["mean", "std", "count"])
        summary.columns = ["_".join(col) for col in summary.columns]
        return seed_info.join(summary).reset_index()


    def atomic_excel_tables(path, sheets):
        """Keep the previous file intact if writing fails; never silently truncate rows."""
        target = Path(path)
        temporary = target.with_name(target.stem + ".writing.xlsx")
        max_rows = 1_048_575  # header occupies the first Excel row
        try:
            with pd.ExcelWriter(temporary, engine="openpyxl") as writer:
                for name, table in sheets.items():
                    if table.empty:
                        continue
                    chunks = max(1, (len(table) + max_rows - 1) // max_rows)
                    for chunk in range(chunks):
                        sheet_name = name[:31] if chunks == 1 else f"{name[:25]}_{chunk + 1:04d}"
                        table.iloc[chunk * max_rows:(chunk + 1) * max_rows].to_excel(
                            writer, sheet_name=sheet_name, index=False)
            os.replace(temporary, target)
        finally:
            if temporary.exists():
                temporary.unlink()


    def atomic_csv(path, table):
        target = Path(path)
        temporary = target.with_name(target.name + ".writing")
        try:
            table.to_csv(temporary, index=False)
            os.replace(temporary, target)
        finally:
            if temporary.exists():
                temporary.unlink()


    def export_summary(results):
        validate_v8_configuration()
        if results.empty:
            return
        results = results.drop_duplicates("run_id", keep="last").sort_values(
            ["experiment", "student_init_mode", "seed"]).reset_index(drop=True)
        for record in results.to_dict("records"):
            validate_completed_result(record, "in-memory consolidation")
        numeric = set(results.select_dtypes(include=[np.number]).columns)
        metrics = [name for name in results if name in numeric and (
            name.startswith(("student_val_", "student_test_", "teacher_val_", "teacher_test_", "gain_vs_teacher_"))
            or name.endswith("_at_best"))]
        scaling_columns = ["amount_apply_to", "where_apply_to", "amount_enabled", "where_enabled",
            "amount_scope_definition", "amount_expected_weight", "amount_expected_weight_base",
            "amount_applied_weight_mean_at_best", "amount_applied_weight_valid_n_at_best", "observed_training_epoch",
            "teacher_raw_cam_mean_sq", "cam_scale_eps", "cam_scale_reference", "cam_scale_applied",
            "cam_scale_enabled", "cam_scale_scope", "cam_global_coefficient",
            "amount_factor_outside_selection", "cam_effective_coefficient_outside_amount_selection"]
        scaling_columns += [name for name in results if name.startswith(("amount_factor_base_",
            "amount_factor_if_selected_", "amount_factor_applied_", "amount_factor_positive_", "amount_factor_negative_",
            "amount_factor_zero_", "amount_positive_fraction_", "amount_negative_fraction_", "amount_zero_fraction_", "cam_effective_coefficient_",
            "amount_applied_fraction_", "amount_applied_n_", "amount_selected_outcome_",
            "where_selected_outcome_", "where_applied_fraction_", "where_applied_n_"))]
        scaling_columns = list(dict.fromkeys(scaling_columns))
        metrics += [name for name in scaling_columns if name in numeric]
        metrics += ["mean_train_epoch_seconds", "best_epoch", "epochs_ran"]
        summary = summarize_seed_metrics(results, ["experiment", "student_init_mode"], metrics)
        policy_columns = ["amount_apply_to", "where_apply_to", "amount_enabled", "where_enabled",
            "amount_scope_definition", "cam_scale_enabled", "cam_scale_scope"]
        for _, part in results.groupby(["experiment", "student_init_mode"], dropna=False):
            if any(part[name].nunique(dropna=False) != 1 for name in policy_columns):
                raise RuntimeError("A seed summary mixes different outcome-selection policies.")
        summary = summary.merge(results[["experiment", "student_init_mode"] + policy_columns].drop_duplicates(),
            on=["experiment", "student_init_mode"], validate="one_to_one")
        scaling_table = results[[name for name in ["run_id", "experiment", "student_init_mode", "seed",
            "config_hash", "method_version", "cam_alpha"] + scaling_columns if name in results]].copy()
        comparisons = paired_comparisons(results)
        paired_summary = summarize_seed_metrics(comparisons,
            ["student_init_mode", "comparison", "metric"], ["delta"]) if not comparisons.empty else pd.DataFrame()
        if not paired_summary.empty:
            keys = ["student_init_mode", "comparison", "metric"]
            flags = ["paired_initial_state", "paired_teacher_checkpoint", "paired_split", "paired_global_scale"]
            paired_summary = paired_summary.merge(comparisons.groupby(keys, dropna=False)[flags].all().reset_index(),
                on=keys, validate="one_to_one")
            paired_summary = paired_summary.merge(comparisons[keys + ["amount_apply_to", "where_apply_to", "comparison_scope"]].drop_duplicates(),
                on=keys, validate="one_to_one")
        records = results.to_dict("records")

        def collect(filename, best_only=False):
            frames = []
            for record in records:
                frame = read_run_table(record, filename)
                if best_only and not frame.empty and "epoch" in frame:
                    frame = frame[frame.epoch == record["best_epoch"]]
                frames.append(frame)
            return concatenate_tables(frames)

        def numeric_summary(frame, keys):
            fields = [name for name in frame.select_dtypes(include=[np.number]) if name not in {"seed", "epoch"}]
            return summarize_seed_metrics(frame, keys, fields)

        train_groups = collect("train_diagnostics_per_group.csv", True)
        train_outcomes = collect("train_diagnostics_group_outcome.csv", True)
        train_outcome_summary = numeric_summary(train_outcomes,
            ["experiment", "student_init_mode", "group", "teacher_correct"])
        classification_outcomes = collect("classification_outcome_diagnostics.csv")
        classification_summary = summarize_seed_metrics(classification_outcomes,
            ["experiment", "student_init_mode", "split", "group", "teacher_correct"],
            ["student_correct_rate", "student_correct_rate_class_balanced", "recovery_rate", "regression_rate",
             "recovery_rate_class_balanced", "regression_rate_class_balanced"])
        class_results, probe_classes = collect("per_class_classification_diagnostics.csv"), collect("probe_class_summary.csv")
        if not class_results.empty and not probe_classes.empty:
            keys = ["run_id", "experiment", "seed", "student_init_mode", "split", "class_id", "group", "config_hash"]
            class_results = class_results.merge(probe_classes, on=keys, how="left",
                suffixes=("", "_probe"), validate="many_to_one")

        gradients = collect("feature_gradient_diagnostics.csv", True)
        gradient_seed, gradient_summary = pd.DataFrame(), pd.DataFrame()
        if not gradients.empty:
            keys = ["experiment", "student_init_mode", "seed", "group", "teacher_outcome", "branch"]
            fields = [name for name in gradients if name.endswith(("_feature_grad_norm",
                "_to_ce_feature_grad_ratio", "_feature_grad_cosine"))]
            gradient_seed = gradients.groupby(keys, dropna=False, as_index=False)[fields].mean()
            coverage = gradients.groupby(keys, dropna=False).agg(n_sampled_batches=("batch", "nunique"),
                n_sampled_images=("n_samples", "sum")).reset_index()
            gradient_seed = gradient_seed.merge(coverage, on=keys, validate="one_to_one")
            gradient_seed["diagnostic_scope"] = "sampled_batches_last_feature_tensor"
            gradient_seed["batch_aggregation"] = "mean_of_sampled_batch_diagnostics_not_full_epoch_gradient"
            gradient_summary = summarize_seed_metrics(gradient_seed,
                ["experiment", "student_init_mode", "group", "teacher_outcome", "branch"],
                fields + ["n_sampled_batches", "n_sampled_images"])
            gradient_summary["diagnostic_scope"] = "sampled_batches_last_feature_tensor"
            gradient_summary["seed_aggregation"] = "equal_seed_mean_of_sampled_batch_means"

        balance_epochs, balance_best = collect("train_loss_balance.csv"), collect("train_loss_balance_best.csv")
        balance_keys = ["experiment", "student_init_mode", "group", "partition", "stratum"]
        balance_metrics = [name for name in balance_best.select_dtypes(include=[np.number]) if name not in {"epoch", "seed"}]
        balance_summary = summarize_seed_metrics(balance_best, balance_keys, balance_metrics)
        balance_epoch_summary = summarize_seed_metrics(balance_epochs, balance_keys + ["epoch"], balance_metrics)
        trajectory_metrics = ["n_samples", "n_classes_observed", "n_epoch_samples", "n_group_samples",
            "ce_objective_sum", "method_objective_sum", "correct_objective_sum", "error_objective_sum",
            "raw_objective_sum", "difference_objective_sum", "coincidence_objective_sum",
            "method_objective_population_mean_contribution", "method_to_ce_ratio_of_sums",
            "guidance_before_q_sum", "guidance_after_q_sum", "guidance_share_of_epoch_before_q",
            "amount_factor_base_conditional_mean", "q_conditional_mean", "q_valid_n",
            "amount_selected_outcome_conditional_mean", "amount_applied_conditional_mean",
            "where_selected_outcome_conditional_mean", "where_applied_conditional_mean",
            "cam_scale_reference_conditional_mean", "cam_scale_applied_conditional_mean",
            "cam_global_coefficient_conditional_mean", "cam_effective_coefficient_conditional_mean",
            "guidance_share_of_epoch_after_q", "teacher_raw_target_cam_l1_conditional_mean",
            "q_times_teacher_raw_target_cam_l1_conditional_mean", "teacher_raw_target_cam_l1_class_balanced_mean",
            "q_times_teacher_raw_target_cam_l1_class_balanced_mean",
            "teacher_spatial_weighted_target_cam_l1_conditional_mean",
            "q_times_teacher_spatial_weighted_target_cam_l1_conditional_mean"]
        identifiers = ["run_id", "experiment", "student_init_mode", "seed", "epoch", "group", "partition", "stratum"]
        trajectory = balance_epochs[[name for name in identifiers + trajectory_metrics if name in balance_epochs]]
        summary_columns = balance_keys + ["epoch", "n_seeds", "seeds"] + [
            name + suffix for name in trajectory_metrics for suffix in ("_mean", "_std", "_count")]
        epoch_summary_view = balance_epoch_summary[[name for name in summary_columns if name in balance_epoch_summary]]

        sheets = {"summary": summary, "per_seed": results, "cam_scaling": scaling_table, "paired_deltas": comparisons,
            "paired_summary": paired_summary, "parameters": parameter_table_df,
            "config": pd.DataFrame({"parameter": list(CONFIG),
                "value": [json.dumps(value, ensure_ascii=False) for value in CONFIG.values()]}),
            "teacher_q": collect("teacher_group_cam_statistics.csv"), "train_group_best": train_groups,
            "train_class_outcome_best": collect("train_diagnostics_class_outcome.csv", True),
            "train_group_outcome_best": train_outcomes, "train_outcome_summary": train_outcome_summary,
            "gradients_best": gradients, "gradient_seed": gradient_seed, "gradient_summary": gradient_summary,
            "loss_balance_trajectory": trajectory, "loss_balance_best": balance_best,
            "loss_balance_seed_summary": balance_summary, "loss_balance_epoch_summary": epoch_summary_view,
            "classification_outcomes": classification_outcomes, "classification_outcome_summary": classification_summary,
            "probe_group": collect("probe_group_summary.csv"), "probe_correct_wrong": collect("probe_group_correct_wrong.csv"),
            "class_results_and_CAM": class_results, "diagnostic_guide": diagnostic_guide_df}
        # Every seed has equal weight. Metric-specific *_count is the number of
        # finite seed values, not the number of images or sampled gradient batches.
        tables = {"complete_results": results, "summary_metrics": summary, "cam_scaling": scaling_table, "paired_deltas": comparisons,
            "paired_summary": paired_summary, "gradient_seed": gradient_seed, "gradient_summary": gradient_summary,
            "loss_balance_epochs": balance_epochs, "loss_balance_best": balance_best,
            "loss_balance_summary": balance_summary, "loss_balance_epoch_summary": balance_epoch_summary,
            "train_outcome_summary": train_outcome_summary, "classification_outcome_summary": classification_summary}
        for filename, table in tables.items():
            atomic_csv(Path(RUNS_ROOT, filename + ".csv"), table)
            atomic_json(Path(RUNS_ROOT, filename + ".json"), table.to_dict("split"))
        atomic_excel_tables(RESULTS_XLSX_PATH, {"Sheet1": results})
        atomic_excel_tables(SUMMARY_XLSX_PATH, sheets)
        display(summary.round(5))
        if not comparisons.empty:
            display(comparisons[comparisons.metric == "best_val_f1_macro"].round(6))
        print("Complete results:", RESULTS_XLSX_PATH)
        print("Summary (all completed seeds):", SUMMARY_XLSX_PATH)


    validate_v8_configuration()
    results_df = collect_completed_results()
    export_current = False
    for experiment in EXPERIMENTS_TO_RUN:
        for student_init_mode in STUDENT_INIT_MODES:
            for seed in SEEDS:
                run_id = f"{CONFIG_HASH}|{experiment}|seed={seed}|init={student_init_mode}"
                if SKIP_COMPLETED_RUNS and not results_df.empty and results_df.run_id.eq(run_id).any():
                    print("Already completed:", run_id)
                    continue
                result, _ = run_experiment(experiment, seed, student_init_mode)
                results_df = pd.concat([results_df, pd.DataFrame([result])], ignore_index=True)
                results_df = results_df.drop_duplicates("run_id", keep="last").reset_index(drop=True)
                export_summary(results_df)
                export_current = True
    if not export_current:
        export_summary(results_df)
    print("All requested runs finished.")
else:  # mobilenetv2_imagenet_lt
    # Consolidate only completed V8_mobilenet runs with matching provenance and outcome controls.
    COMPARISON_METRICS = ["best_val_f1_macro", "student_val_few_shot_f1", "student_test_acc",
        "student_test_balanced_acc", "student_test_f1_macro", "student_test_few_shot_f1",
        "student_test_many_shot_f1", "student_test_medium_shot_f1", "train_method_to_ce_ratio_at_best"]


    def paired_comparisons(results):
        for record in results.to_dict("records"):
            validate_completed_result(record, "paired comparison")
        rows = []
        for (seed, init), part in results.groupby(["seed", "student_init_mode"], dropna=False):
            indexed = part.set_index("experiment")
            if not indexed.index.is_unique:
                raise ValueError("Duplicate method/seed/init in consolidated results.")
            metrics = [metric for metric in COMPARISON_METRICS if metric in indexed]
            def flags(selected):
                return {"paired_" + name: bool(selected[col].notna().all() and selected[col].nunique() == 1)
                    for name,col in (("initial_state", "student_initial_sha256"),
                        ("teacher_checkpoint", "teacher_checkpoint_sha256"), ("split", "split_manifests_sha256"),
                        ("global_scale", "cam_scale_reference"))}
            scopes = {
                ("QW1_AMOUNT_ONLY", "QW0_BASE_RAW"): f"QUANTO group factors on {AMOUNT_APPLY_TO}; common global sT",
                ("QW2_WHERE_ONLY", "QW0_BASE_RAW"): f"WHERE on {WHERE_APPLY_TO}; common global sT",
                ("QW3_AMOUNT_WHERE", "QW0_BASE_RAW"): f"QUANTO on {AMOUNT_APPLY_TO} plus WHERE on {WHERE_APPLY_TO}; common global sT",
                ("QW3_AMOUNT_WHERE", "QW1_AMOUNT_ONLY"): f"WHERE on {WHERE_APPLY_TO}; same outcome-selected QUANTO and global sT",
                ("QW3_AMOUNT_WHERE", "QW2_WHERE_ONLY"): f"QUANTO group factors on {AMOUNT_APPLY_TO}; same WHERE and global sT"}
            for left, right in (("QW1_AMOUNT_ONLY", "QW0_BASE_RAW"),
                                ("QW2_WHERE_ONLY", "QW0_BASE_RAW"),
                                ("QW3_AMOUNT_WHERE", "QW0_BASE_RAW"),
                                ("QW3_AMOUNT_WHERE", "QW1_AMOUNT_ONLY"),
                                ("QW3_AMOUNT_WHERE", "QW2_WHERE_ONLY")):
                if left not in indexed.index or right not in indexed.index:
                    continue
                for metric in metrics:
                    rows.append({"seed": seed, "student_init_mode": init, "comparison": f"{left} - {right}",
                        "metric": metric, "delta": indexed.at[left, metric] - indexed.at[right, metric],
                        "amount_apply_to": AMOUNT_APPLY_TO, "where_apply_to": WHERE_APPLY_TO,
                        "comparison_scope": scopes[(left, right)],
                        **flags(indexed.loc[[left, right]])})
            if set(EXPERIMENT_DEFINITIONS) <= set(indexed.index):
                for metric in metrics:
                    interaction = (indexed.at["QW3_AMOUNT_WHERE", metric] - indexed.at["QW1_AMOUNT_ONLY", metric]
                        - indexed.at["QW2_WHERE_ONLY", metric] + indexed.at["QW0_BASE_RAW", metric])
                    rows.append({"seed": seed, "student_init_mode": init,
                        "comparison": "QW3-QW1-QW2+QW0 (interaction)", "metric": metric,
                        "amount_apply_to": AMOUNT_APPLY_TO, "where_apply_to": WHERE_APPLY_TO,
                        "comparison_scope": "Interaction of outcome-selected group factors and spatial weighting; common global sT",
                        "delta": interaction, **flags(indexed.loc[list(EXPERIMENT_DEFINITIONS)])})
        return pd.DataFrame(rows)


    def read_run_table(result, filename):
        path = Path(result["save_dir"], filename)
        if not path.exists() or path.stat().st_size < 2:
            return pd.DataFrame()
        try:
            table = pd.read_csv(path)
        except pd.errors.EmptyDataError:
            return pd.DataFrame()
        for key in ("run_id", "experiment", "seed", "student_init_mode", "config_hash", "method_version"):
            table[key] = result[key]
        return table


    def concatenate_tables(tables):
        tables = [table for table in tables if not table.empty]
        return pd.concat(tables, ignore_index=True) if tables else pd.DataFrame()


    def validate_completed_result(record, source):
        """A completed run may be reused only for the same method, configuration, seed and init."""
        if record.get("config_hash") != CONFIG_HASH or record.get("method_version") != CONFIG["method_version"]:
            raise RuntimeError(f"Result belongs to another configuration or method version: {source}")
        for key, expected in current_split_metadata().items():
            if record.get(key) != expected:
                raise RuntimeError(f"Split oficial/proveniência alterado ({key}): {source}")
        teacher_path = TEACHER_CKPT_TEMPLATE.format(seed=int(record["seed"]))
        if record.get("teacher_checkpoint_sha256") != file_sha256(teacher_path):
            raise RuntimeError(f"Checkpoint do professor alterado: {source}")
        experiment = record.get("experiment")
        if experiment not in EXPERIMENT_DEFINITIONS:
            raise RuntimeError(f"Unknown experiment in completed result: {source}")
        seed = record.get("seed")
        if seed is None or not np.isfinite(float(seed)) or int(seed) != float(seed):
            raise RuntimeError(f"Invalid seed in completed result: {source}")
        init = record.get("student_init_mode")
        if init != "random":
            # The builder remains authoritative; no silent reuse of an unknown init.
            if init not in STUDENT_INIT_MODES:
                raise RuntimeError(f"Unknown initialization in completed result: {source}")
        expected_id = f"{CONFIG_HASH}|{experiment}|seed={int(seed)}|init={init}"
        if record.get("run_id") != expected_id:
            raise RuntimeError(f"Completed run_id disagrees with its seed/experiment/init: {source}")
        required_metrics = ("best_epoch", "best_val_f1_macro", "student_test_f1_macro",
                            "student_test_balanced_acc", "student_test_few_shot_f1")
        if any(record.get(key) is None or not np.isfinite(float(record[key])) for key in required_metrics):
            raise RuntimeError(f"Completed run has missing final metrics: {source}")
        if record.get("amount_apply_to") != AMOUNT_APPLY_TO or record.get("where_apply_to") != WHERE_APPLY_TO:
            raise RuntimeError(f"Completed run has different teacher-outcome controls: {source}")
        if record.get("amount_scope_definition") != "signed_group_target_factors_on_selected_teacher_outcome_without_renormalization":
            raise RuntimeError(f"Completed run has another QUANTO selection policy: {source}")
        if (record.get("cam_scale_scope") != "all_experiments"
                or not isinstance(record.get("cam_scale_enabled"), (bool, np.bool_))
                or not bool(record["cam_scale_enabled"])):
            raise RuntimeError(f"Completed run does not use the common global CAM scale: {source}")
        cfg = EXPERIMENT_DEFINITIONS[experiment]
        if (not isinstance(record.get("amount_enabled"), (bool, np.bool_))
                or not isinstance(record.get("where_enabled"), (bool, np.bool_))
                or bool(record["amount_enabled"]) != bool(cfg["use_amount"])
                or bool(record["where_enabled"]) != bool(cfg["use_where"])):
            raise RuntimeError(f"Completed run has inconsistent QUANTO/WHERE enable flags: {source}")

        def number(name, allow_missing=False):
            value = record.get(name)
            if value is None or not np.isfinite(float(value)):
                if allow_missing:
                    return np.nan
                raise RuntimeError(f"Completed run has missing/nonfinite {name}: {source}")
            return float(value)

        def close(name, expected):
            if not np.isclose(number(name), expected, rtol=5e-6, atol=1e-10):
                raise RuntimeError(f"Completed run has inconsistent {name}: {source}")

        scale = max(number("teacher_raw_cam_mean_sq"), float(CAM_SCALE_EPS))
        close("cam_alpha", float(CAM_ALPHA))
        close("cam_scale_eps", float(CAM_SCALE_EPS))
        close("cam_scale_reference", scale)
        close("cam_scale_applied", scale)  # Never disabled by the QUANTO flag.
        coefficient = float(CAM_ALPHA) / scale
        close("cam_global_coefficient", coefficient)
        close("amount_factor_outside_selection", 1.)
        close("cam_effective_coefficient_outside_amount_selection", coefficient)
        close("amount_expected_weight", number("amount_expected_weight_base"))
        group_n, group_q_sum, group_coefficient_sum = 0, 0., 0.
        for group_id in range(3):
            group = GROUP_ID_TO_NAME[group_id]
            base = number("amount_factor_computed_" + group)
            close("amount_factor_base_" + group, base)
            potential = base if cfg["use_amount"] else 1.
            close("amount_factor_if_selected_" + group, potential)
            for sign in ("positive","negative","zero"):
                sign_base=number("amount_factor_"+sign+"_base_"+group)
                close("amount_factor_"+sign+"_if_selected_"+group,sign_base if cfg["use_amount"] else 1.)
            close("cam_effective_coefficient_if_selected_" + group, coefficient)
            n = number("amount_factor_applied_valid_n_" + group)
            if n <= 0 or n != int(n):
                raise RuntimeError(f"Completed run has no observed training images in {group}: {source}")
            count, weighted_q = 0, 0.
            for outcome in ("correct", "error"):
                suffix = "teacher_" + outcome + "_" + group
                valid = number("amount_factor_applied_valid_n_" + suffix)
                if valid < 0 or valid != int(valid):
                    raise RuntimeError(f"Completed run has invalid conditional counts: {source}")
                close("cam_effective_coefficient_valid_n_" + suffix, valid)
                selected = AMOUNT_APPLY_TO in {"all", outcome}
                expected_q = potential if selected else 1.
                if valid:
                    expected_sign_q={sign:(number("amount_factor_"+sign+"_if_selected_"+group)
                        if selected else 1.) for sign in ("positive","negative","zero")}
                    fractions={sign:number("amount_"+sign+"_fraction_"+suffix)
                        for sign in ("positive","negative","zero")}
                    if not np.isclose(sum(fractions.values()),1.,rtol=5e-6,atol=1e-7):
                        raise RuntimeError(f"Teacher sign fractions do not close: {source}")
                    for sign in expected_sign_q:
                        close("amount_factor_"+sign+"_applied_"+suffix,expected_sign_q[sign])
                    expected_q=sum(expected_sign_q[sign]*fractions[sign] for sign in expected_sign_q)
                    close("amount_factor_applied_" + suffix, expected_q)
                    close("cam_effective_coefficient_" + suffix, coefficient)
                    close("amount_applied_fraction_" + suffix, float(bool(cfg["use_amount"]) and selected))
                    close("amount_applied_n_" + suffix, valid * float(bool(cfg["use_amount"]) and selected))
                    close("amount_selected_outcome_n_" + suffix, valid * float(selected))
                    close("where_selected_outcome_fraction_" + suffix, float(WHERE_APPLY_TO in {"all", outcome}))
                else:
                    if np.isfinite(number("amount_factor_applied_" + suffix, True)) or np.isfinite(number("cam_effective_coefficient_" + suffix, True)):
                        raise RuntimeError(f"Completed run invents a mean for an empty outcome stratum: {source}")
                count += int(valid)
                weighted_q += valid * expected_q
            if count != int(n):
                raise RuntimeError(f"Conditional counts do not close in {group}: {source}")
            expected_mean = weighted_q / n
            close("amount_factor_applied_" + group, expected_mean)
            # The observed mean already closes across outcomes; sign composition can vary.
            close("cam_effective_coefficient_" + group, coefficient)
            close("cam_effective_coefficient_valid_n_" + group, n)
            group_n += int(n)
            group_q_sum += weighted_q
            group_coefficient_sum += n * coefficient
        close("amount_factor_applied_valid_n_all", group_n)
        close("amount_factor_applied_all", group_q_sum / group_n)
        close("amount_applied_weight_mean_at_best", group_q_sum / group_n)
        close("amount_applied_weight_valid_n_at_best", group_n)
        close("cam_effective_coefficient_all", group_coefficient_sum / group_n)


    def collect_completed_results():
        # JSON is the completion marker. A stale XLSX row without its completed JSON
        # never causes a run to be skipped. Re-export rebuilds both CSV and XLSX.
        records = []
        for path in sorted(Path(RUNS_ROOT).glob("*/seed_*_init_*/result.json")):
            with path.open(encoding="utf-8") as file:
                record = json.load(file)
            validate_completed_result(record, path)
            expected_folder = Path(RUNS_ROOT, record["experiment"],
                f"seed_{int(record['seed'])}_init_{record['student_init_mode']}")
            if path.parent.resolve() != expected_folder.resolve():
                raise RuntimeError(f"Completed result is in the wrong run folder: {path}")
            config_path = path.parent / "config.json"
            if not config_path.exists():
                raise RuntimeError(f"Missing per-run configuration: {config_path}")
            saved_config = json.loads(config_path.read_text(encoding="utf-8"))
            if (saved_config.get("run_id") != record["run_id"] or saved_config.get("config_hash") != CONFIG_HASH
                    or saved_config.get("method_version") != CONFIG["method_version"]):
                raise RuntimeError(f"Per-run configuration does not match completed result: {path}")
            saved_payload = saved_config.get("config", {})
            saved_hash_payload = {key: value for key, value in saved_payload.items()
                if key not in {"SEEDS", "EXPERIMENTS_TO_RUN", "STUDENT_INIT_MODES"}}
            saved_hash = hashlib.sha256(json.dumps(saved_hash_payload, sort_keys=True).encode()).hexdigest()[:12]
            if saved_hash != CONFIG_HASH:
                raise RuntimeError(f"Per-run configuration content has a different hash: {path}")
            record["save_dir"] = str(path.parent)
            records.append(record)
        if not records:
            return pd.DataFrame()
        results = pd.DataFrame(records)
        if results.run_id.duplicated().any():
            raise RuntimeError("Duplicate completed run IDs.")
        return results.sort_values(["experiment", "student_init_mode", "seed"]).reset_index(drop=True)


    def summarize_seed_metrics(table, keys, metrics):
        """Equal seed weights; sample std is undefined (NaN) with only one seed."""
        if table.empty:
            return pd.DataFrame()
        metrics = [metric for metric in dict.fromkeys(metrics) if metric in table]
        if table.duplicated(keys + ["seed"]).any():
            raise ValueError(f"More than one observation per seed for summary keys {keys}.")
        grouped = table.groupby(keys, dropna=False)
        seed_info = grouped["seed"].agg(n_seeds="nunique",
            seeds=lambda value: ", ".join(str(int(x)) for x in sorted(value.unique())))
        if not metrics:
            return seed_info.reset_index()
        summary = grouped[metrics].agg(["mean", "std", "count"])
        summary.columns = ["_".join(col) for col in summary.columns]
        return seed_info.join(summary).reset_index()


    def atomic_excel_tables(path, sheets):
        """Keep the previous file intact if writing fails; never silently truncate rows."""
        target = Path(path)
        temporary = target.with_name(target.stem + ".writing.xlsx")
        max_rows = 1_048_575  # header occupies the first Excel row
        try:
            with pd.ExcelWriter(temporary, engine="openpyxl") as writer:
                for name, table in sheets.items():
                    if table.empty:
                        continue
                    chunks = max(1, (len(table) + max_rows - 1) // max_rows)
                    for chunk in range(chunks):
                        sheet_name = name[:31] if chunks == 1 else f"{name[:25]}_{chunk + 1:04d}"
                        table.iloc[chunk * max_rows:(chunk + 1) * max_rows].to_excel(
                            writer, sheet_name=sheet_name, index=False)
            os.replace(temporary, target)
        finally:
            if temporary.exists():
                temporary.unlink()


    def atomic_csv(path, table):
        target = Path(path)
        temporary = target.with_name(target.name + ".writing")
        try:
            table.to_csv(temporary, index=False)
            os.replace(temporary, target)
        finally:
            if temporary.exists():
                temporary.unlink()



    validate_v8_mobilenet_configuration()
    results_df = collect_completed_results()
    for experiment in EXPERIMENTS_TO_RUN:
        for student_init_mode in STUDENT_INIT_MODES:
            for seed in SEEDS:
                run_id = f"{CONFIG_HASH}|{experiment}|seed={seed}|init={student_init_mode}"
                if SKIP_COMPLETED_RUNS and not results_df.empty and results_df.run_id.eq(run_id).any():
                    print("Already completed:", run_id)
                    continue
                result, _ = run_experiment(experiment, seed, student_init_mode)
                results_df = pd.concat([results_df, pd.DataFrame([result])], ignore_index=True)
                results_df = results_df.drop_duplicates("run_id", keep="last").reset_index(drop=True)
    export_summary_essential(results_df)
    print("All requested runs finished. Essential output profile.")


## Comparações e reprodução

Para comparar QW0–QW3 dentro de um perfil, mantenha professores, dados, seeds, inicialização e parâmetros comuns. A lista de variantes pode ser definida em `EXPERIMENTS_OVERRIDE`; `None` preserva as variantes inicialmente selecionadas no respectivo anexo.

Uma comparação entre as duas arquiteturas também envolve datasets, classes, entradas e protocolos diferentes. Não reúna seus valores em uma mesma média entre seeds. A organização de resultados e os metadados mantêm esses contextos separados.

Após trocar `EXPERIMENT_PROFILE`, reinicie o kernel e execute desde o início. Não tente reaproveitar na memória modelos, funções ou caches do perfil anterior. Ao publicar novas execuções, registre as versões das bibliotecas e preserve a implementação dos dados e os checkpoints associados.
